## Phase 0 — Already Done

- data collection
  - NERRS (1995–2025, most regions minus hudson river and 2x great lakes, all stations)
    - originally 15min resolution
    - mostly intact water conditions
    - very sparse nutrient data
      - nutrient data extended ("as-of" forward-fill, 7-day cap)
      - this just in case a record at 945am was skipped over by picking the 1000am record instead to collate at 1hr resolution
    - entirely blank meteorlogical data?
  - ERA5 atmospherics
    - get that atmospheric data back
    - 1hr resolution
    - 0.25 degree resolution
      - not a perfect match to each station in nerrs
      - median centroid was calculated for all stations in a region
      - though most times that centroid ended up nearby but over land (so switched to skin temp instead of water surface temp)
- data cleaning and collation
- split into two datasets
  - full (with nutrients, less complete timeline)
  - and core (without nutrients, more complete)
  - script can be rerun for 1hr resolution
  - but 12hr resolution is included in this repo (zipped)
- various metrics analysis, exploration

## Logic

### Modular Chain
- Model A: ERA5 drivers -> water_temp
- Model B: ERA5 drivers + predicted water_temp -> water properties (salinity, oxygen, pH, etc.)
- Model C: ERA5 drivers + predicted water state -> nutrients (only where nutrient data exists)
  - C can be sacrificed if we run short on time, but it would be a shame

### Why?
- Logical process, re: forcing -> state -> chemistry (deltas from local mean)
- The super sparse nutrient data, if we can keep it, won't impact this as badly as it would one gigantic model
- Easier to read
- Data 'leakage' comes up a lot in reading about this, and this should help reduce it

### To consider...
- When training model B, feed **out-of-fold** predictions (deltas, etc, rather than raw original)
- Then again, if we have time, from B (state) to C (nutrients)

In [ ]:
# first... dependencies
import numpy as np                  # for arrays and math
import pandas as pd                 # for dataframes and csv I/O
import matplotlib.pyplot as plt     # basic visualizations
import seaborn as sns               # for quick readable charts
from sklearn.cluster import KMeans      # simple clustering
from sklearn.decomposition import PCA   # quick 2d display of clusters
from sklearn.preprocessing import StandardScaler  # normalize features
from sklearn.metrics import silhouette_score  # cluster quality check

# keep charts easy to read
sns.set_theme( style = 'whitegrid' )

# the files (thee have been collated and cleaned already)
res = 1 # hours - alternatives 4 and 12

# https://www.youtube.com/watch?v=_W7K79FjI58
# mandatory listening while working on this project

In [ ]:
water = pd.read_csv( f'../data/{res}hr/t4d.{res}hr.water.history.csv' )

water[ 'region' ] = water[ 'region' ].astype( str ).str.strip( ).str.lower( )
water[ 'station' ] = water[ 'station' ].astype( str ).str.strip( ).str.lower( )

# HEE is too sparse for this analysis, so drop it globally here
water = water.loc[ water[ 'region' ] != 'hee' ].copy( )

In [ ]:
# lightweight station and region lookup for labels only
station_lookup = pd.read_csv( '../data/reference/nerrs_station_index.csv' )

station_lookup[ 'region' ] = station_lookup[ 'region_code' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station_full' ] = station_lookup[ 'station' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station' ] = station_lookup[ 'station_full' ].str[ -2: ]

station_lookup = station_lookup[ [ 
    'region',
    'station',
    'station_full',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
] ].drop_duplicates( subset = [ 'region', 'station' ] )

region_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'region_name' ] )
    .drop_duplicates( subset = [ 'region' ] )
    .set_index( 'region' )[ 'region_name' ]
    .to_dict( )
)

station_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'station_name' ] )
    .set_index( [ 'region', 'station' ] )[ 'station_name' ]
    .to_dict( )
)

station_context_cols = [ 
    'region',
    'station',
    'station_full',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
]


def attach_station_context( frame ):
    if not all( [ col in frame.columns for col in [ 'region', 'station' ] ] ):
        return frame.copy( )

    frame_local = frame.copy( )
    overlap_cols = [ 
        col for col in station_context_cols
        if col in frame_local.columns and col not in [ 'region', 'station' ]
    ]

    if len( overlap_cols ) > 0:
        frame_local = frame_local.drop( columns = overlap_cols )

    return frame_local.merge( station_lookup[ station_context_cols ], on = [ 'region', 'station' ], how = 'left' )


def region_label( region ):
    region_key = str( region ).strip( ).lower( )
    return region_name_lookup.get( region_key, region_key )


def station_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]

    name = station_name_lookup.get( ( region_key, station_key ) )

    if pd.isna( name ) or name is None or str( name ).strip( ) == '':
        return station_key

    return f'{station_key} - {name}'


def station_row_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]
    station_name = station_name_lookup.get( ( region_key, station_key ) )
    region_name = region_name_lookup.get( region_key, region_key )

    if pd.isna( station_name ) or station_name is None or str( station_name ).strip( ) == '':
        return f'({region_key}{station_key}) {region_name}'

    return f'({region_key}{station_key}) {station_name}, {region_name}'


def add_station_row_labels( frame, label_col = 'row_label' ):
    frame_local = attach_station_context( frame )
    frame_local[ label_col ] = frame_local.apply( lambda row: station_row_label( row[ 'region' ], row[ 'station' ] ), axis = 1 )
    return frame_local


water = attach_station_context( water )

lookup_missing = ( 
    water[ [ 'region', 'station', 'station_name' ] ]
    .drop_duplicates( )
    [ 'station_name' ]
    .isna( )
    .sum( )
)

print( f'lookup missing station names: {int( lookup_missing )}' )

In [ ]:
water.describe( ).round( 3 ).T


## Phase 1 — Characterization & Classification
Goal: understand what we have before modeling anything

- 1.1 compute per-station summary statistics over a defined baseline period (suggest 1995–2005)
  - mean annual water temp, seasonal amplitude, mean salinity, mean DO
- 1.2 cluster stations in (mean temp × seasonal amplitude) space 
  - k-means, try k=3 or 4, use elbow/silhouette to let the data suggest the right number of groups
  - example in nutrient analysis work
- 1.3 run a parallel k-means pass on atmospheric/climate forcing signatures
  - air temp, wind, solar radiation, precipitation (baseline means + seasonal amplitudes)
- 1.4 enrich clusters with any available station metadata (estuary type, distance from mouth, watershed area)
  - confirm clusters are physically meaningful, not just statistical artifacts
- 1.5 assign each station a baseline regime label
  - see if kmeans self-identify... 
  - otherwise probably get a rolling means (temp) per station for 1995-2000 to classify
- 1.6 visualize stations on a map colored by regime
  - sanity check that PR/HI are warm, alaska/maine are cold, etc.
- 1.7 document baseline period statistics per regime as a reference table
  - this will be needed for the paper/poster later, too


In [ ]:
# lets make this more readable
water = water.rename( columns = { 
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxy_saturation',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temp_c': 'air_temp'
} )

# 1.0 - a description
water.describe( ).round( 3 ).T

In [ ]:
# 1.1 - station character baseline (first valid years, not fixed 1995-2000)
# this keeps the idea simple: each station gets its own early baseline window

base_all = water.copy( )
base_all[ 'datetime' ] = pd.to_datetime( base_all[ 'datetime' ], errors = 'coerce' )

base_all = base_all.loc[ 
    :,
    [ 
        'region',
        'station',
        'datetime',
        'water_temp',
        'salinity',
        'oxygen',
        'oxy_saturation',
        'ph',
        'depth',
    ],
].dropna( subset = [ 'datetime' ] )

base_all[ 'year' ] = base_all[ 'datetime' ].dt.year

In [ ]:
#base_all.describe( ).round( 3 ).T

In [ ]:
# annual coverage check so thin years do not define the baseline
year_obs = ( 
    base_all
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

# leap years and all that jazz
year_obs[ 'expected_obs' ] = np.where( 
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

In [ ]:
year_obs.describe( ).round( 2 ).T

In [ ]:
# try to identify years with enough coverage to be considered valid for defining the baseline
year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70 

valid_years = year_obs.loc[ year_obs[ 'year_is_valid' ] ].copy( )
valid_years = valid_years.sort_values( [ 'region', 'station', 'year' ] ).reset_index( drop = True )
valid_years[ 'valid_year_rank' ] = valid_years.groupby( [ 'region', 'station' ] ).cumcount( ) + 1

# first 5 valid years per station
# turned out too many didn't even start operating between 95 and 00
baseline_years = valid_years.loc[ valid_years[ 'valid_year_rank' ] <= 5 ].copy( )

baseline_window = ( 
    baseline_years
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        baseline_start_year = ( 'year', 'min' ),
        baseline_end_year = ( 'year', 'max' ),
        n_valid_years = ( 'year', 'size' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

baseline_window[ 'is_partial_baseline' ] = baseline_window[ 'n_valid_years' ] < 5

In [ ]:
baseline_window.sample( 15 ).round( 3 ).T

In [ ]:
# keep stations with at least 3 valid years
eligible_stations = baseline_window.loc[ 
    baseline_window[ 'n_valid_years' ] >= 3,
    [ 'region', 'station' ],
]

# pull only rows from each station's selected baseline years
base = base_all.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)

# then keep only stations that passed the >=3-year minimum
base = base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

In [ ]:
#base.describe( ).round( 3 ).T

In [ ]:
# okay, now build station character features from the selected baseline window
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = ( 
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

# each station gets its own, yo
annual_means = ( 
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

In [ ]:
annual_means.describe( ).round( 2 ).T

In [ ]:
# seasonal amplitudes from monthly climatology
monthly = ( 
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

In [ ]:
monthly.describe( ).round( 2 ).T

In [ ]:
monthly.sample( 10 ).round( 3 ).T

In [ ]:
# build both a simple amplitude table and a richer warm / cold season signature table
seasonal_amp = ( 
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)


def build_season_signature( group ):
    group = group.sort_values( 'month' ).copy( )

    temp_series = group[ 'water_temp_mmean' ]
    oxygen_series = group[ 'oxygen_mmean' ]

    if temp_series.notna( ).any( ):
        temp_mean = float( temp_series.mean( ) )
        warm_mask = ( temp_series > temp_mean ).fillna( False )
        cold_mask = ( temp_series < temp_mean ).fillna( False )

        if not warm_mask.any( ):
            warm_mask = ( temp_series == temp_series.max( ) ).fillna( False )

        if not cold_mask.any( ):
            cold_mask = ( temp_series == temp_series.min( ) ).fillna( False )

        warm_peak_month = int( group.loc[ temp_series.idxmax( ), 'month' ] )

    else:
        temp_mean = np.nan
        warm_mask = pd.Series( False, index = group.index )
        cold_mask = pd.Series( False, index = group.index )
        warm_peak_month = np.nan

    warm_oxygen = group.loc[ warm_mask, 'oxygen_mmean' ].dropna( )
    cold_oxygen = group.loc[ cold_mask, 'oxygen_mmean' ].dropna( )

    if oxygen_series.notna( ).any( ):
        oxygen_nadir_month = int( group.loc[ oxygen_series.idxmin( ), 'month' ] )

    else:
        oxygen_nadir_month = np.nan

    if pd.notna( warm_peak_month ) and pd.notna( oxygen_nadir_month ):
        temp_oxygen_phase_lag = float( ( oxygen_nadir_month - warm_peak_month ) % 12 )

    else:
        temp_oxygen_phase_lag = np.nan

    return pd.Series( { 
        'warm_peak_month': warm_peak_month,
        'oxygen_nadir_month': oxygen_nadir_month,
        'temp_oxygen_phase_lag': temp_oxygen_phase_lag,
        'warm_len_months': int( warm_mask.sum( ) ),
        'cold_len_months': int( cold_mask.sum( ) ),
        'warm_intensity_temp': float( group.loc[ warm_mask, 'water_temp_mmean' ].sub( temp_mean ).sum( ) ) if pd.notna( temp_mean ) and warm_mask.any( ) else np.nan,
        'cold_intensity_temp': float( temp_mean * int( cold_mask.sum( ) ) - group.loc[ cold_mask, 'water_temp_mmean' ].sum( ) ) if cold_mask.any( ) else 0.0,
        'warm_oxygen_mean': float( warm_oxygen.mean( ) ) if len( warm_oxygen ) > 0 else np.nan,
        'warm_oxygen_min': float( warm_oxygen.min( ) ) if len( warm_oxygen ) > 0 else np.nan,
        'cold_oxygen_mean': float( cold_oxygen.mean( ) ) if len( cold_oxygen ) > 0 else np.nan,
    } )


seasonal_signature = ( 
    monthly
    .groupby( [ 'region', 'station' ] )
    .apply( build_season_signature )
    .reset_index( )
)

In [ ]:
seasonal_amp.describe( ).round( 2 ).T

In [ ]:
seasonal_amp.sample( 10 ).round( 3 ).T

In [ ]:
# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

In [ ]:
depth_stats.describe( ).round( 2 ).T

In [ ]:
coverage = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

# lets snap these together into a baseline ...
station_baseline = ( 
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( seasonal_signature, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .merge( 
        baseline_window[ [ 
            'region',
            'station',
            'baseline_start_year',
            'baseline_end_year',
            'n_valid_years',
            'mean_year_coverage',
            'is_partial_baseline',
        ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# placeholder flag in case we later add nearest-neighbor feature fallback
station_baseline[ 'character_imputed' ] = False
station_baseline_display = ( 
    station_baseline
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

In [ ]:
station_baseline.describe( ).round( 2 ).T

In [ ]:
# quick baseline summary
n_total_stations = baseline_window[ [ 'region', 'station' ] ].drop_duplicates( ).shape[ 0 ]
n_eligible_stations = eligible_stations.shape[ 0 ]
n_full_five_year = int( ( baseline_window[ 'n_valid_years' ] >= 5 ).sum( ) )

print( f'stations with >=3 valid baseline years: {n_eligible_stations} of {n_total_stations}' )
print( f'stations with full 5-year baseline: {n_full_five_year} of {n_total_stations}' )
del ( 
    n_total_stations,
    n_eligible_stations,
    n_full_five_year
)

#eligible_stations.sample( 10 ).round( 3 ).T

In [ ]:
# tall small-multiples: one subplot per region, one line per station

water[ 'datetime' ] = pd.to_datetime( water[ 'datetime' ], errors = 'coerce' )
water[ 'year' ] = water[ 'datetime' ].dt.year

annual_station = ( 
    water
    .groupby( [ 'region', 'station', 'year' ], as_index = False )[ 'water_temp' ]
    .mean( )
)

regions = sorted( annual_station[ 'region' ].dropna( ).unique( ) )
n_regions = len( regions ) # need for number o'plots

fig, axes = plt.subplots( 
    n_regions,
    1,
    figsize = ( 16, 2 * n_regions ),
    sharex = True,
    constrained_layout = True
)

for ax, region in zip( axes, regions ):
    sub = annual_station.loc[ annual_station[ 'region' ] == region ].sort_values( [ 'station', 'year' ] )

    for station, g in sub.groupby( 'station' ):
        ax.plot( 
            g[ 'year' ],
            g[ 'water_temp' ],
            linewidth = 1.5,
            alpha = 0.85,
            label = station_label( region, station )
        )

    region_title = region_name_lookup.get( region, region )
    ax.set_title( f'{region.upper( )} ( {region_title} ): Mean Annual Water Temp by Station' )
    ax.set_ylabel( 'Temp ( C )' )

    # keep legends readable
    n_stations = sub[ 'station' ].nunique( )

    ax.legend( 
        ncol = 1,
        fontsize = 7,
        frameon = False,
        loc = 'upper left',
        bbox_to_anchor = ( 1.01, 1.0 ),
        borderaxespad = 0
    )

axes[ -1 ].set_xlabel( 'Year' )
plt.show( )

del ( # lil bit of cleanup, for ememory
    annual_station,
    regions,
    n_regions
)

### 1.2 - Domain-Driven KMeans (Temperature, Salinity, Oxygen, Depth)

single slim clustering pass with interpretable station-character features


In [ ]:
# single, slim KMeans pass with a domain-driven feature subset
# selected features:
# - broad baseline hydroclimate structure only
# - keep the richer warm / cold season signature fields in station_baseline,
#   but do not let them define the primary regime geometry
domain_feature_cols = [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
]

print( 'domain features:', domain_feature_cols )

domain_input = station_baseline[ [ 'region', 'station' ] + domain_feature_cols ].copy( )

# simple missing-value strategy: region median, then global median
for col in domain_feature_cols:
    domain_input[ col ] = domain_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    domain_input[ col ] = domain_input[ col ].fillna( domain_input[ col ].median( ) )

scaler_domain = StandardScaler( )
X_scaled_domain = scaler_domain.fit_transform( domain_input[ domain_feature_cols ] )

# k scan for elbow + silhouette
k_scan_rows = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_domain )

    k_scan_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_domain, labels_scan ) ),
    } )

k_scan_domain = pd.DataFrame( k_scan_rows )
k_scan_domain[ 'inertia_drop' ] = k_scan_domain[ 'inertia' ].shift( 1 ) - k_scan_domain[ 'inertia' ]
k_scan_domain[ 'inertia_drop_pct' ] = k_scan_domain[ 'inertia_drop' ] / k_scan_domain[ 'inertia' ].shift( 1 )

recommended_k = int( k_scan_domain.loc[ k_scan_domain[ 'silhouette' ].idxmax( ), 'k' ] )

# keep the final pick interpretable for section 1.2
k_min_interpretable = 3
k_max_interpretable = 6
k_scan_interpretable = k_scan_domain.loc[ k_scan_domain[ 'k' ].between( k_min_interpretable, k_max_interpretable ) ].copy( )

if k_scan_interpretable.empty:
    recommended_k_interpretable = 4

else:
    recommended_k_interpretable = int( 
        k_scan_interpretable.loc[ k_scan_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
    )

print( '\ndomain-feature k scan:' )
print( k_scan_domain.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k }' )
print( f'interpretable-window K ( { k_min_interpretable }-{ k_max_interpretable } ): { recommended_k_interpretable }' )

fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

axes[ 0 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'inertia' ],
    marker = 'o'
)
axes[ 0 ].set_title( 'Domain Features: K vs Inertia' )
axes[ 0 ].set_xlabel( 'K' )
axes[ 0 ].set_ylabel( 'Inertia' )

axes[ 1 ].plot( 
    k_scan_domain[ 'k' ],
    k_scan_domain[ 'silhouette' ],
    marker = 'o'
)
axes[ 1 ].set_title( 'Domain Features: K vs Silhouette Score' )
axes[ 1 ].set_xlabel( 'K' )
axes[ 1 ].set_ylabel( 'Silhouette Score' )

plt.tight_layout( )
plt.show( )


In [ ]:
# not a really DISTINCT elbow, but a clear flattening after 4 or 5 clusters, and a silhouette max at 5

# use the bounded recommendation by default; override manually if needed
k_clusters = recommended_k_interpretable
kmeans_model = KMeans( n_clusters = k_clusters, random_state = 42, n_init = 20 )
domain_input[ 'cluster' ] = kmeans_model.fit_predict( X_scaled_domain )

# keep this alias so downstream sections can still use the same naming
domain_input[ 'cluster_domain' ] = domain_input[ 'cluster' ]

domain_silhouette = float( silhouette_score( X_scaled_domain, domain_input[ 'cluster' ] ) )

print( f'\nchosen K: { k_clusters }' )
print( 'domain-feature silhouette:', round( domain_silhouette, 4 ) )

print( '\ncluster sizes:' )
print( domain_input[ 'cluster' ].value_counts( ).sort_index( ) )

cluster_profile_raw = domain_input.groupby( 'cluster' )[ domain_feature_cols ].mean( )
cluster_profile = cluster_profile_raw.round( 3 )

cluster_profile_mean = cluster_profile_raw.mean( )
cluster_profile_std = cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
cluster_profile_z = ( cluster_profile_raw - cluster_profile_mean ) / cluster_profile_std


def bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_cluster_name( z_row ):
    temp_tag = bucket_tag( z_row[ 'mean_annual_water_temp' ], 'Cool', 'Temp', 'Warm' )
    sal_tag = bucket_tag( z_row[ 'mean_annual_salinity' ], 'Fresh', 'Brackish', 'Saline' )
    oxy_tag = bucket_tag( z_row[ 'mean_annual_oxygen' ], 'Low O2', 'Mid O2', 'High O2' )

    amp_tag = bucket_tag( z_row[ 'seasonal_amp_water_temp' ], 'Stable', 'Mixed', 'Seasonal' )
    depth_tag = bucket_tag( z_row[ 'mean_annual_depth' ], 'Shallow', 'Mid', 'Deep' )

    name = f'{ temp_tag } / { sal_tag } / { oxy_tag }'
    short_name = f'{ temp_tag }-{ sal_tag }'
    note = f'{ amp_tag }, { depth_tag }'

    return pd.Series( { 'cluster_name': name, 'cluster_label': short_name, 'cluster_note': note } )


cluster_name_map = cluster_profile_z.apply( build_cluster_name, axis = 1 ).reset_index( )

print( '\ncluster names:' )
print( cluster_name_map )

In [ ]:
# keep labels categorical and ordered by cluster id for readability + compact memory
cluster_order = sorted( cluster_name_map[ 'cluster' ].astype( int ).tolist( ) )
cluster_name_order = list( dict.fromkeys( cluster_name_map.sort_values( 'cluster' )[ 'cluster_name' ].tolist( ) ) )
cluster_label_order = list( dict.fromkeys( cluster_name_map.sort_values( 'cluster' )[ 'cluster_label' ].tolist( ) ) )

domain_input = domain_input.merge( cluster_name_map, on = 'cluster', how = 'left' )

# write cluster labels back to the station tables
for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ]:
    if cluster_col in station_baseline.columns:
        station_baseline = station_baseline.drop( columns = [ cluster_col ] )

station_baseline = station_baseline.merge( 
    domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

station_baseline[ 'cluster' ] = pd.Categorical( 
    station_baseline[ 'cluster' ],
    categories = cluster_order,
    ordered = True,
)
station_baseline[ 'cluster_name' ] = pd.Categorical( 
    station_baseline[ 'cluster_name' ],
    categories = cluster_name_order,
    ordered = True,
)
station_baseline[ 'cluster_label' ] = pd.Categorical( 
    station_baseline[ 'cluster_label' ],
    categories = cluster_label_order,
    ordered = True,
)

for cluster_col in [ 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ]:
    if cluster_col in station_baseline_display.columns:
        station_baseline_display = station_baseline_display.drop( columns = [ cluster_col ] )

station_baseline_display = station_baseline_display.merge( 
    domain_input[ [ 'region', 'station', 'cluster', 'cluster_domain', 'cluster_name', 'cluster_label', 'cluster_note' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

# cluster mean feature profiles
plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Cluster Mean Station-Character Features ( Domain Set )' )
plt.xlabel( 'Features' )
plt.ylabel( 'Cluster' )
plt.tight_layout( )
plt.show( )

In [ ]:
# correlation matrix for the domain feature set
feature_corr_domain = domain_input[ domain_feature_cols ].corr( )
tri_mask = np.triu( np.ones_like( feature_corr_domain, dtype = bool ), k = 1 )

plt.figure( figsize = ( 10, 8 ) )
sns.heatmap( 
    feature_corr_domain,
    mask = tri_mask,
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm',
    center = 0,
    vmin = -1,
    vmax = 1,
    square = True,
    linewidths = 0.35,
    annot_kws = { 'size': 8 },
    cbar_kws = { 'label': 'Pearson r', 'shrink': 0.85 },
)
plt.title( 'Domain Feature Correlation Matrix ( Triangle + Labels )' )
plt.tight_layout( )
plt.show( )

In [ ]:
corr_pairs_domain = ( 
    feature_corr_domain
    .where( np.triu( np.ones( feature_corr_domain.shape ), k = 1 ).astype( bool ) )
    .stack( )
    .reset_index( )
    .rename( columns = { 'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'corr' } )
)

corr_pairs_domain[ 'abs_corr' ] = corr_pairs_domain[ 'corr' ].abs( )
corr_pairs_domain = corr_pairs_domain.sort_values( 'abs_corr', ascending = False )

print( '\ntop correlated domain-feature pairs:' )
display( corr_pairs_domain.head( 12 ) )

from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.spatial import ConvexHull, QhullError


# and now, let's build a reusable PCA cluster view that keeps the 3d geometry,
# while also giving flat pane projections that are easier to read in a paper or screenshot


def safe_hull_edges_3d( points ):
    if len( points ) < 4:
        return [ ]

    try:
        hull = ConvexHull( points )

    except QhullError:
        return [ ]

    edge_pairs = set( )

    for simplex in hull.simplices:
        for idx in range( len( simplex ) ):
            a = int( simplex[ idx ] )
            b = int( simplex[ ( idx + 1 ) % len( simplex ) ] )
            edge_pairs.add( tuple( sorted( ( a, b ) ) ) )

    return [ points[ [ a, b ] ] for a, b in sorted( edge_pairs ) ]


def safe_hull_outline_2d( points ):
    if len( points ) < 3:
        return None

    try:
        hull = ConvexHull( points )

    except QhullError:
        return None

    return points[ hull.vertices ]


def render_cluster_geometry_views( 
    plot_frame,
    cluster_col,
    title_prefix,
    x_col = 'pc1',
    y_col = 'pc2',
    z_col = 'pc3',
    center_label_col = None,
    x_label = 'PC1',
    y_label = 'PC2',
    z_label = 'PC3',
    elev = 22,
    azim = -58,
    title_fontsize = 18,
    label_fontsize = 13,
    tick_fontsize = 11,
    centroid_fontsize = 13,
    legend_fontsize = 11,
    show_projection_panes = False,
    centroid_label_lift = 0.05,
):
    plot_local = plot_frame.copy( )
    cluster_ids = sorted( pd.Series( plot_local[ cluster_col ].dropna( ).unique( ) ).tolist( ) )
    palette = sns.color_palette( 'tab10', n_colors = max( len( cluster_ids ), 3 ) )
    color_map = { cid: palette[ idx ] for idx, cid in enumerate( cluster_ids ) }
    axis_index_map = { x_col: 0, y_col: 1, z_col: 2 }

    mins = plot_local[ [ x_col, y_col, z_col ] ].min( )
    maxs = plot_local[ [ x_col, y_col, z_col ] ].max( )
    spans = ( maxs - mins ).replace( 0, 1.0 )
    pads = spans * 0.10

    x_floor = float( mins[ x_col ] - pads[ x_col ] )
    x_ceil = float( maxs[ x_col ] + pads[ x_col ] )
    y_floor = float( mins[ y_col ] - pads[ y_col ] )
    y_ceil = float( maxs[ y_col ] + pads[ y_col ] )
    z_floor = float( mins[ z_col ] - pads[ z_col ] )
    z_ceil = float( maxs[ z_col ] + pads[ z_col ] )
    z_span = max( z_ceil - z_floor, 1.0 )

    if show_projection_panes:
        fig = plt.figure( figsize = ( 18, 12 ) )
        gs = GridSpec( 2, 2, figure = fig, width_ratios = [ 1.7, 1.0 ], height_ratios = [ 1.0, 1.0 ] )
        proj_gs = gs[ 1, 1 ].subgridspec( 1, 2, wspace = 0.30 )

        ax3d = fig.add_subplot( gs[ :, 0 ], projection = '3d' )
        ax_xy = fig.add_subplot( gs[ 0, 1 ] )
        ax_xz = fig.add_subplot( proj_gs[ 0, 0 ] )
        ax_yz = fig.add_subplot( proj_gs[ 0, 1 ] )

        projection_axes = [ 
            ( ax_xy, x_col, y_col, x_label, y_label, 'PC1 / PC2 pane' ),
            ( ax_xz, x_col, z_col, x_label, z_label, 'PC1 / PC3 pane' ),
            ( ax_yz, y_col, z_col, y_label, z_label, 'PC2 / PC3 pane' ),
        ]

    else:
        fig = plt.figure( figsize = ( 13, 10 ) )
        ax3d = fig.add_subplot( 111, projection = '3d' )
        projection_axes = [ ]
        ax_xy = None

    ax3d.set_proj_type( 'ortho' )
    ax3d.view_init( elev = elev, azim = azim )
    ax3d.set_box_aspect( ( 1.0, 1.0, 0.88 ) )
    ax3d.set_xlim( x_floor, x_ceil )
    ax3d.set_ylim( y_floor, y_ceil )
    ax3d.set_zlim( z_floor, z_ceil )
    ax3d.xaxis.pane.set_facecolor( ( 0.95, 0.95, 0.95, 1.0 ) )
    ax3d.yaxis.pane.set_facecolor( ( 0.97, 0.97, 0.97, 1.0 ) )
    ax3d.zaxis.pane.set_facecolor( ( 0.96, 0.96, 0.96, 1.0 ) )
    ax3d.grid( alpha = 0.18 )
    ax3d.set_xlabel( x_label, fontsize = label_fontsize, labelpad = 16 )
    ax3d.set_ylabel( y_label, fontsize = label_fontsize, labelpad = 16 )
    ax3d.set_zlabel( z_label, fontsize = label_fontsize, labelpad = 12 )
    ax3d.set_title( f'{ title_prefix }: 3D PCA Geometry + Centroids + Hulls', fontsize = title_fontsize, pad = 18 )
    ax3d.tick_params( axis = 'both', which = 'major', labelsize = tick_fontsize, pad = 4 )
    ax3d.zaxis.set_tick_params( labelsize = tick_fontsize, pad = 4 )

    legend_handles = [ ]

    for cid in cluster_ids:
        cluster_frame = plot_local.loc[ plot_local[ cluster_col ] == cid ].copy( )
        color = color_map[ cid ]
        points = cluster_frame[ [ x_col, y_col, z_col ] ].to_numpy( )
        centroid = points.mean( axis = 0 )

        ax3d.scatter( 
            points[ :, 0 ],
            points[ :, 1 ],
            points[ :, 2 ],
            s = 46,
            alpha = 0.82,
            color = color,
            edgecolors = 'white',
            linewidths = 0.5,
            depthshade = False,
        )

        # and now, let's drop faint shadows onto the cube walls for orientation
        ax3d.scatter( points[ :, 0 ], points[ :, 1 ], np.full( len( points ), z_floor ), s = 14, alpha = 0.11, color = color, depthshade = False )
        ax3d.scatter( points[ :, 0 ], np.full( len( points ), y_ceil ), points[ :, 2 ], s = 14, alpha = 0.09, color = color, depthshade = False )
        ax3d.scatter( np.full( len( points ), x_floor ), points[ :, 1 ], points[ :, 2 ], s = 14, alpha = 0.09, color = color, depthshade = False )

        hull_edges = safe_hull_edges_3d( points )
        if len( hull_edges ) > 0:
            ax3d.add_collection3d( Line3DCollection( hull_edges, colors = [ color ], linewidths = 1.1, alpha = 0.34 ) )

        ax3d.scatter( 
            centroid[ 0 ],
            centroid[ 1 ],
            centroid[ 2 ],
            marker = 'X',
            s = 190,
            color = color,
            edgecolors = 'black',
            linewidths = 0.8,
            zorder = 6,
            depthshade = False,
        )

        if center_label_col is not None and center_label_col in cluster_frame.columns:
            centroid_label = str( cluster_frame[ center_label_col ].dropna( ).iloc[ 0 ] )

        else:
            centroid_label = f'Cluster { cid }'


        legend_handles.append( Line2D( [ 0 ], [ 0 ], marker = 'o', color = 'none', markerfacecolor = color, markeredgecolor = 'white', markersize = 9, label = str( centroid_label ) ) )

        for axis, axis_x, axis_y, axis_xlabel, axis_ylabel, axis_title in projection_axes:
            axis.scatter( 
                cluster_frame[ axis_x ],
                cluster_frame[ axis_y ],
                s = 34,
                alpha = 0.78,
                color = color,
                edgecolors = 'white',
                linewidths = 0.4,
            )

            outline = safe_hull_outline_2d( cluster_frame[ [ axis_x, axis_y ] ].to_numpy( ) )
            if outline is not None:
                axis.fill( outline[ :, 0 ], outline[ :, 1 ], color = color, alpha = 0.10 )
                axis.plot( np.r_[ outline[ :, 0 ], outline[ 0, 0 ] ], np.r_[ outline[ :, 1 ], outline[ 0, 1 ] ], color = color, linewidth = 1.3, alpha = 0.85 )

            axis.scatter( 
                centroid[ axis_index_map[ axis_x ] ],
                centroid[ axis_index_map[ axis_y ] ],
                marker = 'X',
                s = 90,
                color = color,
                edgecolors = 'black',
                linewidths = 0.6,
                zorder = 5,
            )
            axis.set_xlabel( axis_xlabel, fontsize = label_fontsize )
            axis.set_ylabel( axis_ylabel, fontsize = label_fontsize )
            axis.set_title( axis_title, fontsize = title_fontsize - 3 )
            axis.tick_params( axis = 'both', which = 'major', labelsize = tick_fontsize )
            axis.grid( alpha = 0.20 )

    if ax_xy is None:
        ax3d.legend( handles = legend_handles, title = 'Cluster centroid labels', loc = 'upper left', bbox_to_anchor = ( 1.02, 0.98 ), fontsize = legend_fontsize, title_fontsize = legend_fontsize + 1 )

    else:
        ax_xy.legend( handles = legend_handles, title = 'Cluster centroid labels', bbox_to_anchor = ( 1.03, 1.0 ), loc = 'upper left', fontsize = legend_fontsize, title_fontsize = legend_fontsize + 1 )

    plt.tight_layout( )
    plt.show( )


# PCA scatter of resulting clusters
pca_domain = PCA( n_components = 3 )
pcs_domain = pca_domain.fit_transform( X_scaled_domain )

pca_loadings_domain = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ 'PC1', 'PC2', 'PC3' ],
)

# for eventual readability 
feature_phrase_map = { 
    'mean_annual_water_temp': ( 'warmer mean temp', 'cooler mean temp' ),
    'seasonal_amp_water_temp': ( 'larger temp seasonality', 'smaller temp seasonality' ),
    'mean_annual_salinity': ( 'saltier water', 'fresher water' ),
    'seasonal_amp_salinity': ( 'larger salinity swings', 'smaller salinity swings' ),
    'mean_annual_oxygen': ( 'higher mean oxygen', 'lower mean oxygen' ),
    'seasonal_amp_oxygen': ( 'larger oxygen swings', 'smaller oxygen swings' ),
    'mean_annual_depth': ( 'deeper stations', 'shallower stations' ),
}


# and now, let's let the notebook write a short human meaning onto each PCA axis


def pc_axis_interpretation( component_name ):
    top_features = pca_loadings_domain[ component_name ].abs( ).sort_values( ascending = False ).head( 2 ).index
    text_parts = [ ]

    for feat in top_features:
        pos_text, neg_text = feature_phrase_map[ feat ]
        loading = pca_loadings_domain.loc[ feat, component_name ]
        text_parts.append( pos_text if loading >= 0 else neg_text )

    return ' + '.join( text_parts )


pc1_var_pct = round( float( pca_domain.explained_variance_ratio_[ 0 ] ) * 100, 1 )
pc2_var_pct = round( float( pca_domain.explained_variance_ratio_[ 1 ] ) * 100, 1 )
pc3_var_pct = round( float( pca_domain.explained_variance_ratio_[ 2 ] ) * 100, 1 )
domain_pca_3d_var_pct = round( pc1_var_pct + pc2_var_pct + pc3_var_pct, 1 )
pc1_label_text = pc_axis_interpretation( 'PC1' )
pc2_label_text = pc_axis_interpretation( 'PC2' )
pc3_label_text = pc_axis_interpretation( 'PC3' )

print( f'domain PCA variance: PC1={ pc1_var_pct }%, PC2={ pc2_var_pct }%, PC3={ pc3_var_pct }%, cumulative={ domain_pca_3d_var_pct }%' )
print( f'PC1 interpretation: { pc1_label_text }' )
print( f'PC2 interpretation: { pc2_label_text }' )
print( f'PC3 interpretation: { pc3_label_text }' )

domain_plot = domain_input[ [ 'region', 'station', 'cluster', 'cluster_name', 'cluster_label', 'cluster_note' ] ].copy( )
domain_plot[ 'pc1' ] = pcs_domain[ :, 0 ]
domain_plot[ 'pc2' ] = pcs_domain[ :, 1 ]
domain_plot[ 'pc3' ] = pcs_domain[ :, 2 ]

render_cluster_geometry_views( 
    domain_plot,
    cluster_col = 'cluster',
    title_prefix = f'Domain KMeans Clusters ( K = { k_clusters } )',
    center_label_col = 'cluster_label',
    x_label = f'PC1 ( { pc1_var_pct }% ) | { pc1_label_text }',
    y_label = f'PC2 ( { pc2_var_pct }% ) | { pc2_label_text }',
    z_label = f'PC3 ( { pc3_var_pct }% ) | { pc3_label_text }',
)


In [ ]:
# optional interactive 3d PCA view (plotly)
# the static chart above should now be the paper-friendly view,
# and this one is just for free spinning if you want to inspect odd cases
import plotly.express as px

domain_plot_interactive = domain_plot.copy( )
domain_plot_interactive[ 'cluster' ] = domain_plot_interactive[ 'cluster' ].astype( str )

fig_domain_3d = px.scatter_3d( 
    domain_plot_interactive,
    x = 'pc1',
    y = 'pc2',
    z = 'pc3',
    color = 'cluster',
    hover_data = [ 'region', 'station', 'cluster_name', 'cluster_label', 'cluster_note' ],
    title = f'Domain KMeans 3D PCA (Interactive) | K = { k_clusters }',
    opacity = 0.82,
)
fig_domain_3d.update_traces( marker = dict( size = 4 ) )
fig_domain_3d.update_layout( height = 1000 )
fig_domain_3d.show( )



In [ ]:
# just wanna see soem examples
station_baseline_display[ [ 
    'region',
    'station',
    'station_name',
    'cluster',
    'cluster_name',
    'cluster_label',
    'cluster_note',
    'baseline_start_year',
    'baseline_end_year',
    'n_valid_years',
] ].sort_values( [ 'cluster', 'region', 'station' ] ).sample( 5 ).T

#### PCA Interpretation (Domain Feature Set)

- **PC1** and **PC2** are weighted blends of the selected domain features
- gonna play with PC3, too, just to see if an interactive 3D space helps aplexain some ... odd overlaps
- use **explained variance ratio** to see how much structure each component captures
- use **absolute loadings** to identify which features dominate each component
- nearby points in PCA space still represent similar station character


In [ ]:
# quick PCA interpretation table for the domain feature run
pca_variance = pd.Series( 
    pca_domain.explained_variance_ratio_,
    index = [ f'PC{i+1}' for i in range( len( pca_domain.explained_variance_ratio_ ) ) ],
    name = 'explained_variance_ratio',
)

pca_loadings = pd.DataFrame( 
    pca_domain.components_.T,
    index = domain_feature_cols,
    columns = [ f'PC{i+1}' for i in range( pca_domain.n_components_ ) ],
)

print( 'explained variance ratio:' )
print( pca_variance.round( 3 ) )

pca_loadings.round( 3 )

### 1.3 - Atmospheric/Forcing KMeans (Air Temp, Wind, Solar, Precip)

barebones compare pass: build baseline forcing signatures and cluster them


In [ ]:
# 1.3 - build forcing features from the same baseline years and run a slim KMeans pass

forcing_base = water.copy( )
forcing_base[ 'datetime' ] = pd.to_datetime( forcing_base[ 'datetime' ], errors = 'coerce' )
forcing_base = forcing_base.dropna( subset = [ 'datetime' ] )
forcing_base[ 'year' ] = forcing_base[ 'datetime' ].dt.year
forcing_base[ 'month' ] = forcing_base[ 'datetime' ].dt.month

forcing_base = forcing_base.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)
forcing_base = forcing_base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )

# lets just see if the air above these stations also lend themselves to a separate classification?
forcing_annual = ( 
    forcing_base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        air_temp_ymean = ( 'air_temp', 'mean' ),
        wind_speed_ymean = ( 'wind_speed', 'mean' ),
        solar_radiation_ymean = ( 'solar_radiation', 'mean' ),
        precipitation_ymean = ( 'precipitation', 'mean' ),
    )
)

forcing_annual_means = ( 
    forcing_annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_air_temp = ( 'air_temp_ymean', 'mean' ),
        mean_annual_wind_speed = ( 'wind_speed_ymean', 'mean' ),
        mean_annual_solar_radiation = ( 'solar_radiation_ymean', 'mean' ),
        mean_annual_precipitation = ( 'precipitation_ymean', 'mean' ),
    )
)

forcing_monthly = ( 
    forcing_base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        air_temp_mmean = ( 'air_temp', 'mean' ),
        wind_speed_mmean = ( 'wind_speed', 'mean' ),
        solar_radiation_mmean = ( 'solar_radiation', 'mean' ),
        precipitation_mmean = ( 'precipitation', 'mean' ),
    )
)

forcing_seasonal_amp = ( 
    forcing_monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_air_temp = ( 'air_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_wind_speed = ( 'wind_speed_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_solar_radiation = ( 'solar_radiation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_precipitation = ( 'precipitation_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

forcing_feature_cols = [ 
    'mean_annual_air_temp',
    'seasonal_amp_air_temp',
    'mean_annual_wind_speed',
    'seasonal_amp_wind_speed',
    'mean_annual_solar_radiation',
    'seasonal_amp_solar_radiation',
    'mean_annual_precipitation',
    'seasonal_amp_precipitation',
]

forcing_input = forcing_annual_means.merge( 
    forcing_seasonal_amp,
    on = [ 'region', 'station' ],
    how = 'inner',
)

for col in forcing_feature_cols:
    forcing_input[ col ] = forcing_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    forcing_input[ col ] = forcing_input[ col ].fillna( forcing_input[ col ].median( ) )

In [ ]:
forcing_input.describe( ).round( 3 ).T

In [ ]:
scaler_forcing = StandardScaler( )
X_scaled_forcing = scaler_forcing.fit_transform( forcing_input[ forcing_feature_cols ] )

k_scan_forcing_rows = [ ]
for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_forcing )
    k_scan_forcing_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_forcing, labels_scan ) ),
    } )

k_scan_forcing = pd.DataFrame( k_scan_forcing_rows )
k_scan_forcing[ 'inertia_drop' ] = k_scan_forcing[ 'inertia' ].shift( 1 ) - k_scan_forcing[ 'inertia' ]
k_scan_forcing[ 'inertia_drop_pct' ] = k_scan_forcing[ 'inertia_drop' ] / k_scan_forcing[ 'inertia' ].shift( 1 )

recommended_k_forcing = int( k_scan_forcing.loc[ k_scan_forcing[ 'silhouette' ].idxmax( ), 'k' ] )
k_min_interpretable_forcing = 3
k_max_interpretable_forcing = 6
k_scan_forcing_interpretable = k_scan_forcing.loc[ 
    k_scan_forcing[ 'k' ].between( k_min_interpretable_forcing, k_max_interpretable_forcing )
].copy( )

k_clusters_forcing = int( 
    k_scan_forcing_interpretable.loc[ k_scan_forcing_interpretable[ 'silhouette' ].idxmax( ), 'k' ]
)
kmeans_forcing = KMeans( n_clusters = k_clusters_forcing, random_state = 42, n_init = 20 )
forcing_input[ 'forcing_cluster' ] = kmeans_forcing.fit_predict( X_scaled_forcing )
forcing_silhouette = float( silhouette_score( X_scaled_forcing, forcing_input[ 'forcing_cluster' ] ) )

print( 'forcing features:', forcing_feature_cols )
print( '\nforcing-feature k scan:' )
print( k_scan_forcing.round( 4 ) )
print( f'unbounded silhouette max K: { recommended_k_forcing }' )
print( f'interpretable-window K ( { k_min_interpretable_forcing }-{ k_max_interpretable_forcing } ): { recommended_k_forcing }' )
print( f'forcing silhouette at chosen K: { round( forcing_silhouette, 4 ) }' )
print( '\nforcing cluster sizes:' )
print( forcing_input[ 'forcing_cluster' ].value_counts( ).sort_index( ) )

# side by side
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )
axes[ 0 ].plot( 
    k_scan_forcing[ 'k' ],
    k_scan_forcing[ 'inertia' ],
    marker = 'o'
)
axes[ 0 ].set_title( 'Domain Features: K vs Inertia' )
axes[ 0 ].set_xlabel( 'K' )
axes[ 0 ].set_ylabel( 'Inertia' )

axes[ 1 ].plot( 
    k_scan_forcing[ 'k' ],
    k_scan_forcing[ 'silhouette' ],
    marker = 'o'
)
axes[ 1 ].set_title( 'Domain Features: K vs Silhouette Score' )
axes[ 1 ].set_xlabel( 'K' )
axes[ 1 ].set_ylabel( 'Silhouette Score' )

plt.tight_layout( )
plt.show( )


In [ ]:
# and again ... but for the air
# btw, forcing as in the external components driving change int he water
pca_forcing = PCA( n_components = 3 )
pcs_forcing = pca_forcing.fit_transform( X_scaled_forcing )

pca_loadings_forcing = pd.DataFrame( 
    pca_forcing.components_.T,
    index = forcing_feature_cols,
    columns = [ 'PC1', 'PC2', 'PC3' ],
)

forcing_feature_phrase_map = { 
    'mean_annual_air_temp': ( 'warmer air', 'cooler air' ),
    'seasonal_amp_air_temp': ( 'larger air-temp seasonality', 'smaller air-temp seasonality' ),
    'mean_annual_wind_speed': ( 'windier conditions', 'calmer conditions' ),
    'seasonal_amp_wind_speed': ( 'larger wind seasonality', 'smaller wind seasonality' ),
    'mean_annual_solar_radiation': ( 'higher solar load', 'lower solar load' ),
    'seasonal_amp_solar_radiation': ( 'larger solar seasonality', 'smaller solar seasonality' ),
    'mean_annual_precipitation': ( 'wetter conditions', 'drier conditions' ),
    'seasonal_amp_precipitation': ( 'larger precip seasonality', 'smaller precip seasonality' ),
}


# and now, let's interpret the climate PCA axes in the same plain language


def pc_axis_interpretation_forcing( component_name ):
    top_features = pca_loadings_forcing[ component_name ].abs( ).sort_values( ascending = False ).head( 2 ).index
    text_parts = [ ]

    for feat in top_features:
        pos_text, neg_text = forcing_feature_phrase_map[ feat ]
        loading = pca_loadings_forcing.loc[ feat, component_name ]
        text_parts.append( pos_text if loading >= 0 else neg_text )

    return ' + '.join( text_parts )


forcing_plot = forcing_input[ [ 'region', 'station', 'forcing_cluster' ] ].copy( )
forcing_plot[ 'pc1' ] = pcs_forcing[ :, 0 ]
forcing_plot[ 'pc2' ] = pcs_forcing[ :, 1 ]
forcing_plot[ 'pc3' ] = pcs_forcing[ :, 2 ]

pc1_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 0 ] ) * 100, 1 )
pc2_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 1 ] ) * 100, 1 )
pc3_forcing_var_pct = round( float( pca_forcing.explained_variance_ratio_[ 2 ] ) * 100, 1 )
forcing_pca_3d_var_pct = round( pc1_forcing_var_pct + pc2_forcing_var_pct + pc3_forcing_var_pct, 1 )

pc1_forcing_label_text = pc_axis_interpretation_forcing( 'PC1' )
pc2_forcing_label_text = pc_axis_interpretation_forcing( 'PC2' )
pc3_forcing_label_text = pc_axis_interpretation_forcing( 'PC3' )

print( f'forcing PCA variance: PC1={ pc1_forcing_var_pct }%, PC2={ pc2_forcing_var_pct }%, PC3={ pc3_forcing_var_pct }%, cumulative={ forcing_pca_3d_var_pct }%' )
print( f'PC1 interpretation: { pc1_forcing_label_text }' )
print( f'PC2 interpretation: { pc2_forcing_label_text }' )
print( f'PC3 interpretation: { pc3_forcing_label_text }' )

render_cluster_geometry_views( 
    forcing_plot,
    cluster_col = 'forcing_cluster',
    title_prefix = f'Forcing KMeans Clusters ( K = { k_clusters_forcing } )',
    center_label_col = 'forcing_cluster',
    x_label = f'PC1 ( { pc1_forcing_var_pct }% ) | { pc1_forcing_label_text }',
    y_label = f'PC2 ( { pc2_forcing_var_pct }% ) | { pc2_forcing_label_text }',
    z_label = f'PC3 ( { pc3_forcing_var_pct }% ) | { pc3_forcing_label_text }',
)



In [ ]:
# 1.3 - optional interactive 3d PCA view (plotly)
# same idea here: static readability first, free-spin inspection second
import plotly.express as px

forcing_plot_interactive = forcing_plot.copy( )
forcing_plot_interactive[ 'forcing_cluster' ] = forcing_plot_interactive[ 'forcing_cluster' ].astype( str )

fig_forcing_3d = px.scatter_3d( 
    forcing_plot_interactive,
    x = 'pc1',
    y = 'pc2',
    z = 'pc3',
    color = 'forcing_cluster',
    hover_data = [ 'region', 'station', 'forcing_cluster' ],
    title = f'Forcing KMeans 3D PCA (Interactive) | K = { k_clusters_forcing }',
    opacity = 0.82,
)
fig_forcing_3d.update_traces( marker = dict( size = 4 ) )
fig_forcing_3d.update_layout( 
    height = 700
)
fig_forcing_3d.show( )



In [ ]:
# 1.3 - forcing cluster profiles + auto-naming
forcing_cluster_profile_raw = forcing_input.groupby( 'forcing_cluster' )[ forcing_feature_cols ].mean( )
forcing_cluster_profile = forcing_cluster_profile_raw.round( 3 )

forcing_cluster_profile_mean = forcing_cluster_profile_raw.mean( )
forcing_cluster_profile_std = forcing_cluster_profile_raw.std( ddof = 0 ).replace( 0, np.nan )
forcing_cluster_profile_z = ( forcing_cluster_profile_raw - forcing_cluster_profile_mean ) / forcing_cluster_profile_std


def forcing_bucket_tag( value, low_label, mid_label, high_label, threshold = 0.35 ):
    if pd.isna( value ):
        return mid_label

    if value >= threshold:
        return high_label

    if value <= -threshold:
        return low_label

    return mid_label


def build_forcing_cluster_name( z_row ):
    temp_tag = forcing_bucket_tag( z_row[ 'mean_annual_air_temp' ], 'Cool Air', 'Mild Air', 'Warm Air' )
    rain_tag = forcing_bucket_tag( z_row[ 'mean_annual_precipitation' ], 'Dry', 'Mid Rain', 'Wet' )
    wind_tag = forcing_bucket_tag( z_row[ 'mean_annual_wind_speed' ], 'Calm', 'Mixed Wind', 'Windy' )

    solar_tag = forcing_bucket_tag( z_row[ 'mean_annual_solar_radiation' ], 'Low Sun', 'Mid Sun', 'High Sun' )
    season_tag = forcing_bucket_tag( z_row[ 'seasonal_amp_air_temp' ], 'Stable', 'Mixed', 'Seasonal' )

    name = f'{ temp_tag } / { rain_tag } / { wind_tag }'
    short_name = f'{ temp_tag }-{ rain_tag }'
    note = f'{ solar_tag }, { season_tag }'

    return pd.Series( { 'forcing_cluster_name': name, 'forcing_cluster_label': short_name, 'forcing_cluster_note': note } )


forcing_cluster_name_map = forcing_cluster_profile_z.apply( build_forcing_cluster_name, axis = 1 ).reset_index( )
forcing_input = forcing_input.merge( forcing_cluster_name_map, on = 'forcing_cluster', how = 'left' )
forcing_plot = forcing_plot.merge( forcing_cluster_name_map, on = 'forcing_cluster', how = 'left' )

forcing_cluster_preview = ( 
    forcing_input
    .merge( 
        station_baseline_display[ [ 'region', 'station', 'station_name' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'forcing_cluster', 'region', 'station' ] )
)

print( 'forcing cluster names:' )
display( forcing_cluster_name_map.sort_values( 'forcing_cluster' ) )

print( 'forcing cluster mean feature profiles:' )
forcing_cluster_profile

plt.figure( figsize = ( 12, 5 ) )
sns.heatmap( forcing_cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Forcing Cluster Mean Features' )
plt.xlabel( 'Features' )
plt.ylabel( 'Forcing Cluster' )
plt.tight_layout( )
plt.show( )

forcing_cluster_preview[ [ 'region', 'station', 'station_name', 'forcing_cluster', 'forcing_cluster_name', 'forcing_cluster_label', 'forcing_cluster_note' ] ].sample( 20 )


### 1.4 - Cluster Context and Distribution

prepare cluster-station context and inspect distribution by region


In [ ]:
# 1.4 - cluster context check with available station metadata
# keep a clean station-level table for context and distribution checks

cluster_station = station_baseline_display.copy( )

# fallback labels in case this cell is run before 1.2 naming is created
if 'cluster_name' not in cluster_station.columns:
    cluster_station[ 'cluster_name' ] = 'Cluster ' + cluster_station[ 'cluster' ].astype( str )

if 'cluster_note' not in cluster_station.columns:
    cluster_station[ 'cluster_note' ] = ''

cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

cluster_distribution_region = pd.crosstab( 
    cluster_station[ 'region' ],
    cluster_station[ 'cluster' ],
)

print( 'cluster distribution by region:' )
cluster_distribution_region

plt.figure( figsize = ( 10, 6 ) )
sns.heatmap( cluster_distribution_region, annot = True, fmt = 'd', cmap = 'Blues' )
plt.title( 'Cluster Distribution by Region' )
plt.xlabel( 'Cluster' )
plt.ylabel( 'Region' )
plt.tight_layout( )
plt.show( )


### 1.5 - Assign Regime Labels to the Working Dataset (Slim Join)

add only the cluster id to the large working table, and keep descriptive labels in small lookup tables


In [ ]:
# 1.5 - slim cluster assignment to the large working table
# keep it lean: add water-regime cluster id and forcing-regime cluster id to water

station_cluster_lookup = ( 
    station_baseline[ [ 'region', 'station', 'cluster' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

station_forcing_cluster_lookup = ( 
    forcing_input[ [ 'region', 'station', 'forcing_cluster' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

cluster_label_lookup = ( 
    station_baseline[ [ 'cluster', 'cluster_name', 'cluster_label', 'cluster_note' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# save the small lookup tables for reproducibility
station_cluster_lookup_path = '../data/reference/t4d.station_cluster.lookup.csv'
station_forcing_cluster_lookup_path = '../data/reference/t4d.station_forcing_cluster.lookup.csv'
cluster_label_lookup_path = '../data/reference/t4d.cluster_label.lookup.csv'

station_cluster_lookup.to_csv( station_cluster_lookup_path, index = False )
station_forcing_cluster_lookup.to_csv( station_forcing_cluster_lookup_path, index = False )
cluster_label_lookup.to_csv( cluster_label_lookup_path, index = False )

# add only cluster ids to the large water table
for col in [ 'cluster', 'forcing_cluster' ]:
    if col in water.columns:
        water = water.drop( columns = [ col ] )

water = water.merge( station_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )
water = water.merge( station_forcing_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

n_rows = len( water )
n_clustered = int( water[ 'cluster' ].notna( ).sum( ) )
n_forcing_clustered = int( water[ 'forcing_cluster' ].notna( ).sum( ) )
cluster_coverage = round( 100 * n_clustered / n_rows, 2 ) if n_rows > 0 else np.nan
forcing_cluster_coverage = round( 100 * n_forcing_clustered / n_rows, 2 ) if n_rows > 0 else np.nan

water_memory_mb = round( water.memory_usage( deep = True ).sum( ) / 1024 / 1024, 2 )

print( f'water rows with water-regime cluster assigned: {n_clustered} / {n_rows} ( {cluster_coverage}% )' )
print( f'water rows with forcing-regime cluster assigned: {n_forcing_clustered} / {n_rows} ( {forcing_cluster_coverage}% )' )
print( f'current water table size (MB): {water_memory_mb}' )
print( f'saved: {station_cluster_lookup_path}' )
print( f'saved: {station_forcing_cluster_lookup_path}' )
print( f'saved: {cluster_label_lookup_path}' )

#print( )
#print( 'cluster label lookup:' )
#cluster_label_lookup


In [ ]:
water.sample( 5 ).T

### 1.6 - Reference Tables for Reporting

commit cluster summary and station tables to small reference objects/files


In [ ]:
# 1.6 - save reference tables for reporting
cluster_station[ 'latitude' ] = pd.to_numeric( cluster_station[ 'latitude' ], errors = 'coerce' )
cluster_station[ 'longitude' ] = pd.to_numeric( cluster_station[ 'longitude' ], errors = 'coerce' )

forcing_station_labels = ( 
    forcing_input[ [ 
        'region',
        'station',
        'forcing_cluster',
        'forcing_cluster_name',
        'forcing_cluster_label',
        'forcing_cluster_note',
    ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

for col in [ 'forcing_cluster', 'forcing_cluster_name', 'forcing_cluster_label', 'forcing_cluster_note' ]:
    if col in cluster_station.columns:
        cluster_station = cluster_station.drop( columns = [ col ] )

cluster_station = cluster_station.merge( 
    forcing_station_labels,
    on = [ 'region', 'station' ],
    how = 'left',
)

cluster_specs = ( 
    cluster_station
    .groupby( 'cluster', as_index = False )
    .agg( # gather the following specs for each station cluster: (see kmeans earlier)
        cluster_name = ( 'cluster_name', 'first' ),
        cluster_label = ( 'cluster_label', 'first' ),
        cluster_note = ( 'cluster_note', 'first' ),
        n_stations = ( 'station', 'nunique' ),
        n_regions = ( 'region', 'nunique' ),
        mean_annual_water_temp = ( 'mean_annual_water_temp', 'mean' ),
        seasonal_amp_water_temp = ( 'seasonal_amp_water_temp', 'mean' ),
        mean_annual_salinity = ( 'mean_annual_salinity', 'mean' ),
        seasonal_amp_salinity = ( 'seasonal_amp_salinity', 'mean' ),
        mean_annual_oxygen = ( 'mean_annual_oxygen', 'mean' ),
        seasonal_amp_oxygen = ( 'seasonal_amp_oxygen', 'mean' ),
        mean_annual_depth = ( 'mean_annual_depth', 'mean' ),
        mean_latitude = ( 'latitude', 'mean' ),
        mean_longitude = ( 'longitude', 'mean' ),
    )
    .sort_values( 'cluster' )
    .reset_index( drop = True )
)

# a tad more readable ... 
for col in [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_depth',
    'mean_latitude',
    'mean_longitude',
]:
    cluster_specs[ col ] = cluster_specs[ col ].round( 3 )

cluster_station_table = cluster_station[ [ 
    'region',
    'region_name',
    'station',
    'station_name',
    'latitude',
    'longitude',
    'cluster',
    'cluster_name',
    'cluster_label',
    'cluster_note',
    'forcing_cluster',
    'forcing_cluster_name',
    'forcing_cluster_label',
    'forcing_cluster_note',
] ].sort_values( [ 'cluster', 'region', 'station' ] )

# caching the cluster specifications as a reference guide later
# if we reconfigure anyhting about the k-means work, will have to delete these
cluster_specs_path = '../data/reference/t4d.domain.cluster.specs.csv'
cluster_station_table_path = '../data/reference/t4d.cluster.station.table.csv'

cluster_specs.to_csv( cluster_specs_path, index = False )
cluster_station_table.to_csv( cluster_station_table_path, index = False )

#print( f'saved: {cluster_specs_path}' )
#print( f'saved: {cluster_station_table_path}' )

print( 'station table sample:' )
cluster_station_table.sample( 6 ).round( 3 ).T


## Phase 2 — Temporal Diagnostics
Goal: characterize lag structure and trends before building predictive features

- 2.1 run STL decomposition on water temperature per station
  - see: https://www.youtube.com/watch?v=1NXryMoU7Ho
  - annual + diurnal cycles 
  - extract trend components
  - see trends analysis for example
- 2.2 compute cross-correlations between air temp and water temp across a range of lags (0–28 days)
  - identify lag-at-peak-correlation per station
- 2.3 repeat cross-correlation for wind/precip → salinity, air temp → DO
  - reading suggests DO may be longer lag times
- 2.4 summarize lag structure by regime
  - do cold estuaries respond more slowly than warm ones?
  - other oddities? 
  - stuff we'd report on in paper/poster
- 2.5 identify stations showing trend drift in the STL trend component
  - lag candidates for regime-transition analysis
  - feeds into phase 5, btw...

In [ ]:
# helper function to find best lag correlation for a given driver-response pair


def best_lag_table( daily_df, driver_col, response_col, lag_max = 28, min_pairs = 40 ):
    rows = [ ]

    for ( region, station ), g in daily_df.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )

        driver = s[ driver_col ]
        response = s[ response_col ]

        best_lag = np.nan
        best_corr = np.nan
        best_n = 0

        for lag in range( lag_max + 1 ):
            # lag means driver leads response by `lag` days
            pair = pd.concat( [ response, driver.shift( lag ) ], axis = 1 ).dropna( )

            if len( pair ) < min_pairs:
                continue

            # how strongly do they align?
            corr = pair.iloc[ :, 0 ].corr( pair.iloc[ :, 1 ] )

            if pd.isna( corr ):
                continue

            # hold on to the strong correlation
            if pd.isna( best_corr ) or abs( corr ) > abs( best_corr ):
                best_lag = lag
                best_corr = corr
                best_n = len( pair )

        # save it
        rows.append( { 
            'region': region,
            'station': station,
            'best_lag_days': best_lag,
            'peak_corr': best_corr,
            'n_pairs': best_n,
        } )

    out = pd.DataFrame( rows )

    out = out.merge( 
        daily_df[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
        on = [ 'region', 'station' ],
        how = 'left',
    )

    return out

# prolly a few things before this that could benefit from
# being function wrapped

In [ ]:
# 2.1 - temporal diagnostics ( first working pass )
# simple outputs: daily_air diagnostics + STL trend summary + lag-at-peak-correlation tables
from pathlib import Path
from statsmodels.tsa.seasonal import STL


phase2_out_dir = '../data/reference/phase2_v5'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

# this is a bit of work so the results will be cached to file and checked
# for if more runs are done that don't need to redo this
# to redo it... delete the cache files
daily_cache_path = f'{phase2_out_dir}/t4d.phase2.daily.csv'
stl_summary_path = f'{phase2_out_dir}/t4d.phase2.stl.summary.csv'
lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'
seasonal_break_summary_path = f'{phase2_out_dir}/t4d.phase2.seasonal_break.summary.csv'
seasonal_break_keys_path = f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv'
lag_cluster_summary_path = f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv'
trend_drift_candidates_path = f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv'

core_cache_paths = [ 
    daily_cache_path,
    stl_summary_path,
    lag_air_to_water_path,
    lag_wind_to_salinity_path,
    lag_precip_to_salinity_path,
    lag_air_to_oxygen_path,
    seasonal_break_summary_path,
    seasonal_break_keys_path,
]

# basically.. it took about 10 min per run,
# and that's a long time to wait just to change a font in a chart later...
phase2_cache_loaded = all( [ Path( fp ).exists( ) for fp in core_cache_paths ] )
if phase2_cache_loaded:
    print( 'loading cached phase 2 tables before recompute...' )

    daily_air = pd.read_csv( daily_cache_path )
    daily_air[ 'date' ] = pd.to_datetime( daily_air[ 'date' ], errors = 'coerce' )

    stl_summary = pd.read_csv( stl_summary_path )
    lag_air_to_water = pd.read_csv( lag_air_to_water_path )
    lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
    lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
    lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )
    seasonal_break_summary = pd.read_csv( seasonal_break_summary_path )
    seasonal_break_station_keys = pd.read_csv( seasonal_break_keys_path )

    for frame in [ seasonal_break_summary, seasonal_break_station_keys ]:
        if 'region' in frame.columns:
            frame[ 'region' ] = frame[ 'region' ].astype( str ).str.strip( ).str.lower( )
        if 'station' in frame.columns:
            frame[ 'station' ] = frame[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

    if Path( lag_cluster_summary_path ).exists( ):
        lag_cluster_summary = pd.read_csv( lag_cluster_summary_path )

    if Path( trend_drift_candidates_path ).exists( ):
        trend_drift_candidates = pd.read_csv( trend_drift_candidates_path )

    print( 'cached row counts:' )
    print( 'daily rows:', len( daily_air ) )
    print( 'stl summary:', len( stl_summary ) )
    print( 'air -> water lag:', len( lag_air_to_water ) )
    print( 'wind -> salinity lag:', len( lag_wind_to_salinity ) )
    print( 'precip -> salinity lag:', len( lag_precip_to_salinity ) )
    print( 'air -> oxygen lag:', len( lag_air_to_oxygen ) )
    print( 'seasonal-break stations:', len( seasonal_break_station_keys ) )

else:
    print( 'phase 2 cache not complete; computing fresh outputs...' )

    phase2 = water.copy( )
    phase2[ 'datetime' ] = pd.to_datetime( phase2[ 'datetime' ], errors = 'coerce' )
    phase2 = phase2.dropna( subset = [ 'datetime' ] )

    if 'air_temp' not in phase2.columns:
        if 'm_temp_c' in phase2.columns:
            phase2[ 'air_temp' ] = phase2[ 'm_temp_c' ]

    if 'cluster' not in phase2.columns:
        phase2 = phase2.merge( 
            station_baseline[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( ),
            on = [ 'region', 'station' ],
            how = 'left',
        )

    phase2[ 'date' ] = phase2[ 'datetime' ].dt.floor( 'D' )

    daily_air = ( 
        phase2
        .groupby( [ 'region', 'station', 'cluster', 'date' ], as_index = False )
        .agg( 
            water_temp_daily = ( 'water_temp', 'mean' ),
            salinity_daily = ( 'salinity', 'mean' ),
            oxygen_daily = ( 'oxygen', 'mean' ),
            air_temp_daily = ( 'air_temp', 'mean' ),
            wind_speed_daily = ( 'wind_speed', 'mean' ),
            precip_daily = ( 'precipitation', 'sum' ),
        )
    )

    print( 'daily diagnostic rows:', len( daily_air ) )
    daily_air.to_csv( daily_cache_path, index = False )

    recurring_break_min_full_years = 3
    recurring_break_min_missing_run_months = 2
    recurring_break_year_share_threshold = 0.50
    recurring_break_same_time_share_threshold = 0.60


    def first_full_calendar_year( ts ):
        if pd.isna( ts ):
            return np.nan

        if ts.month == 1 and ts.day == 1:
            return int( ts.year )

        return int( ts.year ) + 1


    def last_full_calendar_year( ts ):
        if pd.isna( ts ):
            return np.nan

        if ts.month == 12 and ts.day == 31:
            return int( ts.year )

        return int( ts.year ) - 1


    def summarize_missing_month_run( month_obs_values ):
        max_run = 0
        best_start_month = np.nan
        current_run = 0
        current_start = None

        for month_idx, obs_days in enumerate( month_obs_values, start = 1 ):
            if obs_days == 0:
                if current_run == 0:
                    current_start = month_idx
                current_run += 1
                if current_run > max_run:
                    max_run = current_run
                    best_start_month = current_start

            else:
                current_run = 0
                current_start = None

        if max_run < recurring_break_min_missing_run_months:
            best_start_month = np.nan

        return pd.Series( { 
            'max_consecutive_missing_months': int( max_run ),
            'long_gap_start_month': best_start_month,
            'has_long_gap': bool( max_run >= recurring_break_min_missing_run_months ),
        } )


    daily_presence = daily_air[ [ 'region', 'station', 'date' ] ].drop_duplicates( ).copy( )
    daily_presence[ 'year' ] = daily_presence[ 'date' ].dt.year
    daily_presence[ 'month' ] = daily_presence[ 'date' ].dt.month

    station_span_p2 = ( 
        daily_presence
        .groupby( [ 'region', 'station' ], as_index = False )
        .agg( 
            first_obs_date = ( 'date', 'min' ),
            last_obs_date = ( 'date', 'max' ),
            n_years_obs = ( 'year', 'nunique' ),
        )
    )
    station_span_p2[ 'first_full_year' ] = station_span_p2[ 'first_obs_date' ].apply( first_full_calendar_year )
    station_span_p2[ 'last_full_year' ] = station_span_p2[ 'last_obs_date' ].apply( last_full_calendar_year )
    station_span_p2[ 'n_full_years' ] = np.maximum( 
        0,
        station_span_p2[ 'last_full_year' ] - station_span_p2[ 'first_full_year' ] + 1,
    ).astype( int )
    station_span_p2[ 'record_span_years' ] = ( 
        ( station_span_p2[ 'last_obs_date' ] - station_span_p2[ 'first_obs_date' ] ).dt.days / 365.25
    ).round( 4 )

    monthly_obs_counts_p2 = ( 
        daily_presence
        .groupby( [ 'region', 'station', 'year', 'month' ], as_index = False )
        .agg( n_obs_days = ( 'date', 'nunique' ) )
    )

    full_year_rows = [ ]
    for row in station_span_p2.itertuples( index = False ):
        if int( row.n_full_years ) <= 0:
            continue

        for year in range( int( row.first_full_year ), int( row.last_full_year ) + 1 ):
            full_year_rows.append( { 
                'region': row.region,
                'station': row.station,
                'year': int( year ),
            } )

    if len( full_year_rows ) > 0:
        station_full_years_p2 = pd.DataFrame( full_year_rows )
        month_grid = pd.DataFrame( { 'month': np.arange( 1, 13, dtype = int ) } )
        station_full_years_p2[ '_tmp' ] = 1
        month_grid[ '_tmp' ] = 1
        station_year_month_grid_p2 = station_full_years_p2.merge( month_grid, on = '_tmp', how = 'inner' ).drop( columns = [ '_tmp' ] )

    else:
        station_full_years_p2 = pd.DataFrame( columns = [ 'region', 'station', 'year' ] )
        station_year_month_grid_p2 = pd.DataFrame( columns = [ 'region', 'station', 'year', 'month' ] )

    station_year_month_p2 = station_year_month_grid_p2.merge( 
        monthly_obs_counts_p2,
        on = [ 'region', 'station', 'year', 'month' ],
        how = 'left',
    )
    station_year_month_p2[ 'n_obs_days' ] = station_year_month_p2[ 'n_obs_days' ].fillna( 0 ).astype( int )
    station_year_month_p2[ 'missing_whole_month' ] = station_year_month_p2[ 'n_obs_days' ] == 0


    def summarize_station_year_gap( frame ):
        frame = frame.sort_values( 'month' )
        run_summary = summarize_missing_month_run( frame[ 'n_obs_days' ].tolist( ) )
        return pd.Series( { 
            'n_missing_whole_months': int( frame[ 'missing_whole_month' ].sum( ) ),
            'n_obs_months': int( ( ~frame[ 'missing_whole_month' ] ).sum( ) ),
            'max_consecutive_missing_months': int( run_summary[ 'max_consecutive_missing_months' ] ),
            'long_gap_start_month': run_summary[ 'long_gap_start_month' ],
            'has_long_gap': bool( run_summary[ 'has_long_gap' ] ),
        } )


    if len( station_year_month_p2 ) > 0:
        station_year_gap_summary_p2 = ( 
            station_year_month_p2
            .groupby( [ 'region', 'station', 'year' ] )
            .apply( summarize_station_year_gap )
            .reset_index( )
        )
        station_year_gap_summary_p2[ 'n_missing_whole_months' ] = station_year_gap_summary_p2[ 'n_missing_whole_months' ].astype( int )
        station_year_gap_summary_p2[ 'n_obs_months' ] = station_year_gap_summary_p2[ 'n_obs_months' ].astype( int )
        station_year_gap_summary_p2[ 'max_consecutive_missing_months' ] = station_year_gap_summary_p2[ 'max_consecutive_missing_months' ].astype( int )
        station_year_gap_summary_p2[ 'has_long_gap' ] = station_year_gap_summary_p2[ 'has_long_gap' ].fillna( False ).astype( bool )

    else:
        station_year_gap_summary_p2 = pd.DataFrame( columns = [ 
            'region',
            'station',
            'year',
            'n_missing_whole_months',
            'n_obs_months',
            'max_consecutive_missing_months',
            'long_gap_start_month',
            'has_long_gap',
        ] )

    long_gap_years_p2 = station_year_gap_summary_p2.loc[ station_year_gap_summary_p2[ 'has_long_gap' ] ].copy( )
    long_gap_mode_p2 = ( 
        long_gap_years_p2
        .groupby( [ 'region', 'station', 'long_gap_start_month' ], as_index = False )
        .agg( n_years_same_start = ( 'year', 'nunique' ) )
        .sort_values( [ 'region', 'station', 'n_years_same_start', 'long_gap_start_month' ], ascending = [ True, True, False, True ] )
        .drop_duplicates( subset = [ 'region', 'station' ] )
        .rename( columns = { 
            'long_gap_start_month': 'modal_long_gap_start_month',
        } )
        if len( long_gap_years_p2 ) > 0
        else pd.DataFrame( columns = [ 'region', 'station', 'modal_long_gap_start_month', 'n_years_same_start' ] )
    )

    seasonal_break_summary = station_span_p2.merge( 
        ( 
            station_year_gap_summary_p2
            .groupby( [ 'region', 'station' ], as_index = False )
            .agg( 
                median_missing_whole_months = ( 'n_missing_whole_months', 'median' ),
                median_max_consecutive_missing_months = ( 'max_consecutive_missing_months', 'median' ),
                n_full_years_long_gap = ( 'has_long_gap', 'sum' ),
                pct_full_years_with_long_gap = ( 'has_long_gap', 'mean' ),
            )
        ) if len( station_year_gap_summary_p2 ) > 0 else pd.DataFrame( columns = [ 
            'region',
            'station',
            'median_missing_whole_months',
            'median_max_consecutive_missing_months',
            'n_full_years_long_gap',
            'pct_full_years_with_long_gap',
        ] ),
        on = [ 'region', 'station' ],
        how = 'left',
    )
    seasonal_break_summary = seasonal_break_summary.merge( long_gap_mode_p2, on = [ 'region', 'station' ], how = 'left' )

    for col in [ 
        'median_missing_whole_months',
        'median_max_consecutive_missing_months',
        'n_full_years_long_gap',
        'pct_full_years_with_long_gap',
        'n_years_same_start',
    ]:
        if col in seasonal_break_summary.columns:
            seasonal_break_summary[ col ] = seasonal_break_summary[ col ].fillna( 0 )

    seasonal_break_summary[ 'median_missing_whole_months' ] = seasonal_break_summary[ 'median_missing_whole_months' ].astype( float ).round( 4 )
    seasonal_break_summary[ 'median_max_consecutive_missing_months' ] = seasonal_break_summary[ 'median_max_consecutive_missing_months' ].astype( float ).round( 4 )
    seasonal_break_summary[ 'n_full_years_long_gap' ] = seasonal_break_summary[ 'n_full_years_long_gap' ].astype( int )
    seasonal_break_summary[ 'modal_long_gap_start_month' ] = seasonal_break_summary[ 'modal_long_gap_start_month' ].fillna( 0 ).astype( int )
    seasonal_break_summary[ 'n_years_same_start' ] = seasonal_break_summary[ 'n_years_same_start' ].astype( int )
    seasonal_break_summary[ 'modal_long_gap_start_share' ] = np.where( 
        seasonal_break_summary[ 'n_full_years_long_gap' ] > 0,
        seasonal_break_summary[ 'n_years_same_start' ] / seasonal_break_summary[ 'n_full_years_long_gap' ],
        0.0,
    )
    seasonal_break_summary[ 'modal_long_gap_start_share' ] = seasonal_break_summary[ 'modal_long_gap_start_share' ].round( 4 )
    seasonal_break_summary[ 'modal_long_gap_start_label' ] = seasonal_break_summary[ 'modal_long_gap_start_month' ].map( { 
        1: 'Jan',
        2: 'Feb',
        3: 'Mar',
        4: 'Apr',
        5: 'May',
        6: 'Jun',
        7: 'Jul',
        8: 'Aug',
        9: 'Sep',
        10: 'Oct',
        11: 'Nov',
        12: 'Dec',
    } ).fillna( '' )

    seasonal_break_summary[ 'seasonal_break_station' ] = ( 
        ( seasonal_break_summary[ 'n_full_years' ] >= recurring_break_min_full_years )
        & ( seasonal_break_summary[ 'pct_full_years_with_long_gap' ] >= recurring_break_year_share_threshold )
        & ( seasonal_break_summary[ 'modal_long_gap_start_share' ] >= recurring_break_same_time_share_threshold )
    )

    seasonal_break_summary[ 'seasonal_break_rule' ] = ( 
        f'>={ recurring_break_min_missing_run_months } consecutive fully missing months in >= { 100.0 * recurring_break_year_share_threshold:.0f}% '
        f'of full years, with the long gap starting in the same month >= { 100.0 * recurring_break_same_time_share_threshold:.0f}% '
        f'of those years; requires >= { recurring_break_min_full_years } full years'
    )

    seasonal_break_summary[ 'region' ] = seasonal_break_summary[ 'region' ].astype( str ).str.strip( ).str.lower( )
    seasonal_break_summary[ 'station' ] = seasonal_break_summary[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

    seasonal_break_station_keys = ( 
        seasonal_break_summary
        .loc[ seasonal_break_summary[ 'seasonal_break_station' ].astype( bool ), [ 'region', 'station' ] ]
        .drop_duplicates( )
        .sort_values( [ 'region', 'station' ] )
        .reset_index( drop = True )
    )

    print( 'seasonal-break station count:', len( seasonal_break_station_keys ) )

    if len( seasonal_break_station_keys ) > 0:
        display( 
            seasonal_break_summary.loc[ 
                seasonal_break_summary[ 'seasonal_break_station' ],
                [ 
                    'region',
                    'station',
                    'record_span_years',
                    'n_years_obs',
                    'n_full_years',
                    'n_full_years_long_gap',
                    'pct_full_years_with_long_gap',
                    'median_max_consecutive_missing_months',
                    'modal_long_gap_start_label',
                    'modal_long_gap_start_share',
                ],
            ].sort_values( 
                [ 'pct_full_years_with_long_gap', 'modal_long_gap_start_share', 'median_max_consecutive_missing_months' ],
                ascending = [ False, False, False ],
            ).head( 20 )
        )

    seasonal_break_summary.to_csv( seasonal_break_summary_path, index = False )
    seasonal_break_station_keys.to_csv( seasonal_break_keys_path, index = False )

    # 2.1 STL on daily water temperature ( annual cycle )
    stl_rows = [ ]

    for ( region, station ), g in daily_air.groupby( [ 'region', 'station' ] ):
        s = g.sort_values( 'date' ).set_index( 'date' )[ 'water_temp_daily' ]
        full_idx = pd.date_range( s.index.min( ), s.index.max( ), freq = 'D' )
        s = s.reindex( full_idx )

        if s.notna( ).sum( ) < 365:
            continue

        s = s.interpolate( limit_direction = 'both' )

        if s.notna( ).sum( ) < 365:
            continue

        stl = STL( s, period = 365, robust = True ).fit( )
        trend = stl.trend
        slope_per_day = np.polyfit( np.arange( len( trend ) ), trend, 1 )[ 0 ]
        slope_per_year = slope_per_day * 365

        stl_rows.append( { 
            'region': region,
            'station': station,
            'cluster': g[ 'cluster' ].dropna( ).iloc[ 0 ] if g[ 'cluster' ].notna( ).any( ) else np.nan,
            'n_days': int( len( s ) ),
            'stl_trend_slope_c_per_year': float( slope_per_year ),
            'stl_seasonal_amp_c': float( stl.seasonal.max( ) - stl.seasonal.min( ) ),
        } )

    stl_summary = pd.DataFrame( stl_rows )
    print( 'stl station count:', len( stl_summary ) )


### 2.2 and 2.3 - build lag tables

In [ ]:
# 2.2 and 2.3 lag tables (with cache)
# don't redo this work if it's done
# but if you WANT to, then you'll have to clean the files out physically and rerun
if phase2_cache_loaded:
    print( 'using lag tables already loaded from cache in 2.1' )

else:
    phase2_out_dir = '../data/reference/phase2_v5'
    Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

    lag_air_to_water_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv'
    lag_wind_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv'
    lag_precip_to_salinity_path = f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv'
    lag_air_to_oxygen_path = f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv'

    lag_paths = [ 
        lag_air_to_water_path,
        lag_wind_to_salinity_path,
        lag_precip_to_salinity_path,
        lag_air_to_oxygen_path,
    ]

    if all( [ Path( fp ).exists( ) for fp in lag_paths ] ):
        print( 'loading cached lag tables from disk...' )

        lag_air_to_water = pd.read_csv( lag_air_to_water_path )
        lag_wind_to_salinity = pd.read_csv( lag_wind_to_salinity_path )
        lag_precip_to_salinity = pd.read_csv( lag_precip_to_salinity_path )
        lag_air_to_oxygen = pd.read_csv( lag_air_to_oxygen_path )

    else:
        print( 'lag cache not found; computing lag tables...' )

        # defined up above
        # rinse and repeat
        lag_air_to_water = best_lag_table( daily_air, 'air_temp_daily', 'water_temp_daily', lag_max = 28 )
        lag_wind_to_salinity = best_lag_table( daily_air, 'wind_speed_daily', 'salinity_daily', lag_max = 28 )
        lag_precip_to_salinity = best_lag_table( daily_air, 'precip_daily', 'salinity_daily', lag_max = 28 )
        lag_air_to_oxygen = best_lag_table( daily_air, 'air_temp_daily', 'oxygen_daily', lag_max = 28 )

        lag_air_to_water.to_csv( lag_air_to_water_path, index = False )
        lag_wind_to_salinity.to_csv( lag_wind_to_salinity_path, index = False )
        lag_precip_to_salinity.to_csv( lag_precip_to_salinity_path, index = False )
        lag_air_to_oxygen.to_csv( lag_air_to_oxygen_path, index = False )

#print( )
#print( 'lag table row counts:' )
#print( 'air -> water:', len( lag_air_to_water ) )
#print( 'wind -> salinity:', len( lag_wind_to_salinity ) )
#print( 'precip -> salinity:', len( lag_precip_to_salinity ) )
#print( 'air -> oxygen:', len( lag_air_to_oxygen ) )


### 2.3b - visualize lag table signals

In [ ]:
lag_tables = { 
    'air -> water': lag_air_to_water,
    'wind -> salinity': lag_wind_to_salinity,
    'precip -> salinity': lag_precip_to_salinity,
    'air -> oxygen': lag_air_to_oxygen,
}

lag_plot_rows = [ ]
for relation, lag_df in lag_tables.items( ):
    lag_tmp = lag_df.copy( )
    lag_tmp[ 'relation' ] = relation
    lag_plot_rows.append( lag_tmp )

lag_plot = pd.concat( lag_plot_rows, ignore_index = True )
lag_plot = lag_plot.dropna( subset = [ 'best_lag_days', 'peak_corr' ] )
lag_plot[ 'best_lag_days' ] = lag_plot[ 'best_lag_days' ].astype( int )

lag_signal_summary = ( 
    lag_plot
    .groupby( 'relation', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        median_best_lag_days = ( 'best_lag_days', 'median' ),
        mean_best_lag_days = ( 'best_lag_days', 'mean' ),
        mean_abs_peak_corr = ( 'peak_corr', lambda s: s.abs( ).mean( ) ),
    )
)

for col in [ 'median_best_lag_days', 'mean_best_lag_days', 'mean_abs_peak_corr' ]:
    lag_signal_summary[ col ] = lag_signal_summary[ col ].round( 3 )

print( 'lag signal summary:' )
display( lag_signal_summary )

In [ ]:
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

sns.histplot( 
    data = lag_plot,
    x = 'best_lag_days',
    hue = 'relation',
    multiple = 'dodge',
    discrete = True,
    shrink = 0.85,
    ax = axes[ 0 ],
)
axes[ 0 ].set_title( 'Best-Lag Distribution by Relationship' )
axes[ 0 ].set_xlabel( 'Best Lag (days)' )
axes[ 0 ].set_ylabel( 'Station Count' )

sns.boxplot( 
    data = lag_plot,
    x = 'relation',
    y = 'peak_corr',
    ax = axes[ 1 ],
)
axes[ 1 ].axhline( 0, color = 'black', linestyle = '--', linewidth = 1 )
axes[ 1 ].set_title( 'Peak Correlation Spread by Relationship' )
axes[ 1 ].set_xlabel( '' )
axes[ 1 ].set_ylabel( 'Peak Correlation' )
axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

plt.tight_layout( )
plt.show( )

In [ ]:
# lets just see how the best lags are distributed across the relationships and clusters
lag_count = ( 
    lag_plot
    .groupby( [ 'relation', 'best_lag_days' ] )
    .size( )
    .reset_index( name = 'n_stations' )
)

lag_heat = lag_count.pivot( index = 'relation', columns = 'best_lag_days', values = 'n_stations' ).fillna( 0 )

plt.figure( figsize = ( 12, 4 ) )
sns.heatmap( lag_heat, annot = True, fmt = '.0f', cmap = 'Blues' )
plt.title( 'Station Counts by Best Lag Day and Relationship' )
plt.xlabel( 'Best Lag (days)' )
plt.ylabel( '' )
plt.tight_layout( )
plt.show( )

### 2.3c - reading the lag visuals (what / how / why)

**What this is:**
- compact view of lag behavior across station-level relationships:
  - air -> water temp
  - wind -> salinity
  - precip -> salinity
  - air -> oxygen
- shows (1) where best lags concentrate, and (2) how strong peak correlations are

**How it was built:**
- phase 2 lag tables already loaded/computed above
- stacks them into one tidy frame with a `relation` label
- keeps station-level `best_lag_days` and `peak_corr`.
- produces ...
  - summary table by relation
  - histogram of best lag day counts
  - boxplot of peak correlations
  - heatmap of station counts by lag day

**Why this matters:**
- confirms whether lag behavior is tight or broad by driver-response pair
- helps justify Phase 3 feature choices (fixed windows vs lag-specific features).
- gives interpretable evidence for writeup/poster about system response timing.
- flags whether some links are weaker/noisier (wider correlation spread), which is useful for model expectations in later phases

### 2.4 - summarize lag structure by cluster type

In [ ]:
lag_cluster_summary = ( 
    lag_air_to_water
    .groupby( 'cluster', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        mean_lag_days_air_to_water = ( 'best_lag_days', 'mean' ),
        median_lag_days_air_to_water = ( 'best_lag_days', 'median' ),
        mean_peak_corr_air_to_water = ( 'peak_corr', 'mean' ),
    )
)

for col in [ 'mean_lag_days_air_to_water', 'median_lag_days_air_to_water', 'mean_peak_corr_air_to_water' ]:
    lag_cluster_summary[ col ] = lag_cluster_summary[ col ].round( 3 )

### 2.5 - trend drift candidates?

In [ ]:
# simple threshold: |slope| >= 0.05 c per year
trend_drift_threshold = 0.05
trend_drift = stl_summary.copy( )
trend_drift[ 'abs_trend_slope' ] = trend_drift[ 'stl_trend_slope_c_per_year' ].abs( )
trend_drift_candidates = trend_drift.loc[ trend_drift[ 'abs_trend_slope' ] >= trend_drift_threshold ].copy( )
trend_drift_candidates = trend_drift_candidates.sort_values( 'abs_trend_slope', ascending = False )

# save phase 2 outputs
phase2_out_dir = '../data/reference/phase2_v5'
Path( phase2_out_dir ).mkdir( parents = True, exist_ok = True )

daily_air.to_csv( f'{phase2_out_dir}/t4d.phase2.daily.csv', index = False )
seasonal_break_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.seasonal_break.summary.csv', index = False )
seasonal_break_station_keys.to_csv( f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv', index = False )
stl_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.stl.summary.csv', index = False )
lag_air_to_water.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_water.csv', index = False )
lag_wind_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.wind_to_salinity.csv', index = False )
lag_precip_to_salinity.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.precip_to_salinity.csv', index = False )
lag_air_to_oxygen.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.air_to_oxygen.csv', index = False )
lag_cluster_summary.to_csv( f'{phase2_out_dir}/t4d.phase2.lag.cluster.summary.csv', index = False )
trend_drift_candidates.to_csv( f'{phase2_out_dir}/t4d.phase2.trend.drift.candidates.csv', index = False )

print( f'saved phase 2 tables under: {phase2_out_dir}' )


In [ ]:

lag_cluster_summary

In [ ]:
# some examples?
trend_drift_candidates.sample( 10 ).round( 3 )

In [ ]:
# let's visualize the lag relationships for the cluster summary
# via a boxplot of the station-level lag values by cluster
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )
sns.boxplot( 
    data = trend_drift_candidates, 
    x = 'cluster', 
    y = 'stl_trend_slope_c_per_year',
    ax = axes[ 0 ]
)
axes[ 0 ].set_title( 'Annual Trend (in C)' )
axes[ 0 ].set_xlabel( 'Cluster' )
axes[ 0 ].set_ylabel( 'Trend' )

sns.boxplot( 
    data = trend_drift_candidates, 
    x = 'cluster', 
    y = 'stl_seasonal_amp_c',
    ax = axes[ 1 ]
)
axes[ 1 ].set_title( 'Seasonal Amplitide (in C)' )
axes[ 1 ].set_xlabel( 'Cluster' )
axes[ 1 ].set_ylabel( 'Amplitude' )

plt.tight_layout( )
plt.show( )

## Phase 3 — Feature Engineering
Goal: build the lagged, accumulated features your models will actually use

- 3.1 construct rolling-mean atmospheric features at multiple windows
  - 24hr, 72hr, 7-day, 14-day air temp averages
  - in retrospect, 1, 7 and 28 days
  - but note that these don't really match the lag analysis done in phase 2
  - so that needs decisions,...
- 3.2 construct rate-of-change features (first diffs) for air temp and wind speed
- 3.3 maybe measure time cyclically?
  - day-of-year as (sin, cos) pair
  - hour-of-day similarly if using 1hr data
  - this just so hours 0 and 23, or days 1 and 365 don't SEEM far apart when they're right next to each other
  - some libraries probably do this already, btw...
### NOTE -- Phase 3 feeds both 6 and 7.

### 3.1 - rolling air temps

In [ ]:
# to get rolling averages we dont want air temp NAs
# but we don't want to delete other data... 
# so here's a temp copy
phase3 = water.copy( )
phase3[ 'datetime' ] = pd.to_datetime( phase3[ 'datetime' ], errors = 'coerce' )
phase3 = phase3.dropna( subset = [ 'datetime' ] )
phase3[ 'date' ] = phase3[ 'datetime' ].dt.floor( 'D' )

# now, get the resolution down to daily means and we'll roll them avergages
daily_air = ( 
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( 
        air_temp = ( 'air_temp', 'mean' ),
        wind_speed = ( 'wind_speed', 'mean' ),
        solar = ( 'solar_radiation', 'mean' ),
        precip = ( 'precipitation', 'sum' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)

# over-engineer a bit and get a whole bunch of rolling averages for different windows and features
for s in [ 'air_temp', 'wind_speed', 'precip', 'solar' ]:
    for w in [ 1, 7, 28 ]: 
        daily_air[ f'{s}_r{w}d' ] = ( 
            daily_air
            .groupby( [ 'region', 'station' ] )[ s ]
            .transform( lambda s: s.shift( 1 ).rolling( window = w, min_periods = 1 ).mean( ) )
        )

# that shift 1 keeps the current day out of the average

In [ ]:
daily_air.describe( ).round( 3 ).T

Hmm... some of those mins and maxes look wild

### 3.3 - assessing the day of year as a circle rather than a line

This one will need some reading... it took a fair bit to get it, especially needing both sin and cos

To wit (not hoid):
- real stations peak at different times (phase shifts) of the year (diff climates)
- with both sin and cos, the model can form a(sin) + b(cos), which is a sine wave with any phase offset

In [ ]:
# 0.25 includes leap days
daily_air[ 'doy_sin' ] = np.sin( 2 * np.pi * daily_air[ 'date' ].dt.dayofyear / 365.25 )
daily_air[ 'doy_cos' ] = np.cos( 2 * np.pi * daily_air[ 'date' ].dt.dayofyear / 365.25 )

## Phase 4 — Water-Property Delta-From-Mean Features
Goal: construct station-relative anomaly features from baseline-period means

- 4.1 pick a baseline period and compute station baseline means
- 4.2 compute one delta feature example (salinity)

### NOTE -- Phase 4 primarily feeds Phase 7+ modeling.

In [ ]:
baseline_start = '1995-01-01'
baseline_end = '2000-12-31'

# daily water metrics used in later phases (including Phase 5 rolling means)
daily_water = ( 
    phase3
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( 
        water_temp = ( 'water_temp', 'mean' ),
        salinity = ( 'salinity', 'mean' ),
        oxygen = ( 'oxygen', 'mean' ),
        ph = ( 'ph', 'mean' ),
        depth = ( 'depth', 'mean' )
    )
    .sort_values( [ 'region', 'station', 'date' ] )
)



In [ ]:
properties_baseline = ( 
    daily_water
    .loc[ daily_water[ 'date' ].between( baseline_start, baseline_end ) ]
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        water_temp_baseline = ( 'water_temp', 'mean' ),
        salinity_baseline = ( 'salinity', 'mean' ),
        oxygen_baseline = ( 'oxygen', 'mean' ),
        ph_baseline = ( 'ph', 'mean' ),
        depth_baseline = ( 'depth', 'mean' )
    )
)

In [ ]:
daily_water = daily_water.merge( 
    properties_baseline,
    on = [ 'region', 'station' ],
    how = 'left',
)

daily_water[ 'delta_water_temp' ] = daily_water[ 'water_temp' ] - daily_water[ 'water_temp_baseline' ]
daily_water[ 'delta_salinity' ] = daily_water[ 'salinity' ] - daily_water[ 'salinity_baseline' ]
daily_water[ 'delta_oxygen' ] = daily_water[ 'oxygen' ] - daily_water[ 'oxygen_baseline' ]
daily_water[ 'delta_ph' ] = daily_water[ 'ph' ] - daily_water[ 'ph_baseline' ]
daily_water[ 'delta_depth' ] = daily_water[ 'depth' ] - daily_water[ 'depth_baseline' ]

In [ ]:
daily_water.describe( ).round( 3 ).T

## Phase 5 — Regime Transition Detection
Goal: identify natural validation cases and a forward-projection threshold

- 5.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record
- 5.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory
- 5.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?
- 5.4 designate confirmed transitioning stations as a held-out validation set
  - do not use in model training

### NOTE -- Phase 5 must complete before Phase 6

### 5.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record

In [ ]:
# start from daily_water, then build monthly summaries for more gap-tolerant rolling windows
fiveyear_water = daily_water.copy( )
fiveyear_water[ 'date' ] = pd.to_datetime( fiveyear_water[ 'date' ], errors = 'coerce' )
fiveyear_water = fiveyear_water.dropna( subset = [ 'date' ] )
fiveyear_water = fiveyear_water.sort_values( [ 'region', 'station', 'date' ] )
fiveyear_water[ 'month' ] = fiveyear_water[ 'date' ].dt.to_period( 'M' ).dt.to_timestamp( )

# keep this explicit and readable for students
water_metric_candidates = [ 
    'water_temp',
    'salinity',
    'oxygen',
    'ph',
    'depth',
]

# only use metrics that are actually present in daily_water
fiveyear_water_metrics = [ m for m in water_metric_candidates if m in fiveyear_water.columns ]

monthly_water = ( 
    fiveyear_water
    .groupby( [ 'region', 'station', 'month' ], as_index = False )[ fiveyear_water_metrics ]
    .mean( )
    .sort_values( [ 'region', 'station', 'month' ] )
)

print( 'monthly metrics used for rolling windows:', fiveyear_water_metrics )

In [ ]:
# 5-year rolling means on monthly data (60 months)
# this is more tolerant to seasonal gaps than strict daily rolling
for metric in fiveyear_water_metrics:
    monthly_water[ f'{metric}_roll5y' ] = ( 
        monthly_water
        .groupby( [ 'region', 'station' ] )[ metric ]
        .transform( lambda values: values.rolling( window = 60, min_periods = 36 ).mean( ) )
    )

# rolling coverage over the same 5-year monthly window
monthly_water[ 'has_any_data' ] = monthly_water[ fiveyear_water_metrics ].notna( ).any( axis = 1 ).astype( int )
monthly_water[ 'months_with_any_data_5y' ] = ( 
    monthly_water
    .groupby( [ 'region', 'station' ] )[ 'has_any_data' ]
    .transform( lambda values: values.rolling( window = 60, min_periods = 1 ).sum( ) )
)
monthly_water[ 'coverage_ratio_5y' ] = monthly_water[ 'months_with_any_data_5y' ] / 60.0

# attach monthly rolling outputs back onto daily rows for plotting and event timing
roll5y_cols = [ f'{m}_roll5y' for m in fiveyear_water_metrics ]
monthly_roll_cols = [ 'region', 'station', 'month' ] + roll5y_cols + [ 'months_with_any_data_5y', 'coverage_ratio_5y' ]
fiveyear_water = fiveyear_water.merge( monthly_water[ monthly_roll_cols ], on = [ 'region', 'station', 'month' ], how = 'left' )

# keep a backward-compatible alias in case later cells use "fiveyear"
fiveyear = fiveyear_water.copy( )

In [ ]:
roll5y_cols = [ f'{m}_roll5y' for m in fiveyear_water_metrics ]

fiveyear_water[ [ 'region', 'station', 'date', 'month', 'coverage_ratio_5y' ] + roll5y_cols ].sample( 10 )

### 5.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

In [ ]:
# build a monthly Phase-5 working table for classification
required_roll_cols = [ 
    'water_temp_roll5y',
    'salinity_roll5y',
    'oxygen_roll5y',
    'depth_roll5y',
]

available_roll_cols = [ col for col in required_roll_cols if col in monthly_water.columns ]

p5_monthly = monthly_water[ [ 'region', 'station', 'month', 'coverage_ratio_5y' ] + available_roll_cols ].copy( )

p5_monthly = p5_monthly.rename( columns = { 
    'month': 'date',
    'water_temp_roll5y': 'mean_annual_water_temp',
    'salinity_roll5y': 'mean_annual_salinity',
    'oxygen_roll5y': 'mean_annual_oxygen',
    'depth_roll5y': 'mean_annual_depth',
} )

In [ ]:
# attach baseline cluster identity
baseline_cluster = cluster_station[ [ 'region', 'station', 'cluster' ] ].drop_duplicates( )
p5_monthly = p5_monthly.merge( baseline_cluster, on = [ 'region', 'station' ], how = 'left' )

In [ ]:
# compare against baseline centroids in standardized feature space
feature_cols = [ 
    'mean_annual_water_temp',
    'mean_annual_salinity',
    'mean_annual_oxygen',
    'mean_annual_depth',
]

# baseline station profiles from phase 1
ref = station_baseline[ [ 'cluster' ] + feature_cols ].dropna( ).copy( )
scaler_p5 = StandardScaler( ).fit( ref[ feature_cols ] )

ref_z = pd.DataFrame( 
    scaler_p5.transform( ref[ feature_cols ] ),
    columns = feature_cols,
    index = ref.index,
)
ref_z[ 'cluster' ] = ref[ 'cluster' ].values

centroids_z = ( 
    ref_z
    .groupby( 'cluster', as_index = True )[ feature_cols ]
    .mean( )
    .sort_index( )
)

In [ ]:
#ref_z
#centroids_z

In [ ]:
# monthly nearest-centroid assignment with coverage + partial-feature tolerance
coverage_threshold = 0.70
min_features_required = 3

feature_mean = pd.Series( scaler_p5.mean_, index = feature_cols )
feature_scale = pd.Series( scaler_p5.scale_, index = feature_cols )


def classify_month_row( row ):
    coverage = row.get( 'coverage_ratio_5y', np.nan )
    if pd.isna( coverage ) or coverage < coverage_threshold:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': 0,
            'assignment_state': 'insufficient_coverage',
        } )

    available = [ col for col in feature_cols if pd.notna( row.get( col, np.nan ) ) ]
    if len( available ) < min_features_required:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'insufficient_features',
        } )

    values = pd.to_numeric( row[ available ], errors = 'coerce' )
    scales = pd.to_numeric( feature_scale[ available ], errors = 'coerce' ).replace( 0, np.nan )
    means = pd.to_numeric( feature_mean[ available ], errors = 'coerce' )

    z_values = ( values - means ) / scales
    if z_values.isna( ).any( ):
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'scaling_issue',
        } )

    centroid_slice = centroids_z[ available ]

    try:
        centroid_array = np.asarray( centroid_slice.to_numpy( ), dtype = float )
        z_array = np.asarray( z_values.to_numpy( ), dtype = float )
        distances = np.sqrt( np.sum( ( centroid_array - z_array[ None, : ] ) ** 2, axis = 1 ) )

    except Exception:
        return pd.Series( { 
            'implied_cluster': np.nan,
            'dist_best': np.nan,
            'dist_second': np.nan,
            'margin': np.nan,
            'n_features_used': len( available ),
            'assignment_state': 'distance_error',
        } )

    best_idx = int( np.argmin( distances ) )
    best_cluster = int( centroid_slice.index[ best_idx ] )
    best_dist = float( distances[ best_idx ] )

    if len( distances ) > 1:
        second_dist = float( np.partition( distances, 1 )[ 1 ] )
        margin = second_dist - best_dist

    else:
        second_dist = np.nan
        margin = np.nan

    state = 'classified_full' if len( available ) == len( feature_cols ) else 'classified_partial'

    return pd.Series( { 
        'implied_cluster': best_cluster,
        'dist_best': best_dist,
        'dist_second': second_dist,
        'margin': margin,
        'n_features_used': len( available ),
        'assignment_state': state,
    } )

In [ ]:
# apply classification row-by-row after coercing feature columns to numeric
for col in feature_cols:
    if col in p5_monthly.columns:
        p5_monthly[ col ] = pd.to_numeric( p5_monthly[ col ], errors = 'coerce' )

p5_monthly = pd.concat( [ p5_monthly, p5_monthly.apply( classify_month_row, axis = 1 ) ], axis = 1 )

p5_monthly[ 'assignment_state' ].value_counts( dropna = False )

In [ ]:
# confirm transitions on monthly assignments, then project onto daily rows
p5_monthly = p5_monthly.sort_values( [ 'region', 'station', 'date' ] ).copy( )

p5_monthly[ 'candidate_flip' ] = ( 
    p5_monthly[ 'implied_cluster' ].notna( )
    & p5_monthly[ 'cluster' ].notna( )
    & ( p5_monthly[ 'implied_cluster' ] != p5_monthly[ 'cluster' ] )
)

monthly_run_id = ( 
    p5_monthly
    .groupby( [ 'region', 'station' ] )[ 'implied_cluster' ]
    .transform( lambda values: values.ne( values.shift( ) ).cumsum( ) )
)

p5_monthly[ 'run_len_months' ] = ( 
    p5_monthly
    .groupby( [ 'region', 'station', monthly_run_id ] )[ 'implied_cluster' ]
    .transform( 'size' )
)

p5_monthly[ 'flip_confirmed_monthly' ] = ( 
    p5_monthly[ 'candidate_flip' ]
    & ( p5_monthly[ 'run_len_months' ] >= 6 )   # ~6 months persistence
    & ( p5_monthly[ 'margin' ] > 0.10 )
)

# bring monthly classification decisions down to each daily observation row
p5 = fiveyear_water[ [ 'region', 'station', 'date', 'month' ] ].copy( )

monthly_decision_cols = [ 
    'region',
    'station',
    'date',
    'cluster',
    'implied_cluster',
    'dist_best',
    'dist_second',
    'margin',
    'n_features_used',
    'assignment_state',
    'coverage_ratio_5y',
    'candidate_flip',
    'run_len_months',
    'flip_confirmed_monthly',
]

p5 = p5.merge( 
    p5_monthly[ monthly_decision_cols ].rename( columns = { 'date': 'month' } ),
    on = [ 'region', 'station', 'month' ],
    how = 'left',
)

p5[ 'flip_confirmed' ] = p5[ 'flip_confirmed_monthly' ].fillna( False ).astype( bool )
p5[ 'run_len_days' ] = p5[ 'run_len_months' ] * 30.4375

p5 = p5.sort_values( [ 'region', 'station', 'date' ] ).copy( )

In [ ]:
p5[ [ 
    'region',
    'station',
    'date',
    'cluster',
    'implied_cluster',
    'assignment_state',
    'coverage_ratio_5y',
    'run_len_days',
    'margin',
    'flip_confirmed',
] ].sample( 10 )

In [ ]:
# anything?
first_event = ( 
    p5.loc[ p5[ 'flip_confirmed' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'event_date' } )
)

first_event

In [ ]:
# may come to nothing, but... 
# let's do a quick survival analysis to see how long stations have gone without flipping, and if there are any patterns in which ones flip first
survival_df = ( 
    p5.groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .agg( start_date = 'min', end_date = 'max' )
    .merge( first_event, on = [ 'region', 'station' ], how = 'left' )
)

# common models like Kaplan-Meier or Cox Proportional Hazards require a duration and an event indicator
survival_df[ 'event' ] = survival_df[ 'event_date' ].notna( ).astype( int )
survival_df[ 'stop_date' ] = survival_df[ 'event_date' ].fillna( survival_df[ 'end_date' ] )
survival_df[ 'time_days' ] = ( survival_df[ 'stop_date' ] - survival_df[ 'start_date' ] ).dt.days

# not yet a plan for using this, but keeping it around in case we want to explore it more
#survival_df
survival_df.describe( ).round( 3 ).T

### 5.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?

In [ ]:

# top panel: implied regime over time
# middle panel: centroid-distance confidence over time
# bottom panel: rolling 5-year water metrics

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

flagged_stations = ( 
    p5.loc[ p5[ 'flip_confirmed' ], [ 'region', 'station', 'date' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'first_flip_date' } )
    .sort_values( [ 'first_flip_date', 'region', 'station' ] )
)

print( f'stations with confirmed flips ({ len( flagged_stations ) }):' )
flagged_stations

In [ ]:
# let's look at one of them ...
station = 6  # <- pick your villain here (there were 24, so 0-23)
focus_region = flagged_stations.iloc[ station ][ 'region' ]
focus_station = flagged_stations.iloc[ station ][ 'station' ]

station_p5 = ( 
    p5.loc[ ( p5[ 'region' ] == focus_region ) & ( p5[ 'station' ] == focus_station ) ]
    .sort_values( 'date' )
    .copy( )
)

roll_plot_cols = [ 
    col
    for col in [ 'water_temp_roll5y', 'salinity_roll5y', 'oxygen_roll5y', 'ph_roll5y', 'depth_roll5y' ]
    if col in fiveyear_water.columns
]

station_roll = ( 
    fiveyear_water
    .loc[ 
        ( fiveyear_water[ 'region' ] == focus_region )
        & ( fiveyear_water[ 'station' ] == focus_station ),
        [ 'date' ] + roll_plot_cols,
    ]
    .sort_values( 'date' )
    .copy( )
)

baseline_cluster_val = np.nan
if len( station_p5 ) > 0 and station_p5[ 'cluster' ].notna( ).any( ):
    baseline_cluster_val = station_p5.loc[ station_p5[ 'cluster' ].notna( ), 'cluster' ].iloc[ 0 ]

fig, axes = plt.subplots( 3, 1, figsize = ( 14, 10 ), sharex = True )

axes[ 0 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'implied_cluster' ],
    color = 'tab:blue',
    linewidth = 1.6,
    label = 'Implied cluster',
)

if pd.notna( baseline_cluster_val ):
    axes[ 0 ].axhline( 
        baseline_cluster_val,
        color = 'tab:gray',
        linestyle = '--',
        linewidth = 1.2,
        label = 'Baseline cluster',
    )

flip_points = station_p5.loc[ station_p5[ 'flip_confirmed' ] ]
axes[ 0 ].scatter( 
    flip_points[ 'date' ],
    flip_points[ 'implied_cluster' ],
    color = 'tab:red',
    s = 20,
    label = 'Confirmed flip days',
)

cluster_ticks = station_p5[ 'implied_cluster' ].dropna( ).astype( int ).unique( ).tolist( )
if pd.notna( baseline_cluster_val ):
    cluster_ticks = cluster_ticks + [ int( baseline_cluster_val ) ]
cluster_ticks = sorted( set( cluster_ticks ) )

if len( cluster_ticks ) > 0:
    axes[ 0 ].set_yticks( cluster_ticks )
    axes[ 0 ].set_yticklabels( [ cluster_label_map.get( cid, f'Cluster {cid}' ) for cid in cluster_ticks ] )

axes[ 0 ].set_ylabel( 'Regime' )
axes[ 0 ].set_title( 'Phase 5.3 Review Panel: Regime Path + Confidence + Rolling Metrics' )
axes[ 0 ].legend( loc = 'best' )

axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'dist_best' ],
    color = 'tab:green',
    linewidth = 1.6,
    label = 'Distance to nearest centroid',
)
axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'dist_second' ],
    color = 'tab:orange',
    linewidth = 1.3,
    label = 'Distance to 2nd nearest centroid',
)
axes[ 1 ].plot( 
    station_p5[ 'date' ],
    station_p5[ 'margin' ],
    color = 'tab:purple',
    linewidth = 1.2,
    label = 'Margin (2nd - best)',
)
axes[ 1 ].axhline( 0.10, color = 'tab:red', linestyle = '--', linewidth = 1.0, label = 'Margin threshold (0.10)' )
axes[ 1 ].set_ylabel( 'Distance' )
axes[ 1 ].legend( loc = 'best' )

for col in roll_plot_cols:
    axes[ 2 ].plot( station_roll[ 'date' ], station_roll[ col ], linewidth = 1.1, label = col )

axes[ 2 ].set_ylabel( '5-year rolling value' )
axes[ 2 ].set_xlabel( 'Date' )
axes[ 2 ].legend( loc = 'best', ncol = 2 )

plt.tight_layout( )
plt.show( )

Okay, the above is confusing ... having manually looked at about 10, some seem to always be the same classification 

But are marked as transitory...

Gonna see about looking at them all at once:

In [ ]:
# 5.3 diagnostic: compare lookup-preexcluded stations against raw record-span coverage
from pathlib import Path

pointless_station_min_record_span_years = 15.0
pointless_station_out_dir = phase2_out_dir if 'phase2_out_dir' in globals( ) else '../data/reference/phase2_v5'
pointless_station_summary_path = f'{pointless_station_out_dir}/t4d.phase2.pointless_station.summary.csv'
pointless_station_keys_path = f'{pointless_station_out_dir}/t4d.phase2.pointless_station.keys.csv'
Path( pointless_station_out_dir ).mkdir( parents = True, exist_ok = True )

pointless_core_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in water.columns ]

pointless_station_source = water[ [ 'region', 'station', 'datetime' ] + pointless_core_cols ].copy( )
pointless_station_source[ 'timestamp' ] = pd.to_datetime( pointless_station_source[ 'datetime' ], errors = 'coerce' )
pointless_station_source = pointless_station_source.dropna( subset = [ 'timestamp' ] )
pointless_station_source[ 'date' ] = pointless_station_source[ 'timestamp' ].dt.floor( 'D' )
pointless_station_source[ 'year' ] = pointless_station_source[ 'timestamp' ].dt.year
pointless_station_source[ 'has_core_signal' ] = ( 
    pointless_station_source[ pointless_core_cols ].notna( ).any( axis = 1 )
    if len( pointless_core_cols ) > 0
    else False
)
pointless_station_source[ 'core_date' ] = pointless_station_source[ 'date' ].where( pointless_station_source[ 'has_core_signal' ] )
pointless_station_source[ 'core_year' ] = pointless_station_source[ 'year' ].where( pointless_station_source[ 'has_core_signal' ] )

pointless_station_history = ( 
    pointless_station_source
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_raw_rows = ( 'datetime', 'size' ),
        n_raw_dates = ( 'date', 'nunique' ),
        n_raw_years = ( 'year', 'nunique' ),
        n_core_rows = ( 'has_core_signal', 'sum' ),
        n_core_dates = ( 'core_date', 'nunique' ),
        n_core_years = ( 'core_year', 'nunique' ),
        first_obs_date = ( 'date', 'min' ),
        last_obs_date = ( 'date', 'max' ),
    )
)
pointless_station_history[ 'record_span_days' ] = ( 
    pointless_station_history[ 'last_obs_date' ] - pointless_station_history[ 'first_obs_date' ]
).dt.days.fillna( 0 )
pointless_station_history[ 'record_span_years' ] = ( pointless_station_history[ 'record_span_days' ] / 365.25 ).round( 4 )

pointless_station_lookup_scope = station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'in_t4d_1hr_water_history' ] ].drop_duplicates( subset = [ 'region', 'station' ] )

pointless_station_summary = pointless_station_lookup_scope.merge( pointless_station_history, on = [ 'region', 'station' ], how = 'left' )
for col in [ 'n_raw_rows', 'n_raw_dates', 'n_raw_years', 'n_core_rows', 'n_core_dates', 'n_core_years', 'record_span_days' ]:
    pointless_station_summary[ col ] = pointless_station_summary[ col ].fillna( 0 ).astype( int )
pointless_station_summary[ 'record_span_years' ] = pointless_station_summary[ 'record_span_years' ].fillna( 0.0 ).astype( float ).round( 4 )

pointless_station_summary[ 'lookup_preexcluded' ] = ~pointless_station_summary[ 'in_t4d_1hr_water_history' ].fillna( False ).astype( bool )
pointless_station_summary[ 'short_record_span_station' ] = pointless_station_summary[ 'record_span_years' ] < pointless_station_min_record_span_years
pointless_station_summary[ 'pointless_station' ] = ( 
    pointless_station_summary[ 'lookup_preexcluded' ]
    | pointless_station_summary[ 'short_record_span_station' ]
)
pointless_station_summary[ 'pointless_reason' ] = np.select( 
    [ 
        pointless_station_summary[ 'lookup_preexcluded' ] & pointless_station_summary[ 'short_record_span_station' ],
        pointless_station_summary[ 'lookup_preexcluded' ],
        pointless_station_summary[ 'short_record_span_station' ],
    ],
    [ 
        'lookup_preexcluded_and_short_record_span',
        'lookup_preexcluded_only',
        'short_record_span_only',
    ],
    default = 'retain',
)

pointless_station_keys = pointless_station_summary.loc[ pointless_station_summary[ 'pointless_station' ], [ 'region', 'station' ] ].drop_duplicates( )

pointless_station_summary.to_csv( pointless_station_summary_path, index = False )
pointless_station_keys.to_csv( pointless_station_keys_path, index = False )

print( f'pointless-station threshold: record span < { pointless_station_min_record_span_years:.0f} years between first and last observation' )
print( f'lookup-preexcluded stations: { int( pointless_station_summary[ "lookup_preexcluded" ].sum( ) ) }' )
print( f'short-record-span stations: { int( pointless_station_summary[ "short_record_span_station" ].sum( ) ) }' )
print( f'combined pointless stations: { int( pointless_station_summary[ "pointless_station" ].sum( ) ) }' )
print( 'lookup-preexcluded vs short-record-span:' )
display( pd.crosstab( pointless_station_summary[ 'lookup_preexcluded' ], pointless_station_summary[ 'short_record_span_station' ] ) )

print( 'lookup-preexcluded but not short-record-span:' )
display( 
    pointless_station_summary.loc[ 
        pointless_station_summary[ 'lookup_preexcluded' ] & ~pointless_station_summary[ 'short_record_span_station' ],
        [ 'region', 'station', 'region_name', 'station_name', 'record_span_years', 'n_core_dates', 'n_core_years', 'first_obs_date', 'last_obs_date' ],
    ].sort_values( [ 'region', 'station' ] )
)

print( 'short-record-span but not lookup-preexcluded:' )
display( 
    pointless_station_summary.loc[ 
        pointless_station_summary[ 'short_record_span_station' ] & ~pointless_station_summary[ 'lookup_preexcluded' ],
        [ 'region', 'station', 'region_name', 'station_name', 'record_span_years', 'n_core_dates', 'n_core_years', 'first_obs_date', 'last_obs_date' ],
    ].sort_values( [ 'record_span_years', 'n_core_dates', 'region', 'station' ] ).head( 40 )
)


In [ ]:
# 5.3 visualization #1a: original observed-data timeline by eventual cohort status
# one row per original station, one column per day
# white = no original observed data on that day
# blue = observed data for stations retained in training
# orange = observed data for eventual holdout stations
# red = observed data for stations evicted before modeling

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

raw_presence_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in water.columns ]

raw_presence_source = water.copy( )
raw_presence_source[ 'timestamp' ] = pd.to_datetime( raw_presence_source[ 'datetime' ], errors = 'coerce' )
raw_presence_source = raw_presence_source.dropna( subset = [ 'timestamp' ] )
raw_presence_source[ 'date' ] = raw_presence_source[ 'timestamp' ].dt.floor( 'D' )
raw_presence_source = raw_presence_source.loc[ 
    raw_presence_source[ 'date' ].between( pd.Timestamp( '1995-01-01' ), pd.Timestamp( '2025-12-31' ) )
].copy( )

raw_daily_presence = ( 
    raw_presence_source
    .assign( 
        has_data = ( 
            raw_presence_source[ raw_presence_cols ].notna( ).any( axis = 1 )
            if len( raw_presence_cols ) > 0
            else True
        )
    )
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( has_data = ( 'has_data', 'max' ) )
)

original_station_keys = ( 
    water[ [ 'region', 'station' ] ]
    .drop_duplicates( )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

if len( raw_presence_cols ) > 0:
    raw_station_signal = ( 
        raw_presence_source
        .groupby( [ 'region', 'station' ] )[ raw_presence_cols ]
        .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
        .rename( 'n_non_null_core_water' )
        .reset_index( )
    )

else:
    raw_station_signal = original_station_keys.copy( )
    raw_station_signal[ 'n_non_null_core_water' ] = 0

if 'seasonal_break_station_keys' in globals( ):
    seasonal_break_keys_vis = seasonal_break_station_keys[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

elif Path( f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv' ).exists( ):
    seasonal_break_keys_vis = pd.read_csv( f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv' )[ [ 'region', 'station' ] ].drop_duplicates( )

else:
    seasonal_break_keys_vis = pd.DataFrame( columns = [ 'region', 'station' ] )

seasonal_break_keys_vis[ 'region' ] = seasonal_break_keys_vis[ 'region' ].astype( str ).str.strip( ).str.lower( )
seasonal_break_keys_vis[ 'station' ] = seasonal_break_keys_vis[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

if 'pointless_station_keys' in globals( ):
    pointless_station_keys_vis = pointless_station_keys[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

elif Path( f'{phase2_out_dir}/t4d.phase2.pointless_station.keys.csv' ).exists( ):
    pointless_station_keys_vis = pd.read_csv( f'{phase2_out_dir}/t4d.phase2.pointless_station.keys.csv' )[ [ 'region', 'station' ] ].drop_duplicates( )

else:
    pointless_station_keys_vis = pd.DataFrame( columns = [ 'region', 'station' ] )

pointless_station_keys_vis[ 'region' ] = pointless_station_keys_vis[ 'region' ].astype( str ).str.strip( ).str.lower( )
pointless_station_keys_vis[ 'station' ] = pointless_station_keys_vis[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

eventual_holdout_keys = ( 
    p5.loc[ p5[ 'flip_confirmed' ].fillna( False ).astype( bool ), [ 'region', 'station' ] ]
    .drop_duplicates( )
)

original_station_status = original_station_keys.merge( raw_station_signal, on = [ 'region', 'station' ], how = 'left' )
original_station_status[ 'n_non_null_core_water' ] = original_station_status[ 'n_non_null_core_water' ].fillna( 0 )
original_station_status[ 'has_raw_signal' ] = original_station_status[ 'n_non_null_core_water' ] > 0

original_station_status = original_station_status.merge( 
    seasonal_break_keys_vis.assign( has_seasonal_break = True ),
    on = [ 'region', 'station' ],
    how = 'left',
)
original_station_status[ 'has_seasonal_break' ] = original_station_status[ 'has_seasonal_break' ].fillna( False ).astype( bool )

original_station_status = original_station_status.merge( 
    pointless_station_keys_vis.assign( has_pointless_history = True ),
    on = [ 'region', 'station' ],
    how = 'left',
)
original_station_status[ 'has_pointless_history' ] = original_station_status[ 'has_pointless_history' ].fillna( False ).astype( bool )

original_station_status = original_station_status.merge( 
    eventual_holdout_keys.assign( is_eventual_holdout = True ),
    on = [ 'region', 'station' ],
    how = 'left',
)
original_station_status[ 'is_eventual_holdout' ] = original_station_status[ 'is_eventual_holdout' ].fillna( False ).astype( bool )

original_station_status[ 'is_evicted' ] = ( 
    ~original_station_status[ 'has_raw_signal' ]
    | original_station_status[ 'has_pointless_history' ]
    | original_station_status[ 'has_seasonal_break' ]
)
original_station_status[ 'cohort_status' ] = 'surviving_train'
original_station_status.loc[ original_station_status[ 'is_eventual_holdout' ] & ~original_station_status[ 'is_evicted' ], 'cohort_status' ] = 'eventual_holdout'
original_station_status.loc[ original_station_status[ 'is_evicted' ], 'cohort_status' ] = 'evicted'

status_rank_map = { 
    'evicted': 0,
    'eventual_holdout': 1,
    'surviving_train': 2,
}
status_value_map = { 
    'surviving_train': 1,
    'eventual_holdout': 2,
    'evicted': 3,
}
status_label_map = { 
    'surviving_train': 'Observed data | retained training station',
    'eventual_holdout': 'Observed data | eventual holdout station',
    'evicted': 'Observed data | evicted before modeling',
}
status_color_map = { 
    'surviving_train': '#4c78a8',
    'eventual_holdout': '#f58518',
    'evicted': '#e45756',
}

original_station_status = original_station_status.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
original_station_status[ 'status_rank' ] = original_station_status[ 'cohort_status' ].map( status_rank_map )
original_station_status = original_station_status.sort_values( [ 'status_rank', 'region', 'station' ] ).reset_index( drop = True )
original_station_status = add_station_row_labels( original_station_status )

print( 'original station counts by eventual cohort:' )
display( original_station_status[ 'cohort_status' ].value_counts( ).rename_axis( 'cohort_status' ).to_frame( 'n_stations' ) )

timeline_start = pd.Timestamp( '1995-01-01' )
timeline_end = pd.Timestamp( '2025-12-31' )
all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

station_grid = ( 
    original_station_status[ [ 'region', 'station', 'row_label', 'cohort_status' ] ]
    .assign( _tmp = 1 )
    .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
    .drop( columns = [ '_tmp' ] )
)

coverage_timeline = station_grid.merge( raw_daily_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
coverage_timeline[ 'has_data' ] = coverage_timeline[ 'has_data' ].fillna( False ).astype( bool )
coverage_timeline[ 'status_plot' ] = 0
coverage_timeline.loc[ coverage_timeline[ 'has_data' ], 'status_plot' ] = ( 
    coverage_timeline.loc[ coverage_timeline[ 'has_data' ], 'cohort_status' ]
    .map( status_value_map )
    .fillna( 0 )
    .astype( int )
)

matrix = ( 
    coverage_timeline
    .pivot( index = 'row_label', columns = 'date', values = 'status_plot' )
    .reindex( original_station_status[ 'row_label' ] )
    .to_numpy( )
)

palette = [ '#ffffff', status_color_map[ 'surviving_train' ], status_color_map[ 'eventual_holdout' ], status_color_map[ 'evicted' ] ]
cmap = ListedColormap( palette )
norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )

fig_height = max( 10, 0.24 * len( original_station_status ) + 2 )
fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

ax.set_yticks( np.arange( len( original_station_status ) ) )
ax.set_yticklabels( original_station_status[ 'row_label' ] )

status_dot_colors = original_station_status[ 'cohort_status' ].map( status_color_map ).fillna( '#000000' ).tolist( )
ax.scatter( 
    np.full( len( original_station_status ), -4.0 ),
    np.arange( len( original_station_status ) ),
    s = 70,
    c = status_dot_colors,
    edgecolors = 'black',
    linewidths = 0.6,
    clip_on = False,
    zorder = 5,
)
ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
ax.set_xticks( tick_positions )
ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

ax.set_xlabel( 'Date' )
ax.set_ylabel( 'Original Station ( Code, Station Name, Region Name )' )
ax.set_title( 'Original Observed Data Presence by Eventual Station Cohort ( Daily, 1995-2025 )' )

legend_handles = [ 
    mpatches.Patch( color = palette[ 0 ], label = 'No original observed data' ),
    mpatches.Patch( color = status_color_map[ 'surviving_train' ], label = status_label_map[ 'surviving_train' ] ),
    mpatches.Patch( color = status_color_map[ 'eventual_holdout' ], label = status_label_map[ 'eventual_holdout' ] ),
    mpatches.Patch( color = status_color_map[ 'evicted' ], label = status_label_map[ 'evicted' ] ),
    mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = station cohort' ),
]

ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Coverage State' )
plt.tight_layout( )
plt.show( )


In [ ]:
# 5.3 visualization #1b: full timeline for flagged stations
# one row per flagged station, one column per day
# white = no data, gray = data present but no implied cluster assignment
# left dot color = baseline regime for that station

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

p5_vis = p5
fiveyear_vis = fiveyear_water

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

flagged_from_p5 = ( 
    p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station', 'date' ] ]
    .groupby( [ 'region', 'station' ], as_index = False )[ 'date' ]
    .min( )
    .rename( columns = { 'date': 'first_flip_date' } )
)

offenders = flagged_from_p5.copy( )

offenders = offenders.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
offenders = offenders.sort_values( [ 'first_flip_date', 'region', 'station' ] ).reset_index( drop = True )
offenders = add_station_row_labels( offenders )

print( f'flagged stations shown: {len( offenders )}' )

if len( offenders ) == 0:
    print( 'no flagged stations found in current cohort scope' )

else:
    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    station_grid = ( 
        offenders[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False ).astype( bool )

    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )

    cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )

    value_map = { -2: 0, -1: 1 }
    for idx, cid in enumerate( cluster_ids, start = 2 ):
        value_map[ int( cid ) ] = idx

    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( offenders[ 'row_label' ] )
        .to_numpy( )
    )

    palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
    cmap = ListedColormap( palette )
    norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )

    fig_height = max( 8, 0.45 * len( offenders ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( offenders ) ) )
    ax.set_yticklabels( offenders[ 'row_label' ] )

    baseline_values = offenders[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]

    ax.scatter( 
        np.full( len( offenders ), -4.0 ),
        np.arange( len( offenders ) ),
        s = 110,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( 'Flagged Station ( Code, Station Name, Region Name )' )
    ax.set_title( 'Flagged Station Regime Timeline ( Daily, 1995-2025 )' )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]

    for cid in cluster_ids:
        label_text = cluster_label_map.get( cid, f'Cluster {cid}' )
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = label_text ) )

    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )

    plt.tight_layout( )
    plt.show( )

In [ ]:
# 5.3 visualization #1c: full timeline for unflagged stations
# one row per unflagged station, one column per day
# white = no data, gray = data present but no implied cluster assignment
# left dot color = baseline regime for that station

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

p5_vis = p5
fiveyear_vis = fiveyear_water

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

all_stations = station_baseline_regime[ [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ].drop_duplicates( )

flagged_keys = p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ].drop_duplicates( )

unflagged_stations = ( 
    all_stations
    .merge( flagged_keys.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

print( f'total stations in current scope: {len( all_stations )}' )
print( f'flagged stations in current scope: {len( flagged_keys )}' )
print( f'unflagged stations in current scope: {len( unflagged_stations )}' )

comparison_rows = unflagged_stations.head( 100 ).copy( )
comparison_rows

if len( comparison_rows ) == 0:
    print( 'no unflagged stations available after current filtering' )

else:
    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    comparison_rows = add_station_row_labels( comparison_rows )

    station_grid = ( 
        comparison_rows[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False ).astype( bool )

    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )

    cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )

    value_map = { -2: 0, -1: 1 }
    for idx, cid in enumerate( cluster_ids, start = 2 ):
        value_map[ int( cid ) ] = idx

    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( comparison_rows[ 'row_label' ] )
        .to_numpy( )
    )

    palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
    cmap = ListedColormap( palette )
    norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )

    fig_height = max( 8, 0.45 * len( comparison_rows ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( comparison_rows ) ) )
    ax.set_yticklabels( comparison_rows[ 'row_label' ] )

    baseline_values = comparison_rows[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]

    ax.scatter( 
        np.full( len( comparison_rows ), -4.0 ),
        np.arange( len( comparison_rows ) ),
        s = 110,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( 'Unflagged Station ( Code, Station Name, Region Name )' )
    ax.set_title( 'Unflagged Station Regime Timeline ( Daily, 1995-2025 )' )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]

    for cid in cluster_ids:
        label_text = cluster_label_map.get( cid, f'Cluster {cid}' )
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = label_text ) )

    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )

    plt.tight_layout( )
    plt.show( )


Okay, but why is job+jb assigned a regime (red), but has no data showing?

In [ ]:
key = ( 'job', 'jb' )

print( 'in baseline:',
      ( ( cluster_station[ 'region' ] == key[ 0 ] ) & ( cluster_station[ 'station' ] == key[ 1 ] ) ).any( ) )

fw = fiveyear_water.loc[ 
    ( fiveyear_water[ 'region' ] == key[ 0 ] ) &
    ( fiveyear_water[ 'station' ] == key[ 1 ] )
]
print( 'fiveyear_water rows:', len( fw ) )

In [ ]:
# 5.3 diagnostic near 5.4: raw water-table metrics for job | jb
raw_region = 'job'
raw_station = 'jb'

raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in water.columns ]

# baseline assignment (phase 1)
base_match = cluster_station.loc[ 
    ( cluster_station[ 'region' ] == raw_region )
    & ( cluster_station[ 'station' ] == raw_station ),
    [ 'cluster', 'cluster_label' ],
].drop_duplicates( )

if len( base_match ) == 0:
    print( 'baseline assignment: none found in cluster_station' )

else:
    print( 'baseline assignment:' )
    display( base_match )

# implied assignment coverage (phase 5)
p5_match = p5.loc[ 
    ( p5[ 'region' ] == raw_region )
    & ( p5[ 'station' ] == raw_station )
].copy( )
print( f'p5 rows for station: {len( p5_match )}' )
if len( p5_match ) > 0:
    print( 'implied-cluster non-null rows:', int( p5_match[ 'implied_cluster' ].notna( ).sum( ) ) )
    print( 'flip_confirmed rows:', int( p5_match[ 'flip_confirmed' ].fillna( False ).sum( ) ) )

job_raw = water.loc[ 
    ( water[ 'region' ] == raw_region )
    & ( water[ 'station' ] == raw_station )
].copy( )

print( f'raw rows in water before timestamp parsing: {len( job_raw )}' )

if len( job_raw ) == 0:
    print( 'no rows found in water for this station key' )

else:
    # choose the timestamp column that actually parses best for THIS station
    time_candidates = [ col for col in [ 'datetime', 'date' ] if col in job_raw.columns ]
    parsed_counts = { }
    parsed_series = { }

    for col in time_candidates:
        parsed = pd.to_datetime( job_raw[ col ], errors = 'coerce' )
        parsed_series[ col ] = parsed
        parsed_counts[ col ] = int( parsed.notna( ).sum( ) )

    if len( parsed_counts ) == 0:
        print( 'no datetime/date column found in water' )

    else:
        print( 'timestamp parse counts by column:', parsed_counts )
        time_col = max( parsed_counts, key = parsed_counts.get )
        print( f'time column used: {time_col}' )

        job_raw[ 'timestamp' ] = parsed_series[ time_col ]
        job_raw = job_raw.dropna( subset = [ 'timestamp' ] ).sort_values( 'timestamp' )

        window_start = pd.Timestamp( '1995-01-01' )
        window_end = pd.Timestamp( '2025-12-31 23:59:59' )
        job_raw = job_raw.loc[ job_raw[ 'timestamp' ].between( window_start, window_end ) ]

        print( f'rows in plotting window: {len( job_raw )}' )

        if len( job_raw ) == 0:
            print( 'no valid timestamped rows in 1995-2025 for this station' )

        else:
            non_null_counts = job_raw[ raw_metric_cols ].notna( ).sum( )
            print( 'non-null counts by metric:' )
            display( non_null_counts.to_frame( 'non_null_count' ) )

            if int( non_null_counts.sum( ) ) == 0:
                print( 'metrics are all NaN for this station in this window (chart will be blank)' )

            else:
                fig, ax = plt.subplots( figsize = ( 16, 5 ) )
                palette = sns.color_palette( 'tab10', n_colors = len( raw_metric_cols ) )

                for metric, color in zip( raw_metric_cols, palette ):
                    sub = job_raw.loc[ job_raw[ metric ].notna( ), [ 'timestamp', metric ] ]
                    ax.scatter( 
                        sub[ 'timestamp' ],
                        sub[ metric ],
                        s = 3,
                        alpha = 0.5,
                        color = color,
                        label = metric,
                    )

                ax.set_xlim( window_start, window_end )
                ax.set_xlabel( 'Date' )
                ax.set_ylabel( 'Raw Observed Value' )
                ax.set_title( f'Raw Water Time Series by Metric: {station_row_label( raw_region, raw_station )} (1995-2025)' )
                ax.legend( loc = 'upper left', ncol = min( 3, len( raw_metric_cols ) ), frameon = True )
                plt.tight_layout( )
                plt.show( )


In [ ]:
# lets just see what the heck some stations are doin' re: whole months off?
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.colors import LinearSegmentedColormap

# turbo is too dark at the extremes...
turbo_trimmed = LinearSegmentedColormap.from_list( 
    'turbo_trimmed',
    plt.cm.turbo( np.linspace( 0.10, 0.90, 256 ) ),
)

rolling_overlay_station_keys = [ 
    ( 'wel', 'ht' ),
    ( 'wqb', 'mp' ),
    ( 'wel', 'in' ),
]

rolling_overlay_window_days = 21
rolling_overlay_year_min = 1995
rolling_overlay_year_max = 2025
rolling_overlay_month_starts = [ 1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335 ]
rolling_overlay_month_labels = [ 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec' ]

rolling_overlay_source = water[ [ 'region', 'station', 'datetime', 'water_temp' ] ].copy( )
rolling_overlay_source[ 'timestamp' ] = pd.to_datetime( rolling_overlay_source[ 'datetime' ], errors = 'coerce' )
rolling_overlay_source = rolling_overlay_source.dropna( subset = [ 'timestamp', 'water_temp' ] )
rolling_overlay_source[ 'date' ] = rolling_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
rolling_overlay_temp_ymin = float( np.floor( rolling_overlay_source[ 'water_temp' ].quantile( 0.001 ) ) )
rolling_overlay_temp_ymax = float( np.ceil( rolling_overlay_source[ 'water_temp' ].quantile( 0.999 ) ) )

rolling_overlay_daily = ( 
    rolling_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( water_temp_daily = ( 'water_temp', 'mean' ) )
)

rolling_overlay_daily[ 'year' ] = rolling_overlay_daily[ 'date' ].dt.year
rolling_overlay_daily[ 'day_of_year' ] = rolling_overlay_daily[ 'date' ].dt.dayofyear
rolling_overlay_daily[ 'month' ] = rolling_overlay_daily[ 'date' ].dt.month

# for each station we're ogling... 
for region, station in rolling_overlay_station_keys:
    station_daily = rolling_overlay_daily.loc[ 
        ( rolling_overlay_daily[ 'region' ] == region )
        & ( rolling_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )

    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed # color scheme, yo!

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'water_temp_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'water_temp_smooth' ] = year_slice[ 'water_temp_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        # first year and last year be standout lines, the rest faint
        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'water_temp_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( rolling_overlay_temp_ymin, rolling_overlay_temp_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Water Temp ( C )' )
    ax.set_title( f'Observed Water Temperature By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


In [ ]:
# same station overlay, but for dissolved oxygen with faint threshold bands

oxygen_overlay_source = water[ [ 'region', 'station', 'datetime', 'oxygen' ] ].copy( )
oxygen_overlay_source[ 'timestamp' ] = pd.to_datetime( oxygen_overlay_source[ 'datetime' ], errors = 'coerce' )
oxygen_overlay_source = oxygen_overlay_source.dropna( subset = [ 'timestamp', 'oxygen' ] )
oxygen_overlay_source[ 'date' ] = oxygen_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
oxygen_overlay_ymin = min( 0.0, float( np.floor( oxygen_overlay_source[ 'oxygen' ].quantile( 0.001 ) ) ) )
oxygen_overlay_ymax = max( 5.0, float( np.ceil( oxygen_overlay_source[ 'oxygen' ].quantile( 0.999 ) ) ) )

oxygen_overlay_daily = ( 
    oxygen_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( oxygen_daily = ( 'oxygen', 'mean' ) )
)

oxygen_overlay_daily[ 'year' ] = oxygen_overlay_daily[ 'date' ].dt.year
oxygen_overlay_daily[ 'day_of_year' ] = oxygen_overlay_daily[ 'date' ].dt.dayofyear
oxygen_overlay_daily[ 'month' ] = oxygen_overlay_daily[ 'date' ].dt.month

oxygen_bands = [ 
    ( oxygen_overlay_ymin, 0.0, 'anoxic', '#7f0000' ),
    ( 0.0, 2.0, 'hypoxic', '#d95f0e' ),
    ( 2.0, 5.0, 'marginal', '#e6ab02' ),
    ( 5.0, oxygen_overlay_ymax, 'optimal', '#1b9e77' ),
]

for region, station in rolling_overlay_station_keys:
    station_daily = oxygen_overlay_daily.loc[ 
        ( oxygen_overlay_daily[ 'region' ] == region )
        & ( oxygen_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )
    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for band_ymin, band_ymax, band_label, band_color in oxygen_bands:
        if band_ymax <= band_ymin:
            continue
        ax.axhspan( band_ymin, band_ymax, color = band_color, alpha = 0.08, zorder = 0 )
        ax.text( 365.2, ( band_ymin + band_ymax ) / 2, band_label, color = band_color, fontsize = 9, ha = 'right', va = 'center', alpha = 0.90 )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'oxygen_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'oxygen_smooth' ] = year_slice[ 'oxygen_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'oxygen_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
            zorder = 2,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( oxygen_overlay_ymin, oxygen_overlay_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Dissolved Oxygen ( mg/L )' )
    ax.set_title( f'Observed Dissolved Oxygen By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


In [ ]:
# same station overlay, but for salinity
salinity_overlay_source = water[ [ 'region', 'station', 'datetime', 'salinity' ] ].copy( )
salinity_overlay_source[ 'timestamp' ] = pd.to_datetime( salinity_overlay_source[ 'datetime' ], errors = 'coerce' )
salinity_overlay_source = salinity_overlay_source.dropna( subset = [ 'timestamp', 'salinity' ] )
salinity_overlay_source[ 'date' ] = salinity_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
salinity_overlay_ymin = float( np.floor( salinity_overlay_source[ 'salinity' ].quantile( 0.001 ) ) )
salinity_overlay_ymax = float( np.ceil( salinity_overlay_source[ 'salinity' ].quantile( 0.999 ) ) )

salinity_overlay_daily = ( 
    salinity_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( salinity_daily = ( 'salinity', 'mean' ) )
)

salinity_overlay_daily[ 'year' ] = salinity_overlay_daily[ 'date' ].dt.year
salinity_overlay_daily[ 'day_of_year' ] = salinity_overlay_daily[ 'date' ].dt.dayofyear
salinity_overlay_daily[ 'month' ] = salinity_overlay_daily[ 'date' ].dt.month

for region, station in rolling_overlay_station_keys:
    station_daily = salinity_overlay_daily.loc[ 
        ( salinity_overlay_daily[ 'region' ] == region )
        & ( salinity_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )
    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'salinity_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'salinity_smooth' ] = year_slice[ 'salinity_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'salinity_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( salinity_overlay_ymin, salinity_overlay_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Salinity ( PSU )' )
    ax.set_title( f'Observed Salinity By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


In [ ]:
# same station overlay, but for solar radiation
solar_overlay_source = water[ [ 'region', 'station', 'datetime', 'solar_radiation' ] ].copy( )
solar_overlay_source[ 'timestamp' ] = pd.to_datetime( solar_overlay_source[ 'datetime' ], errors = 'coerce' )
solar_overlay_source = solar_overlay_source.dropna( subset = [ 'timestamp', 'solar_radiation' ] )
solar_overlay_source[ 'date' ] = solar_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
solar_overlay_ymin = max( 0.0, float( np.floor( solar_overlay_source[ 'solar_radiation' ].quantile( 0.001 ) ) ) )
solar_overlay_ymax = float( np.ceil( solar_overlay_source[ 'solar_radiation' ].quantile( 0.999 ) ) )

solar_overlay_daily = ( 
    solar_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( solar_radiation_daily = ( 'solar_radiation', 'mean' ) )
)

solar_overlay_daily[ 'year' ] = solar_overlay_daily[ 'date' ].dt.year
solar_overlay_daily[ 'day_of_year' ] = solar_overlay_daily[ 'date' ].dt.dayofyear
solar_overlay_daily[ 'month' ] = solar_overlay_daily[ 'date' ].dt.month

for region, station in rolling_overlay_station_keys:
    station_daily = solar_overlay_daily.loc[ 
        ( solar_overlay_daily[ 'region' ] == region )
        & ( solar_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )
    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'solar_radiation_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'solar_radiation_smooth' ] = year_slice[ 'solar_radiation_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'solar_radiation_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( solar_overlay_ymin, solar_overlay_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Solar Radiation ( kWh/m^2 )' )
    ax.set_title( f'Observed Solar Radiation By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


In [ ]:
# same station overlay, but for wind speed
wind_overlay_source = water[ [ 'region', 'station', 'datetime', 'wind_speed' ] ].copy( )
wind_overlay_source[ 'timestamp' ] = pd.to_datetime( wind_overlay_source[ 'datetime' ], errors = 'coerce' )
wind_overlay_source = wind_overlay_source.dropna( subset = [ 'timestamp', 'wind_speed' ] )
wind_overlay_source[ 'date' ] = wind_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
wind_overlay_ymin = max( 0.0, float( np.floor( wind_overlay_source[ 'wind_speed' ].quantile( 0.001 ) ) ) )
wind_overlay_ymax = float( np.ceil( wind_overlay_source[ 'wind_speed' ].quantile( 0.999 ) ) )

wind_overlay_daily = ( 
    wind_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( wind_speed_daily = ( 'wind_speed', 'mean' ) )
)

wind_overlay_daily[ 'year' ] = wind_overlay_daily[ 'date' ].dt.year
wind_overlay_daily[ 'day_of_year' ] = wind_overlay_daily[ 'date' ].dt.dayofyear
wind_overlay_daily[ 'month' ] = wind_overlay_daily[ 'date' ].dt.month

for region, station in rolling_overlay_station_keys:
    station_daily = wind_overlay_daily.loc[ 
        ( wind_overlay_daily[ 'region' ] == region )
        & ( wind_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )
    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'wind_speed_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'wind_speed_smooth' ] = year_slice[ 'wind_speed_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'wind_speed_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( wind_overlay_ymin, wind_overlay_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Wind Speed ( m/s )' )
    ax.set_title( f'Observed Wind Speed By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


In [ ]:
# same station overlay, but for precipitation
precip_overlay_source = water[ [ 'region', 'station', 'datetime', 'precipitation' ] ].copy( )
precip_overlay_source[ 'timestamp' ] = pd.to_datetime( precip_overlay_source[ 'datetime' ], errors = 'coerce' )
precip_overlay_source = precip_overlay_source.dropna( subset = [ 'timestamp', 'precipitation' ] )
precip_overlay_source[ 'date' ] = precip_overlay_source[ 'timestamp' ].dt.floor( 'D' )

# use a robust full-dataset range so one bad spike does not dictate the plot scale
precip_overlay_ymin = max( 0.0, float( np.floor( precip_overlay_source[ 'precipitation' ].quantile( 0.001 ) ) ) )
precip_overlay_ymax = float( np.ceil( precip_overlay_source[ 'precipitation' ].quantile( 0.999 ) ) )

precip_overlay_daily = ( 
    precip_overlay_source
    .groupby( [ 'region', 'station', 'date' ], as_index = False )
    .agg( precipitation_daily = ( 'precipitation', 'mean' ) )
)

precip_overlay_daily[ 'year' ] = precip_overlay_daily[ 'date' ].dt.year
precip_overlay_daily[ 'day_of_year' ] = precip_overlay_daily[ 'date' ].dt.dayofyear
precip_overlay_daily[ 'month' ] = precip_overlay_daily[ 'date' ].dt.month

for region, station in rolling_overlay_station_keys:
    station_daily = precip_overlay_daily.loc[ 
        ( precip_overlay_daily[ 'region' ] == region )
        & ( precip_overlay_daily[ 'station' ] == station )
    ].copy( )

    title_label = station_label( region, station )
    year_order = sorted( station_daily[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_norm = Normalize( vmin = rolling_overlay_year_min, vmax = rolling_overlay_year_max )
    line_cmap = turbo_trimmed

    fig, ax = plt.subplots( figsize = ( 12, 5.5 ) )

    for year in year_order:
        year_slice = station_daily.loc[ station_daily[ 'year' ] == year, [ 'date', 'precipitation_daily' ] ].copy( )
        year_start = pd.Timestamp( year = int( year ), month = 1, day = 1 )
        year_end = pd.Timestamp( year = int( year ), month = 12, day = 31 )
        full_dates = pd.date_range( year_start, year_end, freq = 'D' )

        year_slice = ( 
            year_slice
            .drop_duplicates( subset = [ 'date' ] )
            .set_index( 'date' )
            .reindex( full_dates )
            .rename_axis( 'date' )
            .reset_index( )
        )
        year_slice[ 'day_of_year' ] = year_slice[ 'date' ].dt.dayofyear
        year_slice[ 'precipitation_smooth' ] = year_slice[ 'precipitation_daily' ].rolling( window = rolling_overlay_window_days, center = True, min_periods = 7 ).mean( )

        line_alpha = 1.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.33
        line_width = 3.0 if year in [ year_order[ 0 ], year_order[ -1 ] ] else 0.75

        ax.plot( 
            year_slice[ 'day_of_year' ],
            year_slice[ 'precipitation_smooth' ],
            color = line_cmap( year_norm( year ) ),
            alpha = line_alpha,
            linewidth = line_width,
        )

    ax.set_xlim( 1, 366 )
    ax.set_xticks( rolling_overlay_month_starts )
    ax.set_xticklabels( rolling_overlay_month_labels )
    ax.set_ylim( precip_overlay_ymin, precip_overlay_ymax )
    ax.set_xlabel( 'Month' )
    ax.set_ylabel( f'{rolling_overlay_window_days}-Day Rolling Mean Precipitation ( mm/h )' )
    ax.set_title( f'Observed Precipitation By Year: {region}.{station} | {title_label}' )
    ax.grid( alpha = 0.2 )

    sm = ScalarMappable( norm = year_norm, cmap = line_cmap )
    sm.set_array( [ ] )
    cbar = plt.colorbar( sm, ax = ax, pad = 0.02 )
    cbar.set_label( 'Year ( older to newer )' )
    cbar.set_ticks( [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] )

    if len( year_order ) > 1:
        ax.text( 
            0.995,
            0.02,
            f'first and last years highlighted; middle years shown faintly ({ year_order[ 0 ] } to { year_order[ -1 ] })',
            transform = ax.transAxes,
            ha = 'right',
            va = 'bottom',
            fontsize = 9,
            color = '#3f4852',
            bbox = dict( boxstyle = 'round,pad=0.25', facecolor = 'white', alpha = 0.85, edgecolor = '#cccccc' ),
        )

    plt.tight_layout( )
    plt.show( )


### 5.4 finalize modeling cohorts (data quality + holdout split)
- remove stations with no usable water signal (all-NaN across core water metrics)
- designate confirmed transitioning stations as a held-out validation set
- keep remaining stations as the primary training cohort


In [ ]:
# 5.4 - remove weak stations, then weak station-years, then create train/holdout station cohorts

core_water_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in daily_water.columns ]

station_raw_quality = ( 
    daily_water
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_daily_rows = ( 'date', 'size' ) )
)

if len( core_water_cols ) > 0:
    raw_non_null_counts = ( 
        daily_water
        .groupby( [ 'region', 'station' ] )[ core_water_cols ]
        .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
        .rename( 'n_non_null_core_water' )
        .reset_index( )
    )

else:
    raw_non_null_counts = station_raw_quality[ [ 'region', 'station' ] ].copy( )
    raw_non_null_counts[ 'n_non_null_core_water' ] = 0

station_raw_quality = station_raw_quality.merge( raw_non_null_counts, on = [ 'region', 'station' ], how = 'left' )
station_raw_quality[ 'has_raw_signal' ] = station_raw_quality[ 'n_non_null_core_water' ] > 0

valid_station_keys = station_raw_quality.loc[ station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )
dropped_nan_station_keys = station_raw_quality.loc[ ~station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )

if 'pointless_station_keys' in globals( ):
    pointless_station_keys_local = pointless_station_keys[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

elif Path( f'{phase2_out_dir}/t4d.phase2.pointless_station.keys.csv' ).exists( ):
    pointless_station_keys_local = pd.read_csv( f'{phase2_out_dir}/t4d.phase2.pointless_station.keys.csv' )[ [ 'region', 'station' ] ].drop_duplicates( )

else:
    pointless_station_keys_local = pd.DataFrame( columns = [ 'region', 'station' ] )

pointless_station_keys_local[ 'region' ] = pointless_station_keys_local[ 'region' ].astype( str ).str.strip( ).str.lower( )
pointless_station_keys_local[ 'station' ] = pointless_station_keys_local[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

valid_station_keys_pre_pointless_filter = valid_station_keys.copy( )
valid_station_keys = ( 
    valid_station_keys
    .merge( pointless_station_keys_local.assign( has_pointless_station = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: ~frame[ 'has_pointless_station' ].fillna( False ).astype( bool ), [ 'region', 'station' ] ]
    .drop_duplicates( )
)
dropped_pointless_station_keys = ( 
    valid_station_keys_pre_pointless_filter
    .merge( pointless_station_keys_local.assign( has_pointless_station = True ), on = [ 'region', 'station' ], how = 'inner' )
    [ [ 'region', 'station' ] ]
    .drop_duplicates( )
)

if 'seasonal_break_station_keys' in globals( ):
    seasonal_break_keys_local = seasonal_break_station_keys[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

elif Path( f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv' ).exists( ):
    seasonal_break_keys_local = pd.read_csv( f'{phase2_out_dir}/t4d.phase2.seasonal_break.keys.csv' )[ [ 'region', 'station' ] ].drop_duplicates( )

else:
    seasonal_break_keys_local = pd.DataFrame( columns = [ 'region', 'station' ] )

seasonal_break_keys_local[ 'region' ] = seasonal_break_keys_local[ 'region' ].astype( str ).str.strip( ).str.lower( )
seasonal_break_keys_local[ 'station' ] = seasonal_break_keys_local[ 'station' ].astype( str ).str.strip( ).str.lower( ).str[ -2: ]

valid_station_keys_pre_break_filter = valid_station_keys.copy( )
valid_station_keys = ( 
    valid_station_keys
    .merge( seasonal_break_keys_local.assign( has_seasonal_break = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: ~frame[ 'has_seasonal_break' ].fillna( False ).astype( bool ), [ 'region', 'station' ] ]
    .drop_duplicates( )
)
dropped_seasonal_break_station_keys = ( 
    valid_station_keys_pre_break_filter
    .merge( seasonal_break_keys_local.assign( has_seasonal_break = True ), on = [ 'region', 'station' ], how = 'inner' )
    [ [ 'region', 'station' ] ]
    .drop_duplicates( )
)

print( f'total stations considered: {station_raw_quality.shape[ 0 ]}' )
print( f'stations with usable raw signal: {valid_station_keys_pre_pointless_filter.shape[ 0 ]}' )
print( f'stations dropped (all-NaN core water metrics): {dropped_nan_station_keys.shape[ 0 ]}' )
print( f'stations dropped (lookup-preexcluded or short record span): {dropped_pointless_station_keys.shape[ 0 ]}' )
print( f'stations dropped (recurring 2+ month same-time breaks): {dropped_seasonal_break_station_keys.shape[ 0 ]}' )
print( f'stations retained for modeling before year-level cleanup: {valid_station_keys.shape[ 0 ]}' )

if dropped_nan_station_keys.shape[ 0 ] > 0:
    display( dropped_nan_station_keys.head( 20 ) )

if dropped_pointless_station_keys.shape[ 0 ] > 0:
    display( dropped_pointless_station_keys.head( 20 ) )

if dropped_seasonal_break_station_keys.shape[ 0 ] > 0:
    display( dropped_seasonal_break_station_keys.head( 20 ) )


# keep the station-level screen first
# this removes clearly pointless stations before we look at individual years


def keep_valid_stations( frame ):
    return frame.merge( valid_station_keys, on = [ 'region', 'station' ], how = 'inner' )


daily_air_model = keep_valid_stations( daily_air )
daily_water_model = keep_valid_stations( daily_water )
fiveyear_water_model = keep_valid_stations( fiveyear_water )
p5_model = keep_valid_stations( p5 )
p5_monthly_model = keep_valid_stations( p5_monthly )


# now do a lighter station-year screen
# this catches good stations that still have weak years mixed into otherwise usable records
station_year_min_any_core_days = 120
station_year_min_temp_days = 120
station_year_min_temp_months = 6
station_year_min_warm_oxygen_days = 45
station_year_min_warm_oxygen_months = 3

station_year_source = daily_water_model.copy( )
station_year_source[ 'date' ] = pd.to_datetime( station_year_source[ 'date' ], errors = 'coerce' )
station_year_source = station_year_source.dropna( subset = [ 'date' ] )
station_year_source[ 'year' ] = station_year_source[ 'date' ].dt.year
station_year_source[ 'month_num' ] = station_year_source[ 'date' ].dt.month
station_year_source[ 'has_any_core_obs' ] = station_year_source[ core_water_cols ].notna( ).any( axis = 1 ) if len( core_water_cols ) > 0 else False
station_year_source[ 'has_temp_obs' ] = station_year_source[ 'water_temp' ].notna( ) if 'water_temp' in station_year_source.columns else False
station_year_source[ 'has_oxygen_obs' ] = station_year_source[ 'oxygen' ].notna( ) if 'oxygen' in station_year_source.columns else False

station_year_quality = ( 
    station_year_source
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        n_days = ( 'date', 'size' ),
        n_any_core_days = ( 'has_any_core_obs', 'sum' ),
        n_temp_days = ( 'has_temp_obs', 'sum' ),
        n_oxygen_days = ( 'has_oxygen_obs', 'sum' ),
    )
)

month_any = station_year_source.loc[ station_year_source[ 'has_any_core_obs' ], [ 'region', 'station', 'year', 'month_num' ] ].drop_duplicates( )
month_temp = station_year_source.loc[ station_year_source[ 'has_temp_obs' ], [ 'region', 'station', 'year', 'month_num' ] ].drop_duplicates( )
month_oxygen = station_year_source.loc[ station_year_source[ 'has_oxygen_obs' ], [ 'region', 'station', 'year', 'month_num' ] ].drop_duplicates( )

for out_col, source_df in [ 
    ( 'n_any_core_months', month_any ),
    ( 'n_temp_months', month_temp ),
    ( 'n_oxygen_months', month_oxygen ),
]:
    month_counts = source_df.groupby( [ 'region', 'station', 'year' ] ).size( ).rename( out_col ).reset_index( )
    station_year_quality = station_year_quality.merge( month_counts, on = [ 'region', 'station', 'year' ], how = 'left' )

warm_source = station_year_source.loc[ station_year_source[ 'month_num' ].between( 6, 9 ) ].copy( )
warm_quality = ( 
    warm_source
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        n_warm_days = ( 'date', 'size' ),
        n_warm_temp_days = ( 'has_temp_obs', 'sum' ),
        n_warm_oxygen_days = ( 'has_oxygen_obs', 'sum' ),
    )
)

warm_month_temp = warm_source.loc[ warm_source[ 'has_temp_obs' ], [ 'region', 'station', 'year', 'month_num' ] ].drop_duplicates( )
warm_month_oxygen = warm_source.loc[ warm_source[ 'has_oxygen_obs' ], [ 'region', 'station', 'year', 'month_num' ] ].drop_duplicates( )

for out_col, source_df in [ 
    ( 'n_warm_temp_months', warm_month_temp ),
    ( 'n_warm_oxygen_months', warm_month_oxygen ),
]:
    month_counts = source_df.groupby( [ 'region', 'station', 'year' ] ).size( ).rename( out_col ).reset_index( )
    warm_quality = warm_quality.merge( month_counts, on = [ 'region', 'station', 'year' ], how = 'left' )

station_year_quality = station_year_quality.merge( warm_quality, on = [ 'region', 'station', 'year' ], how = 'left' ).fillna( 0 )
station_year_quality[ 'has_model_year' ] = ( 
    ( station_year_quality[ 'n_any_core_days' ] >= station_year_min_any_core_days )
    & ( station_year_quality[ 'n_temp_days' ] >= station_year_min_temp_days )
    & ( station_year_quality[ 'n_temp_months' ] >= station_year_min_temp_months )
)
station_year_quality[ 'has_warm_oxygen_year' ] = ( 
    station_year_quality[ 'has_model_year' ]
    & ( station_year_quality[ 'n_warm_oxygen_days' ] >= station_year_min_warm_oxygen_days )
    & ( station_year_quality[ 'n_warm_oxygen_months' ] >= station_year_min_warm_oxygen_months )
)

model_station_year_keys = station_year_quality.loc[ station_year_quality[ 'has_model_year' ], [ 'region', 'station', 'year' ] ].drop_duplicates( )
dropped_sparse_station_year_keys = station_year_quality.loc[ ~station_year_quality[ 'has_model_year' ], [ 'region', 'station', 'year', 'n_any_core_days', 'n_temp_days', 'n_temp_months' ] ].copy( )

print( '\nstation-year cleanup thresholds:' )
print( f'  - min any-core days per year: {station_year_min_any_core_days}' )
print( f'  - min water-temp days per year: {station_year_min_temp_days}' )
print( f'  - min water-temp months per year: {station_year_min_temp_months}' )
print( f'  - min warm oxygen days per year: {station_year_min_warm_oxygen_days}' )
print( f'  - min warm oxygen months per year: {station_year_min_warm_oxygen_months}' )
print( f'station-years retained for modeling: {len( model_station_year_keys )}' )
print( f'station-years dropped as too sparse: {len( dropped_sparse_station_year_keys )}' )
print( f'station-years with usable warm oxygen coverage: {int( station_year_quality[ "has_warm_oxygen_year" ].sum( ) )}' )

if len( dropped_sparse_station_year_keys ) > 0:
    display( dropped_sparse_station_year_keys.head( 20 ) )


# apply the year filter to every model table
# use the same station-year keys so phase 6 and phase 7 see the same cleaned scope


def keep_valid_station_years( frame, date_col = 'date', flag_col = 'has_model_year' ):
    frame = frame.copy( )

    actual_date_col = date_col
    if actual_date_col not in frame.columns:
        if 'month' in frame.columns:
            actual_date_col = 'month'

        else:
            return frame

    frame[ actual_date_col ] = pd.to_datetime( frame[ actual_date_col ], errors = 'coerce' )
    frame = frame.dropna( subset = [ actual_date_col ] )
    frame[ 'year' ] = frame[ actual_date_col ].dt.year

    keep_keys = station_year_quality.loc[ 
        station_year_quality[ flag_col ].fillna( False ).astype( bool ),
        [ 'region', 'station', 'year' ],
    ].drop_duplicates( )

    frame = frame.merge( keep_keys, on = [ 'region', 'station', 'year' ], how = 'inner' )
    return frame.drop( columns = [ 'year' ] )


daily_air_model = keep_valid_station_years( daily_air_model )
daily_water_model = keep_valid_station_years( daily_water_model )
fiveyear_water_model = keep_valid_station_years( fiveyear_water_model )
p5_model = keep_valid_station_years( p5_model )
p5_monthly_model = keep_valid_station_years( p5_monthly_model )

transition_station_keys = ( 
    p5_model
    .loc[ p5_model[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ]
    .drop_duplicates( )
)

train_station_keys = ( 
    valid_station_keys
    .merge( transition_station_keys.assign( is_transition = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_transition' ].isna( ), [ 'region', 'station' ] ]
)

holdout_station_keys = transition_station_keys.copy( )


def subset_by_station_keys( frame, station_keys ):
    return frame.merge( station_keys, on = [ 'region', 'station' ], how = 'inner' )


daily_air_train = subset_by_station_keys( daily_air_model, train_station_keys )
daily_water_train = subset_by_station_keys( daily_water_model, train_station_keys )
fiveyear_water_train = subset_by_station_keys( fiveyear_water_model, train_station_keys )
p5_train = subset_by_station_keys( p5_model, train_station_keys )
p5_monthly_train = subset_by_station_keys( p5_monthly_model, train_station_keys )

daily_air_holdout = subset_by_station_keys( daily_air_model, holdout_station_keys )
daily_water_holdout = subset_by_station_keys( daily_water_model, holdout_station_keys )
fiveyear_water_holdout = subset_by_station_keys( fiveyear_water_model, holdout_station_keys )
p5_holdout = subset_by_station_keys( p5_model, holdout_station_keys )
p5_monthly_holdout = subset_by_station_keys( p5_monthly_model, holdout_station_keys )

print( f'train stations: {train_station_keys.shape[ 0 ]}' )
print( f'holdout (transition) stations: {holdout_station_keys.shape[ 0 ]}' )
print( f'training rows (daily_air, daily_water, p5): {daily_air_train.shape[ 0 ]}, {daily_water_train.shape[ 0 ]}, {p5_train.shape[ 0 ]}' )
print( f'holdout rows (daily_air, daily_water, p5): {daily_air_holdout.shape[ 0 ]}, {daily_water_holdout.shape[ 0 ]}, {p5_holdout.shape[ 0 ]}' )

In [ ]:
# 5.4 progress report: post-filter flagged vs unflagged timelines
# shows both cohorts under current 5.4 filtering in one run

from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

core_water_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in daily_water.columns ]

station_raw_quality = ( 
    daily_water
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_daily_rows = ( 'date', 'size' ) )
)

if len( core_water_cols ) > 0:
    raw_non_null_counts = ( 
        daily_water
        .groupby( [ 'region', 'station' ] )[ core_water_cols ]
        .apply( lambda frame: int( frame.notna( ).sum( ).sum( ) ) )
        .rename( 'n_non_null_core_water' )
        .reset_index( )
    )

else:
    raw_non_null_counts = station_raw_quality[ [ 'region', 'station' ] ].copy( )
    raw_non_null_counts[ 'n_non_null_core_water' ] = 0

station_raw_quality = station_raw_quality.merge( raw_non_null_counts, on = [ 'region', 'station' ], how = 'left' )
station_raw_quality[ 'has_raw_signal' ] = station_raw_quality[ 'n_non_null_core_water' ] > 0

if 'valid_station_keys' in globals( ):
    valid_station_keys_vis = valid_station_keys.copy( )

else:
    valid_station_keys_vis = station_raw_quality.loc[ station_raw_quality[ 'has_raw_signal' ], [ 'region', 'station' ] ].drop_duplicates( )


def keep_valid_for_vis( frame ):
    return frame.merge( valid_station_keys_vis, on = [ 'region', 'station' ], how = 'inner' )

p5_vis = keep_valid_for_vis( p5 )
fiveyear_vis = keep_valid_for_vis( fiveyear_water )

station_baseline_regime = ( 
    cluster_station[ [ 'region', 'station', 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'region', 'station' ] )
    .rename( columns = { 
        'cluster': 'baseline_cluster',
        'cluster_label': 'baseline_label',
    } )
)

cluster_label_map = ( 
    cluster_station[ [ 'cluster', 'cluster_label' ] ]
    .drop_duplicates( subset = [ 'cluster' ] )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

all_stations = valid_station_keys_vis.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
flagged_keys = p5_vis.loc[ p5_vis[ 'flip_confirmed' ].fillna( False ), [ 'region', 'station' ] ].drop_duplicates( )

unflagged_keys = ( 
    all_stations[ [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
    .merge( flagged_keys.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station', 'baseline_cluster', 'baseline_label' ] ]
)

print( f'total stations in scope: {len( all_stations )}' )
print( f'flagged stations in scope: {len( flagged_keys )}' )
print( f'unflagged stations in scope: {len( unflagged_keys )}' )

raw_metric_cols = [ col for col in [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ] if col in fiveyear_vis.columns ]
cluster_ids = sorted( cluster_station[ 'cluster' ].dropna( ).astype( int ).unique( ).tolist( ) )
value_map = { -2: 0, -1: 1 }
for idx, cid in enumerate( cluster_ids, start = 2 ):
    value_map[ int( cid ) ] = idx

palette = [ '#ffffff', '#bdbdbd' ] + sns.color_palette( 'tab10', n_colors = len( cluster_ids ) ).as_hex( )
cmap = ListedColormap( palette )
norm = BoundaryNorm( np.arange( -0.5, len( palette ) + 0.5, 1 ), cmap.N )


def render_timeline( station_df, title, y_label, max_rows = 60 ):
    station_df = station_df.copy( )
    station_df = station_df.head( max_rows ).reset_index( drop = True )

    if len( station_df ) == 0:
        print( f'{title}: no stations to show' )
        return

    station_df = add_station_row_labels( station_df )

    timeline_start = pd.Timestamp( '1995-01-01' )
    timeline_end = pd.Timestamp( '2025-12-31' )
    all_dates = pd.date_range( timeline_start, timeline_end, freq = 'D' )

    station_grid = ( 
        station_df[ [ 'region', 'station', 'row_label' ] ]
        .assign( _tmp = 1 )
        .merge( pd.DataFrame( { 'date': all_dates, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
        .drop( columns = [ '_tmp' ] )
    )

    station_data_presence = ( 
        fiveyear_vis[ [ 'region', 'station', 'date' ] + raw_metric_cols ]
        .copy( )
        .assign( has_data = lambda frame: frame[ raw_metric_cols ].notna( ).any( axis = 1 ) if len( raw_metric_cols ) > 0 else True )
        .loc[ :, [ 'region', 'station', 'date', 'has_data' ] ]
    )

    p5_track = p5_vis[ [ 'region', 'station', 'date', 'implied_cluster' ] ].copy( )

    timeline = ( 
        station_grid
        .merge( station_data_presence, on = [ 'region', 'station', 'date' ], how = 'left' )
        .merge( p5_track, on = [ 'region', 'station', 'date' ], how = 'left' )
    )

    timeline[ 'has_data' ] = timeline[ 'has_data' ].fillna( False ).astype( bool )
    timeline[ 'state' ] = -2
    timeline.loc[ timeline[ 'has_data' ], 'state' ] = -1
    timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'state' ] = timeline.loc[ timeline[ 'implied_cluster' ].notna( ), 'implied_cluster' ].astype( int )
    timeline[ 'cluster_plot' ] = timeline[ 'state' ].map( value_map )

    matrix = ( 
        timeline
        .pivot( index = 'row_label', columns = 'date', values = 'cluster_plot' )
        .reindex( station_df[ 'row_label' ] )
        .to_numpy( )
    )

    fig_height = max( 6, 0.42 * len( station_df ) + 2 )
    fig, ax = plt.subplots( figsize = ( 18, fig_height ) )
    ax.imshow( matrix, aspect = 'auto', interpolation = 'nearest', cmap = cmap, norm = norm )

    ax.set_yticks( np.arange( len( station_df ) ) )
    ax.set_yticklabels( station_df[ 'row_label' ] )

    baseline_values = station_df[ 'baseline_cluster' ].apply( lambda val: value_map.get( int( val ), 0 ) if pd.notna( val ) else 0 )
    baseline_colors = [ palette[ int( val ) ] for val in baseline_values ]
    ax.scatter( 
        np.full( len( station_df ), -4.0 ),
        np.arange( len( station_df ) ),
        s = 105,
        c = baseline_colors,
        edgecolors = 'black',
        linewidths = 0.7,
        clip_on = False,
        zorder = 5,
    )
    ax.set_xlim( -8, matrix.shape[ 1 ] - 0.5 )

    tick_dates = pd.date_range( timeline_start, timeline_end, freq = '2YS' )
    tick_positions = [ np.searchsorted( all_dates.values, np.datetime64( dt ) ) for dt in tick_dates ]
    ax.set_xticks( tick_positions )
    ax.set_xticklabels( [ str( dt.year ) for dt in tick_dates ], rotation = 45, ha = 'right' )

    ax.set_xlabel( 'Date' )
    ax.set_ylabel( y_label )
    ax.set_title( title )

    legend_handles = [ 
        mpatches.Patch( color = palette[ 0 ], label = 'No data' ),
        mpatches.Patch( color = palette[ 1 ], label = 'Data, no classification' ),
    ]
    for cid in cluster_ids:
        legend_handles.append( mpatches.Patch( color = palette[ value_map[ cid ] ], label = cluster_label_map.get( cid, f'Cluster {cid}' ) ) )
    legend_handles.append( mpatches.Patch( facecolor = 'white', edgecolor = 'black', label = 'Left dot = baseline regime' ) )

    ax.legend( handles = legend_handles, loc = 'upper left', bbox_to_anchor = ( 1.01, 1.0 ), frameon = True, title = 'Timeline State' )
    plt.tight_layout( )
    plt.show( )


flagged_rows = flagged_keys.merge( station_baseline_regime, on = [ 'region', 'station' ], how = 'left' )
render_timeline( flagged_rows, 'Progress Report: Flagged Stations ( Post-5.4 )', 'Flagged Station ( Code, Station Name, Region Name )', max_rows = 60 )
render_timeline( unflagged_keys, 'Progress Report: Unflagged Stations ( Post-5.4 )', 'Unflagged Station ( Code, Station Name, Region Name )', max_rows = 100 )



In [ ]:
# 5.5 - drop never-classified stations and rebuild final flagged/unflagged cohorts

p5_scope = p5_model
daily_air_scope = daily_air_model
daily_water_scope = daily_water_model
fiveyear_water_scope = fiveyear_water_model
valid_station_scope = valid_station_keys.copy( )

station_classification_quality = ( 
    p5_scope
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        n_classified_days = ( 'implied_cluster', lambda values: int( values.notna( ).sum( ) ) ),
        n_flip_days = ( 'flip_confirmed', lambda values: int( values.fillna( False ).sum( ) ) ),
    )
)

classifiable_station_keys = station_classification_quality.loc[ 
    station_classification_quality[ 'n_classified_days' ] > 0,
    [ 'region', 'station' ],
].drop_duplicates( )

dropped_never_classified_station_keys = ( 
    valid_station_scope
    .merge( classifiable_station_keys.assign( is_classifiable = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_classifiable' ].isna( ), [ 'region', 'station' ] ]
)

print( f'valid stations entering 5.5: {len( valid_station_scope )}' )
print( f'classifiable stations kept: {len( classifiable_station_keys )}' )
print( f'dropped never-classified stations: {len( dropped_never_classified_station_keys )}' )

if len( dropped_never_classified_station_keys ) > 0:
    display( dropped_never_classified_station_keys.head( 20 ) )

flagged_station_keys_final = holdout_station_keys.merge( classifiable_station_keys, on = [ 'region', 'station' ], how = 'inner' )

unflagged_station_keys_final = ( 
    classifiable_station_keys
    .merge( flagged_station_keys_final.assign( is_flagged = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ 'is_flagged' ].isna( ), [ 'region', 'station' ] ]
)


def subset_by_station_keys( frame, station_keys ):
    return frame.merge( station_keys, on = [ 'region', 'station' ], how = 'inner' )


# this is the final scope of stations for all datasets, including both train and holdouts, after all 5.5 filtering
# used in phase 6
daily_air_final = subset_by_station_keys( daily_air_scope, classifiable_station_keys )
daily_water_final = subset_by_station_keys( daily_water_scope, classifiable_station_keys )
fiveyear_water_final = subset_by_station_keys( fiveyear_water_scope, classifiable_station_keys )
p5_final = subset_by_station_keys( p5_scope, classifiable_station_keys )
p5_monthly_final = subset_by_station_keys( p5_monthly_model, classifiable_station_keys )

daily_air_train_final = subset_by_station_keys( daily_air_scope, unflagged_station_keys_final )
daily_water_train_final = subset_by_station_keys( daily_water_scope, unflagged_station_keys_final )
fiveyear_water_train_final = subset_by_station_keys( fiveyear_water_scope, unflagged_station_keys_final )
p5_train_final = subset_by_station_keys( p5_scope, unflagged_station_keys_final )
p5_monthly_train_final = subset_by_station_keys( p5_monthly_model, unflagged_station_keys_final )

daily_air_holdout_final = subset_by_station_keys( daily_air_scope, flagged_station_keys_final )
daily_water_holdout_final = subset_by_station_keys( daily_water_scope, flagged_station_keys_final )
fiveyear_water_holdout_final = subset_by_station_keys( fiveyear_water_scope, flagged_station_keys_final )
p5_holdout_final = subset_by_station_keys( p5_scope, flagged_station_keys_final )
p5_monthly_holdout_final = subset_by_station_keys( p5_monthly_model, flagged_station_keys_final )

print( f'final unflagged/train stations: {len( unflagged_station_keys_final )}' )
print( f'final flagged/holdout stations: {len( flagged_station_keys_final )}' )
print( f'final training rows (daily_air, daily_water, p5): {len( daily_air_train_final )}, {len( daily_water_train_final )}, {len( p5_train_final )}' )
print( f'final holdout rows (daily_air, daily_water, p5): {len( daily_air_holdout_final )}, {len( daily_water_holdout_final )}, {len( p5_holdout_final )}' )


## Phase 6 — Model Development: Air → Water Temperature
Goal: predict ΔT_water given atmospheric forcing history

# also stage 1

- 6.1 establish naive baselines
  - persistence forecast and climatological mean by day-of-year
  - record RMSE and MAE
- 6.2 train ridge/lasso regression using lagged atmospheric features
  - interpret coefficients as a sanity check
- 6.3 train gradient boosted model (XGBoost or LightGBM) on same features
  - compare to linear baseline
- 6.4 use walk-forward temporal cross-validation
  - train on years 1–N, test on N+1
  - since reading suggests standard k-fold doesn't work well with this kind of job
- 6.5 evaluate on held-out transitioning stations from phase 5
  - does the model generalize across space?
- 6.6 select best model, document feature importances
  - lag windows that dominate are a reportable finding

## Note

- for “unknown estuary” inference, use 3 modes:
  - air-only (weakest, fallback).
  - air + baseline water state (recommended): baseline water levels, depth, season, and simple station metadata.
  - air + short calibration window (best): after a new estuary has enough observations, update the station-specific context honestly from that short local record.
- avoid using end-of-record water summaries as if they were available on every historical date.

In [ ]:
# 6.0 - define inference contract + build a simple estuary-context table
# use training stations only here so we do not leak holdout-station signatures into every row

context_source_water = daily_water_train_final.copy( )
context_source_water[ 'date' ] = pd.to_datetime( context_source_water[ 'date' ], errors = 'coerce' )
context_source_water = context_source_water.dropna( subset = [ 'date' ] )

context_metric_candidates = [ 'water_temp', 'salinity', 'oxygen', 'ph', 'depth' ]
context_metrics = [ col for col in context_metric_candidates if col in context_source_water.columns ]

# keep this context honest and simple
# it is a static training-station summary, not a fake rolling window copied onto every date
context_agg = { }
for col in context_metrics:
    context_agg[ f'ctx_{col}_mean' ] = ( col, 'mean' )
    context_agg[ f'ctx_{col}_std' ] = ( col, 'std' )

phase6_context_table = ( 
    context_source_water
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( **context_agg )
)

phase6_air_feature_candidates = [ 
    'air_temp_r1d',
    'air_temp_r7d',
    'air_temp_r28d',
    'wind_speed_r1d',
    'wind_speed_r7d',
    'wind_speed_r28d',
    'precip_r1d',
    'precip_r7d',
    'precip_r28d',
    'solar_r1d',
    'solar_r7d',
    'solar_r28d',
    'air_temp',
    'wind_speed',
    'precip',
    'solar',
    'doy_sin',
    'doy_cos',
    'air_temp_minus_water_temp_baseline',
    'air_temp_r7d_minus_water_temp_baseline',
    'air_temp_r28d_minus_water_temp_baseline',
    'air_temp_r1d_minus_air_temp_r28d',
    'air_temp_r7d_minus_air_temp_r28d',
    'wind_speed_x_air_temp',
    'solar_x_air_temp',
]

phase6_context_feature_candidates = [ 
    'cluster_code',
    'water_temp_baseline',
    'salinity_baseline',
    'oxygen_baseline',
    'ph_baseline',
    'depth_baseline',
] + [ col for col in phase6_context_table.columns if col.startswith( 'ctx_' ) ]

phase6_inference_contract = { 
    'required_keys': [ 'region', 'station', 'date' ],
    'required_air_features': phase6_air_feature_candidates,
    'optional_estuary_context': phase6_context_feature_candidates,
    'target': 'delta_water_temp',
    'absolute_reconstruction': 'water_temp_baseline + predicted_delta_water_temp',
    'context_note': 'static training-station summaries only; no end-of-record tail reuse',
}

print( f'phase6 context rows: {len( phase6_context_table )}' )
print( 'phase6 inference contract:' )
for key, value in phase6_inference_contract.items( ):
    print( f'  - {key}: {value}' )

if len( phase6_context_table ) > 0:
    display( phase6_context_table.sample( min( 10, len( phase6_context_table ) ) ).T )

In [ ]:
# 6.1 to 6.3 - phase-6 delta-water-temp modeling, with a light calibration pass
# the core model is still air -> water delta, but now we also check whether a simple
# intercept/slope correction helps on the holdout set without changing the basic method

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold



class Phase6CalibratedPredictor:


    def __init__( self, base_model, calibrator ):
        self.base_model = base_model
        self.calibrator = calibrator


    def predict( self, X ):
        raw_pred = np.asarray( self.base_model.predict( X ), dtype = 'float64' )

        if isinstance( self.calibrator, IsotonicRegression ):
            return self.calibrator.predict( raw_pred )

        return self.calibrator.predict( raw_pred.reshape( -1, 1 ) )


def append_phase6_score( score_rows, model_name, y_true, y_pred ):
    score_rows.append( { 
        'model': model_name,
        'mae': float( mean_absolute_error( y_true, y_pred ) ),
        'rmse': float( mean_squared_error( y_true, y_pred ) ** 0.5 ),
        'r2': float( r2_score( y_true, y_pred ) ),
        'bias': float( np.mean( y_pred - y_true ) ),
    } )


def build_phase6_group_oof_predictions( estimator, X, y, groups ):
    X = X.copy( )
    y = y.copy( )
    groups = pd.Series( groups ).astype( str )

    unique_groups = groups.dropna( ).unique( )
    if len( unique_groups ) < 2:
        fitted_estimator = clone( estimator )
        fitted_estimator.fit( X, y )
        return pd.Series( fitted_estimator.predict( X ), index = X.index, dtype = 'float64' )

    n_splits = min( 5, len( unique_groups ) )
    splitter = GroupKFold( n_splits = n_splits )
    oof_pred = pd.Series( index = X.index, dtype = 'float64' )

    for fit_idx, valid_idx in splitter.split( X, y, groups ):
        fold_model = clone( estimator )
        fold_model.fit( X.iloc[ fit_idx ], y.iloc[ fit_idx ] )
        oof_pred.iloc[ valid_idx ] = fold_model.predict( X.iloc[ valid_idx ] )

    if oof_pred.isna( ).any( ):
        fallback_model = clone( estimator )
        fallback_model.fit( X, y )
        missing_mask = oof_pred.isna( )
        oof_pred.loc[ missing_mask ] = fallback_model.predict( X.loc[ missing_mask ] )

    return oof_pred


air_train_source = daily_air_train_final.copy( )
water_train_source = daily_water_train_final.copy( )
air_holdout_source = daily_air_holdout_final.copy( )
water_holdout_source = daily_water_holdout_final.copy( )

base_join_cols = [ 'region', 'station', 'date' ]
water_keep_cols = [ 
    'delta_water_temp',
    'water_temp_baseline',
    'salinity_baseline',
    'oxygen_baseline',
    'ph_baseline',
    'depth_baseline',
]

phase6_train = air_train_source.merge( 
    water_train_source[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = 'inner',
)
phase6_holdout = air_holdout_source.merge( 
    water_holdout_source[ base_join_cols + water_keep_cols ],
    on = base_join_cols,
    how = 'inner',
)

# merge the static context table by station
# holdout stations that were not in training keep missing context and then use train medians
phase6_train = phase6_train.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )
phase6_holdout = phase6_holdout.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )

if 'station_baseline' in globals( ) and 'cluster' in station_baseline.columns:
    phase6_cluster_lookup = station_baseline[ [ 'region', 'station', 'cluster' ] ].copy( )
    phase6_cluster_lookup[ 'cluster_code' ] = pd.to_numeric( phase6_cluster_lookup[ 'cluster' ], errors = 'coerce' )
    phase6_cluster_lookup = phase6_cluster_lookup.drop( columns = [ 'cluster' ] )

    phase6_train = phase6_train.merge( phase6_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )
    phase6_holdout = phase6_holdout.merge( phase6_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

phase6_train[ 'date' ] = pd.to_datetime( phase6_train[ 'date' ], errors = 'coerce' )
phase6_holdout[ 'date' ] = pd.to_datetime( phase6_holdout[ 'date' ], errors = 'coerce' )
phase6_train[ 'doy' ] = phase6_train[ 'date' ].dt.dayofyear
phase6_holdout[ 'doy' ] = phase6_holdout[ 'date' ].dt.dayofyear
phase6_train[ 'month' ] = phase6_train[ 'date' ].dt.month
phase6_holdout[ 'month' ] = phase6_holdout[ 'date' ].dt.month

for phase6_frame in [ phase6_train, phase6_holdout ]:
    if 'air_temp' in phase6_frame.columns and 'water_temp_baseline' in phase6_frame.columns:
        phase6_frame[ 'air_temp_minus_water_temp_baseline' ] = phase6_frame[ 'air_temp' ] - phase6_frame[ 'water_temp_baseline' ]

    if 'air_temp_r7d' in phase6_frame.columns and 'water_temp_baseline' in phase6_frame.columns:
        phase6_frame[ 'air_temp_r7d_minus_water_temp_baseline' ] = phase6_frame[ 'air_temp_r7d' ] - phase6_frame[ 'water_temp_baseline' ]

    if 'air_temp_r28d' in phase6_frame.columns and 'water_temp_baseline' in phase6_frame.columns:
        phase6_frame[ 'air_temp_r28d_minus_water_temp_baseline' ] = phase6_frame[ 'air_temp_r28d' ] - phase6_frame[ 'water_temp_baseline' ]

    if 'air_temp_r1d' in phase6_frame.columns and 'air_temp_r28d' in phase6_frame.columns:
        phase6_frame[ 'air_temp_r1d_minus_air_temp_r28d' ] = phase6_frame[ 'air_temp_r1d' ] - phase6_frame[ 'air_temp_r28d' ]

    if 'air_temp_r7d' in phase6_frame.columns and 'air_temp_r28d' in phase6_frame.columns:
        phase6_frame[ 'air_temp_r7d_minus_air_temp_r28d' ] = phase6_frame[ 'air_temp_r7d' ] - phase6_frame[ 'air_temp_r28d' ]

    if 'wind_speed' in phase6_frame.columns and 'air_temp' in phase6_frame.columns:
        phase6_frame[ 'wind_speed_x_air_temp' ] = phase6_frame[ 'wind_speed' ] * phase6_frame[ 'air_temp' ]

    if 'solar' in phase6_frame.columns and 'air_temp' in phase6_frame.columns:
        phase6_frame[ 'solar_x_air_temp' ] = phase6_frame[ 'solar' ] * phase6_frame[ 'air_temp' ]

target_col = 'delta_water_temp'
phase6_train_model = phase6_train.dropna( subset = [ target_col ] ).copy( )
phase6_holdout_model = phase6_holdout.dropna( subset = [ target_col ] ).copy( )
phase6_train_model[ 'station_group' ] = phase6_train_model[ 'region' ].astype( str ) + ' | ' + phase6_train_model[ 'station' ].astype( str )
phase6_holdout_model[ 'station_group' ] = phase6_holdout_model[ 'region' ].astype( str ) + ' | ' + phase6_holdout_model[ 'station' ].astype( str )

air_features = [ col for col in phase6_air_feature_candidates if col in phase6_train_model.columns ]
context_features = [ col for col in phase6_context_feature_candidates if col in phase6_train_model.columns ]
joint_features = list( dict.fromkeys( air_features + context_features ) )

print( f'train rows: {len( phase6_train_model )}' )
print( f'holdout rows: {len( phase6_holdout_model )}' )
print( f'air features ({len( air_features )}): {air_features}' )
print( f'context features ({len( context_features )}): {context_features}' )

if len( phase6_train_model ) == 0:
    raise ValueError( 'phase 6 has no training rows after cleanup; phase 5 filters are too strict or upstream joins failed' )

if len( air_features ) == 0:
    raise ValueError( 'phase 6 has no air features after cleanup; cannot fit the air -> water model' )

if len( phase6_holdout_model ) == 0:
    print( )
    print( 'phase 6 warning: no holdout rows survived the current filters; score tables will be empty and scenario selection will fall back to a calibrated learned model' )

# naive baseline 1: persistence by station (previous observed day)
phase6_holdout_model = phase6_holdout_model.sort_values( [ 'region', 'station', 'date' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model.groupby( [ 'region', 'station' ] )[ target_col ].shift( 1 )

# naive baseline 2: climatology by day-of-year from the training set
clim_by_doy = ( 
    phase6_train_model
    .groupby( 'doy', as_index = False )[ target_col ]
    .median( )
    .rename( columns = { target_col: 'pred_climatology' } )
)

phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'doy' ].map( clim_by_doy.set_index( 'doy' )[ 'pred_climatology' ] )
global_target_median = float( phase6_train_model[ target_col ].median( ) )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( phase6_holdout_model[ 'pred_climatology' ] )
phase6_holdout_model[ 'pred_persistence' ] = phase6_holdout_model[ 'pred_persistence' ].fillna( global_target_median )
phase6_holdout_model[ 'pred_climatology' ] = phase6_holdout_model[ 'pred_climatology' ].fillna( global_target_median )

y_train = phase6_train_model[ target_col ]
phase6_scores_rows = [ ]
phase6_calibration_rows = [ ]
phase6_model_store = { }
phase6_feature_store = { }
phase6_fill_store = { }
phase6_pred_col_store = { 
    'naive_persistence': 'pred_persistence',
    'naive_climatology_doy': 'pred_climatology',
}

X_train_air = phase6_train_model[ air_features ].copy( )
air_fill_values = X_train_air.median( numeric_only = True )
X_train_air = X_train_air.fillna( air_fill_values )

ridge_air_model = Ridge( alpha = 1.0 )
ridge_air_model.fit( X_train_air, y_train )
phase6_model_store[ 'ridge_air_only' ] = ridge_air_model
phase6_feature_store[ 'ridge_air_only' ] = air_features.copy( )
phase6_fill_store[ 'ridge_air_only' ] = air_fill_values.copy( )
phase6_pred_col_store[ 'ridge_air_only' ] = 'pred_ridge_air_only'

if len( joint_features ) > 0:
    X_train_joint = phase6_train_model[ joint_features ].copy( )
    joint_fill_values = X_train_joint.median( numeric_only = True )
    X_train_joint = X_train_joint.fillna( joint_fill_values )

else:
    joint_fill_values = pd.Series( dtype = 'float64' )
    X_train_joint = pd.DataFrame( index = phase6_train_model.index )

if len( joint_features ) > 0:
    ridge_air_context_model = Ridge( alpha = 1.0 )
    ridge_air_context_model.fit( X_train_joint, y_train )
    phase6_model_store[ 'ridge_air_context' ] = ridge_air_context_model
    phase6_feature_store[ 'ridge_air_context' ] = joint_features.copy( )
    phase6_fill_store[ 'ridge_air_context' ] = joint_fill_values.copy( )
    phase6_pred_col_store[ 'ridge_air_context' ] = 'pred_ridge_air_context'

    hgb_air_context_model = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 6,
        max_iter = 250,
        random_state = 42,
    )
    hgb_air_context_model.fit( X_train_joint, y_train )
    phase6_model_store[ 'hgb_air_context' ] = hgb_air_context_model
    phase6_feature_store[ 'hgb_air_context' ] = joint_features.copy( )
    phase6_fill_store[ 'hgb_air_context' ] = joint_fill_values.copy( )
    phase6_pred_col_store[ 'hgb_air_context' ] = 'pred_hgb_air_context'

if len( phase6_holdout_model ) > 0:
    y_holdout = phase6_holdout_model[ target_col ]
    append_phase6_score( phase6_scores_rows, 'naive_persistence', y_holdout, phase6_holdout_model[ 'pred_persistence' ] )
    append_phase6_score( phase6_scores_rows, 'naive_climatology_doy', y_holdout, phase6_holdout_model[ 'pred_climatology' ] )

for model_name in [ 'ridge_air_only', 'ridge_air_context', 'hgb_air_context' ]:
    estimator = phase6_model_store.get( model_name )
    feature_cols = phase6_feature_store.get( model_name, [ ] )
    fill_values = phase6_fill_store.get( model_name, pd.Series( dtype = 'float64' ) )
    pred_col = phase6_pred_col_store.get( model_name )

    if estimator is None or len( feature_cols ) == 0 or pred_col is None:
        continue

    X_train_model = phase6_train_model[ feature_cols ].copy( ).fillna( fill_values )

    if len( phase6_holdout_model ) > 0:
        X_holdout_model = phase6_holdout_model[ feature_cols ].copy( ).fillna( fill_values )
        phase6_holdout_model[ pred_col ] = estimator.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, model_name, y_holdout, phase6_holdout_model[ pred_col ] )

    # calibration is fit on out-of-fold predictions from the training stations only
    # this keeps the correction modest and avoids fitting the intercept/slope on the holdout rows
    oof_pred = build_phase6_group_oof_predictions( 
        estimator,
        X_train_model,
        y_train,
        phase6_train_model[ 'station_group' ],
    )

    calibrator = LinearRegression( )
    calibrator.fit( np.asarray( oof_pred ).reshape( -1, 1 ), y_train )
    calibrated_model = Phase6CalibratedPredictor( estimator, calibrator )
    calibrated_name = f'{model_name}_calibrated'
    calibrated_pred_col = f'{pred_col}_calibrated'

    phase6_model_store[ calibrated_name ] = calibrated_model
    phase6_feature_store[ calibrated_name ] = feature_cols.copy( )
    phase6_fill_store[ calibrated_name ] = fill_values.copy( )
    phase6_pred_col_store[ calibrated_name ] = calibrated_pred_col

    oof_calibrated = calibrator.predict( np.asarray( oof_pred ).reshape( -1, 1 ) )
    phase6_calibration_rows.append( { 
        'model': model_name,
        'calibration_type': 'linear',
        'slope': float( calibrator.coef_[ 0 ] ),
        'intercept': float( calibrator.intercept_ ),
        'oof_bias_raw': float( np.mean( oof_pred - y_train ) ),
        'oof_bias_calibrated': float( np.mean( oof_calibrated - y_train ) ),
        'oof_mae_raw': float( mean_absolute_error( y_train, oof_pred ) ),
        'oof_mae_calibrated': float( mean_absolute_error( y_train, oof_calibrated ) ),
    } )

    if len( phase6_holdout_model ) > 0:
        phase6_holdout_model[ calibrated_pred_col ] = calibrated_model.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, calibrated_name, y_holdout, phase6_holdout_model[ calibrated_pred_col ] )

    isotonic_calibrator = IsotonicRegression( out_of_bounds = 'clip' )
    isotonic_calibrator.fit( np.asarray( oof_pred, dtype = 'float64' ), np.asarray( y_train, dtype = 'float64' ) )
    isotonic_model = Phase6CalibratedPredictor( estimator, isotonic_calibrator )
    isotonic_name = f'{model_name}_isotonic'
    isotonic_pred_col = f'{pred_col}_isotonic'

    phase6_model_store[ isotonic_name ] = isotonic_model
    phase6_feature_store[ isotonic_name ] = feature_cols.copy( )
    phase6_fill_store[ isotonic_name ] = fill_values.copy( )
    phase6_pred_col_store[ isotonic_name ] = isotonic_pred_col

    oof_isotonic = isotonic_calibrator.predict( np.asarray( oof_pred, dtype = 'float64' ) )
    phase6_calibration_rows.append( { 
        'model': model_name,
        'calibration_type': 'isotonic',
        'slope': np.nan,
        'intercept': np.nan,
        'oof_bias_raw': float( np.mean( oof_pred - y_train ) ),
        'oof_bias_calibrated': float( np.mean( oof_isotonic - y_train ) ),
        'oof_mae_raw': float( mean_absolute_error( y_train, oof_pred ) ),
        'oof_mae_calibrated': float( mean_absolute_error( y_train, oof_isotonic ) ),
    } )

    if len( phase6_holdout_model ) > 0:
        phase6_holdout_model[ isotonic_pred_col ] = isotonic_model.predict( X_holdout_model )
        append_phase6_score( phase6_scores_rows, isotonic_name, y_holdout, phase6_holdout_model[ isotonic_pred_col ] )

phase6_scores = pd.DataFrame( phase6_scores_rows )
phase6_scores_operational = phase6_scores.copy( )
phase6_scores_scenario = phase6_scores.loc[ 
    phase6_scores[ 'model' ].isin( [ 
        'ridge_air_only',
        'ridge_air_only_calibrated',
        'ridge_air_only_isotonic',
        'ridge_air_context',
        'ridge_air_context_calibrated',
        'ridge_air_context_isotonic',
        'hgb_air_context',
        'hgb_air_context_calibrated',
        'hgb_air_context_isotonic',
    ] )
].copy( )

if len( phase6_scores_operational ) > 0:
    phase6_scores_operational = phase6_scores_operational.sort_values( 'rmse', ascending = True ).reset_index( drop = True )

if len( phase6_scores_scenario ) > 0:
    phase6_scores_scenario = phase6_scores_scenario.sort_values( 'rmse', ascending = True ).reset_index( drop = True )

phase6_calibration_summary = pd.DataFrame( phase6_calibration_rows )
if len( phase6_calibration_summary ) > 0:
    phase6_calibration_summary = phase6_calibration_summary.round( 4 )

print( )
print( 'phase 6 leaderboard ( operational, includes persistence ):' )
display( phase6_scores_operational.round( 4 ) )
print( 'phase 6 leaderboard (scenario, learned models only):' )
display( phase6_scores_scenario.round( 4 ) )

if len( phase6_calibration_summary ) > 0:
    print( 'phase 6 calibration summary ( out-of-fold linear and isotonic corrections ):' )
    display( phase6_calibration_summary )

if len( phase6_scores_scenario ) > 0:
    phase6_selected_name = str( phase6_scores_scenario.iloc[ 0 ][ 'model' ] )

elif 'hgb_air_context_isotonic' in phase6_model_store:
    phase6_selected_name = 'hgb_air_context_isotonic'

elif 'hgb_air_context_calibrated' in phase6_model_store:
    phase6_selected_name = 'hgb_air_context_calibrated'

elif 'ridge_air_context_isotonic' in phase6_model_store:
    phase6_selected_name = 'ridge_air_context_isotonic'

elif 'ridge_air_context_calibrated' in phase6_model_store:
    phase6_selected_name = 'ridge_air_context_calibrated'

else:
    if 'ridge_air_only_isotonic' in phase6_model_store:
        phase6_selected_name = 'ridge_air_only_isotonic'

    else:
        phase6_selected_name = 'ridge_air_only_calibrated' if 'ridge_air_only_calibrated' in phase6_model_store else 'ridge_air_only'

phase6_selected_model = phase6_model_store[ phase6_selected_name ]
phase6_selected_features = phase6_feature_store[ phase6_selected_name ].copy( )
phase6_selected_fill_values = phase6_fill_store[ phase6_selected_name ].copy( )
phase6_selected_pred_col = phase6_pred_col_store.get( phase6_selected_name )

print( f'selected phase6 model for scenario demos: {phase6_selected_name}' )
print( f'selected feature count: {len( phase6_selected_features )}' )
if phase6_selected_name.endswith( '_calibrated' ):
    print( 'selected model uses an out-of-fold linear correction on top of the base prediction' )

elif phase6_selected_name.endswith( '_isotonic' ):
    print( 'selected model uses an out-of-fold isotonic correction to stretch or compress prediction bands nonlinearly' )

# tiny unseen-estuary style demo: warm one row by +2 C in air-temp features
# this is just a quick sanity check that the selected model moves in the expected direction
phase6_demo_ready = phase6_holdout_model.copy( )
if len( phase6_selected_features ) > 0 and len( phase6_demo_ready ) > 0:
    phase6_demo_ready[ phase6_selected_features ] = phase6_demo_ready[ phase6_selected_features ].fillna( phase6_selected_fill_values )

demo_cols = [ 'region', 'station', 'date', target_col ] + phase6_selected_features
phase6_demo_row = phase6_demo_ready[ demo_cols ].head( 1 ).copy( )

if len( phase6_demo_row ) == 0:
    print( )
    print( 'unknown-estuary style +2 C quick demo skipped: no holdout rows survived the current filters' )

else:
    phase6_demo_warm = phase6_demo_row.copy( )
    warm_cols = [ col for col in phase6_selected_features if col.startswith( 'air_temp' ) ]
    for col in warm_cols:
        phase6_demo_warm[ col ] = phase6_demo_warm[ col ] + 2.0

    baseline_pred = float( phase6_selected_model.predict( phase6_demo_row[ phase6_selected_features ] )[ 0 ] )
    warm_pred = float( phase6_selected_model.predict( phase6_demo_warm[ phase6_selected_features ] )[ 0 ] )

    print( )
    print( 'unknown-estuary style +2 C quick demo:' )
    print( phase6_demo_row[ [ 'region', 'station', 'date', target_col ] ] )
    print( f'baseline predicted delta_water_temp: {baseline_pred:.3f}' )
    print( f'+2 C predicted delta_water_temp: {warm_pred:.3f}' )
    print( f'implied delta shift from warming: {warm_pred - baseline_pred:.3f}' )


In [ ]:
# tiny unseen-estuary style demo: warm one row by +2 C in air-temp features
# keep this cell safe when the stricter cleanup leaves no holdout rows for the demo

demo_cols = [ 'region', 'station', 'date', target_col ] + phase6_selected_features
phase6_demo_ready = phase6_holdout_model.copy( )

if len( phase6_selected_features ) > 0 and len( phase6_demo_ready ) > 0:
    phase6_demo_ready[ phase6_selected_features ] = phase6_demo_ready[ phase6_selected_features ].fillna( phase6_selected_fill_values )

phase6_demo_row = phase6_demo_ready[ demo_cols ].head( 1 ).copy( )

if len( phase6_demo_row ) == 0:
    print( )
    print( 'unknown-estuary style +2 C quick demo skipped: no holdout rows survived the current filters' )

else:
    phase6_demo_warm = phase6_demo_row.copy( )
    warm_cols = [ col for col in phase6_selected_features if col.startswith( 'air_temp' ) ]
    for col in warm_cols:
        phase6_demo_warm[ col ] = phase6_demo_warm[ col ] + 2.0

    baseline_pred = float( phase6_selected_model.predict( phase6_demo_row[ phase6_selected_features ] )[ 0 ] )
    warm_pred = float( phase6_selected_model.predict( phase6_demo_warm[ phase6_selected_features ] )[ 0 ] )

    print( )
    print( 'unknown-estuary style +2 C quick demo:' )
    print( phase6_demo_row[ [ 'region', 'station', 'date', target_col ] ] )
    print( f'baseline predicted delta_water_temp: {baseline_pred:.3f}' )
    print( f'+2 C predicted delta_water_temp: {warm_pred:.3f}' )
    print( f'implied delta shift from warming: {warm_pred - baseline_pred:.3f}' )


In [ ]:
# 6.4 diagnostic: typical air-water spread baseline vs selected model spread
# spread is defined here as ( water_temp - air_temp )
# water_temp is reconstructed from baseline + predicted/observed delta

selected_pred_col_map = { 
    'ridge_air_only': 'pred_ridge_air_only',
    'ridge_air_only_calibrated': 'pred_ridge_air_only_calibrated',
    'ridge_air_context': 'pred_ridge_air_context',
    'ridge_air_context_calibrated': 'pred_ridge_air_context_calibrated',
    'hgb_air_context': 'pred_hgb_air_context',
    'hgb_air_context_calibrated': 'pred_hgb_air_context_calibrated',
}
selected_pred_col = phase6_selected_pred_col if 'phase6_selected_pred_col' in globals( ) else selected_pred_col_map.get( phase6_selected_name )

# for this diagnostic, we want to see how the selected model's predicted spread
# compares to the observed spread, and how both compare to a simple typical spread baseline
spread_train = phase6_train_model[ [ 
    'region',
    'station',
    'doy',
    'air_temp',
    'water_temp_baseline',
    target_col,
] ].copy( )
spread_train = spread_train.dropna( subset = [ 'air_temp', 'water_temp_baseline', target_col ] )
spread_train[ 'water_temp_obs' ] = spread_train[ 'water_temp_baseline' ] + spread_train[ target_col ]
spread_train[ 'spread_obs' ] = spread_train[ 'water_temp_obs' ] - spread_train[ 'air_temp' ]

typical_spread_station_doy = ( 
    spread_train
    .groupby( [ 'region', 'station', 'doy' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_station_doy' } )
)

typical_spread_station = ( 
    spread_train
    .groupby( [ 'region', 'station' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_station' } )
)

typical_spread_region_doy = ( 
    spread_train
    .groupby( [ 'region', 'doy' ], as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_region_doy' } )
)

typical_spread_doy = ( 
    spread_train
    .groupby( 'doy', as_index = False )[ 'spread_obs' ]
    .median( )
    .rename( columns = { 'spread_obs': 'typical_spread_doy' } )
)

typical_spread_global = float( spread_train[ 'spread_obs' ].median( ) )

spread_holdout_pre = phase6_holdout_model[ [ 
    'region',
    'station',
    'date',
    'doy',
    'air_temp',
    'water_temp_baseline',
    target_col,
    selected_pred_col,
] ].copy( )

holdout_station_keys_pre = spread_holdout_pre[ [ 'region', 'station' ] ].drop_duplicates( )
spread_holdout = spread_holdout_pre.dropna( subset = [ 'air_temp', 'water_temp_baseline', target_col, selected_pred_col ] )
holdout_station_keys_post = spread_holdout[ [ 'region', 'station' ] ].drop_duplicates( )

dropped_spread_station_keys = ( 
    holdout_station_keys_pre
    .merge( holdout_station_keys_post.assign( _kept = True ), on = [ 'region', 'station' ], how = 'left' )
    .loc[ lambda frame: frame[ '_kept' ].isna( ), [ 'region', 'station' ] ]
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

spread_holdout[ 'water_temp_obs' ] = spread_holdout[ 'water_temp_baseline' ] + spread_holdout[ target_col ]
spread_holdout[ 'water_temp_pred' ] = spread_holdout[ 'water_temp_baseline' ] + spread_holdout[ selected_pred_col ]
spread_holdout[ 'spread_obs' ] = spread_holdout[ 'water_temp_obs' ] - spread_holdout[ 'air_temp' ]
spread_holdout[ 'spread_pred_model' ] = spread_holdout[ 'water_temp_pred' ] - spread_holdout[ 'air_temp' ]

spread_holdout = spread_holdout.merge( typical_spread_station_doy, on = [ 'region', 'station', 'doy' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_station, on = [ 'region', 'station' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_region_doy, on = [ 'region', 'doy' ], how = 'left' )
spread_holdout = spread_holdout.merge( typical_spread_doy, on = [ 'doy' ], how = 'left' )

spread_holdout[ 'spread_pred_typical' ] = spread_holdout[ 'typical_spread_station_doy' ]
spread_holdout[ 'typical_source' ] = pd.Series( index = spread_holdout.index, dtype = 'object' )
spread_holdout.loc[ spread_holdout[ 'spread_pred_typical' ].notna( ), 'typical_source' ] = 'station_doy'

station_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_station' ].notna( )
spread_holdout.loc[ station_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ station_fill_mask, 'typical_spread_station' ]
spread_holdout.loc[ station_fill_mask, 'typical_source' ] = 'station'

region_doy_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_region_doy' ].notna( )
spread_holdout.loc[ region_doy_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ region_doy_fill_mask, 'typical_spread_region_doy' ]
spread_holdout.loc[ region_doy_fill_mask, 'typical_source' ] = 'region_doy'

doy_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( ) & spread_holdout[ 'typical_spread_doy' ].notna( )
spread_holdout.loc[ doy_fill_mask, 'spread_pred_typical' ] = spread_holdout.loc[ doy_fill_mask, 'typical_spread_doy' ]
spread_holdout.loc[ doy_fill_mask, 'typical_source' ] = 'doy'

global_fill_mask = spread_holdout[ 'spread_pred_typical' ].isna( )
spread_holdout.loc[ global_fill_mask, 'spread_pred_typical' ] = typical_spread_global
spread_holdout.loc[ global_fill_mask, 'typical_source' ] = 'global'

spread_holdout[ 'model_spread_error' ] = spread_holdout[ 'spread_pred_model' ] - spread_holdout[ 'spread_obs' ]
spread_holdout[ 'typical_spread_error' ] = spread_holdout[ 'spread_pred_typical' ] - spread_holdout[ 'spread_obs' ]
spread_holdout[ 'month' ] = spread_holdout[ 'date' ].dt.month

spread_score_rows = [ ]
for model_name, pred_col in [ 
    ( 'selected_model_spread', 'spread_pred_model' ),
    ( 'typical_spread_baseline', 'spread_pred_typical' ),
]:
    pred_values = spread_holdout[ pred_col ]
    truth_values = spread_holdout[ 'spread_obs' ]
    spread_score_rows.append( { 
        'model': model_name,
        'mae': float( mean_absolute_error( truth_values, pred_values ) ),
        'rmse': float( mean_squared_error( truth_values, pred_values ) ** 0.5 ),
        'bias': float( np.mean( pred_values - truth_values ) ),
        'n_rows': int( len( spread_holdout ) ),
    } )

spread_scores = pd.DataFrame( spread_score_rows ).round( 4 )
print( 'spread diagnostic score table:' )
display( spread_scores )

print( f'holdout stations before spread-scope dropna: { len( holdout_station_keys_pre ) }' )
print( f'holdout stations after spread-scope dropna: { len( holdout_station_keys_post ) }' )
if len( dropped_spread_station_keys ) > 0:
    print( 'holdout stations dropped from spread diagnostic scope:' )
    display( dropped_spread_station_keys )


In [ ]:
# finally, let's visualize the spread distribution and an example timeline for a single station
fig, axes = plt.subplots( 1, 2, figsize = ( 14, 5 ) )

spread_bin_edges = np.linspace( -15, 20, 61 )
sns.histplot( 
    spread_holdout[ 'spread_obs' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 2.2,
    color = 'black',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Observed spread',
)
sns.histplot( 
    spread_holdout[ 'spread_pred_typical' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 1.8,
    color = 'tab:blue',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Typical spread baseline',
)
sns.histplot( 
    spread_holdout[ 'spread_pred_model' ],
    bins = spread_bin_edges,
    stat = 'count',
    element = 'step',
    fill = False,
    linewidth = 1.8,
    color = 'tab:orange',
    alpha = 1.0,
    ax = axes[ 0 ],
    label = 'Selected model spread',
)
axes[ 0 ].set_title( 'Holdout Distribution: Air-Water Spread ( C )' )
axes[ 0 ].set_xlabel( 'water_temp - air_temp ( C )' )
axes[ 0 ].set_ylabel( 'Count' )
axes[ 0 ].set_xlim( -15, 20 )
axes[ 0 ].set_ylim( 0, 30000 )
axes[ 0 ].legend( loc = 'best' )

top_station = ( 
    spread_holdout
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( n_rows = ( 'date', 'size' ) )
    .sort_values( 'n_rows', ascending = False )
    .head( 1 )
)

station_region = top_station.iloc[ 0 ][ 'region' ]
station_name = top_station.iloc[ 0 ][ 'station' ]
station_slice = spread_holdout.loc[ 
    ( spread_holdout[ 'region' ] == station_region )
    & ( spread_holdout[ 'station' ] == station_name )
].sort_values( 'date' )

axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_obs' ], 
    linewidth = 1.0, 
    color = 'black', 
    label = 'Observed spread'
)
axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_pred_typical' ], 
    linewidth = 1.0, 
    color = 'tab:blue', 
    alpha = 0.9, 
    label = 'Typical spread baseline'
)
axes[ 1 ].plot( 
    station_slice[ 'date' ], station_slice[ 'spread_pred_model' ], 
    linewidth = 1.0, 
    color = 'tab:orange', 
    alpha = 0.9, 
    label = f'Selected model ({phase6_selected_name})'
)
axes[ 1 ].set_title( f'Spread Timeline Example: {station_row_label( station_region, station_name )}' )
axes[ 1 ].set_xlabel( 'Date' )
axes[ 1 ].set_ylabel( 'water_temp - air_temp ( C )' )
axes[ 1 ].legend( loc = 'best' )

plt.tight_layout( )
plt.show( )

### 6.4b - station-level spread residual offsets
- build per-station prediction-offset dataset ( model spread minus observed spread )


In [ ]:
# 6.4b - station-level offsets from selected model spread vs observed spread
# residual sign: positive means model spread is too warm vs observed spread

spread_holdout_resid = spread_holdout.copy( )
spread_holdout_resid[ 'spread_residual_model_minus_obs' ] = ( 
    spread_holdout_resid[ 'spread_pred_model' ] - spread_holdout_resid[ 'spread_obs' ]
)
spread_holdout_resid[ 'spread_abs_residual_model_minus_obs' ] = spread_holdout_resid[ 'spread_residual_model_minus_obs' ].abs( )

spread_station_offset = ( 
    spread_holdout_resid
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        mean_offset_c = ( 'spread_residual_model_minus_obs', 'mean' ),
        median_offset_c = ( 'spread_residual_model_minus_obs', 'median' ),
        mae_offset_c = ( 'spread_abs_residual_model_minus_obs', 'mean' ),
        rmse_offset_c = ( 'spread_residual_model_minus_obs', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
    )
    .sort_values( 'mean_offset_c' )
    .reset_index( drop = True )
)

spread_station_offset[ 'station_key' ] = spread_station_offset[ 'region' ].astype( str ) + ' | ' + spread_station_offset[ 'station' ].astype( str )

for col in [ 
    'mean_offset_c',
    'median_offset_c',
    'mae_offset_c',
    'rmse_offset_c',
]:
    spread_station_offset[ col ] = spread_station_offset[ col ].round( 4 )

print( f'station rows in offset table: { len( spread_station_offset ) }' )
display( spread_station_offset.sort_values( 'mean_offset_c' ).reset_index( drop = True ) )

plot_df = spread_station_offset.sort_values( 'mean_offset_c' ).reset_index( drop = True )
plot_df[ 'station_key' ] = pd.Categorical( plot_df[ 'station_key' ], categories = plot_df[ 'station_key' ], ordered = True )

region_order = sorted( plot_df[ 'region' ].dropna( ).unique( ).tolist( ) )
region_palette = dict( zip( region_order, sns.color_palette( 'tab20', n_colors = max( len( region_order ), 1 ) ) ) )

fig_height = max( 8, 0.25 * len( plot_df ) + 2 )
fig, axes = plt.subplots( 1, 2, figsize = ( 18, fig_height ), sharey = True )

for region in region_order:
    sub = plot_df.loc[ plot_df[ 'region' ] == region ]
    axes[ 0 ].scatter( 
        sub[ 'mean_offset_c' ],
        sub[ 'station_key' ],
        s = 200,
        alpha = 0.85,
        color = region_palette[ region ],
        label = region,
    )

axes[ 0 ].axvline( 0, color = 'black', linewidth = 1.1, linestyle = '--' )
axes[ 0 ].set_title( 'Station Mean Offset ( Model Spread - Observed Spread )' )
axes[ 0 ].set_xlabel( 'Mean Offset ( C )' )
axes[ 0 ].set_ylabel( 'Station ( Region | Station )' )
axes[ 0 ].legend( title = 'Region', bbox_to_anchor = ( 1.01, 1 ), loc = 'upper left' )

axes[ 1 ].barh( 
    plot_df[ 'station_key' ],
    plot_df[ 'mae_offset_c' ],
    color = [ region_palette[ r ] for r in plot_df[ 'region' ] ],
    alpha = 0.85,
)
axes[ 1 ].set_title( 'Station MAE of Spread Residual' )
axes[ 1 ].set_xlabel( 'MAE ( C )' )

plt.tight_layout( )
plt.show( )


### 6.4c - station bias classes and model-vs-typical MAE delta
- classify station spread bias and compare selected model MAE vs typical spread MAE


In [ ]:
# 6.4c - compare selected model spread error vs typical spread error by station
# negative delta_mae_model_minus_typical_c means selected model beat the typical-spread baseline
# also add direct phase-6 residual diagnostics, since the main concern is thermal under-estimation

spread_holdout_resid[ 'typical_abs_error_c' ] = spread_holdout_resid[ 'typical_spread_error' ].abs( )

spread_station_compare = ( 
    spread_holdout_resid
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( 'date', 'size' ),
        mean_model_offset_c = ( 'spread_residual_model_minus_obs', 'mean' ),
        median_model_offset_c = ( 'spread_residual_model_minus_obs', 'median' ),
        mae_model_c = ( 'spread_abs_residual_model_minus_obs', 'mean' ),
        mae_typical_c = ( 'typical_abs_error_c', 'mean' ),
    )
)

spread_station_compare[ 'delta_mae_model_minus_typical_c' ] = ( 
    spread_station_compare[ 'mae_model_c' ] - spread_station_compare[ 'mae_typical_c' ]
)
spread_station_compare[ 'station_key' ] = spread_station_compare[ 'region' ].astype( str ) + ' | ' + spread_station_compare[ 'station' ].astype( str )

bias_cutoff_c = 0.5
spread_station_compare[ 'bias_class' ] = 'near_zero'
spread_station_compare.loc[ spread_station_compare[ 'mean_model_offset_c' ] <= -bias_cutoff_c, 'bias_class' ] = 'under'
spread_station_compare.loc[ spread_station_compare[ 'mean_model_offset_c' ] >= bias_cutoff_c, 'bias_class' ] = 'over'
spread_station_compare[ 'model_beats_typical' ] = spread_station_compare[ 'delta_mae_model_minus_typical_c' ] < 0

for col in [ 
    'mean_model_offset_c',
    'median_model_offset_c',
    'mae_model_c',
    'mae_typical_c',
    'delta_mae_model_minus_typical_c',
]:
    spread_station_compare[ col ] = spread_station_compare[ col ].round( 4 )

spread_station_compare = spread_station_compare.sort_values( 'mean_model_offset_c' ).reset_index( drop = True )

print( '' )
print( 'stations where selected model beat typical spread baseline ( lower MAE ):' )
print( int( spread_station_compare[ 'model_beats_typical' ].sum( ) ), '/', len( spread_station_compare ) )

print( f'station rows in compare table: { len( spread_station_compare ) }' )
display( spread_station_compare.sort_values( 'mean_model_offset_c' ).reset_index( drop = True ) )

region_bias_summary = ( 
    spread_station_compare
    .groupby( 'region', as_index = False )
    .agg( 
        n_stations = ( 'station', 'nunique' ),
        mean_station_offset_c = ( 'mean_model_offset_c', 'mean' ),
        mean_station_delta_mae_c = ( 'delta_mae_model_minus_typical_c', 'mean' ),
        frac_model_beats_typical = ( 'model_beats_typical', 'mean' ),
    )
    .sort_values( 'mean_station_offset_c' )
    .reset_index( drop = True )
)

for col in [ 
    'mean_station_offset_c',
    'mean_station_delta_mae_c',
    'frac_model_beats_typical',
]:
    region_bias_summary[ col ] = region_bias_summary[ col ].round( 4 )

display( region_bias_summary )

fig, ax = plt.subplots( figsize = ( 10, 5 ) )
sns.histplot( 
    spread_station_compare[ 'mean_model_offset_c' ],
    bins = 16,
    color = 'tab:purple',
    alpha = 0.75,
    edgecolor = 'white',
    ax = ax,
)
ax.axvline( 0, color = 'black', linewidth = 1.2, linestyle = '--', label = 'Zero bias' )
ax.axvline( -bias_cutoff_c, color = 'tab:gray', linewidth = 1.0, linestyle = ':' )
ax.axvline( bias_cutoff_c, color = 'tab:gray', linewidth = 1.0, linestyle = ':', label = 'Bias class cutoff' )
ax.set_title( 'Histogram: Station Mean Spread Bias ( Model - Observed )' )
ax.set_xlabel( 'Mean Spread Bias ( C )' )
ax.set_ylabel( 'Station Count' )
ax.legend( loc = 'best' )
plt.tight_layout( )
plt.show( )

# direct delta residual diagnostics
# these are the ones to check when the model seems to under-estimate warming
phase6_direct_eval = phase6_holdout_model[ [ 
    'region',
    'station',
    'date',
    'month',
    'water_temp_baseline',
    target_col,
    selected_pred_col,
] ].dropna( subset = [ 'date', 'water_temp_baseline', target_col, selected_pred_col ] ).copy( )
phase6_direct_eval[ 'water_temp_obs' ] = phase6_direct_eval[ 'water_temp_baseline' ] + phase6_direct_eval[ target_col ]
phase6_direct_eval[ 'water_temp_pred' ] = phase6_direct_eval[ 'water_temp_baseline' ] + phase6_direct_eval[ selected_pred_col ]
phase6_direct_eval[ 'delta_residual_pred_minus_obs' ] = phase6_direct_eval[ selected_pred_col ] - phase6_direct_eval[ target_col ]
phase6_direct_eval[ 'water_temp_residual_pred_minus_obs' ] = phase6_direct_eval[ 'water_temp_pred' ] - phase6_direct_eval[ 'water_temp_obs' ]
phase6_direct_eval[ 'abs_delta_residual' ] = phase6_direct_eval[ 'delta_residual_pred_minus_obs' ].abs( )

phase6_month_bias = ( 
    phase6_direct_eval
    .groupby( 'month', as_index = False )
    .agg( 
        n_rows = ( target_col, 'size' ),
        mean_obs_delta_c = ( target_col, 'mean' ),
        mean_pred_delta_c = ( selected_pred_col, 'mean' ),
        mean_residual_c = ( 'delta_residual_pred_minus_obs', 'mean' ),
        mae_c = ( 'abs_delta_residual', 'mean' ),
        rmse_c = ( 'delta_residual_pred_minus_obs', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
    )
    .sort_values( 'month' )
    .reset_index( drop = True )
)

phase6_region_delta_bias = ( 
    phase6_direct_eval
    .groupby( 'region', as_index = False )
    .agg( 
        n_rows = ( target_col, 'size' ),
        mean_residual_c = ( 'delta_residual_pred_minus_obs', 'mean' ),
        mae_c = ( 'abs_delta_residual', 'mean' ),
        rmse_c = ( 'delta_residual_pred_minus_obs', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
    )
    .sort_values( 'mean_residual_c' )
    .reset_index( drop = True )
)

phase6_station_delta_bias = ( 
    phase6_direct_eval
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_rows = ( target_col, 'size' ),
        mean_residual_c = ( 'delta_residual_pred_minus_obs', 'mean' ),
        mae_c = ( 'abs_delta_residual', 'mean' ),
        rmse_c = ( 'delta_residual_pred_minus_obs', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
        mean_obs_water_temp_c = ( 'water_temp_obs', 'mean' ),
    )
    .sort_values( 'mean_residual_c' )
    .reset_index( drop = True )
)
phase6_station_delta_bias[ 'station_key' ] = phase6_station_delta_bias[ 'region' ].astype( str ) + ' | ' + phase6_station_delta_bias[ 'station' ].astype( str )

for frame in [ phase6_month_bias, phase6_region_delta_bias, phase6_station_delta_bias ]:
    for col in [ 'mean_obs_delta_c', 'mean_pred_delta_c', 'mean_residual_c', 'mae_c', 'rmse_c', 'mean_obs_water_temp_c' ]:
        if col in frame.columns:
            frame[ col ] = frame[ col ].round( 4 )

print( 'phase 6 direct residual summary by month:' )
display( phase6_month_bias )
print( 'phase 6 direct residual summary by region:' )
display( phase6_region_delta_bias )
print( 'phase 6 stations with strongest under-estimation:' )
display( phase6_station_delta_bias.head( 15 ) )
print( 'phase 6 stations with strongest over-estimation:' )
display( phase6_station_delta_bias.tail( 15 ).sort_values( 'mean_residual_c', ascending = False ) )

fig, axes = plt.subplots( 1, 3, figsize = ( 19, 5 ) )

sns.barplot( 
    data = phase6_month_bias,
    x = 'month',
    y = 'mean_residual_c',
    color = 'tab:blue',
    ax = axes[ 0 ],
)
axes[ 0 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
axes[ 0 ].set_title( 'Phase 6 Mean Residual by Month' )
axes[ 0 ].set_xlabel( 'Month' )
axes[ 0 ].set_ylabel( 'Predicted - Observed ΔT ( C )' )

sns.scatterplot( 
    data = phase6_direct_eval,
    x = 'water_temp_obs',
    y = 'water_temp_residual_pred_minus_obs',
    s = 18,
    alpha = 0.25,
    color = 'tab:orange',
    ax = axes[ 1 ],
)
axes[ 1 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
axes[ 1 ].set_title( 'Phase 6 Residual vs Observed Water Temp' )
axes[ 1 ].set_xlabel( 'Observed Water Temp ( C )' )
axes[ 1 ].set_ylabel( 'Predicted - Observed Water Temp ( C )' )

sns.barplot( 
    data = phase6_region_delta_bias,
    x = 'region',
    y = 'mean_residual_c',
    color = 'tab:green',
    ax = axes[ 2 ],
)
axes[ 2 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
axes[ 2 ].set_title( 'Phase 6 Mean Residual by Region' )
axes[ 2 ].set_xlabel( 'Region' )
axes[ 2 ].set_ylabel( 'Predicted - Observed ΔT ( C )' )
axes[ 2 ].tick_params( axis = 'x', rotation = 30 )

plt.tight_layout( )
plt.show( )


In [ ]:
# confusion-matrix style view for regression holdout
# keep compact B-label ticks, but also print and plot a bin legend so the ranges are obvious

y_true = phase6_holdout_model[ target_col ]
y_pred = phase6_holdout_model[ selected_pred_col ]

phase6_bin_edges = [ -np.inf, -2.0, -1.0, -0.5, 0.5, 1.0, 2.0, np.inf ]
phase6_bin_ranges = [ 
    'ΔT <= -2.0 C',
    '-2.0 to -1.0 C',
    '-1.0 to -0.5 C',
    '-0.5 to 0.5 C',
    '0.5 to 1.0 C',
    '1.0 to 2.0 C',
    'ΔT > 2.0 C',
]
phase6_closed_bin_edges = [ -3.0, -2.0, -1.0, -0.5, 0.5, 1.0, 2.0, 3.0 ]
phase6_closed_bin_ranges = [ 
    '-3.0 to -2.0 C',
    '-2.0 to -1.0 C',
    '-1.0 to -0.5 C',
    '-0.5 to 0.5 C',
    '0.5 to 1.0 C',
    '1.0 to 2.0 C',
    '2.0 to 3.0 C',
]


def build_phase6_binned_cm( y_true_values, y_pred_values, bin_edges, bin_ranges ):
    bin_labels = [ f'B{ i + 1 }' for i in range( len( bin_ranges ) ) ]
    legend_lines = [ 
        f'{ label }: { value_range }'
        for label, value_range in zip( bin_labels, bin_ranges )
    ]
    legend_rows = [ '   |   '.join( legend_lines[ i:i + 3 ] ) for i in range( 0, len( legend_lines ), 3 ) ]
    legend_text = chr( 10 ).join( legend_rows )

    obs_local = pd.cut( y_true_values, bins = bin_edges, labels = bin_labels, include_lowest = True )
    pred_local = pd.cut( y_pred_values, bins = bin_edges, labels = bin_labels, include_lowest = True )
    keep_mask = obs_local.notna( ) & pred_local.notna( )

    cm_local = pd.crosstab( 
        obs_local.loc[ keep_mask ],
        pred_local.loc[ keep_mask ],
        rownames = [ 'Observed ΔT bin' ],
        colnames = [ 'Predicted ΔT bin' ],
        normalize = 'index',
    )
    cm_local = cm_local.reindex( index = bin_labels, columns = bin_labels, fill_value = 0.0 )

    return { 
        'cm': cm_local,
        'labels': bin_labels,
        'legend_lines': legend_lines,
        'legend_text': legend_text,
        'coverage_share': float( keep_mask.mean( ) ),
    }


phase6_cm_open = build_phase6_binned_cm( y_true, y_pred, phase6_bin_edges, phase6_bin_ranges )
phase6_cm_closed = build_phase6_binned_cm( y_true, y_pred, phase6_closed_bin_edges, phase6_closed_bin_ranges )

phase6_scatter_df = pd.DataFrame( { 
    'y_true': y_true,
    'y_pred': y_pred,
} ).dropna( ).copy( )


def plot_phase6_temperature_media_pair( bin_mode = 'open' ):

    if bin_mode == 'open':
        cm_obj = phase6_cm_open
        bin_label = 'open-tail bins'

    elif bin_mode == 'closed':
        cm_obj = phase6_cm_closed
        bin_label = 'closed-tail bins'

    else:
        raise ValueError( "bin_mode must be 'open' or 'closed'" )

    legend_cols = int( np.ceil( len( cm_obj[ 'legend_lines' ] ) / 2 ) )
    legend_rows = [ 
        '      '.join( cm_obj[ 'legend_lines' ][ i:i + legend_cols ] )
        for i in range( 0, len( cm_obj[ 'legend_lines' ] ), legend_cols )
    ]
    legend_text = chr( 10 ).join( legend_rows )

    fig, axes = plt.subplots( 1, 2, figsize = ( 17.5, 7.2 ) )
    fig.suptitle( 
        f'Phase 6 Holdout Temperature Diagnostics | Δ Water Temp ({ bin_label })',
        fontsize = 24,
        fontweight = 'bold',
        y = 0.98,
    )

    sns.heatmap( 
        cm_obj[ 'cm' ],
        annot = True,
        fmt = '.2f',
        cmap = 'Blues',
        vmin = 0,
        vmax = 1,
        cbar = False,
        ax = axes[ 0 ],
    )
    axes[ 0 ].set_title( 'Binned Observed vs Predicted' )
    axes[ 0 ].set_xlabel( 'Predicted bin' )
    axes[ 0 ].set_ylabel( 'Observed bin' )

    sns.scatterplot( data = phase6_scatter_df, x = 'y_pred', y = 'y_true', s = 18, alpha = 0.35, ax = axes[ 1 ] )
    lo = float( min( phase6_scatter_df[ 'y_true' ].min( ), phase6_scatter_df[ 'y_pred' ].min( ) ) )
    hi = float( max( phase6_scatter_df[ 'y_true' ].max( ), phase6_scatter_df[ 'y_pred' ].max( ) ) )
    axes[ 1 ].plot( [ lo, hi ], [ lo, hi ], color = 'black', linestyle = '--', linewidth = 1.0 )
    axes[ 1 ].set_title( 'Predicted vs Observed Scatter' )
    axes[ 1 ].set_xlabel( 'Predicted Δ Water Temp ( C )' )
    axes[ 1 ].set_ylabel( 'Observed Δ Water Temp ( C )' )

    fig.text( 
        0.5,
        0.08,
        legend_text,
        va = 'top',
        ha = 'center',
        fontsize = 10.5,
        fontweight = 'semibold',
        bbox = { 'boxstyle': 'round,pad=0.5', 'facecolor': 'white', 'edgecolor': '#bbbbbb', 'alpha': 0.97 },
    )

    fig.subplots_adjust( top = 0.84, bottom = 0.23, wspace = 0.25 )
    plt.show( )


plot_phase6_temperature_media_pair( 'open' )

print( 'phase 6 open-tail bin legend:' )
for line in phase6_cm_open[ 'legend_lines' ]:
    print( line )
print( f'open-tail matrix row coverage: {phase6_cm_open[ "coverage_share" ]:.3f}' )

print( )
print( 'phase 6 closed-tail bin legend:' )
for line in phase6_cm_closed[ 'legend_lines' ]:
    print( line )
print( f'closed-tail matrix row coverage: {phase6_cm_closed[ "coverage_share" ]:.3f}' )

abs_err = ( y_pred - y_true ).abs( )
for tol in [ 0.5, 1.0, 2.0 ]:
    print( f'within ±{tol:.1f} C: {( abs_err <= tol ).mean():.3f}' )


### 6.4d - interpretation of binned observed vs predicted ΔT matrix
- rows are observed bins; columns are predicted bins; each row sums to ~1.0.
- strong performance at the extremes:
  - observed `<-2`: 0.938 on-diagonal
  - observed `>2`: 0.891 on-diagonal
- mid-range bins are much less stable:
  - observed `-2:-1`: only 0.191 on-diagonal, with 0.555 pushed into `<-2`
  - observed `-1:-0.5`: only 0.124 on-diagonal, with large spill into colder bins
  - observed `-0.5:0.5`: only 0.215 on-diagonal, broad spread across both sides
  - observed `0.5:1`: only 0.105 on-diagonal
  - observed `1:2`: only 0.172 on-diagonal, with 0.383 pushed into `>2`
- model recognizes strong cooling/warming events fairly well, but moderate ΔT regimes are often mis-binned and pulled toward neighboring or extreme bins.

### 6.5 to 6.7 - seasonal sampling-gap diagnostics (no pipeline changes yet)
Goal: test whether changing winter/summer coverage could bias annual means and apparent warming signals.


In [ ]:
# 6.5 - build station-year sampling diagnostics from daily water observations
coverage_diag = daily_water[ [ 'region', 'station', 'date', 'water_temp', 'delta_water_temp' ] ].copy( )
coverage_diag[ 'date' ] = pd.to_datetime( coverage_diag[ 'date' ], errors = 'coerce' )
coverage_diag = coverage_diag.dropna( subset = [ 'date' ] )
coverage_diag[ 'year' ] = coverage_diag[ 'date' ].dt.year
coverage_diag[ 'month' ] = coverage_diag[ 'date' ].dt.month
coverage_diag[ 'has_temp' ] = coverage_diag[ 'water_temp' ].notna( ).astype( int )

monthly_presence = ( 
    coverage_diag
    .groupby( [ 'region', 'station', 'year', 'month' ], as_index = False )[ 'has_temp' ]
    .max( )
)

monthly_presence[ 'winter_obs' ] = monthly_presence[ 'has_temp' ] * monthly_presence[ 'month' ].isin( [ 12, 1, 2 ] ).astype( int )
monthly_presence[ 'summer_obs' ] = monthly_presence[ 'has_temp' ] * monthly_presence[ 'month' ].isin( [ 6, 7, 8 ] ).astype( int )

station_year_sampling = ( 
    monthly_presence
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        n_months_obs = ( 'has_temp', 'sum' ),
        n_winter_months = ( 'winter_obs', 'sum' ),
        n_summer_months = ( 'summer_obs', 'sum' ),
    )
)

annual_temp_by_station = ( 
    coverage_diag
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        annual_mean_water_temp = ( 'water_temp', 'mean' ),
        annual_mean_delta_water_temp = ( 'delta_water_temp', 'mean' ),
        n_daily_rows = ( 'water_temp', 'size' ),
    )
)

station_year_sampling = station_year_sampling.merge( 
    annual_temp_by_station,
    on = [ 'region', 'station', 'year' ],
    how = 'left',
)

station_year_sampling[ 'has_winter_gap' ] = station_year_sampling[ 'n_winter_months' ] == 0
station_year_sampling[ 'has_summer_gap' ] = station_year_sampling[ 'n_summer_months' ] == 0
station_year_sampling[ 'seasonally_balanced' ] = ( 
    ( station_year_sampling[ 'n_months_obs' ] >= 9 )
    & ( station_year_sampling[ 'n_winter_months' ] >= 2 )
    & ( station_year_sampling[ 'n_summer_months' ] >= 2 )
)

station_year_sampling[ 'season_balance_ratio' ] = ( 
    np.minimum( station_year_sampling[ 'n_winter_months' ], station_year_sampling[ 'n_summer_months' ] )
    / np.maximum( station_year_sampling[ 'n_winter_months' ], station_year_sampling[ 'n_summer_months' ] ).replace( 0, np.nan )
)

print( f'station-year rows: {len( station_year_sampling )}' )
print( f'pct with zero winter months: {100.0 * station_year_sampling[ "has_winter_gap" ].mean( ):.2f}%' )
print( f'pct with zero summer months: {100.0 * station_year_sampling[ "has_summer_gap" ].mean( ):.2f}%' )
print( f'pct seasonally balanced (>=9 months + >=2 winter + >=2 summer): {100.0 * station_year_sampling[ "seasonally_balanced" ].mean( ):.2f}%' )

display( 
    station_year_sampling[ [ 
        'n_months_obs',
        'n_winter_months',
        'n_summer_months',
        'season_balance_ratio',
        'annual_mean_delta_water_temp',
    ] ]
    .describe( percentiles = [ 0.10, 0.25, 0.50, 0.75, 0.90 ] )
    .round( 3 )
)


In [ ]:
# 6.6 - visualize sampling coverage shifts over time
yearly_sampling = ( 
    station_year_sampling
    .groupby( 'year', as_index = False )
    .agg( 
        n_station_years = ( 'station', 'size' ),
        mean_months_obs = ( 'n_months_obs', 'mean' ),
        pct_with_winter_gap = ( 'has_winter_gap', lambda values: 100.0 * float( values.mean( ) ) ),
        pct_with_summer_gap = ( 'has_summer_gap', lambda values: 100.0 * float( values.mean( ) ) ),
        pct_seasonally_balanced = ( 'seasonally_balanced', lambda values: 100.0 * float( values.mean( ) ) ),
        mean_annual_delta_all = ( 'annual_mean_delta_water_temp', 'mean' ),
    )
)

yearly_balanced = ( 
    station_year_sampling
    .loc[ station_year_sampling[ 'seasonally_balanced' ] ]
    .groupby( 'year', as_index = False )
    .agg( 
        mean_annual_delta_balanced = ( 'annual_mean_delta_water_temp', 'mean' ),
        n_station_years_balanced = ( 'station', 'size' ),
    )
)

yearly_sampling = yearly_sampling.merge( yearly_balanced, on = 'year', how = 'left' )

fig, axes = plt.subplots( 1, 2, figsize = ( 16, 5 ) )

axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_with_winter_gap' ], label = '% station-years with winter gap', linewidth = 1.6 )
axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_with_summer_gap' ], label = '% station-years with summer gap', linewidth = 1.6 )
axes[ 0 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'pct_seasonally_balanced' ], label = '% seasonally balanced', linewidth = 1.8 )
axes[ 0 ].set_title( 'Seasonal Sampling Coverage Over Time' )
axes[ 0 ].set_xlabel( 'Year' )
axes[ 0 ].set_ylabel( 'Percent of Station-Years' )
axes[ 0 ].set_ylim( 0, 100 )
axes[ 0 ].legend( loc = 'best' )

axes[ 1 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'mean_annual_delta_all' ], label = 'Mean annual delta (all station-years)', linewidth = 1.8, color = 'black' )
axes[ 1 ].plot( yearly_sampling[ 'year' ], yearly_sampling[ 'mean_annual_delta_balanced' ], label = 'Mean annual delta (seasonally balanced only)', linewidth = 1.8, color = 'tab:orange' )
axes[ 1 ].axhline( 0.0, color = 'gray', linestyle = '--', linewidth = 1.0 )
axes[ 1 ].set_title( 'Annual Delta Water Temp: All vs Balanced Coverage' )
axes[ 1 ].set_xlabel( 'Year' )
axes[ 1 ].set_ylabel( 'Mean annual delta_water_temp ( C )' )
axes[ 1 ].legend( loc = 'best' )

plt.tight_layout( )
plt.show( )

display( yearly_sampling.tail( 15 ).round( 3 ) )


In [ ]:
# 6.7 - quick sensitivity tests: trend slope and gap-vs-no-gap contrast


def slope_c_per_year( frame, y_col ):
    fit = frame[ [ 'year', y_col ] ].dropna( ).copy( )
    if len( fit ) < 2:
        return np.nan

    return float( np.polyfit( fit[ 'year' ].astype( float ), fit[ y_col ].astype( float ), 1 )[ 0 ] )


slope_all = slope_c_per_year( yearly_sampling, 'mean_annual_delta_all' )
slope_balanced = slope_c_per_year( yearly_sampling, 'mean_annual_delta_balanced' )

gap_effect_winter = ( 
    station_year_sampling
    .groupby( 'has_winter_gap', as_index = False )
    .agg( 
        mean_delta = ( 'annual_mean_delta_water_temp', 'mean' ),
        median_delta = ( 'annual_mean_delta_water_temp', 'median' ),
        n_rows = ( 'annual_mean_delta_water_temp', 'size' ),
    )
)

gap_effect_summer = ( 
    station_year_sampling
    .groupby( 'has_summer_gap', as_index = False )
    .agg( 
        mean_delta = ( 'annual_mean_delta_water_temp', 'mean' ),
        median_delta = ( 'annual_mean_delta_water_temp', 'median' ),
        n_rows = ( 'annual_mean_delta_water_temp', 'size' ),
    )
)

print( f'slope all station-years (C/year): {slope_all:.4f}' )
print( f'slope seasonally balanced only (C/year): {slope_balanced:.4f}' )
print( f'slope difference (balanced - all): {slope_balanced - slope_all:.4f}' )

print( '' )
print( 'winter-gap contrast (annual_mean_delta_water_temp):' )
display( gap_effect_winter.round( 3 ) )

print( 'summer-gap contrast (annual_mean_delta_water_temp):' )
display( gap_effect_summer.round( 3 ) )

sns.boxplot( 
    data = station_year_sampling.assign( winter_gap_label = station_year_sampling[ 'has_winter_gap' ].map( { False: 'winter present', True: 'winter missing' } ) ),
    x = 'winter_gap_label',
    y = 'annual_mean_delta_water_temp',
    showfliers = False,
)
plt.title( 'Annual Delta Water Temp by Winter Coverage Status' )
plt.xlabel( 'Coverage Group' )
plt.ylabel( 'annual_mean_delta_water_temp ( C )' )
plt.tight_layout( )
plt.show( )



## Phase 7 — Model Development: Water Temp → Water Properties
Goal: predict Δsalinity, ΔDO, ΔpH given ΔT_water and atmospheric context

# also stage 2

- 7.1 for each target variable, repeat baseline 
  - linear → gradient boosted sequence from phase 6
- 7.2 use predicted Δ air temp for water as input
  - keeps the pipeline honest and propagates uncertainty
- 7.3 compare response sensitivity by regime
  - does a +2°C warming suppress dissolved oxy more in warm estuaries than cold ones?
  - prolly a yes ... quantifying it is the contribution
- 7.4 document responses as simple lookup relationships
  - e.g., per-degree sensitivities per regime 
  - these are our core scientific deliverables

### 7.0 - build annual climate-response tables (non-daily)
- purpose: climate-change potential, not next-day forecasting
- aggregate station-year means, then model annual non-nutrient deltas
- keep phase-6 link by using phase-6 predicted `delta_water_temp` as a phase-7 driver


In [ ]:
# 7.0 - scenario-oriented phase 7 prep (annual + warm-season aggregates)
# keep the same broad two-stage idea, but give oxygen several summer-stress targets
# instead of only one warm-season mean

air_source_7 = daily_air_final.copy( )
water_source_7 = daily_water_final.copy( )

air_source_7[ 'date' ] = pd.to_datetime( air_source_7[ 'date' ], errors = 'coerce' )
water_source_7[ 'date' ] = pd.to_datetime( water_source_7[ 'date' ], errors = 'coerce' )

phase7_base_target_candidates = [ 'delta_salinity', 'delta_oxygen', 'delta_ph' ]
phase7_base_targets = [ col for col in phase7_base_target_candidates if col in water_source_7.columns ]

water_keep_cols_7 = [ 'region', 'station', 'date' ] + phase7_base_targets
for col in [ 'oxygen', 'water_temp_baseline', 'salinity_baseline', 'oxygen_baseline', 'ph_baseline', 'depth_baseline' ]:
    if col in water_source_7.columns:
        water_keep_cols_7.append( col )
water_keep_cols_7 = list( dict.fromkeys( water_keep_cols_7 ) )

phase7_daily = air_source_7.merge( 
    water_source_7[ water_keep_cols_7 ],
    on = [ 'region', 'station', 'date' ],
    how = 'inner',
)
phase7_daily = phase7_daily.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )

if 'station_baseline' in globals( ) and 'cluster' in station_baseline.columns:
    phase7_cluster_lookup = station_baseline[ [ 'region', 'station', 'cluster' ] ].copy( )
    phase7_cluster_lookup[ 'cluster_code' ] = pd.to_numeric( phase7_cluster_lookup[ 'cluster' ], errors = 'coerce' )
    phase7_cluster_lookup = phase7_cluster_lookup.drop( columns = [ 'cluster' ] )
    phase7_daily = phase7_daily.merge( phase7_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

if 'air_temp' in phase7_daily.columns and 'water_temp_baseline' in phase7_daily.columns:
    phase7_daily[ 'air_temp_minus_water_temp_baseline' ] = phase7_daily[ 'air_temp' ] - phase7_daily[ 'water_temp_baseline' ]

if 'air_temp_r7d' in phase7_daily.columns and 'water_temp_baseline' in phase7_daily.columns:
    phase7_daily[ 'air_temp_r7d_minus_water_temp_baseline' ] = phase7_daily[ 'air_temp_r7d' ] - phase7_daily[ 'water_temp_baseline' ]

if 'air_temp_r28d' in phase7_daily.columns and 'water_temp_baseline' in phase7_daily.columns:
    phase7_daily[ 'air_temp_r28d_minus_water_temp_baseline' ] = phase7_daily[ 'air_temp_r28d' ] - phase7_daily[ 'water_temp_baseline' ]

if 'air_temp_r1d' in phase7_daily.columns and 'air_temp_r28d' in phase7_daily.columns:
    phase7_daily[ 'air_temp_r1d_minus_air_temp_r28d' ] = phase7_daily[ 'air_temp_r1d' ] - phase7_daily[ 'air_temp_r28d' ]

if 'air_temp_r7d' in phase7_daily.columns and 'air_temp_r28d' in phase7_daily.columns:
    phase7_daily[ 'air_temp_r7d_minus_air_temp_r28d' ] = phase7_daily[ 'air_temp_r7d' ] - phase7_daily[ 'air_temp_r28d' ]

if 'wind_speed' in phase7_daily.columns and 'air_temp' in phase7_daily.columns:
    phase7_daily[ 'wind_speed_x_air_temp' ] = phase7_daily[ 'wind_speed' ] * phase7_daily[ 'air_temp' ]

if 'solar' in phase7_daily.columns and 'air_temp' in phase7_daily.columns:
    phase7_daily[ 'solar_x_air_temp' ] = phase7_daily[ 'solar' ] * phase7_daily[ 'air_temp' ]

phase7_p6_feature_cols = phase6_selected_features.copy( )
for col in phase7_p6_feature_cols:
    if col not in phase7_daily.columns:
        phase7_daily[ col ] = np.nan
phase7_p6_fill_values = phase6_selected_fill_values.reindex( phase7_p6_feature_cols )
phase7_p6_fill_values = phase7_p6_fill_values.fillna( phase6_train_model[ phase7_p6_feature_cols ].median( numeric_only = True ) )

X_phase7_for_p6 = phase7_daily[ phase7_p6_feature_cols ].copy( ).fillna( phase7_p6_fill_values )
phase7_daily[ 'delta_water_temp_pred_p6' ] = phase6_selected_model.predict( X_phase7_for_p6 )

phase7_daily = phase7_daily.dropna( subset = [ 'date' ] )
phase7_daily[ 'year' ] = phase7_daily[ 'date' ].dt.year
phase7_daily[ 'month_num' ] = phase7_daily[ 'date' ].dt.month
phase7_daily[ 'is_warm_season' ] = phase7_daily[ 'month_num' ].between( 6, 9 )

agg_feature_cols = [ ]
for col in [ 'delta_water_temp_pred_p6', 'air_temp', 'precip', 'wind_speed', 'solar' ]:
    if col in phase7_daily.columns:
        agg_feature_cols.append( col )

baseline_daily_feature_cols = [ ]
for col in [ 'water_temp_baseline', 'salinity_baseline', 'oxygen_baseline', 'ph_baseline', 'depth_baseline' ]:
    if col in phase7_daily.columns:
        baseline_daily_feature_cols.append( col )

phase7_annual_agg = { 
    'n_days_annual': ( 'date', 'size' ),
}
for col in agg_feature_cols:
    phase7_annual_agg[ f'{ col }_mean' ] = ( col, 'mean' )
    phase7_annual_agg[ f'{ col }_std' ] = ( col, 'std' )
for col in baseline_daily_feature_cols:
    phase7_annual_agg[ f'{ col }_annual_mean' ] = ( col, 'mean' )
for target in phase7_base_targets:
    phase7_annual_agg[ f'{ target }_annual_mean' ] = ( target, 'mean' )

phase7_station_year = ( 
    phase7_daily
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( **phase7_annual_agg )
)

phase7_warm_daily = phase7_daily.loc[ phase7_daily[ 'is_warm_season' ] ].copy( )
phase7_warm_agg = { 
    'n_days_warm': ( 'date', 'size' ),
}
for col in agg_feature_cols:
    phase7_warm_agg[ f'{ col }_warm_mean' ] = ( col, 'mean' )
    phase7_warm_agg[ f'{ col }_warm_std' ] = ( col, 'std' )
for target in phase7_base_targets:
    phase7_warm_agg[ f'{ target }_warm_mean' ] = ( target, 'mean' )

phase7_warm_station_year = ( 
    phase7_warm_daily
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( **phase7_warm_agg )
)

phase7_station_year = phase7_station_year.merge( phase7_warm_station_year, on = [ 'region', 'station', 'year' ], how = 'left' )


def warm_quantile( values, q ):
    valid = pd.Series( values ).dropna( )
    if len( valid ) == 0:
        return np.nan

    return float( valid.quantile( q ) )


def warm_share_below( values, threshold ):
    valid = pd.Series( values ).dropna( )
    if len( valid ) == 0:
        return np.nan

    return float( ( valid < threshold ).mean( ) )


if 'oxygen' in phase7_warm_daily.columns:
    phase7_oxygen_stress = ( 
        phase7_warm_daily
        .groupby( [ 'region', 'station', 'year' ], as_index = False )
        .agg( 
            oxygen_warm_min = ( 'oxygen', 'min' ),
            oxygen_warm_p10 = ( 'oxygen', lambda values: warm_quantile( values, 0.10 ) ),
            oxygen_warm_hypoxic_day_share = ( 'oxygen', lambda values: warm_share_below( values, 2.0 ) ),
            oxygen_warm_below_optimal_day_share = ( 'oxygen', lambda values: warm_share_below( values, 5.0 ) ),
        )
    )
    phase7_station_year = phase7_station_year.merge( phase7_oxygen_stress, on = [ 'region', 'station', 'year' ], how = 'left' )

if 'station_year_quality' in globals( ):
    phase7_station_year = phase7_station_year.merge( 
        station_year_quality[ [ 
            'region',
            'station',
            'year',
            'has_model_year',
            'has_warm_oxygen_year',
            'n_temp_days',
            'n_temp_months',
            'n_warm_oxygen_days',
            'n_warm_oxygen_months',
        ] ],
        on = [ 'region', 'station', 'year' ],
        how = 'left',
    )

else:
    phase7_station_year[ 'has_model_year' ] = True
    phase7_station_year[ 'has_warm_oxygen_year' ] = True

phase7_station_year[ 'has_model_year' ] = phase7_station_year[ 'has_model_year' ].fillna( False ).astype( bool )
phase7_station_year[ 'has_warm_oxygen_year' ] = phase7_station_year[ 'has_warm_oxygen_year' ].fillna( False ).astype( bool )

phase7_station_context_feature_candidates = [ 
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_saturation',
    'seasonal_amp_saturation',
    'mean_annual_ph',
    'seasonal_amp_ph',
    'mean_annual_depth',
    'seasonal_amp_depth',
    'warm_peak_month',
    'oxygen_nadir_month',
    'temp_oxygen_phase_lag',
    'warm_len_months',
    'cold_len_months',
    'warm_intensity_temp',
    'cold_intensity_temp',
    'warm_oxygen_mean',
    'warm_oxygen_min',
    'cold_oxygen_mean',
    'n_obs_total',
    'n_years_present',
    'mean_year_coverage',
    'is_partial_baseline',
]
phase7_station_context_cols = [ 
    col for col in phase7_station_context_feature_candidates
    if 'station_baseline' in globals( ) and col in station_baseline.columns
]

if len( phase7_station_context_cols ) > 0:
    phase7_station_context = station_baseline[ [ 'region', 'station' ] + phase7_station_context_cols ].copy( )

else:
    phase7_station_context = station_baseline[ [ 'region', 'station' ] ].drop_duplicates( ).copy( ) if 'station_baseline' in globals( ) else phase7_station_year[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

if 'cluster' in station_baseline.columns:
    phase7_station_context = phase7_station_context.merge( 
        station_baseline[ [ 'region', 'station', 'cluster' ] ].copy( ),
        on = [ 'region', 'station' ],
        how = 'left',
    )
    phase7_station_context[ 'cluster_code' ] = pd.to_numeric( phase7_station_context[ 'cluster' ], errors = 'coerce' )
    phase7_station_context = phase7_station_context.drop( columns = [ 'cluster' ] )

phase7_station_year = phase7_station_year.merge( phase7_station_context, on = [ 'region', 'station' ], how = 'left' )

phase7_station_year_train = phase7_station_year.merge( 
    unflagged_station_keys_final[ [ 'region', 'station' ] ].drop_duplicates( ),
    on = [ 'region', 'station' ],
    how = 'inner',
)
phase7_station_year_holdout = phase7_station_year.merge( 
    flagged_station_keys_final[ [ 'region', 'station' ] ].drop_duplicates( ),
    on = [ 'region', 'station' ],
    how = 'inner',
)

phase7_feature_candidates_static = [ 
    'water_temp_baseline_annual_mean',
    'salinity_baseline_annual_mean',
    'oxygen_baseline_annual_mean',
    'ph_baseline_annual_mean',
    'depth_baseline_annual_mean',
    'cluster_code',
    'mean_annual_water_temp',
    'seasonal_amp_water_temp',
    'mean_annual_salinity',
    'seasonal_amp_salinity',
    'mean_annual_oxygen',
    'seasonal_amp_oxygen',
    'mean_annual_saturation',
    'seasonal_amp_saturation',
    'mean_annual_ph',
    'seasonal_amp_ph',
    'mean_annual_depth',
    'seasonal_amp_depth',
    'warm_peak_month',
    'oxygen_nadir_month',
    'temp_oxygen_phase_lag',
    'warm_len_months',
    'cold_len_months',
    'warm_intensity_temp',
    'cold_intensity_temp',
    'warm_oxygen_mean',
    'warm_oxygen_min',
    'cold_oxygen_mean',
    'n_obs_total',
    'n_years_present',
    'mean_year_coverage',
    'is_partial_baseline',
]
phase7_feature_candidates_annual = [ 
    'delta_water_temp_pred_p6_mean',
    'air_temp_mean',
    'precip_mean',
    'wind_speed_mean',
    'solar_mean',
    'delta_water_temp_pred_p6_std',
    'air_temp_std',
    'precip_std',
    'wind_speed_std',
    'solar_std',
] + phase7_feature_candidates_static
phase7_feature_candidates_warm = [ 
    'delta_water_temp_pred_p6_warm_mean',
    'air_temp_warm_mean',
    'precip_warm_mean',
    'wind_speed_warm_mean',
    'solar_warm_mean',
    'delta_water_temp_pred_p6_warm_std',
    'air_temp_warm_std',
    'precip_warm_std',
    'wind_speed_warm_std',
    'solar_warm_std',
] + phase7_feature_candidates_static
phase7_features_annual = [ 
    col for col in dict.fromkeys( phase7_feature_candidates_annual )
    if col in phase7_station_year_train.columns
]
phase7_features_warm = [ 
    col for col in dict.fromkeys( phase7_feature_candidates_warm )
    if col in phase7_station_year_train.columns
]
phase7_use_warm_oxygen_features = len( phase7_features_warm ) > 0

phase7_target_candidates = [ 
    'delta_salinity',
    'delta_oxygen',
    'oxygen_warm_min',
    'oxygen_warm_p10',
    'oxygen_warm_hypoxic_day_share',
    'oxygen_warm_below_optimal_day_share',
    'delta_ph',
]
phase7_oxygen_targets = [ 
    'delta_oxygen',
    'oxygen_warm_min',
    'oxygen_warm_p10',
    'oxygen_warm_hypoxic_day_share',
    'oxygen_warm_below_optimal_day_share',
]
phase7_share_targets = [ 
    'oxygen_warm_hypoxic_day_share',
    'oxygen_warm_below_optimal_day_share',
]

phase7_target_col_map = { 
    'delta_salinity': 'delta_salinity_annual_mean' if 'delta_salinity_annual_mean' in phase7_station_year_train.columns else None,
    'delta_oxygen': 'delta_oxygen_warm_mean' if 'delta_oxygen_warm_mean' in phase7_station_year_train.columns else ( 'delta_oxygen_annual_mean' if 'delta_oxygen_annual_mean' in phase7_station_year_train.columns else None ),
    'oxygen_warm_min': 'oxygen_warm_min' if 'oxygen_warm_min' in phase7_station_year_train.columns else None,
    'oxygen_warm_p10': 'oxygen_warm_p10' if 'oxygen_warm_p10' in phase7_station_year_train.columns else None,
    'oxygen_warm_hypoxic_day_share': 'oxygen_warm_hypoxic_day_share' if 'oxygen_warm_hypoxic_day_share' in phase7_station_year_train.columns else None,
    'oxygen_warm_below_optimal_day_share': 'oxygen_warm_below_optimal_day_share' if 'oxygen_warm_below_optimal_day_share' in phase7_station_year_train.columns else None,
    'delta_ph': 'delta_ph_annual_mean' if 'delta_ph_annual_mean' in phase7_station_year_train.columns else None,
}
phase7_target_feature_map = { 
    'delta_salinity': phase7_features_annual.copy( ),
    'delta_oxygen': phase7_features_warm.copy( ) if phase7_use_warm_oxygen_features else phase7_features_annual.copy( ),
    'oxygen_warm_min': phase7_features_warm.copy( ) if phase7_use_warm_oxygen_features else phase7_features_annual.copy( ),
    'oxygen_warm_p10': phase7_features_warm.copy( ) if phase7_use_warm_oxygen_features else phase7_features_annual.copy( ),
    'oxygen_warm_hypoxic_day_share': phase7_features_warm.copy( ) if phase7_use_warm_oxygen_features else phase7_features_annual.copy( ),
    'oxygen_warm_below_optimal_day_share': phase7_features_warm.copy( ) if phase7_use_warm_oxygen_features else phase7_features_annual.copy( ),
    'delta_ph': phase7_features_annual.copy( ),
}
phase7_target_year_flag_map = { 
    'delta_salinity': 'has_model_year',
    'delta_oxygen': 'has_warm_oxygen_year' if phase7_use_warm_oxygen_features else 'has_model_year',
    'oxygen_warm_min': 'has_warm_oxygen_year' if phase7_use_warm_oxygen_features else 'has_model_year',
    'oxygen_warm_p10': 'has_warm_oxygen_year' if phase7_use_warm_oxygen_features else 'has_model_year',
    'oxygen_warm_hypoxic_day_share': 'has_warm_oxygen_year' if phase7_use_warm_oxygen_features else 'has_model_year',
    'oxygen_warm_below_optimal_day_share': 'has_warm_oxygen_year' if phase7_use_warm_oxygen_features else 'has_model_year',
    'delta_ph': 'has_model_year',
}
phase7_target_window_label_map = { 
    'delta_salinity': 'annual mean',
    'delta_oxygen': 'warm-season mean' if phase7_target_col_map.get( 'delta_oxygen' ) == 'delta_oxygen_warm_mean' else 'annual mean',
    'oxygen_warm_min': 'warm-season minimum',
    'oxygen_warm_p10': 'warm-season 10th percentile',
    'oxygen_warm_hypoxic_day_share': 'warm-season share < 2 mg/L',
    'oxygen_warm_below_optimal_day_share': 'warm-season share < 5 mg/L',
    'delta_ph': 'annual mean',
}
phase7_target_display_name_map = { 
    'delta_salinity': 'delta salinity',
    'delta_oxygen': 'delta oxygen',
    'oxygen_warm_min': 'oxygen minimum',
    'oxygen_warm_p10': 'oxygen p10',
    'oxygen_warm_hypoxic_day_share': 'oxygen hypoxic share',
    'oxygen_warm_below_optimal_day_share': 'oxygen <5 share',
    'delta_ph': 'delta pH',
}
phase7_targets = [ 
    target for target in phase7_target_candidates
    if phase7_target_col_map.get( target ) is not None and len( phase7_target_feature_map.get( target, [ ] ) ) > 0
]

print( f'phase7 targets: { phase7_targets }' )
print( f'phase7 daily rows: { len( phase7_daily ) }' )
print( f'phase7 station-year rows (all): { len( phase7_station_year ) }' )
print( f'phase7 station-year rows (train): { len( phase7_station_year_train ) }' )
print( f'phase7 station-year rows (holdout): { len( phase7_station_year_holdout ) }' )
print( f'phase7 annual features ({ len( phase7_features_annual ) }): { phase7_features_annual }' )
print( f'phase7 warm features ({ len( phase7_features_warm ) }): { phase7_features_warm }' )
print( f'phase7 oxygen stress targets available: {[ target for target in phase7_targets if target in phase7_oxygen_targets ]}' )
print( 'phase7 target setup:' )
for target in phase7_targets:
    print( f'  - {target}: target_col = {phase7_target_col_map.get( target )}, window = {phase7_target_window_label_map.get( target )}, year_flag = {phase7_target_year_flag_map.get( target )}, n_features = {len( phase7_target_feature_map.get( target, [ ] ) )}' )


### 7.1 - fit annual response models (no persistence)
- compare `naive_global_median`, `ridge_phase7_climate`, `hgb_phase7_climate`


In [ ]:
# 7.1 - mixed annual and warm-season response model stack
# salinity and pH stay annual here
# oxygen uses warm-season means when that table exists



class Phase7TargetPredictor:


    def __init__( self, base_model, target_name, calibrator = None ):
        self.base_model = base_model
        self.target_name = target_name
        self.calibrator = calibrator


    def predict( self, X ):
        pred = np.asarray( self.base_model.predict( X ), dtype = 'float64' )

        if self.calibrator is not None:
            pred = self.calibrator.predict( pred.reshape( -1, 1 ) )

        if self.target_name in phase7_share_targets:
            pred = np.clip( pred, 0.0, 1.0 )

        return pred


def append_phase7_score( score_rows, target, target_window, model_name, y_true, y_pred, n_train, n_holdout, n_features, coverage_flag, target_col ):
    score_rows.append( { 
        'target': target,
        'target_window': target_window,
        'model': model_name,
        'mae': float( mean_absolute_error( y_true, y_pred ) ),
        'rmse': float( mean_squared_error( y_true, y_pred ) ** 0.5 ),
        'r2': float( r2_score( y_true, y_pred ) ),
        'bias': float( np.mean( np.asarray( y_pred ) - np.asarray( y_true ) ) ),
        'n_train': int( n_train ),
        'n_holdout': int( n_holdout ),
        'n_features': int( n_features ),
        'coverage_flag': coverage_flag,
        'target_col': target_col,
    } )


def build_phase7_sample_weights( y, target ):
    y = pd.Series( y, copy = False ).astype( 'float64' )
    weights = pd.Series( 1.0, index = y.index, dtype = 'float64' )
    valid = y.dropna( )

    if len( valid ) < 8:
        return weights

    if target in [ 'delta_oxygen', 'oxygen_warm_min', 'oxygen_warm_p10' ]:
        q25 = float( valid.quantile( 0.25 ) )
        q10 = float( valid.quantile( 0.10 ) )
        weights.loc[ y <= q25 ] = 2.0
        weights.loc[ y <= q10 ] = 3.0

    elif target in phase7_share_targets:
        q75 = float( valid.quantile( 0.75 ) )
        q90 = float( valid.quantile( 0.90 ) )
        weights.loc[ y >= q75 ] = 2.0
        weights.loc[ y >= q90 ] = 3.0

    else:
        q20 = float( valid.quantile( 0.20 ) )
        q80 = float( valid.quantile( 0.80 ) )
        weights.loc[ ( y <= q20 ) | ( y >= q80 ) ] = 1.75

    return weights


def build_phase7_group_oof_predictions( estimator, X, y, groups, sample_weight = None ):
    X = X.copy( )
    y = y.copy( )
    groups = pd.Series( groups, index = X.index ).astype( str )

    if sample_weight is not None:
        sample_weight = pd.Series( sample_weight, index = X.index ).astype( 'float64' )

    unique_groups = groups.dropna( ).unique( )
    if len( unique_groups ) < 2:
        fitted_estimator = clone( estimator )

        if sample_weight is None:
            fitted_estimator.fit( X, y )

        else:
            fitted_estimator.fit( X, y, sample_weight = sample_weight )

        return pd.Series( fitted_estimator.predict( X ), index = X.index, dtype = 'float64' )

    n_splits = min( 5, len( unique_groups ) )
    splitter = GroupKFold( n_splits = n_splits )
    oof_pred = pd.Series( index = X.index, dtype = 'float64' )

    for fit_idx, valid_idx in splitter.split( X, y, groups ):
        fold_model = clone( estimator )
        X_fit = X.iloc[ fit_idx ]
        y_fit = y.iloc[ fit_idx ]

        if sample_weight is None:
            fold_model.fit( X_fit, y_fit )

        else:
            fold_model.fit( X_fit, y_fit, sample_weight = sample_weight.iloc[ fit_idx ] )

        oof_pred.iloc[ valid_idx ] = fold_model.predict( X.iloc[ valid_idx ] )

    if oof_pred.isna( ).any( ):
        fallback_model = clone( estimator )

        if sample_weight is None:
            fallback_model.fit( X, y )

        else:
            fallback_model.fit( X, y, sample_weight = sample_weight )

        missing_mask = oof_pred.isna( )
        oof_pred.loc[ missing_mask ] = fallback_model.predict( X.loc[ missing_mask ] )

    return oof_pred


phase7_scores_rows = [ ]
phase7_best_rows = [ ]
phase7_model_store = { }
phase7_feature_store = { }
phase7_fill_store = { }

for target in phase7_targets:
    target_col = phase7_target_col_map.get( target )
    feature_cols_t = phase7_target_feature_map.get( target, [ ] )
    year_flag_col = phase7_target_year_flag_map.get( target, 'has_model_year' )

    if target_col is None or len( feature_cols_t ) == 0:
        continue

    train_t = phase7_station_year_train.loc[ phase7_station_year_train[ year_flag_col ].fillna( False ).astype( bool ) ].copy( )
    holdout_t = phase7_station_year_holdout.loc[ phase7_station_year_holdout[ year_flag_col ].fillna( False ).astype( bool ) ].copy( )

    train_t = train_t.dropna( subset = [ target_col ] ).copy( )
    holdout_t = holdout_t.dropna( subset = [ target_col ] ).copy( )

    if len( train_t ) == 0 or len( holdout_t ) == 0:
        continue

    y_train_t = train_t[ target_col ]
    y_holdout_t = holdout_t[ target_col ]

    X_train_t = train_t[ feature_cols_t ].copy( )
    X_holdout_t = holdout_t[ feature_cols_t ].copy( )

    fill_values_t = X_train_t.median( numeric_only = True )
    X_train_t = X_train_t.fillna( fill_values_t )
    X_holdout_t = X_holdout_t.fillna( fill_values_t )

    station_groups_t = train_t[ 'region' ].astype( str ) + ' | ' + train_t[ 'station' ].astype( str )
    sample_weight_t = build_phase7_sample_weights( y_train_t, target )

    pred_naive = np.full( len( y_holdout_t ), float( y_train_t.median( ) ) )
    pred_naive = np.clip( pred_naive, 0.0, 1.0 ) if target in phase7_share_targets else pred_naive
    append_phase7_score( 
        phase7_scores_rows,
        target,
        phase7_target_window_label_map.get( target, 'annual mean' ),
        'naive_global_median',
        y_holdout_t,
        pred_naive,
        len( train_t ),
        len( holdout_t ),
        len( feature_cols_t ),
        year_flag_col,
        target_col,
    )

    model_candidates_t = { }

    ridge_t = Ridge( alpha = 1.0 )
    ridge_t.fit( X_train_t, y_train_t )
    model_candidates_t[ 'ridge_phase7_climate' ] = Phase7TargetPredictor( ridge_t, target )

    ridge_weighted_t = Ridge( alpha = 1.0 )
    ridge_weighted_t.fit( X_train_t, y_train_t, sample_weight = sample_weight_t )
    ridge_oof_t = build_phase7_group_oof_predictions( 
        Ridge( alpha = 1.0 ),
        X_train_t,
        y_train_t,
        station_groups_t,
        sample_weight = sample_weight_t,
    )
    ridge_calibrator_t = LinearRegression( )
    ridge_calibrator_t.fit( np.asarray( ridge_oof_t ).reshape( -1, 1 ), y_train_t, sample_weight = sample_weight_t )
    model_candidates_t[ 'ridge_phase7_weighted_calibrated' ] = Phase7TargetPredictor( ridge_weighted_t, target, ridge_calibrator_t )

    hgb_t = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 4,
        max_iter = 300,
        min_samples_leaf = 20,
        random_state = 42,
    )
    hgb_t.fit( X_train_t, y_train_t )
    model_candidates_t[ 'hgb_phase7_climate' ] = Phase7TargetPredictor( hgb_t, target )

    hgb_weighted_t = HistGradientBoostingRegressor( 
        learning_rate = 0.05,
        max_depth = 4,
        max_iter = 300,
        min_samples_leaf = 20,
        random_state = 42,
    )
    hgb_weighted_t.fit( X_train_t, y_train_t, sample_weight = sample_weight_t )
    hgb_oof_t = build_phase7_group_oof_predictions( 
        HistGradientBoostingRegressor( 
            learning_rate = 0.05,
            max_depth = 4,
            max_iter = 300,
            min_samples_leaf = 20,
            random_state = 42,
        ),
        X_train_t,
        y_train_t,
        station_groups_t,
        sample_weight = sample_weight_t,
    )
    hgb_calibrator_t = LinearRegression( )
    hgb_calibrator_t.fit( np.asarray( hgb_oof_t ).reshape( -1, 1 ), y_train_t, sample_weight = sample_weight_t )
    model_candidates_t[ 'hgb_phase7_weighted_calibrated' ] = Phase7TargetPredictor( hgb_weighted_t, target, hgb_calibrator_t )

    for model_name, model_obj in model_candidates_t.items( ):
        pred_t = model_obj.predict( X_holdout_t )
        append_phase7_score( 
            phase7_scores_rows,
            target,
            phase7_target_window_label_map.get( target, 'annual mean' ),
            model_name,
            y_holdout_t,
            pred_t,
            len( train_t ),
            len( holdout_t ),
            len( feature_cols_t ),
            year_flag_col,
            target_col,
        )

    target_scores = pd.DataFrame( phase7_scores_rows )
    target_scores = target_scores.loc[ target_scores[ 'target' ] == target ].sort_values( 'rmse', ascending = True )

    target_scores_non_naive = ( 
        target_scores
        .loc[ target_scores[ 'model' ] != 'naive_global_median' ]
        .sort_values( 'rmse', ascending = True )
    )

    if len( target_scores_non_naive ) > 0:
        best_row_for_scenarios = target_scores_non_naive.iloc[ 0 ]

    else:
        best_row_for_scenarios = target_scores.iloc[ 0 ]

    best_model_name = str( best_row_for_scenarios[ 'model' ] )
    best_model_obj = model_candidates_t.get( best_model_name )

    if best_model_obj is None:
        best_model_name = 'ridge_phase7_climate'
        best_model_obj = model_candidates_t[ best_model_name ]
        best_row_for_scenarios = target_scores.loc[ target_scores[ 'model' ] == best_model_name ].iloc[ 0 ]

    phase7_model_store[ target ] = best_model_obj
    phase7_feature_store[ target ] = feature_cols_t
    phase7_fill_store[ target ] = fill_values_t

    phase7_best_rows.append( { 
        'target': target,
        'target_window': phase7_target_window_label_map.get( target, 'annual mean' ),
        'best_model_any': str( target_scores.iloc[ 0 ][ 'model' ] ),
        'best_model_for_scenarios': best_model_name,
        'best_rmse_for_scenarios': float( best_row_for_scenarios[ 'rmse' ] ),
        'best_mae_for_scenarios': float( best_row_for_scenarios[ 'mae' ] ),
        'best_r2_for_scenarios': float( best_row_for_scenarios[ 'r2' ] ),
        'best_bias_for_scenarios': float( best_row_for_scenarios[ 'bias' ] ),
        'n_train': int( len( train_t ) ),
        'n_holdout': int( len( holdout_t ) ),
        'n_features': int( len( feature_cols_t ) ),
        'coverage_flag': year_flag_col,
        'target_col': target_col,
    } )

phase7_scores = pd.DataFrame( phase7_scores_rows )
phase7_scores = phase7_scores.sort_values( [ 'target', 'rmse' ], ascending = [ True, True ] ).reset_index( drop = True )

phase7_best = pd.DataFrame( phase7_best_rows )
phase7_best = phase7_best.sort_values( 'best_rmse_for_scenarios' ).reset_index( drop = True )

print( 'phase7 response leaderboard (holdout):' )
display( phase7_scores.round( 4 ) )

print( 'phase7 best model per target:' )
display( phase7_best.round( 4 ).T )


In [ ]:
display( phase7_best.round( 3 ) )

### 7.1b - holdout model success/failure diagnostics
- evaluate the best phase-7 scenario model on holdout station-year rows
- oxygen uses the warm-season target when that table exists
- plot MAE/RMSE, residual spread, observed-vs-predicted parity,
  plus residual bias by baseline regime and signature month

In [ ]:
# 7.1b - holdout diagnostics for phase 7 scenario models

phase7_eval_rows = [ ]

for target in phase7_targets:
    target_col = phase7_target_col_map.get( target )
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = 'float64' ) )
    year_flag_col = phase7_target_year_flag_map.get( target, 'has_model_year' )
    target_window = phase7_target_window_label_map.get( target, 'annual mean' )

    if target_col is None:
        continue

    holdout_t = phase7_station_year_holdout.loc[ phase7_station_year_holdout[ year_flag_col ].fillna( False ).astype( bool ) ].copy( )
    holdout_t = holdout_t.dropna( subset = [ target_col ] ).copy( )

    if model_t is None or len( feature_cols_t ) == 0 or len( holdout_t ) == 0:
        continue

    X_holdout_t = holdout_t[ feature_cols_t ].copy( ).fillna( fill_values_t )
    pred_t = model_t.predict( X_holdout_t )

    eval_t = holdout_t[ [ 'region', 'station', 'year' ] ].copy( )
    eval_t[ 'target' ] = target
    eval_t[ 'target_window' ] = target_window
    eval_t[ 'target_col' ] = target_col
    eval_t[ 'coverage_flag' ] = year_flag_col
    eval_t[ 'target_display' ] = phase7_target_display_name_map.get( target, target )
    eval_t[ 'y_true' ] = holdout_t[ target_col ].values
    eval_t[ 'y_pred' ] = pred_t
    eval_t[ 'residual' ] = eval_t[ 'y_pred' ] - eval_t[ 'y_true' ]
    eval_t[ 'abs_error' ] = eval_t[ 'residual' ].abs( )

    phase7_eval_rows.append( eval_t )

if len( phase7_eval_rows ) == 0:
    print( 'phase7_eval_rows is empty; no diagnostics generated' )

else:
    phase7_eval_long = pd.concat( phase7_eval_rows, ignore_index = True )

    phase7_station_meta_cols = [ 
        'region',
        'station',
        'cluster',
        'cluster_name',
        'cluster_label',
        'warm_peak_month',
        'oxygen_nadir_month',
        'warm_len_months',
        'warm_intensity_temp',
        'warm_oxygen_min',
    ]
    phase7_station_meta_cols = [ col for col in phase7_station_meta_cols if col in station_baseline.columns ]

    if len( phase7_station_meta_cols ) > 0:
        phase7_station_meta = station_baseline[ phase7_station_meta_cols ].drop_duplicates( subset = [ 'region', 'station' ] ).copy( )
        phase7_eval_long = phase7_eval_long.merge( phase7_station_meta, on = [ 'region', 'station' ], how = 'left' )


    def format_phase7_cluster_display( row ):
        if 'cluster' not in row.index or pd.isna( row[ 'cluster' ] ):
            return np.nan

        cluster_id = int( row[ 'cluster' ] )
        cluster_name = row.get( 'cluster_name', np.nan )

        if pd.notna( cluster_name ):
            return f'C{ cluster_id }: { cluster_name }'

        return f'C{ cluster_id }'


    def phase7_signature_month( row ):
        oxygen_month = row.get( 'oxygen_nadir_month', np.nan )
        warm_month = row.get( 'warm_peak_month', np.nan )

        if row[ 'target' ] in phase7_oxygen_targets and pd.notna( oxygen_month ):
            return int( oxygen_month )

        if pd.notna( warm_month ):
            return int( warm_month )

        return np.nan


    phase7_eval_long[ 'cluster_display' ] = phase7_eval_long.apply( format_phase7_cluster_display, axis = 1 )
    phase7_eval_long[ 'signature_month' ] = phase7_eval_long.apply( phase7_signature_month, axis = 1 )

    cluster_display_order = [ ]
    if 'cluster' in station_baseline.columns:
        cluster_display_order_frame = station_baseline[ [ 'cluster', 'cluster_name' ] ].dropna( subset = [ 'cluster' ] ).drop_duplicates( ).copy( )
        cluster_display_order_frame[ 'cluster' ] = cluster_display_order_frame[ 'cluster' ].astype( int )
        cluster_display_order_frame = cluster_display_order_frame.sort_values( 'cluster' )
        cluster_display_order = [ 
            f'C{ int( row.cluster ) }: { row.cluster_name }' if pd.notna( row.cluster_name ) else f'C{ int( row.cluster ) }'
            for row in cluster_display_order_frame.itertuples( index = False )
        ]

    month_label_map = { 
        1: 'Jan',
        2: 'Feb',
        3: 'Mar',
        4: 'Apr',
        5: 'May',
        6: 'Jun',
        7: 'Jul',
        8: 'Aug',
        9: 'Sep',
        10: 'Oct',
        11: 'Nov',
        12: 'Dec',
    }
    phase7_eval_long[ 'signature_month_label' ] = phase7_eval_long[ 'signature_month' ].map( month_label_map )

    phase7_eval_summary = ( 
        phase7_eval_long
        .groupby( [ 'target', 'target_window' ], as_index = False )
        .agg( 
            n_holdout = ( 'y_true', 'size' ),
            mae = ( 'abs_error', 'mean' ),
            rmse = ( 'residual', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
            r2 = ( 'y_true', lambda values: np.nan ),
            bias = ( 'residual', 'mean' ),
            median_abs_error = ( 'abs_error', 'median' ),
        )
    )
    phase7_eval_summary[ 'target_display' ] = phase7_eval_summary[ 'target' ].map( phase7_target_display_name_map ).fillna( phase7_eval_summary[ 'target' ] )

    for idx, row in phase7_eval_summary.iterrows( ):
        t = row[ 'target' ]
        sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == t ]
        phase7_eval_summary.loc[ idx, 'r2' ] = float( r2_score( sub[ 'y_true' ], sub[ 'y_pred' ] ) )

    phase7_eval_summary = phase7_eval_summary.sort_values( 'rmse' ).reset_index( drop = True )

    print( 'phase7 holdout diagnostics summary:' )
    display( phase7_eval_summary.round( 4 ) )

    score_plot_df = phase7_eval_summary.melt( 
        id_vars = [ 'target', 'target_window', 'target_display' ],
        value_vars = [ 'mae', 'rmse', 'median_abs_error' ],
        var_name = 'metric',
        value_name = 'value',
    )
    score_plot_df[ 'target_label' ] = score_plot_df[ 'target_display' ] + ' (' + score_plot_df[ 'target_window' ] + ')'
    phase7_eval_long[ 'target_label' ] = phase7_eval_long[ 'target_display' ] + ' (' + phase7_eval_long[ 'target_window' ] + ')'

    fig, axes = plt.subplots( 1, 2, figsize = ( 18, 5 ) )

    sns.barplot( 
        data = score_plot_df,
        x = 'target_label',
        y = 'value',
        hue = 'metric',
        ax = axes[ 0 ],
    )
    axes[ 0 ].set_title( 'Phase 7 Holdout Error Metrics by Target' )
    axes[ 0 ].set_xlabel( 'Target' )
    axes[ 0 ].set_ylabel( 'Error ( target units )' )
    axes[ 0 ].tick_params( axis = 'x', rotation = 20 )

    sns.boxplot( 
        data = phase7_eval_long,
        x = 'target_label',
        y = 'residual',
        ax = axes[ 1 ],
    )
    axes[ 1 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
    axes[ 1 ].set_title( 'Phase 7 Holdout Residual Spread ( Pred - Obs )' )
    axes[ 1 ].set_xlabel( 'Target' )
    axes[ 1 ].set_ylabel( 'Residual ( target units )' )
    axes[ 1 ].tick_params( axis = 'x', rotation = 20 )

    plt.tight_layout( )
    plt.show( )

    phase7_bias_by_cluster = ( 
        phase7_eval_long
        .dropna( subset = [ 'cluster_display' ] )
        .groupby( [ 'target', 'target_window', 'target_display', 'cluster_display' ], as_index = False )
        .agg( 
            n_holdout = ( 'y_true', 'size' ),
            mae = ( 'abs_error', 'mean' ),
            rmse = ( 'residual', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
            bias = ( 'residual', 'mean' ),
        )
        .sort_values( [ 'target', 'cluster_display' ] )
        .reset_index( drop = True )
    )

    if len( phase7_bias_by_cluster ) > 0 and len( cluster_display_order ) > 0:
        phase7_target_order = phase7_eval_summary[ [ 'target', 'target_window', 'target_display' ] ].drop_duplicates( ).copy( )
        phase7_bias_by_cluster = phase7_target_order.merge( 
            pd.DataFrame( { 'cluster_display': cluster_display_order } ),
            how = 'cross',
        ).merge( 
            phase7_bias_by_cluster,
            on = [ 'target', 'target_window', 'target_display', 'cluster_display' ],
            how = 'left',
        )
        phase7_bias_by_cluster[ 'n_holdout' ] = phase7_bias_by_cluster[ 'n_holdout' ].fillna( 0 ).astype( int )
        phase7_bias_by_cluster = phase7_bias_by_cluster.sort_values( [ 'target', 'cluster_display' ] ).reset_index( drop = True )

    phase7_bias_by_signature_month = ( 
        phase7_eval_long
        .dropna( subset = [ 'signature_month' ] )
        .groupby( [ 'target', 'target_window', 'target_display', 'signature_month', 'signature_month_label' ], as_index = False )
        .agg( 
            n_holdout = ( 'y_true', 'size' ),
            mae = ( 'abs_error', 'mean' ),
            rmse = ( 'residual', lambda values: float( np.sqrt( np.mean( np.square( values ) ) ) ) ),
            bias = ( 'residual', 'mean' ),
        )
        .sort_values( [ 'target', 'signature_month' ] )
        .reset_index( drop = True )
    )

    print( 'phase7 residual bias by baseline regime:' )
    display( phase7_bias_by_cluster.round( 4 ) )

    print( 'phase7 residual bias by baseline signature month:' )
    display( phase7_bias_by_signature_month.round( 4 ) )

    phase7_cluster_heat = pd.DataFrame( )
    if len( phase7_bias_by_cluster ) > 0:
        phase7_bias_by_cluster[ 'target_label' ] = phase7_bias_by_cluster[ 'target_display' ] + ' (' + phase7_bias_by_cluster[ 'target_window' ] + ')'
        phase7_cluster_heat = phase7_bias_by_cluster.pivot( index = 'target_label', columns = 'cluster_display', values = 'bias' )
        if len( cluster_display_order ) > 0:
            phase7_cluster_heat = phase7_cluster_heat.reindex( columns = [ col for col in cluster_display_order if col in phase7_cluster_heat.columns ] )

    phase7_month_heat = pd.DataFrame( )
    if len( phase7_bias_by_signature_month ) > 0:
        phase7_bias_by_signature_month[ 'target_label' ] = phase7_bias_by_signature_month[ 'target_display' ] + ' (' + phase7_bias_by_signature_month[ 'target_window' ] + ')'
        phase7_month_heat = phase7_bias_by_signature_month.pivot( index = 'target_label', columns = 'signature_month_label', values = 'bias' )
        month_order = [ month_label_map[ month ] for month in sorted( month_label_map.keys( ) ) ]
        phase7_month_heat = phase7_month_heat.reindex( columns = [ col for col in month_order if col in phase7_month_heat.columns ] )

    if len( phase7_cluster_heat ) > 0 or len( phase7_month_heat ) > 0:
        fig, axes = plt.subplots( 1, 2, figsize = ( 20, 6 ) )

        if len( phase7_cluster_heat ) > 0:
            sns.heatmap( 
                phase7_cluster_heat,
                annot = True,
                fmt = '.3f',
                cmap = 'coolwarm',
                center = 0,
                ax = axes[ 0 ],
                cbar_kws = { 'label': 'Residual Bias ( Pred - Obs )' },
            )
            axes[ 0 ].set_title( 'Phase 7 Residual Bias by Baseline Regime' )
            axes[ 0 ].set_xlabel( 'Baseline Regime' )
            axes[ 0 ].set_ylabel( 'Target' )

        else:
            axes[ 0 ].axis( 'off' )

        if len( phase7_month_heat ) > 0:
            sns.heatmap( 
                phase7_month_heat,
                annot = True,
                fmt = '.3f',
                cmap = 'coolwarm',
                center = 0,
                ax = axes[ 1 ],
                cbar_kws = { 'label': 'Residual Bias ( Pred - Obs )' },
            )
            axes[ 1 ].set_title( 'Phase 7 Residual Bias by Baseline Signature Month' )
            axes[ 1 ].set_xlabel( 'Signature Month' )
            axes[ 1 ].set_ylabel( 'Target' )

        else:
            axes[ 1 ].axis( 'off' )

        plt.tight_layout( )
        plt.show( )

    targets_plot = phase7_eval_summary[ 'target' ].tolist( )
    n_targets = len( targets_plot )
    n_cols = min( 3, n_targets )
    n_rows = int( np.ceil( n_targets / n_cols ) )
    fig, axes = plt.subplots( n_rows, n_cols, figsize = ( 6 * n_cols, 5 * n_rows ) )
    flat_axes = np.atleast_1d( axes ).ravel( )

    for ax, t in zip( flat_axes, targets_plot ):
        sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == t ]
        target_window = str( sub[ 'target_window' ].iloc[ 0 ] )
        target_display = str( sub[ 'target_display' ].iloc[ 0 ] )
        sns.scatterplot( data = sub, x = 'y_pred', y = 'y_true', s = 35, alpha = 0.65, ax = ax )
        lo = float( min( sub[ 'y_true' ].min( ), sub[ 'y_pred' ].min( ) ) )
        hi = float( max( sub[ 'y_true' ].max( ), sub[ 'y_pred' ].max( ) ) )
        ax.plot( [ lo, hi ], [ lo, hi ], color = 'black', linestyle = '--', linewidth = 1.0 )
        ax.set_title( f'{ target_display }: Predicted vs Observed ({ target_window })' )
        ax.set_xlabel( 'Predicted target value' )
        ax.set_ylabel( 'Observed target value' )

    for ax in flat_axes[ n_targets: ]:
        ax.axis( 'off' )

    plt.tight_layout( )
    plt.show( )


### 7.1c - confusion-matrix style bins for regression holdout
- binned observed vs predicted target values (row-normalized)
- salinity and pH stay annual here
- oxygen now includes warm-season mean plus summer stress targets when those tables exist


In [ ]:
# 7.1c - build per-target confusion-style matrices + a reusable media plot helper

phase7_cm_store = { }
phase7_bin_legend_store = { }
phase7_targets_present = set( phase7_eval_long[ 'target' ].dropna( ).unique( ).tolist( ) )
targets_cm = [ target for target in phase7_targets if target in phase7_targets_present ]

for target in targets_cm:
    sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == target ].copy( )

    if len( sub ) < 20 or sub[ 'y_true' ].nunique( ) < 4:
        continue

    n_bins = min( 6, int( sub[ 'y_true' ].nunique( ) ) )

    try:
        _, bins = pd.qcut( sub[ 'y_true' ], q = n_bins, retbins = True, duplicates = 'drop' )

    except ValueError:
        continue

    bins = np.unique( bins ) 
    if len( bins ) < 3:
        continue

    labels = [ f'B{ i + 1 }' for i in range( len( bins ) - 1 ) ]
    range_lines = [ ]
    for i, label in enumerate( labels ):
        left_edge = bins[ i ]
        right_edge = bins[ i + 1 ]
        range_lines.append( f'{ label }: { left_edge:.3f} to { right_edge:.3f}' )

    obs_bin = pd.cut( sub[ 'y_true' ], bins = bins, labels = labels, include_lowest = True )
    pred_bin = pd.cut( sub[ 'y_pred' ], bins = bins, labels = labels, include_lowest = True )

    cm = pd.crosstab( 
        obs_bin,
        pred_bin,
        rownames = [ 'Observed bin' ],
        colnames = [ 'Predicted bin' ],
        normalize = 'index',
    )
    cm = cm.reindex( index = labels, columns = labels, fill_value = 0.0 )

    phase7_cm_store[ target ] = cm
    phase7_bin_legend_store[ target ] = range_lines

print( f'phase7 media-pair targets available: { [ target for target in phase7_targets if target in phase7_targets_present ] }' )
print( f'phase7 confusion-style targets available: { list( phase7_cm_store.keys( ) ) }' )


# a helper function to plot the media pair for a given target, including the confusion-style matrix if available


def plot_phase7_target_media_pair( target ):

    if target not in phase7_cm_store:
        print( f'No confusion-style matrix available for { target }.' )
        return

    sub = phase7_eval_long.loc[ phase7_eval_long[ 'target' ] == target ].copy( )
    if len( sub ) == 0:
        print( f'No evaluation rows available for { target }.' )
        return

    target_window = str( sub[ 'target_window' ].iloc[ 0 ] )
    target_display = str( sub[ 'target_display' ].iloc[ 0 ] )
    fig, axes = plt.subplots( 1, 2, figsize = ( 16.5, 6.6 ) )
    fig.suptitle( 
        f'{ target_display } Holdout Diagnostics ({ target_window })',
        fontsize = 24,
        fontweight = 'bold',
        y = 0.98,
    )

    cm = phase7_cm_store[ target ]
    legend_cols = int( np.ceil( len( phase7_bin_legend_store[ target ] ) / 2 ) )
    legend_rows = [ 
        '      '.join( phase7_bin_legend_store[ target ][ i:i + legend_cols ] )
        for i in range( 0, len( phase7_bin_legend_store[ target ] ), legend_cols )
    ]
    legend_text = chr( 10 ).join( legend_rows )
    sns.heatmap( 
        cm,
        annot = True,
        fmt = '.2f',
        cmap = 'Blues',
        vmin = 0,
        vmax = 1,
        cbar = False,
        ax = axes[ 0 ],
    )
    axes[ 0 ].set_title( 'Binned Observed vs Predicted' )
    axes[ 0 ].set_xlabel( 'Predicted bin' )
    axes[ 0 ].set_ylabel( 'Observed bin' )

    sns.scatterplot( data = sub, x = 'y_pred', y = 'y_true', s = 35, alpha = 0.65, ax = axes[ 1 ] )
    lo = float( min( sub[ 'y_true' ].min( ), sub[ 'y_pred' ].min( ) ) )
    hi = float( max( sub[ 'y_true' ].max( ), sub[ 'y_pred' ].max( ) ) )
    axes[ 1 ].plot( [ lo, hi ], [ lo, hi ], color = 'black', linestyle = '--', linewidth = 1.0 )
    axes[ 1 ].set_title( 'Predicted vs Observed Scatter' )
    axes[ 1 ].set_xlabel( 'Predicted target value' )
    axes[ 1 ].set_ylabel( 'Observed target value' )

    fig.text( 
        0.5,
        0.08,
        legend_text,
        va = 'top',
        ha = 'center',
        fontsize = 10.5,
        fontweight = 'semibold',
        bbox = { 'boxstyle': 'round,pad=0.5', 'facecolor': 'white', 'edgecolor': '#bbbbbb', 'alpha': 0.97 },
    )

    fig.subplots_adjust( top = 0.84, bottom = 0.24, wspace = 0.25 )
    plt.show( )


### 7.1d - media-ready per-target diagnostics
- one target per cell
- left: confusion-style observed vs predicted bins
- right: predicted vs observed parity scatter


#### Delta Salinity


In [ ]:
plot_phase7_target_media_pair( 'delta_salinity' )


#### Delta Oxygen


In [ ]:
plot_phase7_target_media_pair( 'delta_oxygen' )


#### Oxygen Warm Minimum


In [ ]:
plot_phase7_target_media_pair( 'oxygen_warm_min' )


#### Oxygen Warm P10


In [ ]:
plot_phase7_target_media_pair( 'oxygen_warm_p10' )


#### Oxygen Warm Hypoxic Share


In [ ]:
plot_phase7_target_media_pair( 'oxygen_warm_hypoxic_day_share' )


#### Oxygen Warm Below-Optimal Share


In [ ]:
plot_phase7_target_media_pair( 'oxygen_warm_below_optimal_day_share' )


#### Delta pH


In [ ]:
plot_phase7_target_media_pair( 'delta_ph' )


### 7.2 - climate-change benchmark scenarios
- scenario examples: `+1.5C` with `precip x0.9 / x1.0 / x1.1`
- output: projected target deltas overall and by region
- oxygen follows the warm-season response table when that model exists

In [ ]:
# 7.2 - scenario projections from phase 7 response models

# estimate a simple phase6 air->water transfer coefficient (per +1C air shift)
phase7_phase6_air_cols = [ col for col in phase7_p6_feature_cols if col.startswith( 'air_temp' ) ]

if len( phase7_p6_feature_cols ) > 0 and len( phase7_phase6_air_cols ) > 0 and len( phase6_holdout_model ) > 0:
    X_p6_holdout_base = phase6_holdout_model[ phase7_p6_feature_cols ].copy( )
    X_p6_holdout_base = X_p6_holdout_base.fillna( phase7_p6_fill_values )

    X_p6_holdout_warm = X_p6_holdout_base.copy( )
    for col in phase7_phase6_air_cols:
        X_p6_holdout_warm[ col ] = X_p6_holdout_warm[ col ] + 1.0

    pred_base = phase6_selected_model.predict( X_p6_holdout_base )
    pred_warm = phase6_selected_model.predict( X_p6_holdout_warm )

    phase7_water_temp_transfer_per_c = float( np.mean( pred_warm - pred_base ) )

else:
    phase7_water_temp_transfer_per_c = 1.0

print( f'phase7 inferred phase6 transfer: +1C air -> { phase7_water_temp_transfer_per_c:.4f}C in predicted delta_water_temp' )

phase7_scenarios = pd.DataFrame( [ 
    { 'scenario': 'baseline_ref', 'air_temp_add_c': 0.0, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p5C_precip_minus10pct', 'air_temp_add_c': 1.5, 'precip_mult': 0.90 },
    { 'scenario': 'plus1p5C_precip_same', 'air_temp_add_c': 1.5, 'precip_mult': 1.00 },
    { 'scenario': 'plus1p5C_precip_plus10pct', 'air_temp_add_c': 1.5, 'precip_mult': 1.10 },
] )

projection_rows = [ ]
projection_region_rows = [ ]
phase7_projected_targets = [ ]
phase7_skipped_targets = [ ]

projection_base = phase7_station_year.copy( )

for target in phase7_targets:
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = 'float64' ) )

    if model_t is None or len( feature_cols_t ) == 0:
        phase7_skipped_targets.append( target )
        continue

    phase7_projected_targets.append( target )

    X_base = projection_base[ feature_cols_t ].copy( ).fillna( fill_values_t )
    target_projection_df = projection_base[ [ 'region', 'station', 'year' ] ].copy( )
    target_window = phase7_target_window_label_map.get( target, 'annual mean' )
    target_display = phase7_target_display_name_map.get( target, target )

    for _, scen in phase7_scenarios.iterrows( ):
        X_scen = X_base.copy( )

        for col in [ 'air_temp_mean', 'air_temp_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] + float( scen[ 'air_temp_add_c' ] )

        for col in [ 'precip_mean', 'precip_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] * float( scen[ 'precip_mult' ] )

        for col in [ 'delta_water_temp_pred_p6_mean', 'delta_water_temp_pred_p6_warm_mean' ]:
            if col in X_scen.columns:
                X_scen[ col ] = X_scen[ col ] + float( scen[ 'air_temp_add_c' ] ) * phase7_water_temp_transfer_per_c

        pred_scen = model_t.predict( X_scen )
        scen_name = str( scen[ 'scenario' ] )

        target_projection_df[ scen_name ] = pred_scen

        projection_rows.append( { 
            'target': target,
            'target_display': target_display,
            'target_window': target_window,
            'scenario': scen_name,
            'mean_pred': float( np.mean( pred_scen ) ),
            'median_pred': float( np.median( pred_scen ) ),
            'n_station_years': int( len( pred_scen ) ),
        } )

        region_tmp = target_projection_df.groupby( 'region', as_index = False )[ scen_name ].mean( )
        for _, r in region_tmp.iterrows( ):
            projection_region_rows.append( { 
                'target': target,
                'target_display': target_display,
                'target_window': target_window,
                'scenario': scen_name,
                'region': r[ 'region' ],
                'mean_pred': float( r[ scen_name ] ),
            } )

phase7_projection_summary = pd.DataFrame( projection_rows )

baseline_lookup = ( 
    phase7_projection_summary
    .loc[ phase7_projection_summary[ 'scenario' ] == 'baseline_ref', [ 'target', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)

phase7_projection_summary = phase7_projection_summary.merge( baseline_lookup, on = 'target', how = 'left' )
phase7_projection_summary[ 'delta_vs_baseline' ] = phase7_projection_summary[ 'mean_pred' ] - phase7_projection_summary[ 'baseline_mean_pred' ]
phase7_projection_summary = phase7_projection_summary.sort_values( [ 'target', 'scenario' ] ).reset_index( drop = True )

phase7_projection_by_region = pd.DataFrame( projection_region_rows )
baseline_region_lookup = ( 
    phase7_projection_by_region
    .loc[ phase7_projection_by_region[ 'scenario' ] == 'baseline_ref', [ 'target', 'region', 'mean_pred' ] ]
    .rename( columns = { 'mean_pred': 'baseline_mean_pred' } )
)
phase7_projection_by_region = phase7_projection_by_region.merge( baseline_region_lookup, on = [ 'target', 'region' ], how = 'left' )
phase7_projection_by_region[ 'delta_vs_baseline' ] = phase7_projection_by_region[ 'mean_pred' ] - phase7_projection_by_region[ 'baseline_mean_pred' ]
phase7_projection_by_region = phase7_projection_by_region.sort_values( [ 'target', 'region', 'scenario' ] ).reset_index( drop = True )

print( 'phase7 climate benchmark summary (all station-years):' )
display( phase7_projection_summary.round( 4 ) )

print( 'phase7 climate benchmark summary by region:' )
display( phase7_projection_by_region.round( 4 ) )

print( f'phase7 projected targets: { sorted( set( phase7_projected_targets ) ) }' )
if len( phase7_skipped_targets ) > 0:
    print( f'phase7 skipped targets (no scenario model available): { sorted( set( phase7_skipped_targets ) ) }' )

scenario_delta_plot = phase7_projection_summary.loc[ phase7_projection_summary[ 'scenario' ] != 'baseline_ref' ].copy( )
scenario_delta_plot[ 'target_label' ] = scenario_delta_plot[ 'target_display' ] + ' (' + scenario_delta_plot[ 'target_window' ] + ')'

if len( scenario_delta_plot ) > 0:
    fig, axes = plt.subplots( 1, 2, figsize = ( 18, 6 ) )

    sns.barplot( 
        data = scenario_delta_plot,
        x = 'target_label',
        y = 'delta_vs_baseline',
        hue = 'scenario',
        ax = axes[ 0 ],
    )
    axes[ 0 ].axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
    axes[ 0 ].set_title( 'Phase 7 Scenario Response by Target' )
    axes[ 0 ].set_xlabel( 'Target' )
    axes[ 0 ].set_ylabel( 'Delta vs Baseline Prediction' )
    axes[ 0 ].tick_params( axis = 'x', rotation = 25 )

    region_focus = phase7_projection_by_region.loc[ 
        phase7_projection_by_region[ 'scenario' ] == 'plus1p5C_precip_same'
    ].copy( )

    if len( region_focus ) > 0:
        region_focus[ 'target_label' ] = region_focus[ 'target_display' ] + ' (' + region_focus[ 'target_window' ] + ')'
        region_heat = region_focus.pivot( index = 'region', columns = 'target_label', values = 'delta_vs_baseline' )
        sns.heatmap( 
            region_heat,
            annot = True,
            fmt = '.3f',
            cmap = 'coolwarm',
            center = 0,
            ax = axes[ 1 ],
            cbar_kws = { 'label': 'Delta vs Baseline' },
        )
        axes[ 1 ].set_title( 'Regional Response Heatmap ( +1.5C, precip same )' )
        axes[ 1 ].set_xlabel( 'Target' )
        axes[ 1 ].set_ylabel( 'Region' )

    else:
        axes[ 1 ].axis( 'off' )

    plt.tight_layout( )
    plt.show( )

oxygen_region_plot = phase7_projection_by_region.loc[ 
    ( phase7_projection_by_region[ 'target' ].isin( phase7_oxygen_targets ) )
    & ( phase7_projection_by_region[ 'scenario' ] != 'baseline_ref' )
].copy( )

if len( oxygen_region_plot ) > 0:
    oxygen_targets_present = [ target for target in phase7_oxygen_targets if target in oxygen_region_plot[ 'target' ].unique( ) ]
    n_targets = len( oxygen_targets_present )
    n_cols = min( 3, n_targets )
    n_rows = int( np.ceil( n_targets / n_cols ) )
    fig, axes = plt.subplots( n_rows, n_cols, figsize = ( 6 * n_cols, 5 * n_rows ) )
    flat_axes = np.atleast_1d( axes ).ravel( )

    for ax, target in zip( flat_axes, oxygen_targets_present ):
        sub = oxygen_region_plot.loc[ oxygen_region_plot[ 'target' ] == target ].copy( )
        sns.barplot( 
            data = sub,
            x = 'region',
            y = 'delta_vs_baseline',
            hue = 'scenario',
            ax = ax,
        )
        ax.axhline( 0, color = 'black', linewidth = 1.0, linestyle = '--' )
        ax.set_title( f'{ phase7_target_display_name_map.get( target, target ) } Scenario Shift by Region' )
        ax.set_xlabel( 'Region' )
        ax.set_ylabel( 'Delta vs Baseline' )
        ax.tick_params( axis = 'x', rotation = 35 )

    for ax in flat_axes[ n_targets: ]:
        ax.axis( 'off' )

    plt.tight_layout( )
    plt.show( )


### 7.2b - interpretation guide for climate benchmark tables
- use `delta_vs_baseline` as the scenario signal.
- positive `delta_vs_baseline` means the target mean delta is projected higher than baseline.
- negative `delta_vs_baseline` means projected suppression relative to baseline.
- this is still a coarse response emulator.
- salinity and pH stay annual here.
- oxygen uses the warm-season table when that target exists.

### 7.3 - regime character spaghetti plots, 1995 onward
- monthly annual-cycle spaghetti by baseline water regime
- one row per topic, one column per regime
- each colored line is a 5-year centered rolling annual-cycle; black line is the 1995+ mean

In [ ]:
# 7.3 - monthly annual-cycle spaghetti plots by baseline regime, 1995 onward


turbo_trimmed = LinearSegmentedColormap.from_list( 
    'turbo_trimmed',
    plt.cm.turbo( np.linspace( 0.10, 0.90, 256 ) ),
)

regime_lookup = station_baseline[ [ 'region', 'station', 'cluster' ] ].dropna( subset = [ 'cluster' ] ).drop_duplicates( ).copy( )
regime_lookup[ 'cluster' ] = pd.to_numeric( regime_lookup[ 'cluster' ], errors = 'coerce' )
regime_lookup = regime_lookup.dropna( subset = [ 'cluster' ] ).copy( )
regime_lookup[ 'cluster' ] = regime_lookup[ 'cluster' ].astype( int )

cluster_label_col = 'cluster_label' if 'cluster_label' in station_baseline.columns else ( 'cluster_name' if 'cluster_name' in station_baseline.columns else None )
if cluster_label_col is not None:
    regime_labels = station_baseline[ [ 'cluster', cluster_label_col ] ].dropna( subset = [ 'cluster' ] ).drop_duplicates( ).copy( )
    regime_labels[ 'cluster' ] = pd.to_numeric( regime_labels[ 'cluster' ], errors = 'coerce' )
    regime_labels = regime_labels.dropna( subset = [ 'cluster' ] ).copy( )
    regime_labels[ 'cluster' ] = regime_labels[ 'cluster' ].astype( int )
    regime_lookup = regime_lookup.merge( regime_labels, on = 'cluster', how = 'left' )

else:
    regime_lookup[ 'cluster_label' ] = np.nan
    cluster_label_col = 'cluster_label'

regime_lookup[ 'cluster_display' ] = regime_lookup.apply( 
    lambda row: f"C{ int( row[ 'cluster' ] ) }: { row[ cluster_label_col ] }" if pd.notna( row.get( cluster_label_col, np.nan ) ) else f"C{ int( row[ 'cluster' ] ) }",
    axis = 1,
)

cluster_order = sorted( regime_lookup[ 'cluster' ].dropna( ).unique( ).tolist( ) )
cluster_display_order = [ 
    regime_lookup.loc[ regime_lookup[ 'cluster' ] == cluster_id, 'cluster_display' ].dropna( ).iloc[ 0 ]
    for cluster_id in cluster_order
]
cluster_station_counts = regime_lookup.groupby( [ 'cluster', 'cluster_display' ] ).size( ).reset_index( name = 'n_stations' )
cluster_station_counts = cluster_station_counts.sort_values( 'cluster' ).reset_index( drop = True )

air_cycle_source = daily_air_final.copy( )
water_cycle_source = daily_water_final.copy( )
air_cycle_source[ 'date' ] = pd.to_datetime( air_cycle_source[ 'date' ], errors = 'coerce' )
water_cycle_source[ 'date' ] = pd.to_datetime( water_cycle_source[ 'date' ], errors = 'coerce' )


def build_regime_monthly_cycle( df, value_col, topic_key ):

    plot_df = df[ [ 'region', 'station', 'date', value_col ] ].copy( )
    plot_df = plot_df.dropna( subset = [ 'date', value_col ] )
    plot_df = plot_df.merge( regime_lookup[ [ 'region', 'station', 'cluster', 'cluster_display' ] ], on = [ 'region', 'station' ], how = 'inner' )
    plot_df[ 'year' ] = plot_df[ 'date' ].dt.year
    plot_df[ 'month' ] = plot_df[ 'date' ].dt.month
    plot_df = plot_df.loc[ plot_df[ 'year' ] >= 1995 ].copy( )

    plot_df = ( 
        plot_df
        .groupby( [ 'cluster', 'cluster_display', 'year', 'month' ], as_index = False )[ value_col ]
        .mean( )
        .rename( columns = { value_col: 'value' } )
    )
    plot_df[ 'topic' ] = topic_key
    return plot_df


air_water_join = air_cycle_source.merge( 
    water_cycle_source[ [ 'region', 'station', 'date', 'water_temp' ] ],
    on = [ 'region', 'station', 'date' ],
    how = 'inner',
)
if 'air_temp' in air_water_join.columns and 'water_temp' in air_water_join.columns:
    air_water_join[ 'air_water_temp_diff' ] = air_water_join[ 'water_temp' ] - air_water_join[ 'air_temp' ]

topic_config = [ 
    { 'topic': 'water_temp', 'label': 'Water Temp ( C )', 'source': water_cycle_source, 'value_col': 'water_temp' },
    { 'topic': 'air_water_temp_diff', 'label': 'Water - Air Temp ( C )', 'source': air_water_join, 'value_col': 'air_water_temp_diff' },
    { 'topic': 'precip', 'label': 'Precip ( mean daily )', 'source': air_cycle_source, 'value_col': 'precip' },
    { 'topic': 'salinity', 'label': 'Salinity', 'source': water_cycle_source, 'value_col': 'salinity' },
    { 'topic': 'oxygen', 'label': 'Dissolved Oxygen ( mg/L )', 'source': water_cycle_source, 'value_col': 'oxygen' },
]

regime_cycle_parts = [ ]
for cfg in topic_config:
    regime_cycle_part = build_regime_monthly_cycle( cfg[ 'source' ], cfg[ 'value_col' ], cfg[ 'topic' ] )
    if len( regime_cycle_part ) > 0:
        regime_cycle_parts.append( regime_cycle_part )

if len( regime_cycle_parts ) == 0:
    print( 'No regime monthly-cycle data available for plotting.' )

else:
    regime_cycle_long = pd.concat( regime_cycle_parts, ignore_index = True )
    rolling_window_years = 5
    topic_label_map = { cfg[ 'topic' ]: cfg[ 'label' ] for cfg in topic_config }
    month_label_map = { 
        1: 'Jan',
        2: 'Feb',
        3: 'Mar',
        4: 'Apr',
        5: 'May',
        6: 'Jun',
        7: 'Jul',
        8: 'Aug',
        9: 'Sep',
        10: 'Oct',
        11: 'Nov',
        12: 'Dec',
    }

    regime_cycle_plot_long = regime_cycle_long.sort_values( [ 'topic', 'cluster', 'month', 'year' ] ).copy( )
    regime_cycle_plot_long[ 'value_plot' ] = ( 
        regime_cycle_plot_long
        .groupby( [ 'topic', 'cluster', 'cluster_display', 'month' ] )[ 'value' ]
        .transform( lambda s: s.rolling( window = rolling_window_years, center = True, min_periods = 3 ).mean( ) )
    )
    regime_cycle_plot_long[ 'value_plot' ] = regime_cycle_plot_long[ 'value_plot' ].fillna( regime_cycle_plot_long[ 'value' ] )

    regime_cycle_mean = ( 
        regime_cycle_plot_long
        .groupby( [ 'topic', 'cluster', 'cluster_display', 'month' ], as_index = False )[ 'value' ]
        .mean( )
        .rename( columns = { 'value': 'value_mean' } )
    )

    topic_y_limits = { }
    for cfg in topic_config:
        topic_key = cfg[ 'topic' ]
        topic_values = regime_cycle_plot_long.loc[ regime_cycle_plot_long[ 'topic' ] == topic_key, 'value_plot' ].dropna( )
        if len( topic_values ) == 0:
            continue

        ymin = float( topic_values.quantile( 0.01 ) )
        ymax = float( topic_values.quantile( 0.99 ) )
        if np.isclose( ymin, ymax ):
            pad = max( 0.5, abs( ymin ) * 0.08 + 0.25 )

        else:
            pad = max( 0.15, ( ymax - ymin ) * 0.08 )

        if topic_key in [ 'precip', 'salinity', 'oxygen' ]:
            ymin = min( ymin, 0.0 )

        topic_y_limits[ topic_key ] = ( ymin - pad, ymax + pad )

    years_present = sorted( regime_cycle_plot_long[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
    year_min = min( years_present )
    year_max = max( years_present )
    year_norm = Normalize( vmin = year_min, vmax = year_max )
    line_cmap = turbo_trimmed

    n_rows = len( topic_config )
    n_cols = len( cluster_order )
    fig, axes = plt.subplots( n_rows, n_cols, figsize = ( 4.25 * n_cols, 2.95 * n_rows ), sharex = True )
    axes = np.atleast_2d( axes )

    for row_idx, cfg in enumerate( topic_config ):
        topic_key = cfg[ 'topic' ]
        y_limits = topic_y_limits.get( topic_key )
        for col_idx, cluster_id in enumerate( cluster_order ):
            ax = axes[ row_idx, col_idx ]
            sub = regime_cycle_plot_long.loc[ 
                ( regime_cycle_plot_long[ 'topic' ] == topic_key )
                & ( regime_cycle_plot_long[ 'cluster' ] == cluster_id )
            ].copy( )

            if len( sub ) == 0:
                ax.axis( 'off' )
                continue

            year_order = sorted( sub[ 'year' ].dropna( ).astype( int ).unique( ).tolist( ) )
            first_year = year_order[ 0 ]
            last_year = year_order[ -1 ]

            for year in year_order:
                year_sub = sub.loc[ sub[ 'year' ] == year ].sort_values( 'month' )
                line_alpha = 1.0 if year in [ first_year, last_year ] else 0.4
                line_width = 2.0 if year in [ first_year, last_year ] else 0.75
                ax.plot( 
                    year_sub[ 'month' ],
                    year_sub[ 'value_plot' ],
                    color = line_cmap( year_norm( year ) ),
                    alpha = line_alpha,
                    linewidth = line_width,
                    zorder = 2,
                )

            #mean_sub = regime_cycle_mean.loc[ 
            #    ( regime_cycle_mean[ 'topic' ] == topic_key )
            #    & ( regime_cycle_mean[ 'cluster' ] == cluster_id )
            #].sort_values( 'month' )
            #ax.plot( 
            #    mean_sub[ 'month' ],
            #    mean_sub[ 'value_mean' ],
            #    color = 'black',
            #    linewidth = 2.2,
            #    zorder = 3,
            #)

            ax.set_xlim( 1, 12 )
            ax.set_xticks( range( 1, 13 ) )
            ax.set_xticklabels( [ month_label_map[ month ] for month in range( 1, 13 ) ], rotation = 45 )
            if y_limits is not None:
                ax.set_ylim( y_limits[ 0 ], y_limits[ 1 ] )
            ax.grid( alpha = 0.4, linewidth = 0.75 )

            if row_idx == 0:
                station_count = int( cluster_station_counts.loc[ cluster_station_counts[ 'cluster' ] == cluster_id, 'n_stations' ].iloc[ 0 ] )
                cluster_display = cluster_display_order[ col_idx ]
                ax.set_title( f'{ cluster_display }\n{ station_count } stations' )

            if col_idx == 0:
                ax.set_ylabel( topic_label_map[ topic_key ] )

            #if row_idx == n_rows - 1:
            #    ax.set_xlabel( 'Month' )

    scalar_mappable = ScalarMappable( norm = year_norm, cmap = line_cmap )
    scalar_mappable.set_array( [ ] )
    fig.subplots_adjust( left = 0.07, right = 0.90, bottom = 0.07, top = 0.90, wspace = 0.18, hspace = 0.26 )
    cbar_ax = fig.add_axes( [ 0.915, 0.17, 0.012, 0.64 ] )
    colorbar = fig.colorbar( scalar_mappable, cax = cbar_ax )
    #colorbar.set_label( 'Year ( older to newer )' )
    colorbar.set_ticks( [ year for year in [ 1995, 2000, 2005, 2010, 2015, 2020, 2025 ] if year_min <= year <= year_max ] )

    fig.suptitle( 
        'Baseline Water Regime Character: 5-Year Rolling Monthly Annual Cycles, 1995+',
        fontsize = 24,
        fontweight = 'bold'
    )

    plt.show( )

    #print( f'years plotted: { year_min } to { year_max }' )
    #print( f'colored lines = { rolling_window_years }-year centered rolling monthly means within regime' )
    #print( 'black line = 1995+ all-years monthly mean within regime' )
    display( cluster_station_counts )


### 7.4 - flat station map by baseline regime
- land and sea styled separately on a regional map
- station markers colored by baseline water regime
- region labels placed near the average location of stations in each region


In [ ]:
# 7.4 - flat station map by baseline regime

import plotly.graph_objects as go

map_station = station_baseline_display.copy( )
for col in [ 'latitude', 'longitude', 'cluster' ]:
    if col in map_station.columns:
        map_station[ col ] = pd.to_numeric( map_station[ col ], errors = 'coerce' )

label_col = 'cluster_label' if 'cluster_label' in map_station.columns else ( 'cluster_name' if 'cluster_name' in map_station.columns else None )
map_station = map_station.dropna( subset = [ 'latitude', 'longitude', 'cluster' ] ).copy( )
map_station[ 'cluster' ] = map_station[ 'cluster' ].astype( int )
map_station[ 'cluster_display' ] = map_station.apply( 
    lambda row: f"C{ int( row[ 'cluster' ] ) }: { row[ label_col ] }" if label_col is not None and pd.notna( row.get( label_col, np.nan ) ) else f"C{ int( row[ 'cluster' ] ) }",
    axis = 1,
)

cluster_ids = sorted( map_station[ 'cluster' ].dropna( ).unique( ).tolist( ) )
cluster_color_map = { 
    cluster_id: 'rgba' + str( tuple( int( channel * 255 ) for channel in plt.cm.tab10( idx % 10 )[ :3 ] ) + ( 230, ) )
    for idx, cluster_id in enumerate( cluster_ids )
}

fig = go.Figure( )

for cluster_id in cluster_ids:
    sub = map_station.loc[ map_station[ 'cluster' ] == cluster_id ].copy( )
    fig.add_trace( go.Scattergeo( 
        lon = sub[ 'longitude' ],
        lat = sub[ 'latitude' ],
        mode = 'markers',
        name = sub[ 'cluster_display' ].iloc[ 0 ],
        text = sub[ 'station_name' ] if 'station_name' in sub.columns else sub[ 'station' ],
        customdata = np.stack( [ sub[ 'region' ], sub[ 'station' ] ], axis = 1 ),
        hovertemplate = 'Region: %{customdata[0]}<br>Station: %{customdata[1]}<br>Name: %{text}<br>Regime: ' + sub[ 'cluster_display' ].iloc[ 0 ] + '<extra></extra>',
        marker = { 
            'size': 10,
            'color': cluster_color_map[ cluster_id ],
            'line': { 'color': 'white', 'width': 0.7 },
            'opacity': 0.95,
            'symbol': 'circle',
        },
    ) )

region_label_source = map_station.copy( )
if 'region_name' in region_label_source.columns:
    region_label_source[ 'region_label' ] = region_label_source[ 'region_name' ].fillna( region_label_source[ 'region' ] )

else:
    region_label_source[ 'region_label' ] = region_label_source[ 'region' ]

region_label_source[ 'label_key' ] = ( 
    region_label_source[ 'region_label' ]
    .astype( str )
    .str.lower( )
    .str.replace( '&', 'and', regex = False )
    .str.replace( '-', ' ', regex = False )
    .str.replace( r'[^a-z0-9 ]+', '', regex = True )
    .str.replace( r'\s+', ' ', regex = True )
    .str.strip( )
)
region_label_source[ 'label_group' ] = region_label_source[ 'label_key' ]
region_label_source[ 'region_label_display' ] = region_label_source[ 'region_label' ]

chesapeake_mask = region_label_source[ 'label_key' ].str.contains( 'chesapeake bay', na = False )
region_label_source.loc[ chesapeake_mask, 'label_group' ] = 'chesapeake bay combined'
region_label_source.loc[ chesapeake_mask, 'region_label_display' ] = 'Chesapeake Bay'

region_labels = ( 
    region_label_source
    .groupby( [ 'label_group', 'region_label_display' ], as_index = False )
    .agg( 
        latitude = ( 'latitude', 'mean' ),
        longitude = ( 'longitude', 'mean' ),
        n_stations = ( 'station', 'nunique' ),
    )
)

region_nudge = { 
    'mission aransas': ( -2.4, 0.2 ),
    'kachemak bay': ( 2.8, -1.9 ),
    'grand bay': ( 2.9, 1.9 ),
    'weeks bay': ( 2.2, 1.2 ),
    'jacques cousteau': ( 3.1, 1.7 ),
    'delaware': ( 3.3, 0.2 ),
    'chesapeake bay combined': ( 3.7, -1.9 ),
    'north inlet winyah bay': ( 3.8, 0.6 ),
    'ashepoo combahee edisto basin': ( 4.1, 1.5 ),
    'sapelo island': ( 3.7, -0.7 ),
    'wells': ( -0.8, 0.9 ),
}
region_text_position_map = { 
    'mission aransas': 'middle left',
    'kachemak bay': 'bottom right',
    'grand bay': 'top right',
    'weeks bay': 'top right',
    'jacques cousteau': 'top right',
    'delaware': 'middle right',
    'chesapeake bay combined': 'bottom right',
    'north inlet winyah bay': 'top right',
    'ashepoo combahee edisto basin': 'top right',
    'sapelo island': 'middle right',
}
region_labels[ 'label_side' ] = np.where( region_labels[ 'longitude' ] <= -96.0, 'left', 'right' )
region_labels[ 'base_lon_offset' ] = np.where( region_labels[ 'label_side' ] == 'left', -3.0, 2.2 )
region_labels[ 'base_lat_offset' ] = 0.0
region_labels[ 'label_lon' ] = region_labels.apply( lambda row: row[ 'longitude' ] + row[ 'base_lon_offset' ] + region_nudge.get( row[ 'label_group' ], ( 0.0, 0.0 ) )[ 0 ], axis = 1 )
region_labels[ 'label_lat' ] = region_labels.apply( lambda row: row[ 'latitude' ] + row[ 'base_lat_offset' ] + region_nudge.get( row[ 'label_group' ], ( 0.0, 0.0 ) )[ 1 ], axis = 1 )
region_labels[ 'label_text' ] = region_labels[ 'region_label_display' ]
region_labels[ 'text_position' ] = region_labels.apply( lambda row: region_text_position_map.get( row[ 'label_group' ], 'middle left' if row[ 'label_side' ] == 'left' else 'middle right' ), axis = 1 )

leader_lon = [ ]
leader_lat = [ ]
for row in region_labels.itertuples( index = False ):
    leader_lon.extend( [ row.longitude, row.label_lon, None ] )
    leader_lat.extend( [ row.latitude, row.label_lat, None ] )

fig.add_trace( go.Scattergeo( 
    lon = leader_lon,
    lat = leader_lat,
    mode = 'lines',
    hoverinfo = 'skip',
    showlegend = False,
    line = { 'color': 'rgba( 70, 82, 92, 0.55 )', 'width': 1.0 },
) )

fig.add_trace( go.Scattergeo( 
    lon = region_labels[ 'label_lon' ],
    lat = region_labels[ 'label_lat' ],
    mode = 'text',
    text = region_labels[ 'label_text' ],
    textposition = region_labels[ 'text_position' ].tolist( ),
    hoverinfo = 'skip',
    showlegend = False,
    textfont = { 'size': 11, 'color': '#233241' },
) )

fig.update_geos( 
    projection_type = 'equirectangular',
    showland = True,
    landcolor = '#e9dfc9',
    showocean = True,
    oceancolor = '#dcecf7',
    showlakes = True,
    lakecolor = '#dcecf7',
    showcoastlines = True,
    coastlinecolor = '#5f6b73',
    coastlinewidth = 1.0,
    showcountries = True,
    countrycolor = '#9aa5ad',
    countrywidth = 0.6,
    showsubunits = True,
    subunitcolor = '#b8c0c6',
    subunitwidth = 0.4,
    bgcolor = 'white',
    showframe = True,
    framecolor = '#8d9aa3',
    framewidth = 1.0,
    lonaxis = dict( 
        range = [ -170, -60 ],
        showgrid = True,
        gridcolor = 'rgba( 120, 135, 145, 0.20 )',
        gridwidth = 0.5,
    ),
    lataxis = dict( 
        range = [ 15, 66 ],
        showgrid = True,
        gridcolor = 'rgba( 120, 135, 145, 0.20 )',
        gridwidth = 0.5,
    ),
)

fig.update_layout( 
    title = 'Baseline Water Regimes by Station Location',
    width = 1200,
    height = 720,
    margin = { 'l': 20, 'r': 20, 't': 60, 'b': 20 },
    paper_bgcolor = 'white',
    plot_bgcolor = 'white',
    legend = { 
        'title': { 'text': 'Baseline Regime' },
        'orientation': 'v',
        'x': 1.01,
        'y': 0.98,
        'xanchor': 'left',
        'yanchor': 'top',
        'bgcolor': 'rgba( 255, 255, 255, 0.90 )',
        'bordercolor': '#c7cdd1',
        'borderwidth': 1,
    },
    #annotations = [ 
    #    { 
    #        'xref': 'paper',
    #        'yref': 'paper',
    #        'x': 0.0,
    #        'y': 0.0,
    #        'xanchor': 'left',
    #        'yanchor': 'bottom',
    #        'showarrow': False,
    #        'text': f"Stations shown: { map_station[ 'station' ].nunique( ) } | Regions shown: { len( region_labels ) }",
    #        'font': { 'size': 11, 'color': '#44515b' },
    #    },
    #],
)

fig.show( )


### 7.4b - flat station map, dots only
- stripped-down export version with land, sea, and regime-colored station markers only

In [ ]:
# 7.4b - flat station map by baseline regime, dots only

import plotly.graph_objects as go

map_station_simple = station_baseline_display.copy( )
for col in [ 'latitude', 'longitude', 'cluster' ]:
    if col in map_station_simple.columns:
        map_station_simple[ col ] = pd.to_numeric( map_station_simple[ col ], errors = 'coerce' )

label_col = 'cluster_label' if 'cluster_label' in map_station_simple.columns else ( 'cluster_name' if 'cluster_name' in map_station_simple.columns else None )
map_station_simple = map_station_simple.dropna( subset = [ 'latitude', 'longitude', 'cluster' ] ).copy( )
map_station_simple[ 'cluster' ] = map_station_simple[ 'cluster' ].astype( int )
map_station_simple[ 'cluster_display' ] = map_station_simple.apply( 
    lambda row: f"C{ int( row[ 'cluster' ] ) }: { row[ label_col ] }" if label_col is not None and pd.notna( row.get( label_col, np.nan ) ) else f"C{ int( row[ 'cluster' ] ) }",
    axis = 1,
)

cluster_ids = sorted( map_station_simple[ 'cluster' ].dropna( ).unique( ).tolist( ) )
cluster_color_map = { 
    cluster_id: 'rgba' + str( tuple( int( channel * 255 ) for channel in plt.cm.tab10( idx % 10 )[ :3 ] ) + ( 230, ) )
    for idx, cluster_id in enumerate( cluster_ids )
}

fig = go.Figure( )

for cluster_id in cluster_ids:
    sub = map_station_simple.loc[ map_station_simple[ 'cluster' ] == cluster_id ].copy( )
    fig.add_trace( go.Scattergeo( 
        lon = sub[ 'longitude' ],
        lat = sub[ 'latitude' ],
        mode = 'markers',
        name = sub[ 'cluster_display' ].iloc[ 0 ],
        text = sub[ 'station_name' ] if 'station_name' in sub.columns else sub[ 'station' ],
        customdata = np.stack( [ sub[ 'region' ], sub[ 'station' ] ], axis = 1 ),
        hovertemplate = 'Region: %{customdata[0]}<br>Station: %{customdata[1]}<br>Name: %{text}<br>Regime: ' + sub[ 'cluster_display' ].iloc[ 0 ] + '<extra></extra>',
        marker = { 
            'size': 10,
            'color': cluster_color_map[ cluster_id ],
            'line': { 'color': 'white', 'width': 0.7 },
            'opacity': 0.95,
            'symbol': 'circle',
        },
    ) )

fig.update_geos( 
    projection_type = 'equirectangular',
    showland = True,
    landcolor = '#e9dfc9',
    showocean = True,
    oceancolor = '#dcecf7',
    showlakes = True,
    lakecolor = '#dcecf7',
    showcoastlines = True,
    coastlinecolor = '#5f6b73',
    coastlinewidth = 1.0,
    showcountries = True,
    countrycolor = '#9aa5ad',
    countrywidth = 0.6,
    showsubunits = True,
    subunitcolor = '#b8c0c6',
    subunitwidth = 0.4,
    bgcolor = 'white',
    showframe = True,
    framecolor = '#8d9aa3',
    framewidth = 1.0,
    lonaxis = dict( 
        range = [ -170, -60 ],
        showgrid = True,
        gridcolor = 'rgba( 120, 135, 145, 0.20 )',
        gridwidth = 0.5,
    ),
    lataxis = dict( 
        range = [ 15, 66 ],
        showgrid = True,
        gridcolor = 'rgba( 120, 135, 145, 0.20 )',
        gridwidth = 0.5,
    ),
)

fig.update_layout( 
    title = 'Baseline Water Regimes by Station Location',
    width = 1200,
    height = 720,
    margin = { 'l': 20, 'r': 20, 't': 60, 'b': 20 },
    paper_bgcolor = 'white',
    plot_bgcolor = 'white',
    legend = { 
        'title': { 'text': 'Baseline Regime' },
        'orientation': 'v',
        'x': 1.01,
        'y': 0.98,
        'xanchor': 'left',
        'yanchor': 'top',
        'bgcolor': 'rgba( 255, 255, 255, 0.90 )',
        'bordercolor': '#c7cdd1',
        'borderwidth': 1,
    },
)

fig.show( )


## Phase 8 — Nutrient Models
Goal: predict changes in phosphates, nitrates, chlorophyll given water state

# also stage 3 (if time permits)

- 8.1 use the nutrient-inclusive dataset 
  - is okay that coverage is sparser and document that explicitly
- 8.2 same pipeline/process as phase 7, but input features now include water property predictions from earlier
- 8.3 report model skill honestly 
  - nutrient prediction will be noisier; even identifying dominant drivers is a valid result
  - so far, chlorophyll (despite being most obvious to people as blooms) seemed the weakest response?

In [ ]:
# code

## Phase 9 — Forward Projection
Goal: apply response functions to CMIP6 scenarios

- 9.1 obtain CMIP6 projected Δ air temp for your estuary regions under SSP2-4.5 and SSP5-8.5
  - ESGF portal or pre-downscaled products like NASA NEX-GDDP
  - also ote that there's some evidence that CMIP6 has under-estimated climate damage (by overestimating plant growth)
  - and the new estimate thinks it might be about 11% worse than predicted
  - might factor that in (with a *note?)
- 9.2 feed projected Δ air temp into stage 1 model 
  - get projected Δ water temp by decade
- 9.3 Feed projected Δ water temp into stage 2 response functions 
  - get projected Δ salinity, Δ oxygen, Δ pH
- 9.4 Identify projected regime-crossing dates per station under each scenario 
  - *"station X transitions from cool to warm regime between 2045–2060 under SSP5-8.5"* maybe?
- 9.5 propagate and report uncertainty 
  - at minimum, show scenario spread (SSP2 vs SSP5)
  - if time permits, model confidence intervals

In [ ]:
# 9.0 - forward-projection helpers + case-study setup
from pathlib import Path


phase9_case_count = 3
phase9_case_pool_mode = 'station_list'
phase9_case_region_codes = [ 'cbm', 'cbv', 'elk' ]
phase9_case_station_keys = [ 
    { 'region': 'elk', 'station': 'ap' },
    { 'region': 'cbm', 'station': 'ip' },
    { 'region': 'cbv', 'station': 'cb' },
]
phase9_rolling_window_years = 5
phase9_scenario_path_csv = Path( '../data/reference/phase9.scenario.paths.csv' )


def select_phase9_case_studies( case_pool, n_cases = 5 ):
    case_pool = case_pool.copy( )
    case_pool = case_pool.sort_values( [ 'event_date', 'region', 'station' ] ).reset_index( drop = True )

    selected_rows = [ ]
    used_keys = set( )

    for cluster_id in case_pool[ 'cluster' ].dropna( ).astype( int ).sort_values( ).unique( ):
        cluster_slice = case_pool.loc[ case_pool[ 'cluster' ].astype( 'Int64' ) == int( cluster_id ) ].copy( )
        if len( cluster_slice ) == 0:
            continue

        row = cluster_slice.iloc[ 0 ]
        key = ( str( row[ 'region' ] ), str( row[ 'station' ] ) )
        if key in used_keys:
            continue

        selected_rows.append( row )
        used_keys.add( key )

        if len( selected_rows ) >= n_cases:
            break

    if len( selected_rows ) < n_cases:
        for row in case_pool.itertuples( index = False ):
            key = ( str( row.region ), str( row.station ) )
            if key in used_keys:
                continue

            selected_rows.append( pd.Series( row._asdict( ) ) )
            used_keys.add( key )

            if len( selected_rows ) >= n_cases:
                break

    if len( selected_rows ) == 0:
        return case_pool.head( 0 ).copy( )

    out = pd.DataFrame( selected_rows ).reset_index( drop = True )
    out[ 'case_id' ] = [ f'case_{ idx + 1 }' for idx in range( len( out ) ) ]
    return out


def normalize_phase9_station_keys( station_keys ):
    if station_keys is None:
        return None

    rows = [ ]
    for idx, item in enumerate( station_keys ):
        if isinstance( item, dict ):
            region = item.get( 'region' )
            station = item.get( 'station' )

        elif isinstance( item, ( list, tuple ) ) and len( item ) >= 2:
            region, station = item[ 0 ], item[ 1 ]

        else:
            continue

        if pd.isna( region ) or pd.isna( station ):
            continue

        rows.append( { 
            'region': str( region ).strip( ),
            'station': str( station ).strip( ),
            'phase9_case_order': idx,
        } )

    if len( rows ) == 0:
        return None

    return pd.DataFrame( rows ).drop_duplicates( subset = [ 'region', 'station' ] ).reset_index( drop = True )


def build_phase9_case_pool( case_pool_mode, case_station_keys = None ):
    station_meta = station_baseline_display[ [ 'region', 'station', 'region_name', 'station_name', 'cluster', 'cluster_label', 'cluster_name' ] ].drop_duplicates( ).copy( )
    available_keys = ( 
        daily_air_final[ [ 'region', 'station' ] ]
        .drop_duplicates( )
        .merge( daily_water_final[ [ 'region', 'station' ] ].drop_duplicates( ), on = [ 'region', 'station' ], how = 'inner' )
    )
    transition_pool = ( 
        first_event
        .merge( station_meta, on = [ 'region', 'station' ], how = 'left' )
        .merge( flagged_station_keys_final.assign( is_holdout = True ), on = [ 'region', 'station' ], how = 'left' )
        .merge( available_keys.assign( has_history = True ), on = [ 'region', 'station' ], how = 'inner' )
    )
    transition_pool[ 'is_holdout' ] = transition_pool[ 'is_holdout' ].fillna( False )
    transition_pool[ 'is_train' ] = ~transition_pool[ 'is_holdout' ]

    if case_pool_mode == 'holdout_transition':
        case_pool = transition_pool.loc[ transition_pool[ 'is_holdout' ] ].copy( )

    elif case_pool_mode == 'train_transition':
        case_pool = transition_pool.loc[ transition_pool[ 'is_train' ] ].copy( )

    elif case_pool_mode == 'all_transition':
        case_pool = transition_pool.copy( )

    elif case_pool_mode == 'station_list':
        station_keys_frame = normalize_phase9_station_keys( case_station_keys )
        if station_keys_frame is None or len( station_keys_frame ) == 0:
            raise ValueError( 'phase9_case_station_keys must be provided when phase9_case_pool_mode = "station_list".' )

        case_pool = ( 
            station_keys_frame
            .merge( station_meta, on = [ 'region', 'station' ], how = 'left' )
            .merge( first_event, on = [ 'region', 'station' ], how = 'left' )
            .merge( flagged_station_keys_final.assign( is_holdout = True ), on = [ 'region', 'station' ], how = 'left' )
            .merge( available_keys.assign( has_history = True ), on = [ 'region', 'station' ], how = 'inner' )
            .sort_values( [ 'phase9_case_order', 'region', 'station' ] )
            .reset_index( drop = True )
        )
        case_pool[ 'is_holdout' ] = case_pool[ 'is_holdout' ].fillna( False )
        case_pool[ 'is_train' ] = ~case_pool[ 'is_holdout' ]

    else:
        raise ValueError( f'Unknown phase9_case_pool_mode: { case_pool_mode }' )

    return case_pool.sort_values( [ 'event_date', 'region', 'station' ], na_position = 'last' ).reset_index( drop = True )


def build_phase9_demo_paths( start_year = 2026, end_year = 2100 ):
    rows = [ ]
    demo_specs = [ 
        ( 'ssp245_demo', 1.8, 1.00, 1.00, 1.00 ),
        ( 'ssp585_demo', 3.7, 0.97, 1.02, 1.00 ),
    ]

    n_years = max( 1, end_year - start_year )

    for scenario_name, end_air_add_c, end_precip_mult, end_wind_mult, end_solar_mult in demo_specs:
        for year in range( start_year, end_year + 1 ):
            frac = ( year - start_year ) / n_years
            rows.append( { 
                'scenario': scenario_name,
                'year': year,
                'air_temp_add_c': float( end_air_add_c * frac ),
                'precip_mult': float( 1.0 + ( end_precip_mult - 1.0 ) * frac ),
                'wind_speed_mult': float( 1.0 + ( end_wind_mult - 1.0 ) * frac ),
                'solar_mult': float( 1.0 + ( end_solar_mult - 1.0 ) * frac ),
            } )

    return pd.DataFrame( rows )


def add_phase9_driver_rolls( frame ):
    frame = frame.sort_values( [ 'region', 'station', 'scenario', 'date' ] ).copy( )

    for source_col in [ 'air_temp', 'wind_speed', 'precip', 'solar' ]:
        for window in [ 1, 7, 28 ]:
            frame[ f'{ source_col }_r{ window }d' ] = ( 
                frame
                .groupby( [ 'region', 'station', 'scenario' ] )[ source_col ]
                .transform( lambda values: values.shift( 1 ).rolling( window = window, min_periods = 1 ).mean( ) )
            )

    frame[ 'doy' ] = frame[ 'date' ].dt.dayofyear
    frame[ 'doy_sin' ] = np.sin( 2 * np.pi * frame[ 'doy' ] / 365.25 )
    frame[ 'doy_cos' ] = np.cos( 2 * np.pi * frame[ 'doy' ] / 365.25 )

    return frame


def build_phase9_station_year_features( phase9_daily_frame ):
    frame = phase9_daily_frame.copy( )
    frame[ 'date' ] = pd.to_datetime( frame[ 'date' ], errors = 'coerce' )
    frame = frame.dropna( subset = [ 'date' ] )
    frame[ 'year' ] = frame[ 'date' ].dt.year
    frame[ 'month_num' ] = frame[ 'date' ].dt.month
    frame[ 'is_warm_season' ] = frame[ 'month_num' ].between( 6, 9 )

    agg_feature_cols = [ col for col in [ 'delta_water_temp_pred_p6', 'air_temp', 'precip', 'wind_speed', 'solar' ] if col in frame.columns ]
    baseline_daily_feature_cols = [ col for col in [ 'water_temp_baseline', 'salinity_baseline', 'oxygen_baseline', 'ph_baseline', 'depth_baseline' ] if col in frame.columns ]

    annual_agg = { 
        'n_days_annual': ( 'date', 'size' ),
    }
    for col in agg_feature_cols:
        annual_agg[ f'{ col }_mean' ] = ( col, 'mean' )
        annual_agg[ f'{ col }_std' ] = ( col, 'std' )
    for col in baseline_daily_feature_cols:
        annual_agg[ f'{ col }_annual_mean' ] = ( col, 'mean' )

    station_year = ( 
        frame
        .groupby( [ 'region', 'station', 'scenario', 'year' ], as_index = False )
        .agg( **annual_agg )
    )

    warm_daily = frame.loc[ frame[ 'is_warm_season' ] ].copy( )
    warm_agg = { 
        'n_days_warm': ( 'date', 'size' ),
    }
    for col in agg_feature_cols:
        warm_agg[ f'{ col }_warm_mean' ] = ( col, 'mean' )
        warm_agg[ f'{ col }_warm_std' ] = ( col, 'std' )

    warm_year = ( 
        warm_daily
        .groupby( [ 'region', 'station', 'scenario', 'year' ], as_index = False )
        .agg( **warm_agg )
    )
    station_year = station_year.merge( warm_year, on = [ 'region', 'station', 'scenario', 'year' ], how = 'left' )

    if 'phase7_station_context' in globals( ):
        station_year = station_year.merge( phase7_station_context, on = [ 'region', 'station' ], how = 'left' )

    station_year[ 'has_model_year' ] = True
    station_year[ 'has_warm_oxygen_year' ] = True

    return station_year


phase9_regime_feature_cols = list( centroids_z.columns )


def phase9_classify_regime_row( row, regime_feature_cols = None, min_features_required = 3 ):
    if regime_feature_cols is None:
        regime_feature_cols = phase9_regime_feature_cols

    available = [ 
        col
        for col in regime_feature_cols
        if ( 
            col in row.index
            and col in feature_mean.index
            and col in feature_scale.index
            and col in centroids_z.columns
            and pd.notna( row.get( col, np.nan ) )
        )
    ]
    if len( available ) < min_features_required:
        return pd.Series( { 
            'regime_roll5y': np.nan,
            'regime_dist_best': np.nan,
            'regime_margin': np.nan,
            'regime_n_features_used': len( available ),
        } )

    values = pd.to_numeric( pd.Series( { col: row.get( col, np.nan ) for col in available } ), errors = 'coerce' )
    means = pd.to_numeric( feature_mean[ available ], errors = 'coerce' )
    scales = pd.to_numeric( feature_scale[ available ], errors = 'coerce' ).replace( 0, np.nan )
    z_values = ( values - means ) / scales

    if z_values.isna( ).any( ):
        return pd.Series( { 
            'regime_roll5y': np.nan,
            'regime_dist_best': np.nan,
            'regime_margin': np.nan,
            'regime_n_features_used': len( available ),
        } )

    centroid_slice = centroids_z[ available ]
    centroid_array = np.asarray( centroid_slice.to_numpy( ), dtype = 'float64' )
    z_array = np.asarray( z_values.to_numpy( ), dtype = 'float64' )
    distances = np.sqrt( np.sum( ( centroid_array - z_array[ None, : ] ) ** 2, axis = 1 ) )

    best_idx = int( np.argmin( distances ) )
    best_cluster = int( centroid_slice.index[ best_idx ] )
    best_dist = float( distances[ best_idx ] )

    if len( distances ) > 1:
        second_dist = float( np.partition( distances, 1 )[ 1 ] )
        margin = second_dist - best_dist

    else:
        margin = np.nan

    return pd.Series( { 
        'regime_roll5y': best_cluster,
        'regime_dist_best': best_dist,
        'regime_margin': margin,
        'regime_n_features_used': len( available ),
    } )


### 9.1 - build station-level forward paths
- if `../data/reference/phase9.scenario.paths.csv` exists, use it as the external scenario-path input
- otherwise, fall back to clearly-labeled demo ramps so the full projection pipeline still runs
- forecast the major signatures: air temp, precip, water temp, salinity, oxygen, and rolling regime path

In [ ]:
# 9.1 - build case-study forecasts from annual scenario paths
phase9_case_pool = build_phase9_case_pool( phase9_case_pool_mode, case_station_keys = phase9_case_station_keys )

if phase9_case_region_codes is not None and len( phase9_case_region_codes ) > 0:
    phase9_case_region_codes_norm = { str( code ).strip( ).lower( ) for code in phase9_case_region_codes }
    phase9_case_pool = phase9_case_pool.loc[ phase9_case_pool[ 'region' ].astype( str ).str.lower( ).isin( phase9_case_region_codes_norm ) ].copy( )

if phase9_case_pool_mode == 'station_list':
    phase9_case_studies = phase9_case_pool.copy( )

else:
    phase9_case_studies = select_phase9_case_studies( phase9_case_pool, n_cases = phase9_case_count )

if 'case_id' not in phase9_case_studies.columns:
    phase9_case_studies = phase9_case_studies.reset_index( drop = True ).copy( )
    phase9_case_studies[ 'case_id' ] = [ f'case_{ idx + 1 }' for idx in range( len( phase9_case_studies ) ) ]

else:
    phase9_case_studies[ 'case_id' ] = phase9_case_studies[ 'case_id' ].fillna( '' )
    phase9_missing_case_id = phase9_case_studies[ 'case_id' ].astype( str ).str.strip( ) == ''
    if phase9_missing_case_id.any( ):
        phase9_case_studies = phase9_case_studies.reset_index( drop = True ).copy( )
        phase9_case_studies.loc[ phase9_missing_case_id, 'case_id' ] = [ 
            f'case_{ idx + 1 }' for idx in phase9_case_studies.index[ phase9_missing_case_id ]
        ]

phase9_case_keys = phase9_case_studies[ [ 'region', 'station' ] ].drop_duplicates( )

if phase9_scenario_path_csv.exists( ):
    phase9_scenario_paths = pd.read_csv( phase9_scenario_path_csv )
    phase9_path_source = f'external_csv: { phase9_scenario_path_csv }'

else:
    phase9_scenario_paths = build_phase9_demo_paths( start_year = 2026, end_year = 2100 )
    phase9_path_source = 'demo_ramps'

phase9_scenario_paths[ 'scenario' ] = phase9_scenario_paths[ 'scenario' ].astype( str )
phase9_scenario_paths[ 'year' ] = pd.to_numeric( phase9_scenario_paths[ 'year' ], errors = 'coerce' ).astype( 'Int64' )
for col, fill_value in [ 
    ( 'air_temp_add_c', 0.0 ),
    ( 'precip_mult', 1.0 ),
    ( 'wind_speed_mult', 1.0 ),
    ( 'solar_mult', 1.0 ),
]:
    if col not in phase9_scenario_paths.columns:
        phase9_scenario_paths[ col ] = fill_value

    phase9_scenario_paths[ col ] = pd.to_numeric( phase9_scenario_paths[ col ], errors = 'coerce' ).fillna( fill_value )

phase9_scenarios = sorted( phase9_scenario_paths[ 'scenario' ].dropna( ).unique( ).tolist( ) )
phase9_future_year_min = int( phase9_scenario_paths[ 'year' ].dropna( ).min( ) )
phase9_future_year_max = int( phase9_scenario_paths[ 'year' ].dropna( ).max( ) )

phase9_hist_air = daily_air_final.merge( phase9_case_keys, on = [ 'region', 'station' ], how = 'inner' ).copy( )
phase9_hist_water = daily_water_final.merge( phase9_case_keys, on = [ 'region', 'station' ], how = 'inner' ).copy( )
phase9_hist_air[ 'date' ] = pd.to_datetime( phase9_hist_air[ 'date' ], errors = 'coerce' )
phase9_hist_water[ 'date' ] = pd.to_datetime( phase9_hist_water[ 'date' ], errors = 'coerce' )

phase9_hist_join = phase9_hist_air.merge( 
    phase9_hist_water[ [ 'region', 'station', 'date', 'water_temp', 'salinity', 'oxygen', 'depth' ] ],
    on = [ 'region', 'station', 'date' ],
    how = 'inner',
)
phase9_hist_join[ 'year' ] = phase9_hist_join[ 'date' ].dt.year
phase9_hist_join[ 'month_num' ] = phase9_hist_join[ 'date' ].dt.month
phase9_hist_join[ 'is_warm_season' ] = phase9_hist_join[ 'month_num' ].between( 6, 9 )

phase9_hist_annual = ( 
    phase9_hist_join
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        air_temp_mean = ( 'air_temp', 'mean' ),
        precip_mean = ( 'precip', 'mean' ),
        water_temp_abs_annual_mean = ( 'water_temp', 'mean' ),
        salinity_abs_annual_mean = ( 'salinity', 'mean' ),
        oxygen_abs_annual_mean = ( 'oxygen', 'mean' ),
        depth_abs_annual_mean = ( 'depth', 'mean' ),
    )
)
phase9_hist_warm = ( 
    phase9_hist_join.loc[ phase9_hist_join[ 'is_warm_season' ] ]
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( oxygen_warm_min_abs = ( 'oxygen', 'min' ) )
)
phase9_hist_annual = phase9_hist_annual.merge( phase9_hist_warm, on = [ 'region', 'station', 'year' ], how = 'left' )
phase9_hist_annual[ 'scenario' ] = 'observed_history'

phase9_baseline_value_cols = [ 
    'water_temp_baseline',
    'salinity_baseline',
    'oxygen_baseline',
    'ph_baseline',
    'depth_baseline',
]
phase9_properties_baseline_fallback = ( 
    daily_water_final
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        water_temp_baseline = ( 'water_temp_baseline', 'mean' ),
        salinity_baseline = ( 'salinity_baseline', 'mean' ),
        oxygen_baseline = ( 'oxygen_baseline', 'mean' ),
        ph_baseline = ( 'ph_baseline', 'mean' ),
        depth_baseline = ( 'depth_baseline', 'mean' ),
    )
)

if 'properties_baseline' in globals( ):
    phase9_properties_baseline = properties_baseline.copy( )

else:
    phase9_properties_baseline = phase9_properties_baseline_fallback.copy( )

phase9_properties_baseline = phase9_properties_baseline.merge( 
    phase9_properties_baseline_fallback,
    on = [ 'region', 'station' ],
    how = 'outer',
    suffixes = ( '', '_fallback' ),
)
for col in phase9_baseline_value_cols:
    fallback_col = f'{ col }_fallback'
    if col not in phase9_properties_baseline.columns:
        phase9_properties_baseline[ col ] = phase9_properties_baseline[ fallback_col ]

    else:
        phase9_properties_baseline[ col ] = phase9_properties_baseline[ col ].fillna( phase9_properties_baseline[ fallback_col ] )

    if fallback_col in phase9_properties_baseline.columns:
        phase9_properties_baseline = phase9_properties_baseline.drop( columns = [ fallback_col ] )

phase9_properties_baseline_missing = phase9_properties_baseline.loc[ 
    phase9_properties_baseline[ phase9_baseline_value_cols ].isna( ).any( axis = 1 ),
    [ 'region', 'station' ] + phase9_baseline_value_cols,
].sort_values( [ 'region', 'station' ] ).reset_index( drop = True )
phase9_case_meta = phase9_case_studies[ [ 'region', 'station', 'case_id', 'region_name', 'station_name', 'cluster', 'cluster_label', 'cluster_name', 'event_date' ] ].copy( )
phase9_hist_annual = phase9_hist_annual.merge( phase9_case_meta, on = [ 'region', 'station' ], how = 'left' )
phase9_hist_annual = phase9_hist_annual.merge( phase9_properties_baseline, on = [ 'region', 'station' ], how = 'left' )

phase9_climo = phase9_hist_air.copy( )
phase9_climo[ 'doy' ] = phase9_climo[ 'date' ].dt.dayofyear.clip( upper = 365 )
phase9_driver_climatology = ( 
    phase9_climo
    .groupby( [ 'region', 'station', 'doy' ], as_index = False )
    .agg( 
        air_temp = ( 'air_temp', 'mean' ),
        precip = ( 'precip', 'mean' ),
        wind_speed = ( 'wind_speed', 'mean' ),
        solar = ( 'solar', 'mean' ),
    )
)
phase9_driver_station_mean = ( 
    phase9_hist_air
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        air_temp_station_mean = ( 'air_temp', 'mean' ),
        precip_station_mean = ( 'precip', 'mean' ),
        wind_speed_station_mean = ( 'wind_speed', 'mean' ),
        solar_station_mean = ( 'solar', 'mean' ),
    )
)

phase9_future_dates = pd.DataFrame( { 
    'date': pd.date_range( f'{ phase9_future_year_min }-01-01', f'{ phase9_future_year_max }-12-31', freq = 'D' )
} )
phase9_future_dates[ 'year' ] = phase9_future_dates[ 'date' ].dt.year
phase9_future_dates[ 'doy' ] = phase9_future_dates[ 'date' ].dt.dayofyear.clip( upper = 365 )
phase9_future_dates = phase9_future_dates.loc[ phase9_future_dates[ 'year' ].isin( phase9_scenario_paths[ 'year' ].dropna( ).astype( int ) ) ].copy( )

phase9_future_daily = ( 
    phase9_case_keys.assign( _tmp = 1 )
    .merge( pd.DataFrame( { 'scenario': phase9_scenarios, '_tmp': 1 } ), on = '_tmp', how = 'inner' )
    .merge( phase9_future_dates.assign( _tmp = 1 ), on = '_tmp', how = 'inner' )
    .drop( columns = [ '_tmp' ] )
)
phase9_future_daily = phase9_future_daily.merge( phase9_driver_climatology, on = [ 'region', 'station', 'doy' ], how = 'left' )
phase9_future_daily = phase9_future_daily.merge( phase9_driver_station_mean, on = [ 'region', 'station' ], how = 'left' )
phase9_future_daily = phase9_future_daily.merge( phase9_scenario_paths, on = [ 'scenario', 'year' ], how = 'left' )

for col, station_fill_col in [ 
    ( 'air_temp', 'air_temp_station_mean' ),
    ( 'precip', 'precip_station_mean' ),
    ( 'wind_speed', 'wind_speed_station_mean' ),
    ( 'solar', 'solar_station_mean' ),
]:
    phase9_future_daily[ col ] = phase9_future_daily[ col ].fillna( phase9_future_daily[ station_fill_col ] )

phase9_future_daily[ 'air_temp' ] = phase9_future_daily[ 'air_temp' ] + phase9_future_daily[ 'air_temp_add_c' ]
phase9_future_daily[ 'precip' ] = phase9_future_daily[ 'precip' ] * phase9_future_daily[ 'precip_mult' ]
phase9_future_daily[ 'wind_speed' ] = phase9_future_daily[ 'wind_speed' ] * phase9_future_daily[ 'wind_speed_mult' ]
phase9_future_daily[ 'solar' ] = phase9_future_daily[ 'solar' ] * phase9_future_daily[ 'solar_mult' ]
phase9_future_daily = add_phase9_driver_rolls( phase9_future_daily )

phase9_future_daily = phase9_future_daily.merge( phase9_properties_baseline, on = [ 'region', 'station' ], how = 'left' )
phase9_future_daily = phase9_future_daily.merge( phase6_context_table, on = [ 'region', 'station' ], how = 'left' )
phase9_cluster_lookup = station_baseline[ [ 'region', 'station', 'cluster' ] ].copy( )
phase9_cluster_lookup[ 'cluster_code' ] = pd.to_numeric( phase9_cluster_lookup[ 'cluster' ], errors = 'coerce' )
phase9_cluster_lookup = phase9_cluster_lookup.drop( columns = [ 'cluster' ] )
phase9_future_daily = phase9_future_daily.merge( phase9_cluster_lookup, on = [ 'region', 'station' ], how = 'left' )

phase9_future_daily[ 'air_temp_minus_water_temp_baseline' ] = phase9_future_daily[ 'air_temp' ] - phase9_future_daily[ 'water_temp_baseline' ]
phase9_future_daily[ 'air_temp_r7d_minus_water_temp_baseline' ] = phase9_future_daily[ 'air_temp_r7d' ] - phase9_future_daily[ 'water_temp_baseline' ]
phase9_future_daily[ 'air_temp_r28d_minus_water_temp_baseline' ] = phase9_future_daily[ 'air_temp_r28d' ] - phase9_future_daily[ 'water_temp_baseline' ]
phase9_future_daily[ 'air_temp_r1d_minus_air_temp_r28d' ] = phase9_future_daily[ 'air_temp_r1d' ] - phase9_future_daily[ 'air_temp_r28d' ]
phase9_future_daily[ 'air_temp_r7d_minus_air_temp_r28d' ] = phase9_future_daily[ 'air_temp_r7d' ] - phase9_future_daily[ 'air_temp_r28d' ]
phase9_future_daily[ 'wind_speed_x_air_temp' ] = phase9_future_daily[ 'wind_speed' ] * phase9_future_daily[ 'air_temp' ]
phase9_future_daily[ 'solar_x_air_temp' ] = phase9_future_daily[ 'solar' ] * phase9_future_daily[ 'air_temp' ]

for col in phase6_selected_features:
    if col not in phase9_future_daily.columns:
        phase9_future_daily[ col ] = np.nan

phase9_p6_X = phase9_future_daily[ phase6_selected_features ].copy( ).fillna( phase6_selected_fill_values.reindex( phase6_selected_features ) )
phase9_future_daily[ 'delta_water_temp_pred_p6' ] = phase6_selected_model.predict( phase9_p6_X )
phase9_future_daily[ 'water_temp_pred' ] = phase9_future_daily[ 'water_temp_baseline' ] + phase9_future_daily[ 'delta_water_temp_pred_p6' ]

phase9_future_station_year = build_phase9_station_year_features( phase9_future_daily )
for target in phase7_targets:
    model_t = phase7_model_store.get( target )
    feature_cols_t = phase7_feature_store.get( target, [ ] )
    fill_values_t = phase7_fill_store.get( target, pd.Series( dtype = 'float64' ) )

    if model_t is None or len( feature_cols_t ) == 0:
        continue

    X_target_t = phase9_future_station_year[ feature_cols_t ].copy( ).fillna( fill_values_t.reindex( feature_cols_t ) )
    phase9_future_station_year[ f'{ target }_pred' ] = model_t.predict( X_target_t )

phase9_future_station_year[ 'water_temp_abs_annual_mean' ] = phase9_future_station_year[ 'water_temp_baseline_annual_mean' ] + phase9_future_station_year[ 'delta_water_temp_pred_p6_mean' ]
phase9_future_station_year[ 'salinity_abs_annual_mean' ] = phase9_future_station_year[ 'salinity_baseline_annual_mean' ] + phase9_future_station_year.get( 'delta_salinity_pred', 0.0 )
if 'delta_oxygen_pred' in phase9_future_station_year.columns:
    phase9_future_station_year[ 'oxygen_abs_annual_mean' ] = phase9_future_station_year[ 'oxygen_baseline_annual_mean' ] + phase9_future_station_year[ 'delta_oxygen_pred' ]

else:
    phase9_future_station_year[ 'oxygen_abs_annual_mean' ] = np.nan

phase9_future_station_year[ 'oxygen_warm_min_abs' ] = phase9_future_station_year.get( 'oxygen_warm_min_pred', np.nan )
phase9_future_station_year[ 'oxygen_plot_abs' ] = phase9_future_station_year[ 'oxygen_warm_min_abs' ]
phase9_future_station_year.loc[ phase9_future_station_year[ 'oxygen_plot_abs' ].isna( ), 'oxygen_plot_abs' ] = phase9_future_station_year.loc[ phase9_future_station_year[ 'oxygen_plot_abs' ].isna( ), 'oxygen_abs_annual_mean' ]
phase9_future_station_year = phase9_future_station_year.merge( phase9_case_meta, on = [ 'region', 'station' ], how = 'left' )

phase9_history_plot = phase9_hist_annual.copy( )
phase9_history_plot[ 'oxygen_plot_abs' ] = phase9_history_plot[ 'oxygen_warm_min_abs' ]
phase9_history_plot.loc[ phase9_history_plot[ 'oxygen_plot_abs' ].isna( ), 'oxygen_plot_abs' ] = phase9_history_plot.loc[ phase9_history_plot[ 'oxygen_plot_abs' ].isna( ), 'oxygen_abs_annual_mean' ]

phase9_metric_roll_map = { 
    'air_temp_mean': 'air_temp_roll5y',
    'precip_mean': 'precip_roll5y',
    'water_temp_abs_annual_mean': 'water_temp_roll5y',
    'salinity_abs_annual_mean': 'salinity_roll5y',
    'oxygen_abs_annual_mean': 'oxygen_roll5y',
    'oxygen_plot_abs': 'oxygen_plot_roll5y',
}
for source_col, out_col in phase9_metric_roll_map.items( ):
    phase9_history_plot[ out_col ] = ( 
        phase9_history_plot
        .sort_values( [ 'region', 'station', 'year' ] )
        .groupby( [ 'region', 'station' ] )[ source_col ]
        .transform( lambda values: values.rolling( window = phase9_rolling_window_years, min_periods = 3 ).mean( ) )
    )

phase9_history_plot[ 'mean_annual_water_temp' ] = phase9_history_plot[ 'water_temp_roll5y' ]
phase9_history_plot[ 'mean_annual_salinity' ] = phase9_history_plot[ 'salinity_roll5y' ]
phase9_history_plot[ 'mean_annual_oxygen' ] = phase9_history_plot[ 'oxygen_roll5y' ]
phase9_history_plot[ 'mean_annual_depth' ] = phase9_history_plot[ 'depth_baseline' ]
phase9_history_plot = pd.concat( [ phase9_history_plot, phase9_history_plot.apply( phase9_classify_regime_row, axis = 1 ) ], axis = 1 )

phase9_projection_paths = [ ]
for scenario_name in phase9_scenarios:
    hist_path = phase9_hist_annual.copy( )
    hist_path[ 'scenario' ] = scenario_name
    future_path = phase9_future_station_year.loc[ phase9_future_station_year[ 'scenario' ] == scenario_name ].copy( )
    path_frame = pd.concat( [ hist_path, future_path ], ignore_index = True, sort = False )
    path_frame = path_frame.sort_values( [ 'region', 'station', 'year' ] ).reset_index( drop = True )

    for source_col, out_col in phase9_metric_roll_map.items( ):
        path_frame[ out_col ] = ( 
            path_frame
            .groupby( [ 'region', 'station' ] )[ source_col ]
            .transform( lambda values: values.rolling( window = phase9_rolling_window_years, min_periods = 3 ).mean( ) )
        )

    path_frame[ 'mean_annual_water_temp' ] = path_frame[ 'water_temp_roll5y' ]
    path_frame[ 'mean_annual_salinity' ] = path_frame[ 'salinity_roll5y' ]
    path_frame[ 'mean_annual_oxygen' ] = path_frame[ 'oxygen_roll5y' ]
    path_frame[ 'mean_annual_depth' ] = path_frame[ 'depth_baseline' ]
    path_frame = pd.concat( [ path_frame, path_frame.apply( phase9_classify_regime_row, axis = 1 ) ], axis = 1 )
    phase9_projection_paths.append( path_frame )

phase9_projection_plot = pd.concat( phase9_projection_paths, ignore_index = True ) if len( phase9_projection_paths ) > 0 else pd.DataFrame( )
phase9_projection_future_only = phase9_projection_plot.loc[ phase9_projection_plot[ 'year' ] >= phase9_future_year_min ].copy( )

phase9_crossing_summary = ( 
    phase9_projection_future_only
    .loc[ 
        phase9_projection_future_only[ 'regime_roll5y' ].notna( )
        & phase9_projection_future_only[ 'cluster' ].notna( )
        & ( phase9_projection_future_only[ 'regime_roll5y' ].astype( 'Int64' ) != phase9_projection_future_only[ 'cluster' ].astype( 'Int64' ) )
    ]
    .groupby( [ 'region', 'station', 'scenario' ], as_index = False )[ 'year' ]
    .min( )
    .rename( columns = { 'year': 'first_regime_cross_year' } )
)
phase9_crossing_summary = phase9_case_meta.merge( phase9_crossing_summary, on = [ 'region', 'station' ], how = 'left' )
phase9_crossing_summary = phase9_crossing_summary.sort_values( [ 'scenario', 'first_regime_cross_year', 'region', 'station' ] ).reset_index( drop = True )

phase9_signature_2100 = phase9_future_station_year.loc[ phase9_future_station_year[ 'year' ] == phase9_future_year_max, [ 
    'region',
    'station',
    'scenario',
    'water_temp_abs_annual_mean',
    'salinity_abs_annual_mean',
    'oxygen_abs_annual_mean',
    'oxygen_warm_min_abs',
] ].copy( )
phase9_signature_2100 = phase9_case_meta.merge( phase9_signature_2100, on = [ 'region', 'station' ], how = 'left' )

print( f'phase9 path source: { phase9_path_source }' )
print( f'phase9 case pool mode: { phase9_case_pool_mode }' )
print( f'phase9 case-study stations: { len( phase9_case_studies ) }' )
print( f'phase9 region filter: { sorted( phase9_case_region_codes_norm ) if phase9_case_region_codes is not None and len( phase9_case_region_codes ) > 0 else "all" }' )
print( f'phase9 scenarios: { phase9_scenarios }' )
print( f'phase9 future daily rows: { len( phase9_future_daily ) }' )
print( f'phase9 future station-year rows: { len( phase9_future_station_year ) }' )

phase9_future_path_validity = ( 
    phase9_future_station_year
    .groupby( [ 'region', 'station', 'scenario' ], as_index = False )
    .agg( 
        n_years = ( 'year', 'size' ),
        year_min = ( 'year', 'min' ),
        year_max = ( 'year', 'max' ),
        n_water_temp_valid = ( 'water_temp_abs_annual_mean', lambda values: values.notna( ).sum( ) ),
        n_salinity_valid = ( 'salinity_abs_annual_mean', lambda values: values.notna( ).sum( ) ),
        n_oxygen_mean_valid = ( 'oxygen_abs_annual_mean', lambda values: values.notna( ).sum( ) ),
        n_oxygen_warm_min_valid = ( 'oxygen_warm_min_abs', lambda values: values.notna( ).sum( ) ),
    )
)
phase9_future_path_validity[ 'water_temp_complete' ] = phase9_future_path_validity[ 'n_water_temp_valid' ] == phase9_future_path_validity[ 'n_years' ]
phase9_future_path_validity[ 'salinity_complete' ] = phase9_future_path_validity[ 'n_salinity_valid' ] == phase9_future_path_validity[ 'n_years' ]
phase9_future_path_validity[ 'oxygen_mean_complete' ] = phase9_future_path_validity[ 'n_oxygen_mean_valid' ] == phase9_future_path_validity[ 'n_years' ]
phase9_future_path_validity[ 'oxygen_warm_min_complete' ] = phase9_future_path_validity[ 'n_oxygen_warm_min_valid' ] == phase9_future_path_validity[ 'n_years' ]
phase9_future_path_validity = phase9_case_meta.merge( phase9_future_path_validity, on = [ 'region', 'station' ], how = 'left' )
phase9_future_path_incomplete = phase9_future_path_validity.loc[ 
    ~( 
        phase9_future_path_validity[ 'water_temp_complete' ].fillna( False )
        & phase9_future_path_validity[ 'salinity_complete' ].fillna( False )
        & phase9_future_path_validity[ 'oxygen_mean_complete' ].fillna( False )
    )
].sort_values( [ 'scenario', 'region', 'station' ] ).reset_index( drop = True )

print( '\nphase9 selected case studies:' )
display( phase9_case_studies[ [ 'case_id', 'region', 'station', 'station_name', 'region_name', 'cluster_label', 'event_date' ] ] )


if len( phase9_properties_baseline_missing ) > 0:
    print( '\nphase9 stations with incomplete baseline values after fallback:' )
    display( phase9_properties_baseline_missing )
print( '\nphase9 regime crossing summary:' )
display( phase9_crossing_summary[ [ 'case_id', 'region', 'station', 'station_name', 'scenario', 'cluster_label', 'first_regime_cross_year' ] ] )

print( '\nphase9 2100 signature snapshot:' )
display( phase9_signature_2100.round( 3 ) )

if len( phase9_future_path_incomplete ) > 0:
    print( '\nphase9 incomplete future paths by station/scenario:' )
    display( phase9_future_path_incomplete[ [ 
        'case_id',
        'region',
        'station',
        'station_name',
        'scenario',
        'n_years',
        'n_water_temp_valid',
        'n_salinity_valid',
        'n_oxygen_mean_valid',
        'n_oxygen_warm_min_valid',
        'water_temp_complete',
        'salinity_complete',
        'oxygen_mean_complete',
        'oxygen_warm_min_complete',
    ] ] )


### 9.2 - station portraits through 2100
- black = observed rolling history
- colored lines = scenario paths through the same rolling-signature lens
- top panel uses the existing regime geometry, so crossings are directly comparable to Phase 5

In [ ]:
# 9.2 - media-style station portraits from historical data through 2100
phase9_plot_scenarios = phase9_scenarios.copy( )
phase9_scenario_palette = dict( zip( phase9_plot_scenarios, sns.color_palette( 'Set2', n_colors = len( phase9_plot_scenarios ) ) ) )
phase9_cluster_label_map = ( 
    station_baseline_display[ [ 'cluster', 'cluster_label' ] ]
    .dropna( subset = [ 'cluster' ] )
    .drop_duplicates( )
    .assign( cluster = lambda frame: frame[ 'cluster' ].astype( int ) )
    .set_index( 'cluster' )[ 'cluster_label' ]
    .to_dict( )
)

for case_row in phase9_case_studies.itertuples( index = False ):
    hist_case = phase9_history_plot.loc[ 
        ( phase9_history_plot[ 'region' ] == case_row.region )
        & ( phase9_history_plot[ 'station' ] == case_row.station )
    ].copy( )
    scen_case = phase9_projection_plot.loc[ 
        ( phase9_projection_plot[ 'region' ] == case_row.region )
        & ( phase9_projection_plot[ 'station' ] == case_row.station )
    ].copy( )

    if len( hist_case ) == 0 or len( scen_case ) == 0:
        continue

    fig, axes = plt.subplots( 6, 1, figsize = ( 14, 16 ), sharex = True )
    fig.suptitle( 
        f"Phase 9 Forward Portrait | { case_row.station_name } ({ case_row.region_name }) | baseline { case_row.cluster_label } | first observed flip { pd.to_datetime( case_row.event_date ).date( ) }",
        fontsize = 14,
        y = 0.995,
    )

    axes[ 0 ].plot( 
        hist_case[ 'year' ],
        hist_case[ 'regime_roll5y' ],
        color = 'black',
        linewidth = 2.2,
        label = 'observed 5y regime',
    )
    for scenario_name in phase9_plot_scenarios:
        sub = scen_case.loc[ scen_case[ 'scenario' ] == scenario_name ].copy( )
        axes[ 0 ].plot( 
            sub[ 'year' ],
            sub[ 'regime_roll5y' ],
            color = phase9_scenario_palette[ scenario_name ],
            linewidth = 1.8,
            alpha = 0.95,
            label = scenario_name,
        )

    cluster_ticks = sorted( [ int( val ) for val in pd.Series( pd.concat( [ hist_case[ 'regime_roll5y' ], scen_case[ 'regime_roll5y' ] ] ) ).dropna( ).astype( int ).unique( ) ] )
    if len( cluster_ticks ) > 0:
        axes[ 0 ].set_yticks( cluster_ticks )
        axes[ 0 ].set_yticklabels( [ phase9_cluster_label_map.get( tick, f'C{ tick }' ) for tick in cluster_ticks ] )

    axes[ 0 ].axhline( float( case_row.cluster ), color = '#555555', linestyle = '--', linewidth = 1.0 )
    axes[ 0 ].set_ylabel( 'Regime' )
    axes[ 0 ].legend( loc = 'upper left', ncol = 3 )

    plot_specs = [ 
        ( 'air_temp_roll5y', 'Air Temp ( 5y mean )', None ),
        ( 'precip_roll5y', 'Precip ( 5y mean )', None ),
        ( 'water_temp_roll5y', 'Water Temp ( 5y mean )', None ),
        ( 'salinity_roll5y', 'Salinity ( 5y mean )', None ),
        ( 'oxygen_plot_roll5y', 'Oxygen Signature ( 5y mean )', None ),
    ]

    for ax, ( col, ylabel, title_override ) in zip( axes[ 1: ], plot_specs ):
        ax.plot( hist_case[ 'year' ], hist_case[ col ], color = 'black', linewidth = 2.2, label = 'observed' )

        for scenario_name in phase9_plot_scenarios:
            sub = scen_case.loc[ scen_case[ 'scenario' ] == scenario_name ].copy( )
            ax.plot( 
                sub[ 'year' ],
                sub[ col ],
                color = phase9_scenario_palette[ scenario_name ],
                linewidth = 1.8,
                alpha = 0.95,
                label = scenario_name,
            )

        ax.axvline( phase9_future_year_min, color = '#666666', linestyle = '--', linewidth = 0.9 )
        ax.set_ylabel( ylabel )
        if title_override is not None:
            ax.set_title( title_override )

    axes[ -1 ].set_xlabel( 'Year' )
    plt.tight_layout( )
    plt.show( )


In [ ]:
# 9.3 - scan all eligible stations for projected regime crossings + hypoxia risk
import sys

phase9_scan_region_codes = [ 'cbm', 'cbv', 'elk' ]
phase9_scan_top_n = 12
phase9_scan_plot_n = 3
phase9_scan_plot_station_keys = list( phase9_case_station_keys ) if phase9_case_station_keys is not None else None
phase9_scan_hypoxia_threshold = 2.0
phase9_scan_material_share = 0.05
phase9_scan_progress_every = 20

phase9_repo_root = Path.cwd( )
if not ( phase9_repo_root / 'estuaries' ).exists( ):
    phase9_repo_root = phase9_repo_root.parent

if str( phase9_repo_root ) not in sys.path:
    sys.path.append( str( phase9_repo_root ) )

from helpers.phase9_scan import plot_phase9_scan_portraits, run_phase9_station_scan

phase9_scan_result = run_phase9_station_scan( 
    station_baseline_display = station_baseline_display,
    daily_air_final = daily_air_final,
    daily_water_final = daily_water_final,
    first_event = first_event,
    flagged_station_keys_final = flagged_station_keys_final,
    phase9_scenario_path_csv = phase9_scenario_path_csv,
    build_phase9_demo_paths = build_phase9_demo_paths,
    add_phase9_driver_rolls = add_phase9_driver_rolls,
    build_phase9_station_year_features = build_phase9_station_year_features,
    phase9_classify_regime_row = phase9_classify_regime_row,
    phase9_rolling_window_years = phase9_rolling_window_years,
    phase6_context_table = phase6_context_table,
    phase6_selected_features = phase6_selected_features,
    phase6_selected_fill_values = phase6_selected_fill_values,
    phase6_selected_model = phase6_selected_model,
    phase7_targets = phase7_targets,
    phase7_feature_store = phase7_feature_store,
    phase7_fill_store = phase7_fill_store,
    phase7_model_store = phase7_model_store,
    station_baseline = station_baseline,
    properties_baseline = properties_baseline if 'properties_baseline' in globals( ) else None,
    phase9_scenario_paths = phase9_scenario_paths if 'phase9_scenario_paths' in globals( ) else None,
    phase9_scenarios = phase9_scenarios if 'phase9_scenarios' in globals( ) else None,
    phase9_future_year_min = phase9_future_year_min if 'phase9_future_year_min' in globals( ) else None,
    phase9_future_year_max = phase9_future_year_max if 'phase9_future_year_max' in globals( ) else None,
    phase9_future_dates = phase9_future_dates if 'phase9_future_dates' in globals( ) else None,
    phase9_properties_baseline = phase9_properties_baseline if 'phase9_properties_baseline' in globals( ) else None,
    phase9_cluster_lookup = phase9_cluster_lookup if 'phase9_cluster_lookup' in globals( ) else None,
    region_codes = phase9_scan_region_codes,
    top_n = phase9_scan_top_n,
    plot_n = phase9_scan_plot_n,
    hypoxia_threshold = phase9_scan_hypoxia_threshold,
    material_share = phase9_scan_material_share,
    progress_every = phase9_scan_progress_every,
)

if phase9_scan_plot_station_keys is not None:
    phase9_scan_plot_keys_df = normalize_phase9_station_keys( phase9_scan_plot_station_keys )
    if phase9_scan_plot_keys_df is not None and len( phase9_scan_plot_keys_df ) > 0:
        phase9_scan_result[ 'plot_studies' ] = ( 
            phase9_scan_plot_keys_df
            .merge( phase9_scan_result[ 'station_meta' ], on = [ 'region', 'station' ], how = 'inner' )
            .sort_values( [ 'phase9_case_order', 'region', 'station' ] )
            .drop( columns = [ 'phase9_case_order' ] )
            .reset_index( drop = True )
        )
        phase9_scan_missing_plot_keys = phase9_scan_plot_keys_df.merge( 
            phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station' ] ],
            on = [ 'region', 'station' ],
            how = 'left',
            indicator = True,
        )
        phase9_scan_missing_plot_keys = phase9_scan_missing_plot_keys.loc[ phase9_scan_missing_plot_keys[ '_merge' ] != 'both', [ 'region', 'station' ] ].reset_index( drop = True )

    else:
        phase9_scan_missing_plot_keys = pd.DataFrame( columns = [ 'region', 'station' ] )

else:
    phase9_scan_missing_plot_keys = pd.DataFrame( columns = [ 'region', 'station' ] )

print( f"phase9 scan path source: { phase9_scan_result[ 'path_source' ] }" )
print( f"phase9 scan stations projected: { len( phase9_scan_result[ 'station_keys' ] ) }" )
print( f"phase9 scan scenarios: { phase9_scan_result[ 'scenarios' ] }" )
print( f"phase9 scan region filter: { sorted( phase9_scan_result[ 'region_codes_norm' ] ) if phase9_scan_result[ 'region_codes_norm' ] is not None else 'all' }" )
print( f"phase9 scan future station-year rows: { len( phase9_scan_result[ 'future_station_year' ] ) }" )

print( '\nphase9 earliest projected regime-crossing candidates:' )
display( phase9_scan_result[ 'crossing_display' ] )

print( '\nphase9 strongest projected hypoxia candidates:' )
display( phase9_scan_result[ 'hypoxia_display' ].round( 3 ) )

print( '\nphase9 scan plot stations:' )
display( phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station', 'station_name', 'region_name', 'cluster_label', 'is_holdout', 'event_date' ] ] )

if len( phase9_scan_missing_plot_keys ) > 0:
    print( '\nrequested plot stations missing from the scan inputs:' )
    display( phase9_scan_missing_plot_keys )

plot_phase9_scan_portraits( phase9_scan_result, station_baseline_display = station_baseline_display )


In [ ]:
# 9.4 - save trained model bundle to file
import sys

phase9_bundle_out_path = Path( '../artifacts/t4d.model.bundle.joblib' )

phase9_repo_root = Path.cwd( )
if not ( phase9_repo_root / 'estuaries' ).exists( ):
    phase9_repo_root = phase9_repo_root.parent

if str( phase9_repo_root ) not in sys.path:
    sys.path.append( str( phase9_repo_root ) )

from helpers.model_bundle import build_t4d_model_bundle, save_t4d_model_bundle

phase9_model_bundle = build_t4d_model_bundle( 
    domain_feature_cols = domain_feature_cols if 'domain_feature_cols' in globals( ) else None,
    scaler_domain = scaler_domain if 'scaler_domain' in globals( ) else None,
    kmeans_model = kmeans_model if 'kmeans_model' in globals( ) else None,
    cluster_name_map = cluster_name_map if 'cluster_name_map' in globals( ) else None,
    cluster_order = cluster_order if 'cluster_order' in globals( ) else None,
    cluster_name_order = cluster_name_order if 'cluster_name_order' in globals( ) else None,
    cluster_label_order = cluster_label_order if 'cluster_label_order' in globals( ) else None,
    cluster_note_order = cluster_note_order if 'cluster_note_order' in globals( ) else None,
    phase9_regime_feature_cols = phase9_regime_feature_cols if 'phase9_regime_feature_cols' in globals( ) else None,
    centroids_z = centroids_z if 'centroids_z' in globals( ) else None,
    feature_mean = feature_mean if 'feature_mean' in globals( ) else None,
    feature_scale = feature_scale if 'feature_scale' in globals( ) else None,
    phase6_selected_model = phase6_selected_model if 'phase6_selected_model' in globals( ) else None,
    phase6_selected_features = phase6_selected_features if 'phase6_selected_features' in globals( ) else None,
    phase6_selected_fill_values = phase6_selected_fill_values if 'phase6_selected_fill_values' in globals( ) else None,
    phase6_context_table = phase6_context_table if 'phase6_context_table' in globals( ) else None,
    phase7_targets = phase7_targets if 'phase7_targets' in globals( ) else None,
    phase7_model_store = phase7_model_store if 'phase7_model_store' in globals( ) else None,
    phase7_feature_store = phase7_feature_store if 'phase7_feature_store' in globals( ) else None,
    phase7_fill_store = phase7_fill_store if 'phase7_fill_store' in globals( ) else None,
    phase7_station_context = phase7_station_context if 'phase7_station_context' in globals( ) else None,
)

phase9_bundle_info = save_t4d_model_bundle( phase9_model_bundle, phase9_bundle_out_path )
print( f"saved model bundle: { phase9_bundle_info[ 'path' ] }" )
print( f"bundle size: { phase9_bundle_info[ 'megabytes' ]:.2f} MB" )
print( f"phase6 features saved: { len( phase9_model_bundle[ 'phase6_air_to_water_temp' ][ 'selected_features' ] ) if phase9_model_bundle[ 'phase6_air_to_water_temp' ][ 'selected_features' ] is not None else 0 }" )
print( f"phase7 targets saved: { phase9_model_bundle[ 'phase7_signature_models' ][ 'targets' ] }" )


In [ ]:
# 9.5 - naive STL continuation benchmark from historical monthly signatures
import sys

phase9_stl_region_codes = [ 'cbm', 'cbv', 'elk' ]
phase9_stl_forecast_end_year = 2100
phase9_stl_trend_years = 10
phase9_stl_min_months = 84
phase9_stl_plot_station_count = 3
phase9_stl_station_keys = list( phase9_case_station_keys ) if phase9_case_station_keys is not None else None

phase9_repo_root = Path.cwd( )
if not ( phase9_repo_root / 'estuaries' ).exists( ):
    phase9_repo_root = phase9_repo_root.parent

if str( phase9_repo_root ) not in sys.path:
    sys.path.append( str( phase9_repo_root ) )

from helpers.stl_emulation import build_monthly_station_history, run_stl_emulation

phase9_stl_station_meta = station_baseline_display[ [ 'region', 'station', 'region_name', 'station_name', 'cluster_label' ] ].drop_duplicates( ).copy( )
phase9_stl_available_keys = daily_air_final[ [ 'region', 'station' ] ].drop_duplicates( ).merge( daily_water_final[ [ 'region', 'station' ] ].drop_duplicates( ), on = [ 'region', 'station' ], how = 'inner' )
phase9_stl_station_meta = phase9_stl_available_keys.merge( phase9_stl_station_meta, on = [ 'region', 'station' ], how = 'left' )

if phase9_stl_region_codes is not None and len( phase9_stl_region_codes ) > 0:
    phase9_stl_region_codes_norm = { str( code ).strip( ).lower( ) for code in phase9_stl_region_codes }
    phase9_stl_station_meta = phase9_stl_station_meta.loc[ phase9_stl_station_meta[ 'region' ].astype( str ).str.lower( ).isin( phase9_stl_region_codes_norm ) ].copy( )

else:
    phase9_stl_region_codes_norm = None

if phase9_stl_station_keys is None:
    if 'phase9_scan_result' in globals( ) and len( phase9_scan_result.get( 'plot_studies', pd.DataFrame( ) ) ) > 0:
        phase9_stl_station_keys_df = phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station' ] ].drop_duplicates( ).head( phase9_stl_plot_station_count )

    elif 'phase9_case_studies' in globals( ) and len( phase9_case_studies ) > 0:
        phase9_stl_station_keys_df = phase9_case_studies[ [ 'region', 'station' ] ].drop_duplicates( ).head( phase9_stl_plot_station_count )

    else:
        phase9_stl_station_keys_df = phase9_stl_station_meta[ [ 'region', 'station' ] ].drop_duplicates( ).head( phase9_stl_plot_station_count )

else:
    if isinstance( phase9_stl_station_keys, pd.DataFrame ):
        phase9_stl_station_keys_df = phase9_stl_station_keys[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

    else:
        phase9_stl_station_keys_df = normalize_phase9_station_keys( phase9_stl_station_keys )[ [ 'region', 'station' ] ].drop_duplicates( ).copy( )

phase9_stl_station_keys_df = phase9_stl_station_keys_df.merge( phase9_stl_station_meta[ [ 'region', 'station' ] ].drop_duplicates( ), on = [ 'region', 'station' ], how = 'inner' )
phase9_stl_monthly_history = build_monthly_station_history( daily_air_final, daily_water_final, phase9_stl_station_keys_df, start_year = 1995 )
phase9_stl_result = run_stl_emulation( 
    phase9_stl_monthly_history,
    phase9_stl_station_meta,
    phase9_stl_station_keys_df,
    forecast_end_year = phase9_stl_forecast_end_year,
    trend_years = phase9_stl_trend_years,
    min_months = phase9_stl_min_months,
)

print( 'phase9 STL note: this is a naive continuation benchmark, not the learned scenario model.' )
print( f"phase9 STL stations: { len( phase9_stl_result[ 'station_keys' ] ) }" )
print( f"phase9 STL region filter: { sorted( phase9_stl_region_codes_norm ) if phase9_stl_region_codes_norm is not None else 'all' }" )
print( f"phase9 STL monthly rows: { len( phase9_stl_result[ 'projection_long' ] ) }" )

phase9_stl_summary_display = phase9_stl_result[ 'summary' ].sort_values( [ 'station_name', 'variable_label' ] ).reset_index( drop = True )
display( phase9_stl_summary_display.round( 3 ) )

phase9_stl_variable_order = [ 'air_temp', 'precip', 'water_temp', 'salinity', 'oxygen' ]
phase9_stl_variable_label_map = { 
    'air_temp': 'Air Temp',
    'precip': 'Precip',
    'water_temp': 'Water Temp',
    'salinity': 'Salinity',
    'oxygen': 'Dissolved Oxygen',
}

for station_row in phase9_stl_result[ 'station_keys' ].itertuples( index = False ):
    station_projection = phase9_stl_result[ 'projection_long' ].loc[ 
        ( phase9_stl_result[ 'projection_long' ][ 'region' ] == station_row.region )
        & ( phase9_stl_result[ 'projection_long' ][ 'station' ] == station_row.station )
    ].copy( )
    if len( station_projection ) == 0:
        continue

    station_label = station_projection[ 'station_name' ].dropna( ).iloc[ 0 ] if station_projection[ 'station_name' ].dropna( ).shape[ 0 ] > 0 else station_row.station
    region_label = station_projection[ 'region_name' ].dropna( ).iloc[ 0 ] if station_projection[ 'region_name' ].dropna( ).shape[ 0 ] > 0 else station_row.region
    cluster_label = station_projection[ 'cluster_label' ].dropna( ).iloc[ 0 ] if station_projection[ 'cluster_label' ].dropna( ).shape[ 0 ] > 0 else 'unknown regime'

    fig, axes = plt.subplots( len( phase9_stl_variable_order ), 1, figsize = ( 14, 16 ), sharex = True )
    fig.suptitle( f"Naive STL Continuation | { station_label } ({ region_label }) | baseline { cluster_label }", fontsize = 14, y = 0.995 )

    for ax, variable in zip( axes, phase9_stl_variable_order ):
        sub = station_projection.loc[ station_projection[ 'variable' ] == variable ].copy( ).sort_values( 'month_date' )
        if len( sub ) == 0:
            ax.set_visible( False )
            continue

        hist_sub = sub.loc[ ~sub[ 'is_future' ] ].copy( )
        future_sub = sub.loc[ sub[ 'is_future' ] ].copy( )
        ax.plot( hist_sub[ 'month_date' ], hist_sub[ 'observed' ], color = '#1f1f1f', linewidth = 1.1, alpha = 0.65, label = 'observed monthly' )
        ax.plot( hist_sub[ 'month_date' ], hist_sub[ 'trend' ], color = '#2f7f5f', linewidth = 1.6, alpha = 0.95, label = 'STL trend' )
        ax.plot( hist_sub[ 'month_date' ], hist_sub[ 'fitted' ], color = '#7f8c8d', linewidth = 1.0, alpha = 0.9, label = 'trend + seasonal fit' )
        ax.plot( future_sub[ 'month_date' ], future_sub[ 'fitted' ], color = '#c44e52', linewidth = 1.8, alpha = 0.95, label = 'naive continuation' )
        ax.axvline( hist_sub[ 'month_date' ].max( ), color = '#666666', linestyle = '--', linewidth = 0.9 )
        ax.set_ylabel( phase9_stl_variable_label_map[ variable ] )
        ax.legend( loc = 'upper left', ncol = 4 )

    axes[ -1 ].set_xlabel( 'Year' )
    plt.tight_layout( )
    plt.show( )


In [ ]:
# 9.6 - compare learned phase9 forecast vs naive STL continuation for one shared station
phase9_compare_station_key = phase9_case_station_keys[ 0 ] if phase9_case_station_keys is not None and len( phase9_case_station_keys ) > 0 else None
phase9_compare_preferred_regions = None
phase9_compare_scenarios = None

if 'phase9_scan_result' not in globals( ) or 'phase9_stl_result' not in globals( ):
    print( 'phase9_scan_result or phase9_stl_result is missing; run 9.3 and 9.5 first.' )

else:
    phase9_compare_stl_available_keys = phase9_stl_result[ 'projection_long' ][ [ 'region', 'station' ] ].drop_duplicates( ) if len( phase9_stl_result.get( 'projection_long', pd.DataFrame( ) ) ) > 0 else pd.DataFrame( columns = [ 'region', 'station' ] )
    phase9_compare_shared_keys = phase9_scan_result[ 'station_keys' ].merge( phase9_compare_stl_available_keys, on = [ 'region', 'station' ], how = 'inner' )

    if phase9_compare_station_key is not None:
        if isinstance( phase9_compare_station_key, pd.DataFrame ):
            phase9_compare_station_df = phase9_compare_station_key[ [ 'region', 'station' ] ].drop_duplicates( ).head( 1 ).copy( )

        else:
            phase9_compare_station_df = normalize_phase9_station_keys( [ phase9_compare_station_key ] )[ [ 'region', 'station' ] ].drop_duplicates( ).head( 1 ).copy( )

        phase9_compare_station_df = phase9_compare_shared_keys.merge( phase9_compare_station_df, on = [ 'region', 'station' ], how = 'inner' )

    else:
        phase9_compare_station_df = phase9_compare_shared_keys.copy( )
        if phase9_compare_preferred_regions is not None and len( phase9_compare_preferred_regions ) > 0:
            phase9_compare_region_norm = { str( code ).strip( ).lower( ) for code in phase9_compare_preferred_regions }
            phase9_compare_station_pref = phase9_compare_station_df.loc[ phase9_compare_station_df[ 'region' ].astype( str ).str.lower( ).isin( phase9_compare_region_norm ) ].copy( )
            if len( phase9_compare_station_pref ) > 0:
                phase9_compare_station_df = phase9_compare_station_pref

        if len( phase9_compare_station_df ) > 0 and len( phase9_scan_result.get( 'plot_studies', pd.DataFrame( ) ) ) > 0:
            phase9_compare_station_df = phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station' ] ].drop_duplicates( ).merge( phase9_compare_station_df, on = [ 'region', 'station' ], how = 'inner' )
            if len( phase9_compare_station_df ) == 0:
                phase9_compare_station_df = phase9_compare_shared_keys.copy( )

        phase9_compare_station_df = phase9_compare_station_df.head( 1 ).copy( )

    if len( phase9_compare_station_df ) == 0:
        print( 'No shared station with actual STL projection output was found; set phase9_stl_station_keys or phase9_compare_station_key explicitly and rerun 9.5.' )

    else:
        phase9_compare_row = phase9_compare_station_df.iloc[ 0 ]
        phase9_compare_meta = phase9_scan_result[ 'station_meta' ].loc[ 
            ( phase9_scan_result[ 'station_meta' ][ 'region' ] == phase9_compare_row[ 'region' ] )
            & ( phase9_scan_result[ 'station_meta' ][ 'station' ] == phase9_compare_row[ 'station' ] )
        ].head( 1 )

        if len( phase9_compare_meta ) == 0:
            phase9_compare_station_name = str( phase9_compare_row[ 'station' ] )
            phase9_compare_region_name = str( phase9_compare_row[ 'region' ] )
            phase9_compare_cluster_label = 'unknown regime'

        else:
            phase9_compare_station_name = phase9_compare_meta[ 'station_name' ].iloc[ 0 ]
            phase9_compare_region_name = phase9_compare_meta[ 'region_name' ].iloc[ 0 ]
            phase9_compare_cluster_label = phase9_compare_meta[ 'cluster_label' ].iloc[ 0 ]

        phase9_compare_history = phase9_scan_result[ 'history_plot' ].loc[ 
            ( phase9_scan_result[ 'history_plot' ][ 'region' ] == phase9_compare_row[ 'region' ] )
            & ( phase9_scan_result[ 'history_plot' ][ 'station' ] == phase9_compare_row[ 'station' ] )
        ].copy( )
        phase9_compare_projection = phase9_scan_result[ 'projection_plot' ].loc[ 
            ( phase9_scan_result[ 'projection_plot' ][ 'region' ] == phase9_compare_row[ 'region' ] )
            & ( phase9_scan_result[ 'projection_plot' ][ 'station' ] == phase9_compare_row[ 'station' ] )
        ].copy( )

        if phase9_compare_scenarios is None:
            phase9_compare_scenarios_use = phase9_scan_result[ 'scenarios' ]

        else:
            phase9_compare_scenarios_use = [ scenario for scenario in phase9_compare_scenarios if scenario in phase9_scan_result[ 'scenarios' ] ]

        phase9_compare_stl = phase9_stl_result[ 'projection_long' ].loc[ 
            ( phase9_stl_result[ 'projection_long' ][ 'region' ] == phase9_compare_row[ 'region' ] )
            & ( phase9_stl_result[ 'projection_long' ][ 'station' ] == phase9_compare_row[ 'station' ] )
        ].copy( )

        phase9_compare_metric_map = { 
            'air_temp': ( 'air_temp_roll5y', 'Air Temp ( 5y mean )' ),
            'precip': ( 'precip_roll5y', 'Precip ( 5y mean )' ),
            'water_temp': ( 'water_temp_roll5y', 'Water Temp ( 5y mean )' ),
            'salinity': ( 'salinity_roll5y', 'Salinity ( 5y mean )' ),
            'oxygen': ( 'oxygen_roll5y', 'Oxygen Annual Mean ( 5y mean )' ),
        }
        phase9_compare_stl_roll_parts = [ ]
        for variable, ( model_col, ylabel ) in phase9_compare_metric_map.items( ):
            sub = phase9_compare_stl.loc[ phase9_compare_stl[ 'variable' ] == variable ].copy( ).sort_values( 'month_date' )
            if len( sub ) == 0:
                continue

            sub[ 'value_for_annual' ] = sub[ 'observed' ]
            sub.loc[ sub[ 'is_future' ], 'value_for_annual' ] = sub.loc[ sub[ 'is_future' ], 'fitted' ]
            annual = sub.groupby( 'year', as_index = False ).agg( stl_value_annual = ( 'value_for_annual', 'mean' ) )
            annual = annual.sort_values( 'year' ).reset_index( drop = True )
            annual[ 'stl_roll5y' ] = annual[ 'stl_value_annual' ].rolling( window = phase9_rolling_window_years, min_periods = 3 ).mean( )
            annual[ 'variable' ] = variable
            annual[ 'model_col' ] = model_col
            annual[ 'ylabel' ] = ylabel
            phase9_compare_stl_roll_parts.append( annual )

        phase9_compare_stl_roll = pd.concat( phase9_compare_stl_roll_parts, ignore_index = True ) if len( phase9_compare_stl_roll_parts ) > 0 else pd.DataFrame( columns = [ 'year', 'stl_roll5y', 'variable', 'model_col', 'ylabel' ] )

        print( f"phase9/STL compare station: { phase9_compare_station_name } ({ phase9_compare_region_name }) | baseline { phase9_compare_cluster_label }" )
        print( 'oxygen uses annual mean here so the STL side is comparable; it is not the warm-season minimum signature.' )

        fig, axes = plt.subplots( len( phase9_compare_metric_map ), 1, figsize = ( 14, 17 ), sharex = True )
        fig.suptitle( f"Learned Phase 9 Forecast vs Naive STL Continuation | { phase9_compare_station_name } ({ phase9_compare_region_name }) | baseline { phase9_compare_cluster_label }", fontsize = 14, y = 0.995 )

        phase9_compare_palette = dict( zip( phase9_compare_scenarios_use, sns.color_palette( 'Set2', n_colors = len( phase9_compare_scenarios_use ) ) ) )
        for ax, ( variable, ( model_col, ylabel ) ) in zip( axes, phase9_compare_metric_map.items( ) ):
            ax.plot( phase9_compare_history[ 'year' ], phase9_compare_history[ model_col ], color = '#1f1f1f', linewidth = 2.2, label = 'observed 5y history' )

            for scenario_name in phase9_compare_scenarios_use:
                sub = phase9_compare_projection.loc[ 
                    ( phase9_compare_projection[ 'scenario' ] == scenario_name )
                    & ( phase9_compare_projection[ 'year' ] >= phase9_scan_result[ 'future_year_min' ] )
                ].copy( )
                ax.plot( sub[ 'year' ], sub[ model_col ], color = phase9_compare_palette[ scenario_name ], linewidth = 1.8, alpha = 0.95, label = f'learned { scenario_name }' )

            stl_sub = phase9_compare_stl_roll.loc[ phase9_compare_stl_roll[ 'variable' ] == variable ].copy( ) if 'variable' in phase9_compare_stl_roll.columns else pd.DataFrame( )
            if len( stl_sub ) > 0:
                stl_hist = stl_sub.loc[ stl_sub[ 'year' ] < phase9_scan_result[ 'future_year_min' ] ].copy( )
                stl_future = stl_sub.loc[ stl_sub[ 'year' ] >= phase9_scan_result[ 'future_year_min' ] ].copy( )
                ax.plot( stl_hist[ 'year' ], stl_hist[ 'stl_roll5y' ], color = '#7f8c8d', linewidth = 1.2, alpha = 0.85, linestyle = ':', label = 'STL history fit' )
                ax.plot( stl_future[ 'year' ], stl_future[ 'stl_roll5y' ], color = '#c44e52', linewidth = 2.0, alpha = 0.95, linestyle = '--', label = 'STL naive continuation' )

            ax.axvline( phase9_scan_result[ 'future_year_min' ], color = '#666666', linestyle = '--', linewidth = 0.9 )
            ax.set_ylabel( ylabel )
            ax.legend( loc = 'upper left', ncol = 2 )

        axes[ -1 ].set_xlabel( 'Year' )
        plt.tight_layout( )
        plt.show( )


In [ ]:
# 9.7 - compile or reload the NEX-GDDP-CMIP6 asset manifest for Chesapeake + one high-quality wobbling station
import json
import sys
from pathlib import Path

phase9_nex_models = [ 
    'ACCESS-CM2',
    'ACCESS-ESM1-5',
    'BCC-CSM2-MR',
    'CanESM5',
    'MRI-ESM2-0',
]
phase9_nex_variables = [ 'tas', 'pr', 'sfcWind', 'rsds' ]
phase9_nex_scenarios = { 
    'historical': ( 1950, 2014 ),
    'ssp245': ( 2015, 2100 ),
    'ssp585': ( 2015, 2100 ),
}
phase9_nex_rebuild_artifacts = False
phase9_runtime_root = Path.cwd( ).resolve( )

if ( phase9_runtime_root / 'helpers' ).exists( ) and ( phase9_runtime_root / 'artifacts' ).exists( ):
    phase9_estuaries_root = phase9_runtime_root
    phase9_repo_root = phase9_estuaries_root.parent

elif phase9_runtime_root.name == 'code' and ( phase9_runtime_root.parent / 'helpers' ).exists( ):
    phase9_estuaries_root = phase9_runtime_root.parent
    phase9_repo_root = phase9_estuaries_root.parent

elif ( phase9_runtime_root / 'estuaries' / 'helpers' ).exists( ):
    phase9_repo_root = phase9_runtime_root
    phase9_estuaries_root = phase9_repo_root / 'estuaries'

else:
    raise RuntimeError( 'Could not resolve the estuaries project root for phase9 CMIP6 helpers.' )

phase9_nex_out_dir = phase9_estuaries_root / 'artifacts'
phase9_nex_manifest_path = phase9_nex_out_dir / 'nex_gddp_cmip6_station_asset_manifest.csv'
phase9_nex_unique_asset_path = phase9_nex_out_dir / 'nex_gddp_cmip6_unique_assets.csv'
phase9_nex_station_path = phase9_nex_out_dir / 'nex_gddp_cmip6_station_targets.csv'
phase9_nex_summary_path = phase9_nex_out_dir / 'nex_gddp_cmip6_manifest_summary.csv'
phase9_nex_wobble_path = phase9_nex_out_dir / 'nex_gddp_cmip6_wobble_candidates.csv'
phase9_nex_plan_path = phase9_nex_out_dir / 'nex_gddp_cmip6_download_plan.json'
phase9_nex_required_paths = [ 
    phase9_nex_manifest_path,
    phase9_nex_unique_asset_path,
    phase9_nex_station_path,
    phase9_nex_summary_path,
    phase9_nex_wobble_path,
    phase9_nex_plan_path,
]

if str( phase9_estuaries_root ) not in sys.path:
    sys.path.append( str( phase9_estuaries_root ) )

from helpers.nex_gddp_cmip6 import ( 
    DEFAULT_NEX_MODEL_VARIANTS,
    DEFAULT_NEX_VARIABLE_SPECS,
    build_nex_asset_manifest,
    build_nex_download_plan,
    build_priority_station_frame,
    save_manifest_outputs,
)

if ( not phase9_nex_rebuild_artifacts ) and all( path.exists( ) for path in phase9_nex_required_paths ):
    phase9_nex_manifest = pd.read_csv( phase9_nex_manifest_path )
    phase9_nex_unique_assets = pd.read_csv( phase9_nex_unique_asset_path )
    phase9_nex_station_frame = pd.read_csv( phase9_nex_station_path )
    phase9_nex_manifest_summary = pd.read_csv( phase9_nex_summary_path )
    phase9_nex_wobble_ranked = pd.read_csv( phase9_nex_wobble_path )
    phase9_nex_plan = json.loads( phase9_nex_plan_path.read_text( ) )
    phase9_nex_artifact_source = 'existing_files'
    phase9_nex_saved = { 
        'manifest_path': str( phase9_nex_manifest_path.resolve( ) ),
        'unique_asset_path': str( phase9_nex_unique_asset_path.resolve( ) ),
        'station_path': str( phase9_nex_station_path.resolve( ) ),
        'summary_path': str( phase9_nex_summary_path.resolve( ) ),
        'wobble_path': str( phase9_nex_wobble_path.resolve( ) ),
        'n_manifest_rows': int( len( phase9_nex_manifest ) ),
        'n_unique_assets': int( len( phase9_nex_unique_assets ) ),
        'n_station_targets': int( phase9_nex_station_frame[ [ 'region', 'station' ] ].drop_duplicates( ).shape[ 0 ] ),
    }

else:
    phase9_nex_station_frame, phase9_nex_wobble_ranked = build_priority_station_frame( 
        station_baseline_display,
        first_event,
        chesapeake_region_codes = [ 'cbm', 'cbv' ],
        include_extra_wobbling = True,
    )
    phase9_nex_manifest = build_nex_asset_manifest( 
        phase9_nex_station_frame,
        models = phase9_nex_models,
        model_variants = DEFAULT_NEX_MODEL_VARIANTS,
        variables = phase9_nex_variables,
        variable_specs = DEFAULT_NEX_VARIABLE_SPECS,
        scenario_years = phase9_nex_scenarios,
    )
    phase9_nex_plan = build_nex_download_plan( 
        phase9_nex_station_frame,
        models = phase9_nex_models,
        model_variants = DEFAULT_NEX_MODEL_VARIANTS,
        variables = phase9_nex_variables,
        variable_specs = DEFAULT_NEX_VARIABLE_SPECS,
        scenario_years = phase9_nex_scenarios,
    )
    phase9_nex_saved = save_manifest_outputs( phase9_nex_manifest, phase9_nex_station_frame, phase9_nex_wobble_ranked, phase9_nex_out_dir )
    phase9_nex_unique_assets = pd.read_csv( phase9_nex_unique_asset_path )
    phase9_nex_manifest_summary = pd.read_csv( phase9_nex_summary_path )
    phase9_nex_plan_path.write_text( json.dumps( phase9_nex_plan, indent = 2 ) )
    phase9_nex_artifact_source = 'rebuilt'

print( f"phase9 NEX artifact source: { phase9_nex_artifact_source }" )
print( f"nex manifest rows: { phase9_nex_saved[ 'n_manifest_rows' ] }" )
print( f"nex unique asset files: { phase9_nex_saved[ 'n_unique_assets' ] }" )
print( f"nex station targets: { phase9_nex_saved[ 'n_station_targets' ] }" )
print( f"asset manifest: { phase9_nex_saved[ 'manifest_path' ] }" )
print( f"unique assets: { phase9_nex_saved[ 'unique_asset_path' ] }" )
print( f"station targets: { phase9_nex_saved[ 'station_path' ] }" )
print( f"wobble candidates: { phase9_nex_saved[ 'wobble_path' ] }" )
print( f"download plan: { phase9_nex_plan_path.resolve( ) }" )
print( f"chesapeake bbox: { phase9_nex_plan[ 'bbox' ] }" )

print( '\nphase9 NEX target stations:' )
display( phase9_nex_station_frame[ [ 'region', 'station', 'station_name', 'region_name', 'selection_reason', 'cluster_label', 'latitude', 'longitude' ] ].sort_values( [ 'selection_reason', 'region', 'station' ] ).reset_index( drop = True ) )

print( '\nphase9 NEX top wobble candidates:' )
display( phase9_nex_wobble_ranked[ [ col for col in [ 'region', 'station', 'station_name', 'region_name', 'cluster_label', 'event_date', 'n_years_present', 'mean_year_coverage', 'n_obs_total', 'is_partial_baseline' ] if col in phase9_nex_wobble_ranked.columns ] ].head( 10 ) )

print( '\nphase9 NEX manifest summary:' )
display( phase9_nex_manifest.groupby( [ 'model', 'scenario', 'variable' ], as_index = False ).agg( n_station_links = ( 'file_url', 'size' ), n_stations = ( 'station', 'nunique' ), year_start = ( 'year', 'min' ), year_end = ( 'year', 'max' ) ) )

print( '\nphase9 NEX unique asset summary:' )
display( phase9_nex_manifest_summary.sort_values( [ 'model', 'scenario', 'variable' ] ).reset_index( drop = True ) )


### 9.7b - download selected NEX-GDDP-CMIP6 assets

This section wires the saved Chesapeake plus wobble manifest into the helper scripts so the notebook can dry-run, download, and extract nearest-cell daily series without rebuilding the logic by hand.

The defaults are intentionally conservative:
- the download cell starts in dry-run mode
- the extraction cell starts in dry-run mode
- both cells can be narrowed to a smaller year range or model subset before running real work


In [ ]:
# 9.7b - helper paths + subprocess runner for CMIP6 download and extraction
import subprocess

phase9_nex_python = phase9_repo_root / '.venv' / 'bin' / 'python'
if not phase9_nex_python.exists( ):
    phase9_nex_python = phase9_estuaries_root.parent / '.venv' / 'bin' / 'python'

if not phase9_nex_python.exists( ):
    phase9_nex_python = Path( sys.executable )

phase9_nex_download_script = phase9_estuaries_root / 'helpers' / 'download_nex_gddp_cmip6.py'
phase9_nex_extract_script = phase9_estuaries_root / 'helpers' / 'extract_nex_gddp_cmip6_station_series.py'
phase9_nex_manifest_path = phase9_nex_out_dir / 'nex_gddp_cmip6_station_asset_manifest.csv'
phase9_nex_unique_asset_path = phase9_nex_out_dir / 'nex_gddp_cmip6_unique_assets.csv'
phase9_nex_download_root = phase9_estuaries_root / 'data' / 'cmip6'
phase9_nex_series_root = phase9_estuaries_root / 'data' / 'cmip6_station_series'


def phase9_extend_multi_arg( arg_list, flag, values ):
    values = [ str( value ) for value in values if str( value ).strip( ) ]
    if len( values ) == 0:
        return

    arg_list.append( flag )
    arg_list.extend( values )


def phase9_run_python_helper( script_path, args ):
    cmd = [ str( phase9_nex_python ), str( script_path ) ] + [ str( arg ) for arg in args ]
    print( 'running:' )
    print( ' '.join( cmd ) )

    completed = subprocess.run( 
        cmd,
        cwd = str( phase9_repo_root ),
        capture_output = True,
        text = True,
    )

    if completed.stdout.strip( ):
        print( completed.stdout )

    if completed.stderr.strip( ):
        print( completed.stderr )

    if completed.returncode != 0:
        raise RuntimeError( f'helper exited with code { completed.returncode }' )

    return completed


print( f'python helper: { phase9_nex_python }' )
print( f'download script: { phase9_nex_download_script }' )
print( f'extract script: { phase9_nex_extract_script }' )
print( f'download root: { phase9_nex_download_root }' )
print( f'extract root: { phase9_nex_series_root }' )


In [ ]:
# 9.7c - CMIP6 download command builder
# set dry_run to False when you are ready to pull real files
phase9_nex_download_models = list( phase9_nex_models )
phase9_nex_download_variables = list( phase9_nex_variables )
phase9_nex_download_scenarios = list( phase9_nex_scenarios.keys( ) )
phase9_nex_download_year_start = None
phase9_nex_download_year_end = None
phase9_nex_download_limit = None
phase9_nex_download_workers = 2
phase9_nex_download_force = False
phase9_nex_download_dry_run = True

phase9_nex_download_args = [ 
    '--manifest-path', phase9_nex_manifest_path,
    '--out-dir', phase9_nex_download_root,
    '--workers', phase9_nex_download_workers,
]

phase9_extend_multi_arg( phase9_nex_download_args, '--model', phase9_nex_download_models )
phase9_extend_multi_arg( phase9_nex_download_args, '--scenario', phase9_nex_download_scenarios )
phase9_extend_multi_arg( phase9_nex_download_args, '--variable', phase9_nex_download_variables )

if phase9_nex_download_year_start is not None:
    phase9_nex_download_args.extend( [ '--year-start', phase9_nex_download_year_start ] )

if phase9_nex_download_year_end is not None:
    phase9_nex_download_args.extend( [ '--year-end', phase9_nex_download_year_end ] )

if phase9_nex_download_limit is not None:
    phase9_nex_download_args.extend( [ '--limit', phase9_nex_download_limit ] )

if phase9_nex_download_force:
    phase9_nex_download_args.append( '--force' )

if phase9_nex_download_dry_run:
    phase9_nex_download_args.append( '--dry-run' )

phase9_nex_download_result = phase9_run_python_helper( phase9_nex_download_script, phase9_nex_download_args )


In [ ]:
# 9.7d - extract nearest-cell daily series for the selected stations
# leave dry_run on until the needed yearly NetCDF files are local
phase9_nex_extract_models = list( phase9_nex_download_models )
phase9_nex_extract_variables = list( phase9_nex_download_variables )
phase9_nex_extract_scenarios = list( phase9_nex_download_scenarios )
phase9_nex_extract_year_start = phase9_nex_download_year_start
phase9_nex_extract_year_end = phase9_nex_download_year_end
phase9_nex_extract_limit_assets = phase9_nex_download_limit
phase9_nex_extract_overwrite = False
phase9_nex_extract_dry_run = True

phase9_nex_extract_args = [ 
    '--station-target-path', phase9_nex_out_dir / 'nex_gddp_cmip6_station_targets.csv',
    '--asset-table-path', phase9_nex_unique_asset_path,
    '--cmip6-root', phase9_nex_download_root,
    '--out-dir', phase9_nex_series_root,
]

phase9_extend_multi_arg( phase9_nex_extract_args, '--model', phase9_nex_extract_models )
phase9_extend_multi_arg( phase9_nex_extract_args, '--scenario', phase9_nex_extract_scenarios )
phase9_extend_multi_arg( phase9_nex_extract_args, '--variable', phase9_nex_extract_variables )

if phase9_nex_extract_year_start is not None:
    phase9_nex_extract_args.extend( [ '--year-start', phase9_nex_extract_year_start ] )

if phase9_nex_extract_year_end is not None:
    phase9_nex_extract_args.extend( [ '--year-end', phase9_nex_extract_year_end ] )

if phase9_nex_extract_limit_assets is not None:
    phase9_nex_extract_args.extend( [ '--limit-assets', phase9_nex_extract_limit_assets ] )

if phase9_nex_extract_overwrite:
    phase9_nex_extract_args.append( '--overwrite' )

if phase9_nex_extract_dry_run:
    phase9_nex_extract_args.append( '--dry-run' )

phase9_nex_extract_result = phase9_run_python_helper( phase9_nex_extract_script, phase9_nex_extract_args )


In [ ]:
# 9.7e - preview extracted CMIP6 station products if they exist
phase9_nex_extract_summary_path = phase9_nex_series_root / 'extraction_summary.json'
phase9_nex_selection_summary_path = phase9_nex_series_root / 'selection_summary.json'
phase9_nex_lookup_path = phase9_nex_series_root / 'station_grid_lookup.csv'
phase9_nex_selection_aggregate_path = phase9_nex_series_root / 'selection_reason_daily_aggregate.csv'

if phase9_nex_selection_summary_path.exists( ):
    phase9_nex_selection_summary = json.loads( phase9_nex_selection_summary_path.read_text( ) )
    print( 'selection summary:' )
    print( json.dumps( phase9_nex_selection_summary, indent = 2 ) )

else:
    print( 'selection summary is missing; run 9.7d first or turn off dry-run.' )

if phase9_nex_extract_summary_path.exists( ):
    phase9_nex_extract_summary = json.loads( phase9_nex_extract_summary_path.read_text( ) )
    print( '\nextraction summary:' )
    print( json.dumps( phase9_nex_extract_summary, indent = 2 ) )

if phase9_nex_lookup_path.exists( ):
    print( '\nstation grid lookup:' )
    display( pd.read_csv( phase9_nex_lookup_path ).head( 12 ) )

if phase9_nex_selection_aggregate_path.exists( ):
    print( '\nselection aggregate preview:' )
    display( pd.read_csv( phase9_nex_selection_aggregate_path ).head( 12 ) )


In [ ]:
# 9.8 - compact export views for the wobble station plus two Chesapeake stations
phase9_example_station_keys = list( phase9_case_station_keys ) if phase9_case_station_keys is not None else None

if 'phase9_scan_result' not in globals( ):
    print( 'phase9_scan_result is missing; run 9.3 first.' )

else:
    phase9_example_keys_df = normalize_phase9_station_keys( phase9_example_station_keys )

    if phase9_example_keys_df is None or len( phase9_example_keys_df ) == 0:
        phase9_example_keys_df = phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station' ] ].drop_duplicates( ).head( 3 ).copy( )
        phase9_example_keys_df[ 'phase9_case_order' ] = range( len( phase9_example_keys_df ) )

    phase9_example_keys_df = phase9_example_keys_df.sort_values( [ 'phase9_case_order', 'region', 'station' ] ).reset_index( drop = True )
    phase9_example_scenarios = phase9_scan_result[ 'scenarios' ]
    phase9_example_future_year_min = phase9_scan_result[ 'future_year_min' ]
    phase9_example_palette = dict( zip( phase9_example_scenarios, sns.color_palette( 'Set2', n_colors = len( phase9_example_scenarios ) ) ) )

    cluster_label_map = ( 
        station_baseline_display[ [ 'cluster', 'cluster_label' ] ]
        .dropna( subset = [ 'cluster' ] )
        .drop_duplicates( )
        .assign( cluster = lambda frame: frame[ 'cluster' ].astype( int ) )
        .set_index( 'cluster' )[ 'cluster_label' ]
        .to_dict( )
    )

    for station_row in phase9_example_keys_df.itertuples( index = False ):
        phase9_example_history = phase9_scan_result[ 'history_plot' ].loc[ 
            ( phase9_scan_result[ 'history_plot' ][ 'region' ] == station_row.region )
            & ( phase9_scan_result[ 'history_plot' ][ 'station' ] == station_row.station )
        ].copy( )
        phase9_example_projection = phase9_scan_result[ 'projection_plot' ].loc[ 
            ( phase9_scan_result[ 'projection_plot' ][ 'region' ] == station_row.region )
            & ( phase9_scan_result[ 'projection_plot' ][ 'station' ] == station_row.station )
        ].copy( )

        if len( phase9_example_history ) == 0 or len( phase9_example_projection ) == 0:
            print( f"No phase9 portrait data found for { station_row.region } / { station_row.station }." )
            continue

        phase9_example_meta = phase9_scan_result[ 'station_meta' ].loc[ 
            ( phase9_scan_result[ 'station_meta' ][ 'region' ] == station_row.region )
            & ( phase9_scan_result[ 'station_meta' ][ 'station' ] == station_row.station )
        ].head( 1 )

        if len( phase9_example_meta ) == 0:
            phase9_example_station_name = station_row.station
            phase9_example_region_name = station_row.region
            phase9_example_cluster_label = 'unknown regime'
            phase9_example_cluster = np.nan

        else:
            phase9_example_station_name = phase9_example_meta[ 'station_name' ].iloc[ 0 ]
            phase9_example_region_name = phase9_example_meta[ 'region_name' ].iloc[ 0 ]
            phase9_example_cluster_label = phase9_example_meta[ 'cluster_label' ].iloc[ 0 ]
            phase9_example_cluster = phase9_example_meta[ 'cluster' ].iloc[ 0 ]

        fig, axes = plt.subplots( 2, 1, figsize = ( 14, 8.8 ), sharex = True )
        fig.suptitle( f"Phase 9 Export View | { phase9_example_station_name } ({ phase9_example_region_name }) | baseline { phase9_example_cluster_label }", fontsize = 22, fontweight = 'bold', y = 0.98 )

        axes[ 0 ].plot( 
            phase9_example_history[ 'year' ],
            phase9_example_history[ 'regime_roll5y' ],
            color = 'black',
            linewidth = 2.2,
            label = 'observed 5y regime',
        )
        for scenario_name in phase9_example_scenarios:
            sub = phase9_example_projection.loc[ 
                ( phase9_example_projection[ 'scenario' ] == scenario_name )
                & ( phase9_example_projection[ 'year' ] >= phase9_example_future_year_min )
            ].copy( )
            axes[ 0 ].plot( 
                sub[ 'year' ],
                sub[ 'regime_roll5y' ],
                color = phase9_example_palette[ scenario_name ],
                linewidth = 1.8,
                alpha = 0.95,
                label = scenario_name,
            )

        phase9_example_regime_vals = pd.Series( pd.concat( [ phase9_example_history[ 'regime_roll5y' ], phase9_example_projection[ 'regime_roll5y' ] ] ) ).dropna( )
        if len( phase9_example_regime_vals ) > 0:
            phase9_example_ticks = sorted( phase9_example_regime_vals.astype( int ).unique( ).tolist( ) )
            axes[ 0 ].set_yticks( phase9_example_ticks )
            axes[ 0 ].set_yticklabels( [ cluster_label_map.get( tick, f'C{ tick }' ) for tick in phase9_example_ticks ] )

        if pd.notna( phase9_example_cluster ):
            axes[ 0 ].axhline( float( phase9_example_cluster ), color = '#555555', linestyle = '--', linewidth = 1.0 )

        axes[ 0 ].axvline( phase9_example_future_year_min, color = '#666666', linestyle = '--', linewidth = 0.9 )
        axes[ 0 ].set_ylabel( 'Regime' )
        axes[ 0 ].set_title( 'Projected Regime Path' )
        axes[ 0 ].legend( loc = 'upper left', ncol = 3 )

        axes[ 1 ].plot( 
            phase9_example_history[ 'year' ],
            phase9_example_history[ 'oxygen_hypoxic_share_roll5y' ],
            color = 'black',
            linewidth = 2.2,
            label = 'observed',
        )
        for scenario_name in phase9_example_scenarios:
            sub = phase9_example_projection.loc[ 
                ( phase9_example_projection[ 'scenario' ] == scenario_name )
                & ( phase9_example_projection[ 'year' ] >= phase9_example_future_year_min )
            ].copy( )
            axes[ 1 ].plot( 
                sub[ 'year' ],
                sub[ 'oxygen_hypoxic_share_roll5y' ],
                color = phase9_example_palette[ scenario_name ],
                linewidth = 1.8,
                alpha = 0.95,
                label = scenario_name,
            )

        axes[ 1 ].axvline( phase9_example_future_year_min, color = '#666666', linestyle = '--', linewidth = 0.9 )
        axes[ 1 ].set_ylabel( 'Warm Hypoxic Share ( 5y mean )' )
        axes[ 1 ].set_xlabel( 'Year' )
        axes[ 1 ].set_title( 'Projected Warm Hypoxic Share' )

        fig.subplots_adjust( top = 0.90, hspace = 0.22 )
        plt.show( )


### 9.8b - topical export views by variable
- each figure is organized by variable, with one row per selected station
- left column = annual 5-year forecast path used by the regime logic
- right column = seasonalized story view that preserves each station's observed monthly shape and shifts it to the projected annual level
- note: the right column is a presentation layer for salinity and oxygen, not an independently month-trained forecast


In [ ]:
# 9.8b - topical export views by variable with station rows and seasonalized story panels
phase9_topical_station_keys = list( phase9_case_station_keys ) if phase9_case_station_keys is not None else None
phase9_story_target_years = [ 2050, 2100 ]
phase9_story_month_labels = pd.date_range( '2001-01-01', periods = 12, freq = 'MS' ).strftime( '%b' ).tolist( )

if 'phase9_history_plot' not in globals( ) or 'phase9_projection_plot' not in globals( ) or 'phase9_future_station_year' not in globals( ):
    print( 'phase9 history/projection tables are missing; run 9.1 first.' )

else:
    phase9_topical_keys_df = normalize_phase9_station_keys( phase9_topical_station_keys )

    if phase9_topical_keys_df is None or len( phase9_topical_keys_df ) == 0:
        if 'phase9_scan_result' in globals( ) and len( phase9_scan_result.get( 'plot_studies', pd.DataFrame( ) ) ) > 0:
            phase9_topical_keys_df = phase9_scan_result[ 'plot_studies' ][ [ 'region', 'station' ] ].drop_duplicates( ).head( 3 ).copy( )

        else:
            phase9_topical_keys_df = phase9_case_studies[ [ 'region', 'station' ] ].drop_duplicates( ).head( 3 ).copy( )

        phase9_topical_keys_df[ 'phase9_case_order' ] = range( len( phase9_topical_keys_df ) )

    phase9_topical_keys_df = phase9_topical_keys_df.sort_values( [ 'phase9_case_order', 'region', 'station' ] ).reset_index( drop = True )
    phase9_topical_station_meta = phase9_topical_keys_df.merge( 
        phase9_case_studies[ [ 'region', 'station', 'station_name', 'region_name', 'cluster_label' ] ].drop_duplicates( ),
        on = [ 'region', 'station' ],
        how = 'left',
    )

    phase9_topical_scenarios = phase9_scenarios.copy( ) if 'phase9_scenarios' in globals( ) else sorted( phase9_projection_plot[ 'scenario' ].dropna( ).astype( str ).unique( ).tolist( ) )
    phase9_topical_palette = dict( zip( phase9_topical_scenarios, sns.color_palette( 'Set2', n_colors = len( phase9_topical_scenarios ) ) ) )
    phase9_topical_future_year_min = phase9_future_year_min if 'phase9_future_year_min' in globals( ) else int( phase9_projection_plot[ 'year' ].min( ) )
    phase9_topical_future_year_max = phase9_future_year_max if 'phase9_future_year_max' in globals( ) else int( phase9_projection_plot[ 'year' ].max( ) )
    phase9_story_years_use = [ year for year in phase9_story_target_years if phase9_topical_future_year_min <= year <= phase9_topical_future_year_max ]
    if len( phase9_story_years_use ) == 0:
        phase9_story_years_use = [ phase9_topical_future_year_max ]

    phase9_hist_monthly = phase9_hist_join.copy( )
    phase9_hist_monthly[ 'month_num' ] = phase9_hist_monthly[ 'date' ].dt.month
    phase9_hist_monthly_climo = ( 
        phase9_hist_monthly
        .groupby( [ 'region', 'station', 'month_num' ], as_index = False )
        .agg( 
            water_temp_monthly_mean = ( 'water_temp', 'mean' ),
            salinity_monthly_mean = ( 'salinity', 'mean' ),
            oxygen_monthly_mean = ( 'oxygen', 'mean' ),
        )
    )
    phase9_hist_annual_climo = ( 
        phase9_hist_monthly
        .groupby( [ 'region', 'station' ], as_index = False )
        .agg( 
            water_temp_annual_mean = ( 'water_temp', 'mean' ),
            salinity_annual_mean = ( 'salinity', 'mean' ),
            oxygen_annual_mean = ( 'oxygen', 'mean' ),
        )
    )
    phase9_hist_monthly_climo = phase9_hist_monthly_climo.merge( phase9_hist_annual_climo, on = [ 'region', 'station' ], how = 'left' )

    phase9_topical_specs = [ 
        { 
            'name': 'water_temp',
            'title': 'Water Temperature',
            'roll_col': 'water_temp_roll5y',
            'future_col': 'water_temp_abs_annual_mean',
            'monthly_col': 'water_temp_monthly_mean',
            'annual_col': 'water_temp_annual_mean',
            'ylabel': 'Water Temp ( C )',
        },
        { 
            'name': 'salinity',
            'title': 'Salinity',
            'roll_col': 'salinity_roll5y',
            'future_col': 'salinity_abs_annual_mean',
            'monthly_col': 'salinity_monthly_mean',
            'annual_col': 'salinity_annual_mean',
            'ylabel': 'Salinity',
        },
        { 
            'name': 'oxygen',
            'title': 'Dissolved Oxygen',
            'roll_col': 'oxygen_plot_roll5y',
            'future_col': 'oxygen_abs_annual_mean',
            'monthly_col': 'oxygen_monthly_mean',
            'annual_col': 'oxygen_annual_mean',
            'ylabel': 'Oxygen ( mg/L )',
        },
    ]

    print( '9.8b note: the left column is the actual annual projection used for regime classification.' )
    print( "9.8b note: the right column keeps each station's observed monthly shape and shifts it to the projected annual level." )

    for spec in phase9_topical_specs:
        n_rows = max( 1, len( phase9_topical_station_meta ) )
        fig, axes = plt.subplots( 
            n_rows,
            2,
            figsize = ( 16, max( 4.0, 3.5 * n_rows ) ),
            sharex = 'col',
            squeeze = False,
            gridspec_kw = { 'width_ratios': [ 1.4, 1.0 ] },
        )
        fig.suptitle( f"Phase 9 Topical Export | { spec[ 'title' ] }", fontsize = 16, y = 0.995 )

        for row_idx, station_row in enumerate( phase9_topical_station_meta.itertuples( index = False ) ):
            ax_trend = axes[ row_idx, 0 ]
            ax_cycle = axes[ row_idx, 1 ]
            station_label = f"{ station_row.station_name } ({ station_row.region_name })"
            baseline_label = station_row.cluster_label if pd.notna( station_row.cluster_label ) else 'unknown regime'

            hist_case = phase9_history_plot.loc[ 
                ( phase9_history_plot[ 'region' ] == station_row.region )
                & ( phase9_history_plot[ 'station' ] == station_row.station )
            ].copy( )
            proj_case = phase9_projection_plot.loc[ 
                ( phase9_projection_plot[ 'region' ] == station_row.region )
                & ( phase9_projection_plot[ 'station' ] == station_row.station )
            ].copy( )

            if len( hist_case ) > 0:
                ax_trend.plot( 
                    hist_case[ 'year' ],
                    hist_case[ spec[ 'roll_col' ] ],
                    color = '#1f1f1f',
                    linewidth = 2.2,
                    label = 'observed 5y history',
                )

            for scenario_name in phase9_topical_scenarios:
                sub = proj_case.loc[ 
                    ( proj_case[ 'scenario' ] == scenario_name )
                    & ( proj_case[ 'year' ] >= phase9_topical_future_year_min )
                ].copy( )
                if len( sub ) == 0:
                    continue

                ax_trend.plot( 
                    sub[ 'year' ],
                    sub[ spec[ 'roll_col' ] ],
                    color = phase9_topical_palette[ scenario_name ],
                    linewidth = 1.8,
                    alpha = 0.95,
                    label = scenario_name,
                )

            ax_trend.axvline( phase9_topical_future_year_min, color = '#666666', linestyle = '--', linewidth = 0.9 )
            ax_trend.set_ylabel( spec[ 'ylabel' ] )
            ax_trend.set_title( f'{ station_label } | baseline { baseline_label }', loc = 'left', fontsize = 11 )

            phase9_station_monthly = phase9_hist_monthly_climo.loc[ 
                ( phase9_hist_monthly_climo[ 'region' ] == station_row.region )
                & ( phase9_hist_monthly_climo[ 'station' ] == station_row.station )
            ].sort_values( 'month_num' ).copy( )

            if len( phase9_station_monthly ) > 0:
                ax_cycle.plot( 
                    phase9_station_monthly[ 'month_num' ],
                    phase9_station_monthly[ spec[ 'monthly_col' ] ],
                    color = '#1f1f1f',
                    linewidth = 2.2,
                    label = 'observed monthly climatology',
                )

                phase9_monthly_offset = phase9_station_monthly[ spec[ 'monthly_col' ] ] - phase9_station_monthly[ spec[ 'annual_col' ] ]

                for story_idx, target_year in enumerate( phase9_story_years_use ):
                    line_style = '--' if story_idx < len( phase9_story_years_use ) - 1 else '-'
                    line_alpha = 0.80 if line_style == '--' else 0.95

                    for scenario_name in phase9_topical_scenarios:
                        future_sub = phase9_future_station_year.loc[ 
                            ( phase9_future_station_year[ 'region' ] == station_row.region )
                            & ( phase9_future_station_year[ 'station' ] == station_row.station )
                            & ( phase9_future_station_year[ 'scenario' ] == scenario_name )
                        ].copy( )
                        if len( future_sub ) == 0 or spec[ 'future_col' ] not in future_sub.columns:
                            continue

                        future_sub[ 'year_distance' ] = ( future_sub[ 'year' ] - int( target_year ) ).abs( )
                        future_point = future_sub.sort_values( [ 'year_distance', 'year' ] ).head( 1 )
                        if len( future_point ) == 0 or pd.isna( future_point[ spec[ 'future_col' ] ].iloc[ 0 ] ):
                            continue

                        future_year_use = int( future_point[ 'year' ].iloc[ 0 ] )
                        shifted_cycle = future_point[ spec[ 'future_col' ] ].iloc[ 0 ] + phase9_monthly_offset.to_numpy( )
                        ax_cycle.plot( 
                            phase9_station_monthly[ 'month_num' ],
                            shifted_cycle,
                            color = phase9_topical_palette[ scenario_name ],
                            linewidth = 1.8,
                            alpha = line_alpha,
                            linestyle = line_style,
                            label = f'{ scenario_name } { future_year_use }',
                        )

            ax_cycle.set_xticks( range( 1, 13 ) )
            ax_cycle.set_xticklabels( phase9_story_month_labels, rotation = 0 )
            ax_cycle.set_ylabel( spec[ 'ylabel' ] )
            ax_cycle.set_title( 'Seasonalized monthly story', loc = 'left', fontsize = 11 )

            if row_idx == 0:
                ax_trend.legend( loc = 'upper left', ncol = min( 3, max( 1, len( phase9_topical_scenarios ) + 1 ) ) )
                ax_cycle.legend( loc = 'upper left', ncol = 2 )

        axes[ -1, 0 ].set_xlabel( 'Year' )
        axes[ -1, 1 ].set_xlabel( 'Month' )
        plt.tight_layout( )
        plt.show( )


### 9.8c - mean regime seasonalized forecasts
- groups stations by their baseline regime and averages the forecast path across each regime
- left column = mean annual 5-year path per regime
- right column = seasonalized regime story built from the historical monthly shape of that regime
- oxygen panels mark the `<5 mg/L` stressed and `<2 mg/L` hypoxic thresholds for context


In [ ]:
# 9.8c - mean regime seasonalized forecasts
phase9_regime_story_target_years = [ 2050, 2100 ]
phase9_regime_story_month_labels = pd.date_range( '2001-01-01', periods = 12, freq = 'MS' ).strftime( '%b' ).tolist( )

if 'phase9_scan_result' in globals( ) and len( phase9_scan_result.get( 'future_station_year', pd.DataFrame( ) ) ) > 0:
    phase9_regime_station_meta = phase9_scan_result[ 'station_meta' ][ [ 'region', 'station', 'cluster', 'cluster_label' ] ].drop_duplicates( ).copy( )
    phase9_regime_history_roll = phase9_scan_result[ 'history_plot' ].copy( )
    phase9_regime_projection_roll = phase9_scan_result[ 'projection_plot' ].copy( )
    phase9_regime_future_annual = phase9_scan_result[ 'future_station_year' ].copy( )
    phase9_regime_scenarios = phase9_scan_result[ 'scenarios' ]
    phase9_regime_future_year_min = phase9_scan_result[ 'future_year_min' ]
    phase9_regime_future_year_max = phase9_scan_result[ 'future_year_max' ]

elif 'phase9_case_studies' in globals( ) and len( phase9_case_studies ) > 0:
    phase9_regime_station_meta = phase9_case_studies[ [ 'region', 'station', 'cluster', 'cluster_label' ] ].drop_duplicates( ).copy( )
    phase9_regime_history_roll = phase9_history_plot.copy( ) if 'phase9_history_plot' in globals( ) else pd.DataFrame( )
    phase9_regime_projection_roll = phase9_projection_plot.copy( ) if 'phase9_projection_plot' in globals( ) else pd.DataFrame( )
    phase9_regime_future_annual = phase9_future_station_year.copy( ) if 'phase9_future_station_year' in globals( ) else pd.DataFrame( )
    phase9_regime_scenarios = phase9_scenarios.copy( ) if 'phase9_scenarios' in globals( ) else [ ]
    phase9_regime_future_year_min = phase9_future_year_min if 'phase9_future_year_min' in globals( ) else None
    phase9_regime_future_year_max = phase9_future_year_max if 'phase9_future_year_max' in globals( ) else None

else:
    phase9_regime_station_meta = pd.DataFrame( )
    phase9_regime_history_roll = pd.DataFrame( )
    phase9_regime_projection_roll = pd.DataFrame( )
    phase9_regime_future_annual = pd.DataFrame( )
    phase9_regime_scenarios = [ ]
    phase9_regime_future_year_min = None
    phase9_regime_future_year_max = None

if len( phase9_regime_station_meta ) == 0 or len( phase9_hist_join ) == 0:
    print( 'regime-level seasonalized forecasts need 9.1 first, and use 9.3 when you want all scanned stations folded in.' )

else:
    phase9_regime_station_meta = phase9_regime_station_meta.dropna( subset = [ 'cluster' ] ).drop_duplicates( ).copy( )
    phase9_regime_station_meta[ 'cluster' ] = pd.to_numeric( phase9_regime_station_meta[ 'cluster' ], errors = 'coerce' )
    phase9_regime_station_meta = phase9_regime_station_meta.dropna( subset = [ 'cluster' ] ).copy( )
    phase9_regime_station_meta[ 'cluster' ] = phase9_regime_station_meta[ 'cluster' ].astype( int )

    phase9_regime_palette = dict( zip( phase9_regime_scenarios, sns.color_palette( 'Set2', n_colors = len( phase9_regime_scenarios ) ) ) )
    phase9_regime_order = ( 
        phase9_regime_station_meta[ [ 'cluster', 'cluster_label' ] ]
        .drop_duplicates( )
        .sort_values( [ 'cluster', 'cluster_label' ] )
        .reset_index( drop = True )
    )

    phase9_regime_hist_monthly = phase9_hist_join.merge( phase9_regime_station_meta, on = [ 'region', 'station' ], how = 'inner' )
    phase9_regime_hist_monthly[ 'month_num' ] = phase9_regime_hist_monthly[ 'date' ].dt.month
    phase9_regime_monthly_climo = ( 
        phase9_regime_hist_monthly
        .groupby( [ 'cluster', 'cluster_label', 'month_num' ], as_index = False )
        .agg( 
            water_temp_monthly_mean = ( 'water_temp', 'mean' ),
            salinity_monthly_mean = ( 'salinity', 'mean' ),
            oxygen_monthly_mean = ( 'oxygen', 'mean' ),
        )
    )
    phase9_regime_annual_climo = ( 
        phase9_regime_hist_monthly
        .groupby( [ 'cluster', 'cluster_label' ], as_index = False )
        .agg( 
            water_temp_annual_mean = ( 'water_temp', 'mean' ),
            salinity_annual_mean = ( 'salinity', 'mean' ),
            oxygen_annual_mean = ( 'oxygen', 'mean' ),
        )
    )
    phase9_regime_monthly_climo = phase9_regime_monthly_climo.merge( phase9_regime_annual_climo, on = [ 'cluster', 'cluster_label' ], how = 'left' )

    phase9_regime_history_roll = phase9_regime_history_roll.merge( phase9_regime_station_meta, on = [ 'region', 'station', 'cluster', 'cluster_label' ], how = 'inner' )
    phase9_regime_history_roll = ( 
        phase9_regime_history_roll
        .groupby( [ 'cluster', 'cluster_label', 'year' ], as_index = False )
        .agg( 
            water_temp_roll5y = ( 'water_temp_roll5y', 'mean' ),
            salinity_roll5y = ( 'salinity_roll5y', 'mean' ),
            oxygen_plot_roll5y = ( 'oxygen_plot_roll5y', 'mean' ),
        )
    )

    phase9_regime_projection_roll = phase9_regime_projection_roll.merge( phase9_regime_station_meta, on = [ 'region', 'station', 'cluster', 'cluster_label' ], how = 'inner' )
    phase9_regime_projection_roll = ( 
        phase9_regime_projection_roll
        .loc[ phase9_regime_projection_roll[ 'year' ] >= phase9_regime_future_year_min ]
        .groupby( [ 'cluster', 'cluster_label', 'scenario', 'year' ], as_index = False )
        .agg( 
            water_temp_roll5y = ( 'water_temp_roll5y', 'mean' ),
            salinity_roll5y = ( 'salinity_roll5y', 'mean' ),
            oxygen_plot_roll5y = ( 'oxygen_plot_roll5y', 'mean' ),
        )
    )

    phase9_regime_future_annual = phase9_regime_future_annual.merge( phase9_regime_station_meta, on = [ 'region', 'station', 'cluster', 'cluster_label' ], how = 'inner' )
    phase9_regime_future_annual = ( 
        phase9_regime_future_annual
        .groupby( [ 'cluster', 'cluster_label', 'scenario', 'year' ], as_index = False )
        .agg( 
            water_temp_abs_annual_mean = ( 'water_temp_abs_annual_mean', 'mean' ),
            salinity_abs_annual_mean = ( 'salinity_abs_annual_mean', 'mean' ),
            oxygen_abs_annual_mean = ( 'oxygen_abs_annual_mean', 'mean' ),
        )
    )

    phase9_regime_story_years_use = [ year for year in phase9_regime_story_target_years if phase9_regime_future_year_min <= year <= phase9_regime_future_year_max ]
    if len( phase9_regime_story_years_use ) == 0:
        phase9_regime_story_years_use = [ phase9_regime_future_year_max ]

    phase9_regime_specs = [ 
        { 
            'title': 'Water Temperature',
            'roll_col': 'water_temp_roll5y',
            'future_col': 'water_temp_abs_annual_mean',
            'monthly_col': 'water_temp_monthly_mean',
            'annual_col': 'water_temp_annual_mean',
            'ylabel': 'Water Temp ( C )',
            'thresholds': [ ],
        },
        { 
            'title': 'Salinity',
            'roll_col': 'salinity_roll5y',
            'future_col': 'salinity_abs_annual_mean',
            'monthly_col': 'salinity_monthly_mean',
            'annual_col': 'salinity_annual_mean',
            'ylabel': 'Salinity',
            'thresholds': [ ],
        },
        { 
            'title': 'Dissolved Oxygen',
            'roll_col': 'oxygen_plot_roll5y',
            'future_col': 'oxygen_abs_annual_mean',
            'monthly_col': 'oxygen_monthly_mean',
            'annual_col': 'oxygen_annual_mean',
            'ylabel': 'Oxygen ( mg/L )',
            'thresholds': [ ( 5.0, '#c77c2c', '<5 stressed' ), ( 2.0, '#c44e52', '<2 hypoxic' ) ],
        },
    ]

    for spec in phase9_regime_specs:
        n_rows = max( 1, len( phase9_regime_order ) )
        fig, axes = plt.subplots( 
            n_rows,
            2,
            figsize = ( 16, max( 4.0, 3.6 * n_rows ) ),
            sharex = 'col',
            squeeze = False,
            gridspec_kw = { 'width_ratios': [ 1.4, 1.0 ] },
        )
        fig.suptitle( f"Phase 9 Regime Mean Story | { spec[ 'title' ] }", fontsize = 16, y = 0.995 )

        for row_idx, regime_row in enumerate( phase9_regime_order.itertuples( index = False ) ):
            ax_trend = axes[ row_idx, 0 ]
            ax_cycle = axes[ row_idx, 1 ]
            regime_label = regime_row.cluster_label if pd.notna( regime_row.cluster_label ) else f"cluster { regime_row.cluster }"

            hist_sub = phase9_regime_history_roll.loc[ phase9_regime_history_roll[ 'cluster' ] == regime_row.cluster ].copy( )
            proj_sub = phase9_regime_projection_roll.loc[ phase9_regime_projection_roll[ 'cluster' ] == regime_row.cluster ].copy( )
            monthly_sub = phase9_regime_monthly_climo.loc[ phase9_regime_monthly_climo[ 'cluster' ] == regime_row.cluster ].sort_values( 'month_num' ).copy( )
            future_sub = phase9_regime_future_annual.loc[ phase9_regime_future_annual[ 'cluster' ] == regime_row.cluster ].copy( )

            if len( hist_sub ) > 0:
                ax_trend.plot( hist_sub[ 'year' ], hist_sub[ spec[ 'roll_col' ] ], color = '#1f1f1f', linewidth = 2.2, label = 'observed regime mean' )

            for scenario_name in phase9_regime_scenarios:
                scenario_roll = proj_sub.loc[ proj_sub[ 'scenario' ] == scenario_name ].copy( )
                if len( scenario_roll ) == 0:
                    continue

                ax_trend.plot( 
                    scenario_roll[ 'year' ],
                    scenario_roll[ spec[ 'roll_col' ] ],
                    color = phase9_regime_palette.get( scenario_name, '#4c72b0' ),
                    linewidth = 1.8,
                    alpha = 0.95,
                    label = scenario_name,
                )

            ax_trend.axvline( phase9_regime_future_year_min, color = '#666666', linestyle = '--', linewidth = 0.9 )
            ax_trend.set_ylabel( spec[ 'ylabel' ] )
            ax_trend.set_title( regime_label, loc = 'left', fontsize = 11 )

            if len( monthly_sub ) > 0:
                ax_cycle.plot( monthly_sub[ 'month_num' ], monthly_sub[ spec[ 'monthly_col' ] ], color = '#1f1f1f', linewidth = 2.2, label = 'observed regime monthly shape' )
                monthly_offset = monthly_sub[ spec[ 'monthly_col' ] ] - monthly_sub[ spec[ 'annual_col' ] ]

                for story_idx, target_year in enumerate( phase9_regime_story_years_use ):
                    line_style = '--' if story_idx < len( phase9_regime_story_years_use ) - 1 else '-'
                    line_alpha = 0.80 if line_style == '--' else 0.95

                    for scenario_name in phase9_regime_scenarios:
                        scenario_future = future_sub.loc[ future_sub[ 'scenario' ] == scenario_name ].copy( )
                        if len( scenario_future ) == 0:
                            continue

                        scenario_future[ 'year_distance' ] = ( scenario_future[ 'year' ] - int( target_year ) ).abs( )
                        future_point = scenario_future.sort_values( [ 'year_distance', 'year' ] ).head( 1 )
                        if len( future_point ) == 0 or pd.isna( future_point[ spec[ 'future_col' ] ].iloc[ 0 ] ):
                            continue

                        future_year_use = int( future_point[ 'year' ].iloc[ 0 ] )
                        shifted_cycle = future_point[ spec[ 'future_col' ] ].iloc[ 0 ] + monthly_offset.to_numpy( )
                        ax_cycle.plot( 
                            monthly_sub[ 'month_num' ],
                            shifted_cycle,
                            color = phase9_regime_palette.get( scenario_name, '#4c72b0' ),
                            linewidth = 1.8,
                            alpha = line_alpha,
                            linestyle = line_style,
                            label = f'{ scenario_name } { future_year_use }',
                        )

            for threshold_value, threshold_color, threshold_label in spec[ 'thresholds' ]:
                ax_trend.axhline( threshold_value, color = threshold_color, linestyle = ':', linewidth = 1.0 )
                ax_cycle.axhline( threshold_value, color = threshold_color, linestyle = ':', linewidth = 1.0 )

            ax_cycle.set_xticks( range( 1, 13 ) )
            ax_cycle.set_xticklabels( phase9_regime_story_month_labels )
            ax_cycle.set_ylabel( spec[ 'ylabel' ] )
            ax_cycle.set_title( 'Seasonalized regime story', loc = 'left', fontsize = 11 )

            if row_idx == 0:
                ax_trend.legend( loc = 'upper left', ncol = min( 3, max( 1, len( phase9_regime_scenarios ) + 1 ) ) )
                ax_cycle.legend( loc = 'upper left', ncol = 2 )

        axes[ -1, 0 ].set_xlabel( 'Year' )
        axes[ -1, 1 ].set_xlabel( 'Month' )
        plt.tight_layout( )
        plt.show( )


### 9.8d - regime risk table for warming and oxygen stress
- summarizes baseline-regime risk using the scanned station set when available
- compares historical warm-season oxygen minima to a late-century window
- flags regimes where `<5 mg/L` stress or `<2 mg/L` hypoxia newly appears or widens


In [ ]:
# 9.8d - regime risk table for warming and oxygen stress
phase9_regime_stress_threshold = 5.0
phase9_regime_hypoxia_threshold = 2.0

if 'phase9_scan_result' in globals( ) and len( phase9_scan_result.get( 'future_station_year', pd.DataFrame( ) ) ) > 0:
    phase9_risk_station_meta = phase9_scan_result[ 'station_meta' ][ [ 'region', 'station', 'cluster', 'cluster_label' ] ].drop_duplicates( ).copy( )
    phase9_risk_history = phase9_scan_result[ 'history_plot' ].copy( )
    phase9_risk_future = phase9_scan_result[ 'future_station_year' ].copy( )
    phase9_risk_future_year_min = phase9_scan_result[ 'future_year_min' ]
    phase9_risk_future_year_max = phase9_scan_result[ 'future_year_max' ]

elif 'phase9_case_studies' in globals( ) and 'phase9_future_station_year' in globals( ):
    phase9_risk_station_meta = phase9_case_studies[ [ 'region', 'station', 'cluster', 'cluster_label' ] ].drop_duplicates( ).copy( )
    phase9_risk_history = phase9_history_plot.copy( )
    phase9_risk_future = phase9_future_station_year.copy( )
    phase9_risk_future_year_min = phase9_future_year_min
    phase9_risk_future_year_max = phase9_future_year_max

else:
    phase9_risk_station_meta = pd.DataFrame( )
    phase9_risk_history = pd.DataFrame( )
    phase9_risk_future = pd.DataFrame( )
    phase9_risk_future_year_min = None
    phase9_risk_future_year_max = None

if len( phase9_risk_station_meta ) == 0 or len( phase9_risk_future ) == 0:
    print( 'regime risk summary needs 9.1 first, and 9.3 when you want all scanned stations included.' )

else:
    phase9_risk_station_meta = phase9_risk_station_meta.dropna( subset = [ 'cluster' ] ).drop_duplicates( ).copy( )
    phase9_risk_station_meta[ 'cluster' ] = pd.to_numeric( phase9_risk_station_meta[ 'cluster' ], errors = 'coerce' )
    phase9_risk_station_meta = phase9_risk_station_meta.dropna( subset = [ 'cluster' ] ).copy( )
    phase9_risk_station_meta[ 'cluster' ] = phase9_risk_station_meta[ 'cluster' ].astype( int )

    phase9_risk_history = phase9_risk_history.merge( phase9_risk_station_meta, on = [ 'region', 'station', 'cluster', 'cluster_label' ], how = 'inner' )
    phase9_risk_future = phase9_risk_future.merge( phase9_risk_station_meta, on = [ 'region', 'station', 'cluster', 'cluster_label' ], how = 'inner' )

    phase9_risk_history[ 'oxygen_stressed_flag' ] = np.where( phase9_risk_history[ 'oxygen_warm_min_abs' ].notna( ), phase9_risk_history[ 'oxygen_warm_min_abs' ] < phase9_regime_stress_threshold, np.nan )
    phase9_risk_history[ 'oxygen_hypoxic_flag' ] = np.where( phase9_risk_history[ 'oxygen_warm_min_abs' ].notna( ), phase9_risk_history[ 'oxygen_warm_min_abs' ] < phase9_regime_hypoxia_threshold, np.nan )
    phase9_risk_future[ 'oxygen_stressed_flag' ] = np.where( phase9_risk_future[ 'oxygen_warm_min_abs' ].notna( ), phase9_risk_future[ 'oxygen_warm_min_abs' ] < phase9_regime_stress_threshold, np.nan )
    phase9_risk_future[ 'oxygen_hypoxic_flag' ] = np.where( phase9_risk_future[ 'oxygen_warm_min_abs' ].notna( ), phase9_risk_future[ 'oxygen_warm_min_abs' ] < phase9_regime_hypoxia_threshold, np.nan )

    phase9_risk_future_window_start = max( int( phase9_risk_future_year_min ), int( phase9_risk_future_year_max ) - 9 )
    phase9_risk_future_tail = phase9_risk_future.loc[ phase9_risk_future[ 'year' ].between( phase9_risk_future_window_start, int( phase9_risk_future_year_max ) ) ].copy( )

    phase9_regime_hist_summary = ( 
        phase9_risk_history
        .groupby( [ 'cluster', 'cluster_label' ], as_index = False )
        .agg( 
            n_stations = ( 'station', 'nunique' ),
            hist_water_temp_mean = ( 'water_temp_abs_annual_mean', 'mean' ),
            hist_oxygen_mean = ( 'oxygen_abs_annual_mean', 'mean' ),
            hist_warm_min_mean = ( 'oxygen_warm_min_abs', 'mean' ),
            hist_stressed_share = ( 'oxygen_stressed_flag', 'mean' ),
            hist_hypoxic_share = ( 'oxygen_hypoxic_flag', 'mean' ),
        )
    )

    phase9_regime_future_summary = ( 
        phase9_risk_future_tail
        .groupby( [ 'cluster', 'cluster_label', 'scenario' ], as_index = False )
        .agg( 
            future_water_temp_mean = ( 'water_temp_abs_annual_mean', 'mean' ),
            future_oxygen_mean = ( 'oxygen_abs_annual_mean', 'mean' ),
            future_warm_min_mean = ( 'oxygen_warm_min_abs', 'mean' ),
            future_stressed_share = ( 'oxygen_stressed_flag', 'mean' ),
            future_hypoxic_share = ( 'oxygen_hypoxic_flag', 'mean' ),
        )
    )

    phase9_regime_risk = phase9_regime_future_summary.merge( phase9_regime_hist_summary, on = [ 'cluster', 'cluster_label' ], how = 'left' )
    phase9_regime_risk[ 'delta_water_temp_mean' ] = phase9_regime_risk[ 'future_water_temp_mean' ] - phase9_regime_risk[ 'hist_water_temp_mean' ]
    phase9_regime_risk[ 'delta_oxygen_mean' ] = phase9_regime_risk[ 'future_oxygen_mean' ] - phase9_regime_risk[ 'hist_oxygen_mean' ]
    phase9_regime_risk[ 'warm_min_drop_mean' ] = phase9_regime_risk[ 'hist_warm_min_mean' ] - phase9_regime_risk[ 'future_warm_min_mean' ]
    phase9_regime_risk[ 'stressed_share_increase' ] = phase9_regime_risk[ 'future_stressed_share' ].fillna( 0.0 ) - phase9_regime_risk[ 'hist_stressed_share' ].fillna( 0.0 )
    phase9_regime_risk[ 'hypoxic_share_increase' ] = phase9_regime_risk[ 'future_hypoxic_share' ].fillna( 0.0 ) - phase9_regime_risk[ 'hist_hypoxic_share' ].fillna( 0.0 )
    phase9_regime_risk[ 'stressed_instantiation_flag' ] = ( phase9_regime_risk[ 'hist_stressed_share' ].fillna( 0.0 ) == 0.0 ) & ( phase9_regime_risk[ 'future_stressed_share' ].fillna( 0.0 ) > 0.0 )
    phase9_regime_risk[ 'hypoxic_instantiation_flag' ] = ( phase9_regime_risk[ 'hist_hypoxic_share' ].fillna( 0.0 ) == 0.0 ) & ( phase9_regime_risk[ 'future_hypoxic_share' ].fillna( 0.0 ) > 0.0 )
    phase9_regime_risk = phase9_regime_risk.sort_values( 
        [ 'scenario', 'hypoxic_instantiation_flag', 'hypoxic_share_increase', 'stressed_share_increase', 'delta_water_temp_mean' ],
        ascending = [ True, False, False, False, False ],
    ).reset_index( drop = True )

    print( f'regime risk future window: { phase9_risk_future_window_start } to { int( phase9_risk_future_year_max ) }' )
    display( phase9_regime_risk[ [ 
        'scenario',
        'cluster_label',
        'n_stations',
        'delta_water_temp_mean',
        'delta_oxygen_mean',
        'warm_min_drop_mean',
        'hist_stressed_share',
        'future_stressed_share',
        'stressed_share_increase',
        'stressed_instantiation_flag',
        'hist_hypoxic_share',
        'future_hypoxic_share',
        'hypoxic_share_increase',
        'hypoxic_instantiation_flag',
    ] ].round( 3 ) )
